In [0]:
# Journey Model — self-contained production silver pipeline
#
# Reads curated bronze and publishes the silver planes: spine (person, encounter,
# episode), events, clinical facts, text, reference and artifact. Each published
# product gets its own cell below, holding the source tables, column lists,
# helpers and column comments that only it uses.
#
# House rules every flow in this notebook follows:
#   * deterministic expressions only, so a rerun reproduces the same rows;
#   * loaded_at carries the bronze ADC_UPDT, never the time of this run;
#   * expectations record quality, they never drop rows;
#   * generated ids never contain the name of the table they are published to, so
#     a product can be renamed or promoted without re-minting its keys;
#   * production source tables are explicit; the pipeline controls the publication location.
#
# The pipeline's target catalog and schema control where the flattened tables publish.

In [0]:
# ==== Imports ====
import pyspark.sql.functions as F
from pyspark.sql import Column
try:
    from striprtf.striprtf import rtf_to_text as _rtf_to_text
    _RTF_PARSER_VERSION = "striprtf-0.0.32"
except ImportError:
    # No pipeline environment dependency is required. If striprtf is unavailable,
    # preserve the original value and report that accurately in parser_version.
    def _rtf_to_text(value, errors="ignore"):
        return value
    _RTF_PARSER_VERSION = "rtf-preserved-no-parser"

def _strip_rtf_text_value(value):
    """Parse RTF with the same pinned library as Blob v4; never erase on failure."""
    if value is None:
        return None
    try:
        parsed = _rtf_to_text(value, errors="ignore")
        return parsed if parsed and parsed.strip() else value
    except Exception:
        return value

_strip_rtf_text = F.udf(_strip_rtf_text_value, "string")

In [0]:
# ==== Declarative pipelines API ====

# The declarative pipelines API lives in pyspark.pipelines on the serverless CURRENT
# channel. The fallback keeps the notebook runnable on a runtime that still only has
# the dlt module, at the cost of the refresh_policy hint.
try:
    from pyspark import pipelines as dp
    _NEW_API = True
except ImportError:  # fallback: classic dlt module (no refresh_policy kwarg)
    import dlt
    class _DpShim:
        @staticmethod
        def materialized_view(**kw):
            kw.pop("refresh_policy", None)
            return dlt.table(**kw)
    dp = _DpShim()
    _NEW_API = False

def materialized_view(column_comments=None, **options):
    """Declare a published table, attaching per-column comments to its schema.

    Unity Catalog reads the comments off the schema Lakeflow records for the flow,
    the same way the RDE and OMOP pipelines attach StructField metadata. They only
    reach the catalog when the table is first built: an existing table keeps the
    comments it already has, so changing one here also needs a catalog reconcile.

    Private stages pass no comments and go straight through.
    """
    declare = dp.materialized_view(**options)
    if not column_comments:
        return declare

    def decorate(build):
        def with_column_comments():
            df = build()
            for column in df.columns:
                if column in column_comments:
                    df = df.withMetadata(column, {"comment": column_comments[column]})
            return df

        with_column_comments.__name__ = build.__name__
        return declare(with_column_comments)

    return decorate

In [0]:
# ==== Dataset names ====

# Flatten logical planes into table names. The pipeline target catalog/schema supplies
# the physical publication location.
def _n(name):
    logical_schema, table = name.split(".", 1)
    if not logical_schema.startswith("journey_"):
        raise ValueError(f"unexpected flow schema in {name!r}")
    plane = logical_schema[len("journey_"):]
    return "_" + plane + table if table.startswith("_") else plane + "_" + table

In [0]:
# ==== Reading sources ====

def read_source(name):
    return spark.read.table(name)

In [0]:
# ==== Deterministic identity helpers ====

# Identity helpers. These decide what every key in the product hashes to, so a change
# here is a full rebuild of everything downstream, not an edit.
NULL_TOKEN = "~"   # outside the base64 alphabet => can never collide with an encoded value

def _enc(col_or_lit):
    """Encode one part of a key: NULL becomes '~', anything else base64(trim(string)).
    base64 output contains neither ':' nor '~', so parts joined with ':' can only be
    read back one way and two different inputs can never produce the same key."""
    c = col_or_lit if isinstance(col_or_lit, Column) else F.lit(col_or_lit)
    s = F.trim(c.cast("string"))
    return F.when(c.isNull(), F.lit(NULL_TOKEN)).otherwise(F.base64(s.cast("binary")))

def _present(col):
    """True only when identifier evidence is non-null and non-blank."""
    return col.isNotNull() & (F.trim(col.cast("string")) != "")

def _usable_code(col):
    """Gate B coded test: non-null, non-blank, and not the zero sentinel."""
    v = F.trim(col.cast("string"))
    return col.isNotNull() & (v != "") & (v != "0")

def _code_or_display(code_col, display_col, excluded_displays=()):
    """Preserve clinically meaningful display-only rows without admitting placeholders."""
    display = F.trim(display_col.cast("string"))
    display_ok = display_col.isNotNull() & (display != "") & (display != "0")
    if excluded_displays:
        display_ok = display_ok & ~F.upper(display).isin(*excluded_displays)
    return F.when(_usable_code(code_col), code_col.cast("string")).when(display_ok, display)

def stable_id(namespace, *cols):
    """patient_event_id / encounter_id minting: sha2 over encoded namespace + natural-key parts.
    namespace = fact kind + source feed. NEVER the target table name (promotion-safe)."""
    return F.sha2(F.concat_ws(":", _enc(namespace), *[_enc(c) for c in cols]), 256)

def subject_key_with_system(candidates, source_table_lit, source_row_id_col):
    """Build a deterministic subject key from the strongest available source identifier.

    Identifier-less rows receive a deterministic per-source-row key and therefore remain
    non-joinable across feeds. No secret or environment-specific state is involved.
    """
    key = F.sha2(
        F.concat_ws(
            ":",
            _enc("subject"),
            _enc("nosubject"),
            _enc(source_table_lit),
            _enc(source_row_id_col),
        ),
        256,
    )
    system = F.lit("nosubject")
    for sys_lit, col in reversed(candidates):
        value = F.trim(col.cast("string"))
        present = col.isNotNull() & (value != "")
        key = F.when(
            present,
            F.sha2(F.concat_ws(":", _enc("subject"), _enc(sys_lit), _enc(col)), 256),
        ).otherwise(key)
        system = F.when(present, F.lit(sys_lit)).otherwise(system)
    return key, system

In [0]:
# ==== Naming the source a row came from ====

SOURCE_SYSTEM_DISPLAY = {
    "millennium": "Millennium",
    "millennium-pm": "Millennium (patient management)",
    "millennium-scheduling": "Millennium (scheduling)",
    "surginet": "Millennium SurgiNet",
    "luna": "LUNA",
    "jac": "JAC",
    "laboratory": "WinPath",
    "sectra-pacs": "Sectra PACS",
}

def _source_system_display(token_col):
    mapping = F.create_map(*[F.lit(x) for kv in SOURCE_SYSTEM_DISPLAY.items() for x in kv])
    return F.coalesce(mapping[token_col], F.initcap(token_col))

def _source_object_display(table_col):
    tail = F.element_at(F.split(table_col, r"\."), -1)
    unfixtured = F.regexp_replace(tail, r"_s\d+$", "")
    unprefixed = F.regexp_replace(unfixtured, r"^(map_|mill_|ancil_)", "")
    return F.regexp_replace(unprefixed, "_", " ")

In [0]:
# ==== CodeableConcept ====

def coding_obj(system_col, code_col, display_col, is_source, map_source=None, map_version=None):
    return F.struct(
        system_col.cast("string").alias("coding_system"),
        code_col.cast("string").alias("coding_code"),
        display_col.cast("string").alias("coding_display"),
        F.lit(is_source).alias("is_source"),
        (F.lit(map_source) if map_source else F.lit(None).cast("string")).alias("map_source"),
        (F.lit(map_version) if map_version else F.lit(None).cast("string")).alias("map_version"),
    )

def codeable_concept_json(*objs):
    """Deterministic JSON CodeableConcept for aggregate/join boundaries."""
    arr = F.filter(F.array(*objs), lambda o: o["is_source"] | o["coding_code"].isNotNull())
    return F.to_json(arr)

def codeable_concept(*objs):
    """VARIANT CodeableConcept. The source object is ALWAYS kept — 465 live rows have no source
    code, and a display-only source concept is valid; exactly-one-is_source is an invariant.
    Mapped objects are dropped when their code is null. parse_json(to_json(...)) — deterministic."""
    return F.parse_json(codeable_concept_json(*objs))

In [0]:
# ==== Cross-table quality flags ====

# Cross-table quality flags for products that publish a VARIANT column. The private
# stage does the joining with the VARIANT serialized to a JSON string, and the public
# product is a single-source parse_json wrapper over it: a VARIANT column coming out
# of a join-shaped flow crashes incremental planning and forces a full recompute on
# every update.
SILVER_CROSS_QC = {
    "condition": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "allergy_intolerance": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "appointment": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "cancer_treatment": {"fk": (), "dates": ("event_after_death_30d", "event_before_birth")},
    "condition_stage": {"fk": (), "dates": ("event_after_death_30d", "event_before_birth")},
    "device": {"fk": (), "dates": ("event_after_death_30d", "event_before_birth")},
    "family_history": {"fk": (), "dates": ("event_after_death_30d",)},
    "form": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "imaging_exam": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "medication_admin": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "medication_dispense": {"fk": (), "dates": ("event_before_birth",)},
    "pathology_report": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "procedure": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "registry_entry": {"fk": (), "dates": ("event_after_death_30d", "event_before_birth")},
    "rtt_pathway": {"fk": ("person_id",), "dates": ("event_after_death_30d", "event_before_birth")},
    "clinical_finding": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "clinical_score": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "medication_order": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "pathology_result": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "specimen": {"fk": ("person_id",), "dates": ("event_after_death_30d", "event_before_birth")},
    "vital_sign": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
    "document": {"fk": ("encounter_id", "person_id"), "dates": ("event_after_death_30d", "event_before_birth")},
}

def _cross_qc_primitive(df, model_name, variant_json_columns):
    for public_name, json_name in variant_json_columns.items():
        if public_name in df.columns:
            df = df.withColumn(json_name, F.to_json(F.col(public_name))).drop(public_name)
        elif json_name not in df.columns:
            raise RuntimeError(
                f"{model_name}: missing VARIANT column {public_name!r} and JSON bridge {json_name!r}"
            )

    spec = SILVER_CROSS_QC[model_name]
    fk_columns = spec["fk"]
    date_flags = spec["dates"]
    if "person_id" in fk_columns or date_flags:
        person = (
            spark.read.table(_n("journey_spine.person"))
            .select(
                F.col("person_id").alias("_qc_person_id"),
                F.col("birth_datetime").alias("_qc_birth_datetime"),
                F.col("deceased_datetime").alias("_qc_deceased_datetime"),
            )
            .where(F.col("_qc_person_id").isNotNull())
            .dropDuplicates(["_qc_person_id"])
        )
        df = df.join(person, df.person_id == person._qc_person_id, "left")
        if "person_id" in fk_columns:
            df = df.withColumn(
                "person_id_resolved",
                F.col("person_id").isNull() | F.col("_qc_person_id").isNotNull(),
            )
        if "event_before_birth" in date_flags:
            df = df.withColumn(
                "event_before_birth",
                F.coalesce(
                    F.col("event_datetime") < F.col("_qc_birth_datetime"),
                    F.lit(False),
                ),
            )
        if "event_after_death_30d" in date_flags:
            df = df.withColumn(
                "event_after_death_30d",
                F.coalesce(
                    F.col("event_datetime")
                    > F.col("_qc_deceased_datetime") + F.expr("INTERVAL 30 DAYS"),
                    F.lit(False),
                ),
            )
        df = df.drop("_qc_person_id", "_qc_birth_datetime", "_qc_deceased_datetime")
    if "encounter_id" in fk_columns:
        encounter = (
            spark.read.table(_n("journey_spine.encounter"))
            .select(F.col("encounter_id").alias("_qc_encounter_id"))
            .where(F.col("_qc_encounter_id").isNotNull())
            .dropDuplicates(["_qc_encounter_id"])
        )
        df = (
            df.join(encounter, df.encounter_id == encounter._qc_encounter_id, "left")
            .withColumn(
                "encounter_id_resolved",
                F.col("encounter_id").isNull() | F.col("_qc_encounter_id").isNotNull(),
            )
            .drop("_qc_encounter_id")
        )
    return df

def _cross_qc_flag_columns(model_name):
    spec = SILVER_CROSS_QC[model_name]
    present = {f"{column}_resolved" for column in spec["fk"]} | set(spec["dates"])
    return [
        flag
        for flag in (
            "person_id_resolved",
            "encounter_id_resolved",
            "episode_id_resolved",
            "event_before_birth",
            "event_after_death_30d",
        )
        if flag in present
    ]

def _cross_qc_public(df, model_name, variant_json_columns, public_columns):
    for public_name, json_name in variant_json_columns.items():
        df = df.withColumn(
            public_name, F.expr(f"parse_json(`{json_name}`)")
        )
    return df.select(*public_columns, *_cross_qc_flag_columns(model_name))

In [0]:
# ==== Person identity and reference dimensions ====

SRC_PERSON         = "4_prod.bronze.map_person"
SRC_PATIENT_IDENTIFIER = "4_prod.bronze.map_patient_identifier"

def _person_alias_selection(alias_type):
    """Pick the current MRN or NHS number from the governed alias feed.
    Ranking: active alias first (CURRENT_IND), then latest valid end-effective (open-ended
    sorts last via the 2100 sentinel), NULLS LAST via coalesce floors, deterministic
    SOURCE_PK tiebreak. MULTI_ACTIVE pools are never tiebroken: the whole (person, type)
    pool resolves to 'ambiguous' with NULL value."""
    a = read_source(SRC_PATIENT_IDENTIFIER).where(F.col("ALIAS_TYPE") == alias_type)
    ranked = a.select(
        a.PERSON_ID.cast("string").alias("_pid"),
        F.coalesce(a.MULTI_ACTIVE_IND.cast("int"), F.lit(0)).alias("_multi"),
        a.PIPELINE_UPDT_DT_TM.alias("_alias_loaded_at"),
        a.SOURCE_ADC_UPDT.alias("_alias_source_updt"),
        F.struct(
            F.coalesce(a.CURRENT_IND.cast("int"), F.lit(0)).alias("o_current"),
            F.coalesce(a.END_EFFECTIVE_DT_TM_CLEAN,
                       F.lit("2100-01-01").cast("timestamp")).alias("o_valid_to"),
            F.coalesce(a.BEG_EFFECTIVE_DT_TM_CLEAN,
                       F.lit("1900-01-01").cast("timestamp")).alias("o_valid_from"),
            F.coalesce(a.SOURCE_PK, F.lit(-1)).alias("o_tiebreak"),
            a.ALIAS_VALUE.alias("v_value"),
            F.coalesce(a.CURRENT_IND, F.lit(False)).alias("v_current"),
        ).alias("ranked"),
    )
    won = ranked.groupBy("_pid").agg(
        F.max("ranked").alias("w"),
        F.max("_multi").alias("_m"),
        F.max("_alias_loaded_at").alias("_alias_loaded_at"),
        F.max("_alias_source_updt").alias("_alias_source_updt"),
    )
    ambiguous = F.col("_m") == 1
    return won.select(
        F.col("_pid"),
        F.when(ambiguous, F.lit(None).cast("string")).otherwise(F.col("w.v_value")).alias("_value"),
        F.when(ambiguous, F.lit(None).cast("string"))
         .when(F.col("w.v_current"), F.lit("active")).otherwise(F.lit("inactive")).alias("_value_status"),
        F.when(ambiguous, F.lit("ambiguous"))
         .when(F.col("w.v_current"), F.lit("active_alias"))
         .otherwise(F.lit("latest_valid_alias")).alias("_selection_status"),
        F.col("_alias_loaded_at"),
        F.col("_alias_source_updt"),
    )

PERSON_COLUMN_COMMENTS = {
    "person_id": "Millennium person identifier and table primary key.",
    "active": "Source active indicator.",
    "gender_code": "Source administrative gender code.",
    "gender_display": "Source administrative gender display.",
    "ethnicity_code": "Source ethnicity code.",
    "ethnicity_display": "Source ethnicity display.",
    "birth_date": "Calendar birth date.",
    "birth_datetime": "Source-compatible birth timestamp.",
    "birth_precision_code": "Source code describing birth-date precision.",
    "birth_precision_display": "Display for the source birth precision.",
    "birth_precision_flag": "Raw source birth precision flag.",
    "language_code": "Preferred language source code.",
    "language_display": "Preferred language display.",
    "marital_status_code": "Source marital-status code.",
    "marital_status_display": "Source marital-status display.",
    "religion_code": "Source religion code.",
    "religion_display": "Source religion display.",
    "deceased_ind": "Whether the source indicates the person is deceased.",
    "deceased_datetime": "Source death timestamp.",
    "deceased_datetime_precision": "Raw source death-time precision.",
    "confidentiality_code": "Source confidentiality level carried for downstream access control.",
    "vip_ind": "Source VIP indicator carried as data.",
    "current_address_id": "Current source address reference when available.",
    "latest_known_address_id": "Latest-known source address reference.",
    "address_selection_status": "Provenance for current versus latest-known address selection.",
    "current_mrn": "Current hospital MRN selected from map_patient_identifier: active alias first, then latest valid end-effective, NULLS LAST, deterministic SOURCE_PK tiebreak; NULL when selection is ambiguous or no alias exists.",
    "current_mrn_status": "Lifecycle status of the selected hospital MRN alias; NULL when selection is ambiguous or no alias exists.",
    "mrn_selection_status": "Selection outcome for the governed hospital MRN alias pool; ambiguous = multiple active aliases",
    "nhs_number": "Current NHS number selected from map_patient_identifier: active alias first, then latest valid end-effective, NULLS LAST, deterministic SOURCE_PK tiebreak; NULL when selection is ambiguous or no alias exists.",
    "nhs_number_status": "Lifecycle status of the selected NHS-number alias; NULL when selection is ambiguous or no alias exists.",
    "nhs_number_selection_status": "Selection outcome for the governed NHS-number alias pool; ambiguous = multiple active aliases",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source status effective timestamp.",
    "record_status_effective_to": "Source status end timestamp when supplied.",
    "source_update_timestamp": "Native source update timestamp.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load/update timestamp; never pipeline wall-clock time.",
}

@materialized_view(
    name=_n("journey_spine.person"),
    comment="One resolved Millennium person. Silver retains inactive, implausible-looking, and deceased rows. Current MRN / NHS number selected deterministically from map_patient_identifier; multi-active pools publish as ambiguous, never tiebroken.",
    refresh_policy="incremental",
    column_comments=PERSON_COLUMN_COMMENTS,
)
def person():
    s = read_source(SRC_PERSON)
    mrn = _person_alias_selection("MRN").alias("mrn")
    nhs = _person_alias_selection("NHS").alias("nhs")
    ended = F.coalesce(
        s.end_effective_dt_tm < F.lit("2100-01-01").cast("timestamp"),
        F.lit(False),
    )
    joined = (
        s.join(mrn, s.person_id.cast("string") == F.col("mrn._pid"), "left")
         .join(nhs, s.person_id.cast("string") == F.col("nhs._pid"), "left")
    )
    return joined.select(
        s.person_id.cast("string").alias("person_id"),
        (s.active_ind == F.lit(1)).alias("active"),
        s.gender_cd.cast("string").alias("gender_code"),
        s.gender_display.alias("gender_display"),
        s.ethnicity_cd.cast("string").alias("ethnicity_code"),
        s.ethnicity_display.alias("ethnicity_display"),
        s.birth_date.alias("birth_date"),
        s.birth_datetime.alias("birth_datetime"),
        s.birth_dt_cd.cast("string").alias("birth_precision_code"),
        s.birth_dt_display.alias("birth_precision_display"),
        s.birth_precision_flag.cast("string").alias("birth_precision_flag"),
        s.language_cd.cast("string").alias("language_code"),
        s.language_display.alias("language_display"),
        s.marital_type_cd.cast("string").alias("marital_status_code"),
        s.marital_type_display.alias("marital_status_display"),
        s.religion_cd.cast("string").alias("religion_code"),
        s.religion_display.alias("religion_display"),
        (s.deceased_dt_tm.isNotNull() | (F.coalesce(s.deceased_cd, F.lit(0)) != 0)).alias("deceased_ind"),
        s.deceased_dt_tm.alias("deceased_datetime"),
        s.deceased_dt_tm_precision_flag.cast("string").alias("deceased_datetime_precision"),
        s.confid_level_cd.cast("string").alias("confidentiality_code"),
        (F.coalesce(s.vip_cd, F.lit(0)) != 0).alias("vip_ind"),
        s.current_address_id.cast("string").alias("current_address_id"),
        s.latest_known_address_id.cast("string").alias("latest_known_address_id"),
        s.address_selection_status.alias("address_selection_status"),
        F.col("mrn._value").alias("current_mrn"),
        F.col("mrn._value_status").alias("current_mrn_status"),
        F.coalesce(F.col("mrn._selection_status"), F.lit("no_alias")).alias("mrn_selection_status"),
        F.col("nhs._value").alias("nhs_number"),
        F.col("nhs._value_status").alias("nhs_number_status"),
        F.coalesce(F.col("nhs._selection_status"), F.lit("no_alias")).alias("nhs_number_selection_status"),
        F.when((s.active_ind == 1) & ~ended, F.lit("active"))
         .otherwise(F.lit("superseded")).alias("record_status"),
        s.active_status_dt_tm.alias("record_status_effective_from"),
        F.when(ended, s.end_effective_dt_tm).alias("record_status_effective_to"),
        F.greatest(
            s.source_updt_dt_tm,
            F.coalesce(F.col("mrn._alias_source_updt"), s.source_updt_dt_tm),
            F.coalesce(F.col("nhs._alias_source_updt"), s.source_updt_dt_tm),
        ).alias("source_update_timestamp"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.greatest(
            s.ADC_UPDT,
            F.coalesce(F.col("mrn._alias_loaded_at"), s.ADC_UPDT),
            F.coalesce(F.col("nhs._alias_loaded_at"), s.ADC_UPDT),
        ).alias("loaded_at"),
    )

In [0]:
SRC_COMMUNITY_LINK = "4_prod.bronze.map_community_patient_link"
SRC_PACS_PATIENT_LINK = "4_prod.bronze.map_pacs_patient_link"
SRC_ENDOBASE_PATIENT = "4_prod.bronze.map_endobase_patient"
SRC_CANCER_PTL_LINKAGE = "4_prod.bronze.map_cancer_ptl_linkage"

PERSON_IDENTIFIER_COLUMN_COMMENTS = {
    "person_identifier_id": "Deterministic identifier-assignment primary key.",
    "subject_key": "deterministic join key over the strongest available subject evidence.",
    "subject_id_system": "Identifier system used to derive subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identifier_system": "Namespace for the source identifier.",
    "identifier_value": "Source identifier value.",
    "identifier_type_code": "Identifier type code.",
    "status": "Source assignment or matching status.",
    "valid_from": "Identifier validity start.",
    "valid_to": "Identifier validity end.",
    "source_system": "Source system name.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "identity_status": "Resolution state.",
    "load_batch_id": "Bronze batch token when the source exposes one.",
    "loaded_at": "Bronze load timestamp when exposed.",
}

@materialized_view(
    name=_n("journey_spine.person_identifier"),
    comment="One source identifier assignment. subject_key is person-keyed where resolution exists and source-keyed otherwise.",
    refresh_policy="incremental",
    column_comments=PERSON_IDENTIFIER_COLUMN_COMMENTS,
)
def person_identifier():
    p = read_source(SRC_PERSON)
    p_skey, p_ssys = subject_key_with_system(
        [("urn:cerner:person_id", p.person_id)],
        SRC_PERSON,
        p.person_id,
    )
    p_rows = p.select(
        stable_id("person_identifier:mill:person_id", p.person_id).alias("person_identifier_id"),
        p_skey.alias("subject_key"),
        p_ssys.alias("subject_id_system"),
        p.person_id.cast("string").alias("person_id"),
        F.lit("urn:cerner:person_id").alias("identifier_system"),
        p.person_id.cast("string").alias("identifier_value"),
        F.lit("PI").alias("identifier_type_code"),
        F.when(p.active_ind == 1, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
        p.beg_effective_dt_tm.alias("valid_from"),
        p.end_effective_dt_tm.alias("valid_to"),
        F.lit("millennium").alias("source_system"),
        F.lit(SRC_PERSON).alias("source_table"),
        p.person_id.cast("string").alias("source_row_id"),
        F.lit("resolved").alias("identity_status"),
        F.date_format(p.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        p.ADC_UPDT.alias("loaded_at"),
    )

    c = read_source(SRC_COMMUNITY_LINK)
    c_skey, c_ssys = subject_key_with_system(
        [("urn:cerner:person_id", c.person_id),
         ("urn:barts:community_patient_key", c.community_patient_key)],
        SRC_COMMUNITY_LINK,
        c.community_patient_key,
    )
    common = [
        c_skey.alias("subject_key"),
        c_ssys.alias("subject_id_system"),
        c.person_id.cast("string").alias("person_id"),
        c.person_match_status.alias("status"),
        c.community_registration_date.cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        F.lit("bh-community").alias("source_system"),
        F.lit(SRC_COMMUNITY_LINK).alias("source_table"),
        c.community_patient_key.alias("source_row_id"),
        F.when(c.person_id.isNotNull(), F.lit("resolved"))
         .when(c.community_patient_key.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("loaded_at"),
    ]
    c_key_rows = c.select(
        stable_id("person_identifier:community:key", c.community_patient_key).alias("person_identifier_id"),
        *common[:3],
        F.lit("urn:barts:community_patient_key").alias("identifier_system"),
        c.community_patient_key.alias("identifier_value"),
        F.lit("COMMUNITY_REGISTRATION").alias("identifier_type_code"),
        *common[3:],
    )
    c_number_rows = c.where(c.source_patient_number.isNotNull()).select(
        stable_id("person_identifier:community:number", c.source_database_id, c.source_patient_number)
          .alias("person_identifier_id"),
        *common[:3],
        F.concat(F.lit("urn:barts:community_patient_number:"), c.source_database_id.cast("string"))
          .alias("identifier_system"),
        c.source_patient_number.cast("string").alias("identifier_value"),
        F.lit("MR").alias("identifier_type_code"),
        *common[3:],
    )
    pacs = read_source(SRC_PACS_PATIENT_LINK)
    pacs_skey, pacs_ssys = subject_key_with_system(
        [("urn:cerner:person_id", pacs.PERSON_ID),
         ("urn:sectra:pacs-patient-id", pacs.PACS_PATIENT_ID)],
        SRC_PACS_PATIENT_LINK,
        pacs.PACS_PATIENT_ID,
    )
    pacs_rows = pacs.select(
        stable_id("person_identifier:pacs:patient", pacs.PACS_PATIENT_ID)
        .alias("person_identifier_id"),
        pacs_skey.alias("subject_key"), pacs_ssys.alias("subject_id_system"),
        pacs.PERSON_ID.cast("string").alias("person_id"),
        F.lit("urn:sectra:pacs-patient-id").alias("identifier_system"),
        pacs.PACS_PATIENT_ID.cast("string").alias("identifier_value"),
        F.lit("PACS_INTERNAL").alias("identifier_type_code"),
        pacs.PERSON_MATCH_STATUS.alias("status"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.when(~F.coalesce(pacs.SOURCE_PRESENT_IND, F.lit(True)), pacs.ADC_UPDT)
        .alias("valid_to"),
        F.lit("sectra-pacs").alias("source_system"),
        F.lit(SRC_PACS_PATIENT_LINK).alias("source_table"),
        pacs.PACS_PATIENT_ID.cast("string").alias("source_row_id"),
        F.when(pacs.PERSON_ID.isNotNull(), F.lit("resolved"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.date_format(pacs.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        pacs.ADC_UPDT.alias("loaded_at"),
    )

    ali = read_source(SRC_PATIENT_IDENTIFIER)
    ali_skey, ali_ssys = subject_key_with_system(
        [("urn:cerner:person_id", ali.PERSON_ID)], SRC_PATIENT_IDENTIFIER, ali.SOURCE_PK
    )
    ali_rows = ali.select(
        stable_id("person_identifier:mill_alias", ali.SOURCE_PK).alias("person_identifier_id"),
        ali_skey.alias("subject_key"),
        ali_ssys.alias("subject_id_system"),
        ali.PERSON_ID.cast("string").alias("person_id"),
        F.when(ali.ALIAS_TYPE == F.lit("NHS"), F.lit("https://fhir.nhs.uk/Id/nhs-number"))
         .otherwise(F.lit("urn:barts:mrn")).alias("identifier_system"),
        ali.ALIAS_VALUE.alias("identifier_value"),
        F.when(ali.ALIAS_TYPE == F.lit("NHS"), F.lit("NH")).otherwise(F.lit("MR")).alias("identifier_type_code"),
        F.when(F.coalesce(ali.CURRENT_IND, F.lit(False)), F.lit("active"))
         .otherwise(F.lit("inactive")).alias("status"),
        ali.BEG_EFFECTIVE_DT_TM.alias("valid_from"),
        F.when(ali.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
               ali.END_EFFECTIVE_DT_TM).alias("valid_to"),
        F.lit("millennium").alias("source_system"),
        F.lit(SRC_PATIENT_IDENTIFIER).alias("source_table"),
        ali.SOURCE_PK.cast("string").alias("source_row_id"),
        F.lit("resolved").alias("identity_status"),
        F.date_format(ali.PIPELINE_UPDT_DT_TM, "yyyyMMddHHmmss").alias("load_batch_id"),
        ali.PIPELINE_UPDT_DT_TM.alias("loaded_at"),
    )

    e = read_source(SRC_ENDOBASE_PATIENT)
    e_skey, e_ssys = subject_key_with_system(
        [("urn:cerner:person_id", e.PERSON_ID),
         ("urn:barts:endobase:patient-id", e.ENDOBASE_PATIENT_ID)],
        SRC_ENDOBASE_PATIENT,
        e.ENDOBASE_PATIENT_ID,
    )
    e_present = F.coalesce(e.SOURCE_PRESENT_IND, F.lit(True))
    e_rows = e.select(
        stable_id("person_identifier:endobase", e.ENDOBASE_PATIENT_ID)
        .alias("person_identifier_id"),
        e_skey.alias("subject_key"),
        e_ssys.alias("subject_id_system"),
        e.PERSON_ID.cast("string").alias("person_id"),
        F.lit("urn:barts:endobase:patient-id").alias("identifier_system"),
        e.ENDOBASE_PATIENT_ID.cast("string").alias("identifier_value"),
        F.lit("ENDOBASE_INTERNAL").alias("identifier_type_code"),
        F.when(e_present, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
        e.SOURCE_CREATE_DT.cast("timestamp").alias("valid_from"),
        F.when(~e_present, e.ADC_UPDT).alias("valid_to"),
        F.lit("endobase").alias("source_system"),
        F.lit(SRC_ENDOBASE_PATIENT).alias("source_table"),
        e.ENDOBASE_PATIENT_ID.cast("string").alias("source_row_id"),
        F.when(e.PERSON_ID.isNotNull(), F.lit("resolved"))
         .when(e.PERSON_LINK_STATUS.isin("CONSENSUS", "MRN_ONLY", "NHS_ONLY"),
               F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.date_format(e.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        e.ADC_UPDT.alias("loaded_at"),
    )

    lk = read_source(SRC_CANCER_PTL_LINKAGE)
    lk_skey, lk_ssys = subject_key_with_system(
        [("urn:cerner:person_id", lk.PERSON_ID)],
        SRC_CANCER_PTL_LINKAGE,
        lk.ROW_ID.cast("string"),
    )

    def _ptl_identifier(namespace, system, value_col, type_code):
        return lk.where(F.trim(F.coalesce(value_col.cast("string"), F.lit(""))) != "").select(
            stable_id(namespace, lk.ROW_ID).alias("person_identifier_id"),
            lk_skey.alias("subject_key"),
            lk_ssys.alias("subject_id_system"),
            lk.PERSON_ID.cast("string").alias("person_id"),
            F.lit(system).alias("identifier_system"),
            value_col.cast("string").alias("identifier_value"),
            F.lit(type_code).alias("identifier_type_code"),
            F.when(F.coalesce(lk.PERSON_VALID_IND, F.lit(False)), F.lit("MATCHED"))
             .otherwise(F.lit("UNVALIDATED")).alias("status"),
            lk.REFERRAL_DATE_CLEAN.cast("timestamp").alias("valid_from"),
            F.lit(None).cast("timestamp").alias("valid_to"),
            F.lit("luna-pathfinder").alias("source_system"),
            F.lit(SRC_CANCER_PTL_LINKAGE).alias("source_table"),
            lk.ROW_ID.cast("string").alias("source_row_id"),
            F.lit("resolved").alias("identity_status"),
            F.date_format(lk.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
            lk.ADC_UPDT.alias("loaded_at"),
        )

    ptl_mrn_rows = _ptl_identifier(
        "person_identifier:luna_ptl:mrn", "urn:barts:mrn", lk.HOSPITAL_NUMBER, "MR"
    )
    ptl_nhs_rows = _ptl_identifier(
        "person_identifier:luna_ptl:nhs",
        "https://fhir.nhs.uk/Id/nhs-number",
        lk.NHS_NUMBER,
        "NH",
    )
    return (
        p_rows.unionByName(c_key_rows).unionByName(c_number_rows)
        .unionByName(pacs_rows).unionByName(ali_rows).unionByName(e_rows)
        .unionByName(ptl_mrn_rows).unionByName(ptl_nhs_rows)
    )

In [0]:
# ==== Episode spine containers — source-backed only, never event-indexed ====

SRC_EPISODE = "4_prod.bronze.map_episode"

EPISODE_COLUMN_COMMENTS = {
    "episode_id": "Deterministic id minted from the Millennium episode identifier.",
    "person_id": "Millennium person identifier.",
    "subject_key": "deterministic subject key.",
    "episode_display": "Source episode display name.",
    "episode_type_code": "Source episode type code.",
    "episode_type_display": "Source episode type display.",
    "status_code": "Source episode status code.",
    "status_display": "Source episode status display.",
    "period_start": "Sentinel-cleaned episode begin timestamp (bronze DQ triplet).",
    "period_end": "Sentinel-cleaned episode end timestamp when supplied.",
    "breach_datetime": "Sentinel-cleaned episode breach timestamp when supplied.",
    "pause_days": "Source pause-day count.",
    "close_reason_code": "Source close-reason code.",
    "close_reason_display": "Source close-reason display.",
    "service_category_code": "Source service-category code.",
    "service_category_display": "Source service-category display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "contributor_system_code": "Source contributor-system code.",
    "contributor_system_display": "Source contributor-system display.",
    "direct_encounter_id": "Agreement-gated direct encounter reference published by bronze alongside the N-to-M relation; evidence only",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source status effective timestamp.",
    "record_status_effective_to": "Source status end timestamp when superseded.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load timestamp; never pipeline wall-clock time.",
}

@materialized_view(
    name=_n("journey_spine.episode"),
    comment="One Millennium episode container (map_episode). Source-backed only; the direct "
            "encounter reference is agreement-gated evidence, never containment.",
    cluster_by=["person_id", "period_start"],
    refresh_policy="incremental",
    column_comments=EPISODE_COLUMN_COMMENTS,
)
def episode():
    s = read_source(SRC_EPISODE)
    skey, _ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_EPISODE, s.EPISODE_ID
    )
    superseded = (~F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))) | (
        F.coalesce(s.ACTIVE_IND, F.lit(1)) != 1
    )
    return s.select(
        stable_id("episode:mill", s.EPISODE_ID).alias("episode_id"),
        s.PERSON_ID.cast("string").alias("person_id"),
        skey.alias("subject_key"),
        s.DISPLAY.alias("episode_display"),
        s.EPISODE_TYPE_CD.cast("string").alias("episode_type_code"),
        s.EPISODE_TYPE_DESC.alias("episode_type_display"),
        s.EPISODE_STATUS_CD.cast("string").alias("status_code"),
        s.EPISODE_STATUS_DESC.alias("status_display"),
        s.EPISODE_START_DT_TM_CLEAN.alias("period_start"),
        s.EPISODE_STOP_DT_TM_CLEAN.alias("period_end"),
        s.EPISODE_BREACH_DT_TM_CLEAN.alias("breach_datetime"),
        s.EPISODE_PAUSE_DAYS_CNT.cast("long").alias("pause_days"),
        s.EPISODE_CLOSE_REASON_CD.cast("string").alias("close_reason_code"),
        s.EPISODE_CLOSE_REASON_DESC.alias("close_reason_display"),
        s.SERVICE_CATEGORY_CD.cast("string").alias("service_category_code"),
        s.SERVICE_CATEGORY_DESC.alias("service_category_display"),
        s.REFER_FACILITY_CD.cast("string").alias("referring_facility_code"),
        s.REFER_FACILITY_DESC.alias("referring_facility_display"),
        s.CONTRIBUTOR_SYSTEM_CD.cast("string").alias("contributor_system_code"),
        s.CONTRIBUTOR_SYSTEM_DESC.alias("contributor_system_display"),
        F.when(s.ENCNTR_ID.isNotNull(),
               stable_id("encounter:mill", s.ENCNTR_ID)).alias("direct_encounter_id"),
        F.when(superseded, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(
            superseded,
            F.coalesce(
                F.when(s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
                       s.END_EFFECTIVE_DT_TM),
                s.ACTIVE_STATUS_DT_TM,
                s.ADC_UPDT,
            ),
        ).alias("record_status_effective_to"),
        F.lit(SRC_EPISODE).alias("source_table"),
        s.EPISODE_ID.cast("string").alias("source_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
SRC_EPISODE_ENCOUNTER = "4_prod.bronze.map_episode_encounter"

EPISODE_ENCOUNTER_COLUMN_COMMENTS = {
    "episode_encounter_id": "Deterministic id minted from the source relation row.",
    "episode_id": "Episode container reference (minted; may reference a source-absent episode — see relation_status_code).",
    "encounter_id": "Member encounter reference (minted).",
    "relation_status_code": "Source relation classification including the orphan marker for episode IDs absent from mill_episode.",
    "source_duplicate_count": "Raw byte-identical duplicate rows collapsed into this canonical row by bronze.",
    "valid_from": "Membership effective-from timestamp.",
    "valid_to": "Membership effective-to timestamp when supplied.",
    "record_status": "Normalized source lifecycle status.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_spine.episode_encounter"),
    comment="N:M episode-encounter membership (map_episode_encounter). Multi-relation pairs "
            "preserved; orphan episode references retained with their source classification.",
    cluster_by=["episode_id"],
    refresh_policy="incremental",
    column_comments=EPISODE_ENCOUNTER_COLUMN_COMMENTS,
)
def episode_encounter():
    s = read_source(SRC_EPISODE_ENCOUNTER)
    superseded = (~F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))) | (
        F.coalesce(s.ACTIVE_IND, F.lit(1)) != 1
    )
    return s.select(
        stable_id("episode_encounter:mill", s.EPISODE_ENCNTR_RELTN_ID).alias("episode_encounter_id"),
        stable_id("episode:mill", s.EPISODE_ID).alias("episode_id"),
        stable_id("encounter:mill", s.ENCNTR_ID).alias("encounter_id"),
        s.EPISODE_LINK_STATUS.alias("relation_status_code"),
        s.SOURCE_DUPLICATE_COUNT.cast("long").alias("source_duplicate_count"),
        s.BEG_EFFECTIVE_DT_TM.alias("valid_from"),
        F.when(s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
               s.END_EFFECTIVE_DT_TM).alias("valid_to"),
        F.when(superseded, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.lit(SRC_EPISODE_ENCOUNTER).alias("source_table"),
        s.EPISODE_ENCNTR_RELTN_ID.cast("string").alias("source_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
SRC_ENCOUNTER_IDENTIFIER = "4_prod.bronze.map_encounter_identifier"

ENCOUNTER_IDENTIFIER_COLUMN_COMMENTS = {
    "encounter_identifier_id": "Deterministic id minted from the source alias row key.",
    "encounter_id": "Encounter reference (minted).",
    "identifier_system": "Type-scoped identifier system URI (urn:cerner:encntr_alias:<normalized alias type>) — 23.17M values live under multiple alias types",
    "identifier_type_code": "Source alias type verbatim.",
    "identifier_value": "Identifier value; published and IG-governed at serve time",
    "status": "Alias lifecycle status.",
    "current_ind": "Bronze current-alias indicator.",
    "multi_active_ind": "Bronze multiple-active-aliases indicator for this encounter and type.",
    "valid_from": "Alias effective-from timestamp.",
    "valid_to": "Alias effective-to timestamp when supplied.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze pipeline write timestamp (this feed family has no ADC_UPDT).",
}

@materialized_view(
    name=_n("journey_spine.encounter_identifier"),
    comment="One source encounter identifier assignment (map_encounter_identifier); reference "
            "surface parallel to person_identifier. Feed family has no ADC_UPDT: loaded_at rides "
            "the bronze pipeline write timestamp.",
    refresh_policy="incremental",
    column_comments=ENCOUNTER_IDENTIFIER_COLUMN_COMMENTS,
)
def encounter_identifier():
    # 22 live rows carry a NULL ALIAS_VALUE (8 of them current). An identifier row with
    # no value says nothing, so those rows are excluded and the row-count check compares
    # against the non-null source count.
    a = read_source(SRC_ENCOUNTER_IDENTIFIER).where(F.col("ALIAS_VALUE").isNotNull())
    return a.select(
        stable_id("encounter_identifier:mill_alias", a.SOURCE_PK).alias("encounter_identifier_id"),
        stable_id("encounter:mill", a.ENCNTR_ID).alias("encounter_id"),
        # Type-scoped system: 23.17M ALIAS_VALUEs live under MULTIPLE alias types — a single
        # system string would collide (system, value) across types.
        F.concat(
            F.lit("urn:cerner:encntr_alias:"),
            F.lower(F.regexp_replace(F.coalesce(a.ALIAS_TYPE, F.lit("unknown")),
                                     "[^A-Za-z0-9]+", "_")),
        ).alias("identifier_system"),
        a.ALIAS_TYPE.alias("identifier_type_code"),
        a.ALIAS_VALUE.alias("identifier_value"),
        F.when(F.coalesce(a.CURRENT_IND, F.lit(False)), F.lit("active"))
         .otherwise(F.lit("inactive")).alias("status"),
        a.CURRENT_IND.alias("current_ind"),
        a.MULTI_ACTIVE_IND.alias("multi_active_ind"),
        a.BEG_EFFECTIVE_DT_TM.alias("valid_from"),
        F.when(a.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
               a.END_EFFECTIVE_DT_TM).alias("valid_to"),
        F.lit(SRC_ENCOUNTER_IDENTIFIER).alias("source_table"),
        a.SOURCE_PK.cast("string").alias("source_row_id"),
        F.date_format(a.PIPELINE_UPDT_DT_TM, "yyyyMMddHHmmss").alias("load_batch_id"),
        a.PIPELINE_UPDT_DT_TM.alias("loaded_at"),
    )

In [0]:
SRC_CARE_SITE      = "4_prod.bronze.map_care_site"

def _location_projection():
    s = read_source(SRC_CARE_SITE)

    facilities = (
        s.where(s.facility_cd.isNotNull())
         .groupBy("facility_cd")
         .agg(
             F.max("facility_name").alias("name"),
             F.max("facility_code_active_ind").alias("active_ind"),
             F.min("facility_relation_beg_effective_dt_tm").alias("valid_from"),
             F.max("facility_relation_end_effective_dt_tm").alias("valid_to"),
             F.max("ORGANIZATION_ID").alias("organization_id"),
             F.max("masked_zipcode").alias("address_postcode_masked"),
             F.max("city").alias("address_city"),
             F.max("latitude").alias("latitude"),
             F.max("longitude").alias("longitude"),
             F.max("ADC_UPDT").alias("loaded_at"),
         )
         .select(
             stable_id("location:mill:facility", F.col("facility_cd")).alias("location_id"),
             F.lit(None).cast("string").alias("parent_location_id"),
             F.lit("facility").alias("location_level"),
             F.col("facility_cd").cast("string").alias("source_location_code"),
             F.col("name"),
             F.when(F.col("active_ind") == 1, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
             F.when(F.col("organization_id").isNotNull(),
                    stable_id("organization:mill", F.col("organization_id"))).alias("organization_id"),
             F.lit("si").alias("physical_type_code"),
             F.col("valid_from"), F.col("valid_to"),
             F.col("latitude"), F.col("longitude"),
             F.col("address_city"), F.col("address_postcode_masked"),
             F.lit(SRC_CARE_SITE).alias("source_table"),
             F.col("facility_cd").cast("string").alias("source_row_id"),
             F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
             F.col("loaded_at"),
         )
    )

    buildings = (
        s.where(s.building_cd.isNotNull())
         .groupBy("building_cd")
         .agg(
             F.max("building_name").alias("name"),
             F.max("building_code_active_ind").alias("active_ind"),
             F.min("building_relation_beg_effective_dt_tm").alias("valid_from"),
             F.max("building_relation_end_effective_dt_tm").alias("valid_to"),
             F.min("facility_cd").alias("facility_cd"),
             F.max("ORGANIZATION_ID").alias("organization_id"),
             F.max("masked_zipcode").alias("address_postcode_masked"),
             F.max("city").alias("address_city"),
             F.max("latitude").alias("latitude"),
             F.max("longitude").alias("longitude"),
             F.max("ADC_UPDT").alias("loaded_at"),
         )
         .select(
             stable_id("location:mill:building", F.col("building_cd")).alias("location_id"),
             F.when(F.col("facility_cd").isNotNull(),
                    stable_id("location:mill:facility", F.col("facility_cd"))).alias("parent_location_id"),
             F.lit("building").alias("location_level"),
             F.col("building_cd").cast("string").alias("source_location_code"),
             F.col("name"),
             F.when(F.col("active_ind") == 1, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
             F.when(F.col("organization_id").isNotNull(),
                    stable_id("organization:mill", F.col("organization_id"))).alias("organization_id"),
             F.lit("bu").alias("physical_type_code"),
             F.col("valid_from"), F.col("valid_to"),
             F.col("latitude"), F.col("longitude"),
             F.col("address_city"), F.col("address_postcode_masked"),
             F.lit(SRC_CARE_SITE).alias("source_table"),
             F.col("building_cd").cast("string").alias("source_row_id"),
             F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
             F.col("loaded_at"),
         )
    )

    units = s.select(
        stable_id("location:mill:nurse_unit", s.care_site_cd).alias("location_id"),
        F.when(s.building_cd.isNotNull(), stable_id("location:mill:building", s.building_cd))
         .when(s.facility_cd.isNotNull(), stable_id("location:mill:facility", s.facility_cd))
         .alias("parent_location_id"),
        F.lit("nurse_unit").alias("location_level"),
        s.care_site_cd.cast("string").alias("source_location_code"),
        s.care_site_name.alias("name"),
        F.when(s.location_active_ind == 1, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
        F.when(s.ORGANIZATION_ID.isNotNull(),
               stable_id("organization:mill", s.ORGANIZATION_ID)).alias("organization_id"),
        F.lit("wa").alias("physical_type_code"),
        s.location_beg_effective_dt_tm.alias("valid_from"),
        s.location_end_effective_dt_tm.alias("valid_to"),
        s.latitude, s.longitude,
        s.city.alias("address_city"),
        s.masked_zipcode.alias("address_postcode_masked"),
        F.lit(SRC_CARE_SITE).alias("source_table"),
        s.care_site_cd.cast("string").alias("source_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("loaded_at"),
    )
    return facilities.unionByName(buildings).unionByName(units)

In [0]:
LOCATION_COLUMN_COMMENTS = {
    "location_id": "Deterministic location primary key.",
    "parent_location_id": "Parent facility or building location.",
    "location_level": "Derived hierarchy level.",
    "source_location_code": "Source Millennium location code.",
    "name": "Source location name.",
    "status": "Source-derived location status.",
    "organization_id": "Managing organization reference.",
    "physical_type_code": "FHIR physical location type code.",
    "valid_from": "Source validity start.",
    "valid_to": "Source validity end.",
    "latitude": "Source address latitude.",
    "longitude": "Source address longitude.",
    "address_city": "Source address city.",
    "address_postcode_masked": "Privacy-aware source postcode.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_reference.location"),
    comment="Effective-dated facility, building, and nurse-unit hierarchy derived from bronze care sites.",
    refresh_policy="incremental",
    column_comments=LOCATION_COLUMN_COMMENTS,
)
def location():
    return _location_projection()

In [0]:
SRC_PRACTITIONER   = "4_prod.bronze.map_medical_personnel"
SRC_ENCOUNTER      = "4_prod.bronze.map_encounter"
SRC_DIAGNOSIS = "4_prod.bronze.map_diagnosis"
SRC_PROBLEM = "4_prod.bronze.map_problem"
SRC_IMPLANT_DETAILS = "4_prod.bronze.map_implant_details"
SRC_FORM_ACTIVITY = "4_prod.bronze.mill_form_activity"
SRC_NUMERIC_EVENTS = "4_prod.bronze.map_numeric_events"
SRC_MED_ADMIN = "4_prod.bronze.map_med_admin"
SRC_MEDICATION_ORDER = "4_prod.bronze.map_medication_order"
SRC_MEDICATION_ORDER_ACTION = "4_prod.bronze.map_medication_order_action"
SRC_CODED_EVENTS = "4_prod.bronze.map_coded_events"
SRC_NOMEN_EVENTS = "4_prod.bronze.map_nomen_events"
SRC_DATE_EVENTS = "4_prod.bronze.map_date_events"
SRC_TEXT_EVENTS = "4_prod.bronze.map_text_events"
SRC_MILL_BLOB_TEXT = "4_prod.bronze.mill_blob_text"
SRC_APPOINTMENT = "4_prod.bronze.map_appointment"
SRC_APPOINTMENT_RESOURCE = "4_prod.bronze.map_appointment_resource"
SRC_THEATRE_CASE = "4_prod.bronze.map_theatre_case"
SRC_THEATRE_CASE_PROCEDURE = "4_prod.bronze.map_theatre_case_procedure"

In [0]:
PRACTITIONER_COLUMN_COMMENTS = {
    "practitioner_id": "Deterministic practitioner primary key.",
    "source_practitioner_id": "Millennium personnel identifier.",
    "active": "Source active indicator.",
    "name": "Source formatted practitioner name.",
    "physician_ind": "Source physician indicator.",
    "position_code": "Source position code.",
    "position_display": "Source position display.",
    "practitioner_type_code": "Source personnel type code.",
    "practitioner_type_display": "Source personnel type display.",
    "primary_location_id": "Primary assigned or inferred location reference.",
    "medical_service_id": "Selected medical service reference.",
    "npi": "Selected NPI identifier.",
    "doctor_number": "Selected organization doctor number.",
    "gdp_number": "Selected dental practitioner number.",
    "external_provider_id": "Selected external provider identifier.",
    "record_status": "Normalized lifecycle status.",
    "valid_from": "Source validity start.",
    "valid_to": "Source validity end.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_reference.practitioner"),
    comment="One effective-dated Millennium practitioner/personnel row; inactive rows are retained.",
    refresh_policy="incremental",
    column_comments=PRACTITIONER_COLUMN_COMMENTS,
)
def practitioner():
    s = read_source(SRC_PRACTITIONER)
    directory = s.select(
        stable_id("practitioner:mill", s.PERSON_ID).alias("practitioner_id"),
        s.PERSON_ID.cast("string").alias("source_practitioner_id"),
        (s.ACTIVE_IND == 1).alias("active"),
        s.NAME_FULL_FORMATTED.alias("name"),
        s.PHYSICIAN_IND.alias("physician_ind"),
        s.POSITION_CD.cast("string").alias("position_code"),
        s.position_name.alias("position_display"),
        s.PRSNL_TYPE_CD.cast("string").alias("practitioner_type_code"),
        s.prsnl_type_name.alias("practitioner_type_display"),
        F.when(s.primary_care_site_cd.isNotNull(),
               stable_id("location:mill:nurse_unit", s.primary_care_site_cd)).alias("primary_location_id"),
        F.when(s.MEDSERVICE_GROUP_ID.isNotNull(),
               stable_id("service:mill:medical", s.MEDSERVICE_GROUP_ID)).alias("medical_service_id"),
        s.NPI.alias("npi"),
        s.DOCNBR.alias("doctor_number"),
        s.GDP_NUMBER.alias("gdp_number"),
        s.EXTERNAL_PROVIDER_ID.alias("external_provider_id"),
        F.when(s.ACTIVE_IND == 1, F.lit("active")).otherwise(F.lit("superseded")).alias("record_status"),
        s.BEG_EFFECTIVE_DT_TM.alias("valid_from"),
        s.END_EFFECTIVE_DT_TM.alias("valid_to"),
        F.lit(SRC_PRACTITIONER).alias("source_table"),
        s.PERSON_ID.cast("string").alias("source_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("loaded_at"),
    )
    encounter = read_source(SRC_ENCOUNTER)
    referenced = (
        encounter.where(encounter.REG_PRSNL_ID.isNotNull())
                 .select(encounter.REG_PRSNL_ID.cast("string").alias("source_practitioner_id"),
                         encounter.ADC_UPDT.alias("loaded_at"))
        .unionByName(
            encounter.where(encounter.CREATE_PRSNL_ID.isNotNull())
                     .select(encounter.CREATE_PRSNL_ID.cast("string").alias("source_practitioner_id"),
                             encounter.ADC_UPDT.alias("loaded_at"))
        )
        .unionByName(
            encounter.where(encounter.DISCH_PRSNL_ID.isNotNull())
                     .select(encounter.DISCH_PRSNL_ID.cast("string").alias("source_practitioner_id"),
                             encounter.ADC_UPDT.alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_DIAGNOSIS).where(F.col("DIAG_PRSNL_ID").isNotNull())
              .select(F.col("DIAG_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_PROBLEM).where(F.col("ACTIVE_STATUS_PRSNL_ID").isNotNull())
              .select(F.col("ACTIVE_STATUS_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_IMPLANT_DETAILS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.coalesce(F.col("SOURCE_MAX_ADC_UPDT"), F.col("ADC_UPDT")).alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_FORM_ACTIVITY).where(F.col("PERFORMED_PRSNL_ID_LONG").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID_LONG").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_NUMERIC_EVENTS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MED_ADMIN).where(F.col("PRSNL_ID").isNotNull())
              .select(F.col("PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MED_ADMIN).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MED_ADMIN).where(F.col("CE_VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("CE_VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MED_ADMIN).where(F.col("MAE_VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("MAE_VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MEDICATION_ORDER).where(F.col("LAST_UPDATE_PROVIDER_ID").isNotNull())
              .select(F.col("LAST_UPDATE_PROVIDER_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MEDICATION_ORDER_ACTION).where(F.col("ACTION_PERSONNEL_ID").isNotNull())
              .select(F.col("ACTION_PERSONNEL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MEDICATION_ORDER_ACTION).where(F.col("ORDER_PROVIDER_ID").isNotNull())
              .select(F.col("ORDER_PROVIDER_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MEDICATION_ORDER_ACTION).where(F.col("SUPERVISING_PROVIDER_ID").isNotNull())
              .select(F.col("SUPERVISING_PROVIDER_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_CODED_EVENTS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_CODED_EVENTS).where(F.col("VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_NOMEN_EVENTS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_NOMEN_EVENTS).where(F.col("VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_DATE_EVENTS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_DATE_EVENTS).where(F.col("VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_TEXT_EVENTS).where(F.col("PERFORMED_PRSNL_ID").isNotNull())
              .select(F.col("PERFORMED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_TEXT_EVENTS).where(F.col("VERIFIED_PRSNL_ID").isNotNull())
              .select(F.col("VERIFIED_PRSNL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_MILL_BLOB_TEXT).where(F.col("UPDT_ID").isNotNull())
              .select(F.col("UPDT_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_APPOINTMENT).where(F.col("REQUESTED_PERSONNEL_ID").isNotNull())
              .select(F.col("REQUESTED_PERSONNEL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_APPOINTMENT_RESOURCE).where(F.col("ALLOCATED_PERSONNEL_ID").isNotNull())
              .select(F.col("ALLOCATED_PERSONNEL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_THEATRE_CASE).where(F.col("SURGEON_PERSONNEL_ID").isNotNull())
              .select(F.col("SURGEON_PERSONNEL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_THEATRE_CASE).where(F.col("ANAESTHETIST_PERSONNEL_ID").isNotNull())
              .select(F.col("ANAESTHETIST_PERSONNEL_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .unionByName(
            read_source(SRC_THEATRE_CASE_PROCEDURE).where(F.col("PRIMARY_SURGEON_ID").isNotNull())
              .select(F.col("PRIMARY_SURGEON_ID").cast("string").alias("source_practitioner_id"),
                      F.col("ADC_UPDT").alias("loaded_at"))
        )
        .groupBy("source_practitioner_id")
        .agg(F.max("loaded_at").alias("loaded_at"))
    )
    known = s.select(s.PERSON_ID.cast("string").alias("source_practitioner_id"))
    missing = referenced.join(known, "source_practitioner_id", "left_anti")
    placeholders = missing.select(
        stable_id("practitioner:mill", F.col("source_practitioner_id")).alias("practitioner_id"),
        F.col("source_practitioner_id"),
        F.lit(None).cast("boolean").alias("active"),
        F.lit(None).cast("string").alias("name"),
        F.lit(None).cast("boolean").alias("physician_ind"),
        F.lit(None).cast("string").alias("position_code"),
        F.lit(None).cast("string").alias("position_display"),
        F.lit(None).cast("string").alias("practitioner_type_code"),
        F.lit(None).cast("string").alias("practitioner_type_display"),
        F.lit(None).cast("string").alias("primary_location_id"),
        F.lit(None).cast("string").alias("medical_service_id"),
        F.lit(None).cast("string").alias("npi"),
        F.lit(None).cast("string").alias("doctor_number"),
        F.lit(None).cast("string").alias("gdp_number"),
        F.lit(None).cast("string").alias("external_provider_id"),
        F.lit("active").alias("record_status"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        F.lit(SRC_ENCOUNTER).alias("source_table"),
        F.col("source_practitioner_id").alias("source_row_id"),
        F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("loaded_at"),
    )
    return directory.unionByName(placeholders)

In [0]:
SERVICE_COLUMN_COMMENTS = {
    "service_id": "Deterministic HealthcareService primary key.",
    "source_service_code": "Source personnel-group service code.",
    "name": "Source service name.",
    "status": "Source-derived service status.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source service identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_reference.service"),
    comment="Thin v1 medical-service dimension from practitioner group assignments.",
    refresh_policy="incremental",
    column_comments=SERVICE_COLUMN_COMMENTS,
)
def service():
    s = read_source(SRC_PRACTITIONER)
    return (
        s.where(s.MEDSERVICE_GROUP_ID.isNotNull())
         .groupBy("MEDSERVICE_GROUP_ID")
         .agg(
             F.max(F.coalesce("MEDSERVICE_GROUP_NAME", "MEDSERVICE")).alias("name"),
             F.max("MEDSERVICE_ACTIVE_ASSIGNMENT_COUNT").alias("active_assignment_count"),
             F.max(F.col("MEDSERVICE_SELECTED_INACTIVE_IND").cast("int")).alias("selected_inactive_ind"),
             F.max("ADC_UPDT").alias("loaded_at"),
         )
         .select(
             stable_id("service:mill:medical", F.col("MEDSERVICE_GROUP_ID")).alias("service_id"),
             F.col("MEDSERVICE_GROUP_ID").cast("string").alias("source_service_code"),
             F.col("name"),
             F.when((F.coalesce("active_assignment_count", F.lit(0)) > 0) &
                    (F.coalesce("selected_inactive_ind", F.lit(0)) == 0),
                    F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
             F.lit(SRC_PRACTITIONER).alias("source_table"),
             F.col("MEDSERVICE_GROUP_ID").cast("string").alias("source_row_id"),
             F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
             F.col("loaded_at"),
         )
    )

In [0]:
ORGANIZATION_COLUMN_COMMENTS = {
    "organization_id": "Deterministic organization primary key.",
    "source_organization_id": "Millennium organization identifier.",
    "name": "Source organization name.",
    "status": "Source-derived organization status.",
    "valid_from": "Source validity start.",
    "valid_to": "Source validity end.",
    "address_city": "Source organization address city.",
    "address_postcode_masked": "Privacy-aware source postcode.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source organization identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_reference.organization"),
    comment="Thin v1 provider organization dimension derived from bronze care sites.",
    refresh_policy="incremental",
    column_comments=ORGANIZATION_COLUMN_COMMENTS,
)
def organization():
    s = read_source(SRC_CARE_SITE)
    return (
        s.where(s.ORGANIZATION_ID.isNotNull())
         .groupBy("ORGANIZATION_ID")
         .agg(
             F.max("organization_name").alias("name"),
             F.max("organization_active_ind").alias("active_ind"),
             F.min("organization_beg_effective_dt_tm").alias("valid_from"),
             F.max("organization_end_effective_dt_tm").alias("valid_to"),
             F.max("masked_zipcode").alias("address_postcode_masked"),
             F.max("city").alias("address_city"),
             F.max("ADC_UPDT").alias("loaded_at"),
         )
         .select(
             stable_id("organization:mill", F.col("ORGANIZATION_ID")).alias("organization_id"),
             F.col("ORGANIZATION_ID").cast("string").alias("source_organization_id"),
             F.col("name"),
             F.when(F.col("active_ind") == 1, F.lit("active")).otherwise(F.lit("inactive")).alias("status"),
             F.col("valid_from"), F.col("valid_to"),
             F.col("address_city"), F.col("address_postcode_masked"),
             F.lit(SRC_CARE_SITE).alias("source_table"),
             F.col("ORGANIZATION_ID").cast("string").alias("source_row_id"),
             F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
             F.col("loaded_at"),
         )
    )

In [0]:
def _family_history_canonical_pregate():
    s = read_source(SRC_FAMILY_HISTORY_PRODUCT)
    event_id = stable_id("family_history:mill", s.FHX_ACTIVITY_ID)
    raw_source_code = F.coalesce(s.SOURCE_IDENTIFIER, s.FOUND_CUI).cast("string")
    source_display = s.CONDITION_DESC
    source_code = _code_or_display(raw_source_code, source_display)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:mrn", s.MRN)],
        SRC_FAMILY_HISTORY_PRODUCT,
        s.FHX_ACTIVITY_ID,
    )
    ended = F.coalesce(
        s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
        F.lit(False),
    )
    return s.select(
        event_id.alias("patient_event_id"),
        event_id.alias("fact_row_id"),
        skey.alias("subject_key"),
        ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .when(s.MRN.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(s.ORIGINATING_ENCNTR_ID.isNotNull(),
               stable_id("encounter:mill", s.ORIGINATING_ENCNTR_ID)).alias("encounter_id"),
        s.BEG_EFFECTIVE_DT_TM.alias("event_datetime"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("event_end_datetime"),
        s.SOURCE_VOCABULARY_DESC.alias("source_coding_system"),
        source_code.alias("source_code"),
        source_display.alias("source_display"),
        codeable_concept(
            coding_obj(s.SOURCE_VOCABULARY_DESC, source_code,
                       F.coalesce(s.CONDITION_DESC_CODED, source_display), True),
            coding_obj(F.lit("http://snomed.info/sct"), s.SNOMED_CODE, s.SNOMED_TERM, False,
                       "bronze.map_family_history", None),
            coding_obj(F.lit("http://hl7.org/fhir/sid/icd-10"), s.ICD10_CODE, s.ICD10_TERM, False,
                       "bronze.map_family_history", None),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_CONCEPT_ID, s.OMOP_CONCEPT_NAME, False,
                       "bronze.map_family_history", None),
        ).alias("condition_code"),
        s.RELATION_CD.cast("string").alias("relationship_code"),
        s.RELATION_DESC.alias("relationship_display"),
        s.RELATION_TYPE_CD.cast("string").alias("relationship_type_code"),
        s.RELATION_TYPE_DESC.alias("relationship_type_display"),
        s.ONSET_AGE.cast("decimal(18,4)").alias("onset_age"),
        s.ONSET_AGE_UNIT.alias("onset_age_unit"),
        s.SEVERITY_CD.cast("string").alias("severity_code"),
        s.SEVERITY.alias("severity_display"),
        s.LIFE_CYCLE_STATUS.alias("source_lifecycle_status"),
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.FHX_ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit(None).cast("string").alias("asserter_practitioner_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_FAMILY_HISTORY_PRODUCT).alias("_source_table"),
        s.FHX_ACTIVITY_ID.cast("string").alias("_source_row_id"),
        s.SNOMED_CODE.cast("string").alias("_snomed_code"),
        s.SNOMED_TERM.alias("_snomed_display"),
        s.ICD10_CODE.cast("string").alias("_icd10_code"),
        s.ICD10_TERM.alias("_icd10_display"),
        s.OMOP_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.OMOP_CONCEPT_NAME.alias("_omop_display"),
    )

def _family_history_canonical():
    return _family_history_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_FAMILY_HISTORY_PRODUCT = "4_prod.bronze.map_family_history"

@materialized_view(
    name=_n("journey_clinical._qc_family_history"),
    comment="Private JSON bridge and Gold cross-rule flags for family_history.",
    refresh_policy="incremental",
)
def _qc_family_history():
    return _cross_qc_primitive(
        _family_history_canonical(),
        "family_history",
        {"condition_code": "_qc_condition_code_json"},
    )

In [0]:
# ==== Event backbone — typed facts and the event index ====

FAMILY_HISTORY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "condition_code",
    "relationship_code", "relationship_display", "relationship_type_code",
    "relationship_type_display", "onset_age", "onset_age_unit", "severity_code",
    "severity_display", "source_lifecycle_status", "record_status",
    "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind",
    "asserter_practitioner_id", "load_batch_id", "source_update_timestamp", "loaded_at",
]

FAMILY_HISTORY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide event identifier.",
    "fact_row_id": "Storage-row identifier; equal to patient_event_id for this fact.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime": "Source family-history effective start.",
    "event_end_datetime": "Source effective end when the assertion is no longer current.",
    "source_coding_system": "Verbatim source code system.",
    "source_code": "Source condition code",
    "source_display": "Verbatim source condition display.",
    "condition_code": "Source and mapped condition codings as a one-level CodeableConcept VARIANT.",
    "relationship_code": "Source family relationship code.",
    "relationship_display": "Source family relationship display.",
    "relationship_type_code": "Source relationship record type code.",
    "relationship_type_display": "Source relationship record type display.",
    "onset_age": "Source recorded age at onset.",
    "onset_age_unit": "Unit for recorded onset age.",
    "severity_code": "Source severity code.",
    "severity_display": "Source severity display.",
    "source_lifecycle_status": "Verbatim source life-cycle status.",
    "record_status": "Normalized silver lifecycle status.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "asserter_practitioner_id": "Asserting practitioner reference when supplied.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Bronze source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.family_history"),
    comment="One family-history assertion with the Journey Model standard event block and CodeableConcept mappings.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=FAMILY_HISTORY_COLUMN_COMMENTS,
)
def family_history():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_family_history")),
        "family_history",
        {"condition_code": "_qc_condition_code_json"},
        FAMILY_HISTORY_PUBLIC_COLUMNS,
    )

In [0]:
def _condition_diagnosis_canonical_pregate():
    s = read_source(SRC_DIAGNOSIS)
    event_id = stable_id("condition:mill:diagnosis", s.DIAGNOSIS_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_DIAGNOSIS, s.DIAGNOSIS_ID
    )
    raw_source_code = F.coalesce(
        s.SOURCE_IDENTIFIER, s.CONCEPT_CKI_IDENTIFIER, s.NOMENCLATURE_ID.cast("string")
    )
    source_display = F.coalesce(s.SOURCE_STRING, s.DIAGNOSIS_DISPLAY, s.DIAGNOSIS_TEXT)
    source_code = _code_or_display(raw_source_code, source_display)
    ended = F.coalesce(
        s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"), F.lit(False)
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.DIAG_DT_TM, s.ASSERTED_DT_TM, s.BEG_EFFECTIVE_DT_TM).alias("event_datetime"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("event_end_datetime"),
        F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                   F.lit("urn:cerner:nomenclature")).alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                                  F.lit("urn:cerner:nomenclature")),
                       source_code, source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), s.SNOMED_CODE, s.SNOMED_TERM, False,
                       "bronze.map_diagnosis", None),
            coding_obj(F.lit("http://hl7.org/fhir/sid/icd-10"), s.ICD10_CODE, s.ICD10_TERM, False,
                       "bronze.map_diagnosis", None),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_CONCEPT_ID, s.OMOP_CONCEPT_NAME, False,
                       "bronze.map_diagnosis", None),
        ).alias("_condition_code_json"),
        s.DIAG_TYPE_CD.cast("string").alias("category_code"),
        s.diag_type_desc.alias("category_display"),
        s.ACTIVE_STATUS_CD.cast("string").alias("clinical_status_code"),
        F.lit(None).cast("string").alias("clinical_status_display"),
        s.CONFIRMATION_STATUS_CD.cast("string").alias("verification_status_code"),
        s.confirmation_status_desc.alias("verification_status_display"),
        F.coalesce(s.earliest_diagnosis_date, s.DIAG_DT_TM).alias("onset_datetime"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("abatement_datetime"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        s.SEVERITY_CD.cast("string").alias("severity_code"),
        s.SEVERITY_FTDESC.alias("severity_display"),
        s.LATERALITY_CD.cast("string").alias("laterality_code"),
        F.lit(None).cast("string").alias("laterality_display"),
        s.ASSERTED_DT_TM.alias("asserted_datetime"),
        F.when(s.DIAG_PRSNL_ID.isNotNull(), stable_id("practitioner:mill", s.DIAG_PRSNL_ID))
         .alias("asserter_practitioner_id"),
        F.lit(None).cast("string").alias("recorder_practitioner_id"),
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("diagnosis").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_DIAGNOSIS).alias("_source_table"),
        s.DIAGNOSIS_ID.cast("string").alias("_source_row_id"),
        s.SNOMED_CODE.cast("string").alias("_snomed_code"), s.SNOMED_TERM.alias("_snomed_display"),
        s.ICD10_CODE.cast("string").alias("_icd10_code"), s.ICD10_TERM.alias("_icd10_display"),
        s.OMOP_CONCEPT_ID.cast("string").alias("_omop_code"), s.OMOP_CONCEPT_NAME.alias("_omop_display"),
    )

def _condition_diagnosis_canonical():
    return _condition_diagnosis_canonical_pregate().where(_usable_code(F.col("source_code")))

def _condition_problem_canonical_pregate():
    s = read_source(SRC_PROBLEM)
    event_id = stable_id("condition:mill:problem", s.PROBLEM_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_PROBLEM, s.PROBLEM_ID
    )
    raw_source_code = F.coalesce(
        s.SOURCE_IDENTIFIER, s.CONCEPT_CKI_IDENTIFIER, s.NOMENCLATURE_ID.cast("string")
    )
    source_display = F.coalesce(s.SOURCE_STRING, s.PROBLEM_DISPLAY, s.PROBLEM_FTDESC)
    source_code = _code_or_display(raw_source_code, source_display)
    ended = F.coalesce(
        s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"), F.lit(False)
    )
    encounter_key = F.coalesce(s.ENCNTR_ID, s.ORIGINATING_ENCNTR_ID)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(encounter_key.isNotNull(), stable_id("encounter:mill", encounter_key)).alias("encounter_id"),
        F.coalesce(s.ASSERTED_DT_TM, s.ONSET_DT_TM, s.BEG_EFFECTIVE_DT_TM,
                   s.earliest_problem_date).alias("event_datetime"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("event_end_datetime"),
        F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                   F.lit("urn:cerner:nomenclature")).alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                                  F.lit("urn:cerner:nomenclature")),
                       source_code, source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), s.SNOMED_CODE, s.SNOMED_TERM, False,
                       "bronze.map_problem", None),
            coding_obj(F.lit("http://hl7.org/fhir/sid/icd-10"), s.ICD10_CODE, s.ICD10_TERM, False,
                       "bronze.map_problem", None),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_CONCEPT_ID, s.OMOP_CONCEPT_NAME, False,
                       "bronze.map_problem", None),
        ).alias("_condition_code_json"),
        s.CLASSIFICATION_CD.cast("string").alias("category_code"),
        s.classification_desc.alias("category_display"),
        s.LIFE_CYCLE_STATUS_CD.cast("string").alias("clinical_status_code"),
        s.life_cycle_status_desc.alias("clinical_status_display"),
        s.CONFIRMATION_STATUS_CD.cast("string").alias("verification_status_code"),
        s.confirmation_status_desc.alias("verification_status_display"),
        s.ONSET_DT_TM.alias("onset_datetime"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("abatement_datetime"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        s.SEVERITY_CD.cast("string").alias("severity_code"), s.severity_desc.alias("severity_display"),
        s.LATERALITY_CD.cast("string").alias("laterality_code"), s.laterality_desc.alias("laterality_display"),
        s.ASSERTED_DT_TM.alias("asserted_datetime"),
        F.lit(None).cast("string").alias("asserter_practitioner_id"),
        F.when(s.ACTIVE_STATUS_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.ACTIVE_STATUS_PRSNL_ID)).alias("recorder_practitioner_id"),
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("problem").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_PROBLEM).alias("_source_table"),
        s.PROBLEM_ID.cast("string").alias("_source_row_id"),
        s.SNOMED_CODE.cast("string").alias("_snomed_code"), s.SNOMED_TERM.alias("_snomed_display"),
        s.ICD10_CODE.cast("string").alias("_icd10_code"), s.ICD10_TERM.alias("_icd10_display"),
        s.OMOP_CONCEPT_ID.cast("string").alias("_omop_code"), s.OMOP_CONCEPT_NAME.alias("_omop_display"),
    )

def _condition_problem_canonical():
    return _condition_problem_canonical_pregate().where(_usable_code(F.col("source_code")))

def _waiting_list_index_representative():
    # Incrementalizable collapse: MAX over an orderable struct (is_current, version id,
    # version time, fact_row_id tiebreak), payload as JSON — the _mill_blob_document pattern.
    base = _waiting_list_entry_canonical()
    payload_schema = base.schema
    ranked = base.select(
        F.col("patient_event_id").alias("_entry"),
        F.struct(
            F.coalesce(F.col("is_current").cast("int"), F.lit(0)).alias("o_current"),
            F.coalesce(F.col("source_version_id"), F.lit(-2)).alias("o_version"),
            F.coalesce(
                F.col("version_datetime"), F.lit("1900-01-01").cast("timestamp")
            ).alias("o_time"),
            F.col("fact_row_id").alias("o_tiebreak"),
            F.to_json(F.struct(*[F.col(c) for c in base.columns])).alias("payload"),
        ).alias("ranked"),
    )
    winner = ranked.groupBy("_entry").agg(F.max("ranked").alias("w"))
    return winner.select(
        F.from_json(F.col("w.payload"), payload_schema).alias("r")
    ).select("r.*")

def _endobase_procedure_canonical_pregate():
    s = read_source(SRC_ENDOBASE_EXAM)
    event_id = stable_id("procedure:endobase_exam", s.ENDOBASE_EXAM_ID)
    source_display = s.EXAM_TYPE_DESC
    source_code = _code_or_display(s.ENDOBASE_EXAM_TYPE_ID.cast("string"), source_display)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_ENDOBASE_EXAM, s.ENDOBASE_EXAM_ID
    )
    performed = F.coalesce(s.PERFORMED_TS_CLEAN, s.EXAM_TS_CLEAN,
                           s.TRUE_START_TS_CLEAN, s.START_TS_CLEAN)
    performed_end = F.coalesce(s.TRUE_END_TS_CLEAN, s.END_TS_CLEAN)
    inactive = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    status = F.when(performed.isNotNull(), F.lit("completed")).otherwise(F.lit("preparation"))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(s.MILL_ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.MILL_ENCNTR_ID))
        .alias("encounter_id"),
        performed.alias("event_datetime"), performed_end.alias("event_end_datetime"),
        F.lit("urn:barts:endobase:exam-type").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:barts:endobase:exam-type"), source_code,
                       source_display, True)
        ).alias("_procedure_code_json"),
        status.alias("status_code"), status.alias("status_display"),
        performed.alias("performed_start"), performed_end.alias("performed_end"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("laterality_code"),
        F.lit(None).cast("string").alias("laterality_display"),
        F.lit(None).cast("string").alias("performer_practitioner_id"),
        s.DEPARTMENT_ID.cast("string").alias("procedure_location_code"),
        F.lit(None).cast("string").alias("procedure_location_display"),
        F.lit(None).cast("string").alias("procedure_note"),
        F.lit(None).cast("string").alias("implant_description"),
        F.lit(None).cast("string").alias("device_code"),
        F.lit(None).cast("string").alias("device_display"),
        F.lit(None).cast("string").alias("manufacturer"),
        F.lit(None).cast("string").alias("serial_number"),
        F.lit(None).cast("string").alias("batch_number"),
        F.lit(None).cast("string").alias("udi_di"),
        F.lit(None).cast("string").alias("udi_standard"),
        F.lit(None).cast("decimal(38,6)").alias("quantity"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        performed.alias("record_status_effective_from"),
        F.when(inactive, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("endobase_exam").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("endobase").alias("_source_system"), F.lit(SRC_ENDOBASE_EXAM).alias("_source_table"),
        s.ENDOBASE_EXAM_ID.cast("string").alias("_source_row_id"),
        F.lit(None).cast("string").alias("_snomed_code"),
        F.lit(None).cast("string").alias("_snomed_display"),
        F.lit(None).cast("string").alias("_opcs4_code"),
        F.lit(None).cast("string").alias("_opcs4_display"),
        F.lit(None).cast("string").alias("_omop_code"),
        F.lit(None).cast("string").alias("_omop_display"),
        F.lit(None).cast("string").alias("_device_snomed_code"),
        F.lit(None).cast("string").alias("_device_snomed_display"),
    )

def _endobase_procedure_canonical():
    return _endobase_procedure_canonical_pregate().where(_usable_code(F.col("source_code")))

def _cc_procedure_canonical_pregate():
    s = read_source(SRC_CC_PROCEDURE)
    event_id = stable_id("procedure:nccmds", s.ROW_HASH)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_CC_PROCEDURE, s.ROW_HASH
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    note = F.concat_ws(
        " | ",
        F.concat(F.lit("period_link_status="), F.coalesce(s.PERIOD_LINK_STATUS, F.lit("UNKNOWN"))),
        F.concat(F.lit("period_business_key="), s.PERIOD_BUSINESS_KEY.cast("string")),
        F.concat(F.lit("cds_apc_id="), s.CDS_APC_ID),
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.OPCS_Proc_Dt_CLEAN.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("http://fhir.hl7.org.uk/CodeSystem/OPCS-4").alias("source_coding_system"),
        s.OPCS_Proc_Code.cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        codeable_concept_json(
            coding_obj(
                F.lit("http://fhir.hl7.org.uk/CodeSystem/OPCS-4"),
                s.OPCS_Proc_Code.cast("string"), F.lit(None).cast("string"), True,
            )
        ).alias("_procedure_code_json"),
        F.lit("completed").alias("status_code"),
        s.CC_TYPE_DESC.alias("status_display"),
        s.OPCS_Proc_Dt_CLEAN.alias("performed_start"),
        F.lit(None).cast("timestamp").alias("performed_end"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("laterality_code"),
        F.lit(None).cast("string").alias("laterality_display"),
        F.lit(None).cast("string").alias("performer_practitioner_id"),
        F.lit(None).cast("string").alias("procedure_location_code"),
        F.lit(None).cast("string").alias("procedure_location_display"),
        note.alias("procedure_note"),
        F.lit(None).cast("string").alias("implant_description"),
        F.lit(None).cast("string").alias("device_code"),
        F.lit(None).cast("string").alias("device_display"),
        F.lit(None).cast("string").alias("manufacturer"),
        F.lit(None).cast("string").alias("serial_number"),
        F.lit(None).cast("string").alias("batch_number"),
        F.lit(None).cast("string").alias("udi_di"),
        F.lit(None).cast("string").alias("udi_standard"),
        F.lit(None).cast("decimal(38,6)").alias("quantity"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        s.OPCS_Proc_Dt_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("critical_care_procedure").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("cds-nccmds").alias("_source_system"),
        F.lit(SRC_CC_PROCEDURE).alias("_source_table"),
        s.ROW_HASH.cast("string").alias("_source_row_id"),
        F.lit(None).cast("string").alias("_snomed_code"),
        F.lit(None).cast("string").alias("_snomed_display"),
        s.OPCS_Proc_Code.cast("string").alias("_opcs4_code"),
        F.lit(None).cast("string").alias("_opcs4_display"),
        F.lit(None).cast("string").alias("_omop_code"),
        F.lit(None).cast("string").alias("_omop_display"),
        F.lit(None).cast("string").alias("_device_snomed_code"),
        F.lit(None).cast("string").alias("_device_snomed_display"),
        F.lit(None).cast("string").alias("_theatre_case_id"),
    )

def _cc_procedure_canonical():
    return _cc_procedure_canonical_pregate().where(_usable_code(F.col("source_code")))

def _maternity_diagnosis_canonical_pregate():
    d = _maternity_join(SRC_MATERNITY_DIAGNOSIS)
    person = F.coalesce(F.col("_spine_person_id"), F.col("s.PERSON_ID"))
    event_id = stable_id(
        "condition:msds", F.col("s.PREGNANCYID"), F.col("s.DIAGSCHEME"),
        F.col("s.DIAG"), F.col("s.DIAGDATE"),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", person)], SRC_MATERNITY_DIAGNOSIS,
        F.concat_ws("|", F.col("s.PREGNANCYID"), F.col("s.DIAGSCHEME"),
                    F.col("s.DIAG"), F.col("s.DIAGDATE")),
    )
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"), person.cast("string").alias("person_id"),
        _maternity_identity_status(F.col("s.PERSON_ID"), F.col("_spine_person_id"))
         .alias("identity_status"), F.lit(None).cast("string").alias("encounter_id"),
        F.col("s.DIAGDATE_CLEAN").alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("http://snomed.info/sct").alias("source_coding_system"),
        F.col("s.DIAG").alias("source_code"), F.col("s.DIAG_SNOMED_DESC").alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("http://snomed.info/sct"), F.col("s.DIAG"),
                       F.col("s.DIAG_SNOMED_DESC"), True)
        ).alias("_condition_code_json"),
        F.col("s.DIAGSCHEME").alias("category_code"), F.lit("maternity diagnosis").alias("category_display"),
        F.lit("active").alias("clinical_status_code"), F.lit("active").alias("clinical_status_display"),
        F.lit("confirmed").alias("verification_status_code"),
        F.lit("confirmed").alias("verification_status_display"),
        F.col("s.DIAGDATE_CLEAN").alias("onset_datetime"),
        F.lit(None).cast("timestamp").alias("abatement_datetime"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("severity_code"),
        F.lit(None).cast("string").alias("severity_display"),
        F.lit(None).cast("string").alias("laterality_code"),
        F.lit(None).cast("string").alias("laterality_display"),
        F.col("s.DIAGDATE_CLEAN").alias("asserted_datetime"),
        F.lit(None).cast("string").alias("asserter_practitioner_id"),
        F.lit(None).cast("string").alias("recorder_practitioner_id"),
        F.lit("active").alias("record_status"),
        F.col("s.DIAGDATE_CLEAN").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("maternity_diagnosis").alias("source_feed"),
        F.date_format(F.col("s.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("s.RECORD_UPDATED_DT").alias("source_update_timestamp"),
        F.col("s.ADC_UPDT").alias("loaded_at"), F.lit("msds").alias("_source_system"),
        F.lit(SRC_MATERNITY_DIAGNOSIS).alias("_source_table"),
        F.col("s.ROW_HASH").cast("string").alias("_source_row_id"),
        F.col("s.DIAG").alias("_snomed_code"),
        F.col("s.DIAG_SNOMED_DESC").alias("_snomed_display"),
        F.lit(None).cast("string").alias("_icd10_code"),
        F.lit(None).cast("string").alias("_icd10_display"),
        F.lit(None).cast("string").alias("_omop_code"),
        F.lit(None).cast("string").alias("_omop_display"),
    )

def _maternity_diagnosis_canonical():
    return _maternity_diagnosis_canonical_pregate().where(_usable_code(F.col("source_code")))

def _mill_radiology_exam_canonical_pregate():
    s = read_source(SRC_RADIOLOGY_EVENT)
    event_id = stable_id("imaging_exam:mill", s.EVENT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_RADIOLOGY_EVENT, s.EVENT_ID
    )
    source_code = _code_or_display(s.EXAM_TYPE_CODE, s.EVENT_TITLE_TEXT)
    status = (
        F.when(F.coalesce(s.IN_ERROR_IND, F.lit(False)), F.lit("entered-in-error"))
        .otherwise(F.coalesce(s.RESULT_STATUS_DESC, F.lit("available")))
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        s.PERFORMED_DT_TM_CLEAN.alias("event_datetime"),
        s.EVENT_END_DT_TM_CLEAN.alias("event_end_datetime"),
        F.lit("urn:cerner:radiology-exam").alias("source_coding_system"),
        source_code.alias("source_code"), s.EVENT_TITLE_TEXT.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:radiology-exam"), source_code,
                       s.EVENT_TITLE_TEXT, True)
        ).alias("exam_code"),
        status.alias("status_code"), s.REFERENCE_NBR.alias("accession_identifier"),
        F.lit(None).cast("string").alias("study_instance_uid"),
        s.NHSI_MODALITY_CATEGORY.alias("modality_code"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("report_patient_event_id"),
        F.lit(None).cast("string").alias("requester_practitioner_id"),
        F.lit(None).cast("string").alias("performer_practitioner_id"),
        F.when(F.coalesce(s.IN_ERROR_IND, F.lit(False)), F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.VALID_FROM_DT_TM_CLEAN.alias("record_status_effective_from"),
        F.when(F.coalesce(s.IN_ERROR_IND, F.lit(False)), s.UPDT_DT_TM)
         .otherwise(s.VALID_UNTIL_DT_TM_CLEAN).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("radiology_event").alias("source_feed"),
        F.date_format(s.PIPELINE_UPDT_DT_TM, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDT_DT_TM.alias("source_update_timestamp"),
        s.PIPELINE_UPDT_DT_TM.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_RADIOLOGY_EVENT).alias("_source_table"),
        s.EVENT_ID.cast("string").alias("_source_row_id"),
    )

def _mill_radiology_exam_canonical():
    return _mill_radiology_exam_canonical_pregate().where(
        _usable_code(F.col("source_code"))
    )

In [0]:
SRC_CC_PROCEDURE = "4_prod.bronze.map_critical_care_procedure"
SRC_MATERNITY_DIAGNOSIS = "4_prod.bronze.map_maternity_diagnosis"
SRC_RADIOLOGY_EVENT = "4_prod.bronze.map_radiology_event"

def _event_index_projection(
    df,
    event_type,
    fact_table,
    fact_category,
    source_system=None,
    source_object=None,
    source_row_key=None,
):
    sys_col = _source_system_display(F.col("_source_system")) if source_system is None else source_system
    obj_col = _source_object_display(F.col("_source_table")) if source_object is None else source_object
    key_col = F.col("_source_row_id").cast("string") if source_row_key is None else source_row_key
    return df.select(
        "patient_event_id", "subject_key", "subject_id_system", "person_id", "identity_status",
        "encounter_id", "event_datetime", "event_end_datetime",
        F.lit(event_type).alias("event_type"),
        F.lit(fact_category).alias("fact_category"),
        sys_col.alias("source_system"), obj_col.alias("source_object"),
        key_col.alias("source_row_key"),
        "source_coding_system", "source_code", "source_display",
        "record_status", "confidentiality_code", "vip_ind", "withheld_identity_ind",
        F.lit(fact_table).alias("fact_table"), "fact_row_id",
        "load_batch_id", "loaded_at",
    )

@materialized_view(
    name=_n("journey_events._event_index"),
    private=True,
    comment="INTERNAL event-grain stage (one row per admitted event) behind patient_event. "
            "Internal consumers needing event grain read THIS, never the public N-row table.",
    refresh_policy="incremental",
)
def _event_index():
    lanes = [
        _event_index_projection(_family_history_canonical(), "family_history",
                                "journey_clinical.family_history", "clinical"),
        _event_index_projection(_allergy_canonical(), "allergy_intolerance",
                                "journey_clinical.allergy_intolerance", "clinical"),
        _event_index_projection(_transfusion_canonical(), "transfusion",
                                "journey_clinical.transfusion", "clinical"),
        _event_index_projection(_cancer_treatment_canonical(), "cancer_treatment",
                                "journey_clinical.cancer_treatment", "clinical"),
        _event_index_projection(_condition_stage_canonical(), "condition_stage",
                                "journey_clinical.condition_stage", "clinical"),
        _event_index_projection(_endoscopy_finding_canonical(), "endoscopy_finding",
                                "journey_clinical.endoscopy_finding", "clinical"),
        _event_index_projection(_registry_entry_canonical(), "registry_entry",
                                "journey_clinical.registry_entry", "clinical"),
        _event_index_projection(_device_canonical(), "device",
                                "journey_clinical.device", "clinical"),
        _event_index_projection(_condition_diagnosis_canonical(), "condition",
                                "journey_clinical.condition", "clinical"),
        _event_index_projection(_condition_problem_canonical(), "condition",
                                "journey_clinical.condition", "clinical"),
        _event_index_projection(_procedure_canonical(), "procedure",
                                "journey_clinical.procedure", "clinical"),
        _event_index_projection(_implant_procedure_canonical(), "procedure",
                                "journey_clinical.procedure", "clinical"),
        _event_index_projection(_theatre_procedure_canonical(), "procedure",
                                "journey_clinical.procedure", "clinical"),
        _event_index_projection(_endobase_procedure_canonical(), "procedure",
                                "journey_clinical.procedure", "clinical"),
        _event_index_projection(_pathology_requested_test_canonical(), "pathology_order",
                                "journey_clinical.pathology_order", "clinical"),
        _event_index_projection(_pathology_specimen_canonical(), "specimen",
                                "journey_clinical.specimen", "clinical"),
        _event_index_projection(_pathology_report_series_canonical(), "pathology_report",
                                "journey_clinical.pathology_report", "clinical"),
        _event_index_projection(_pathology_result_canonical(), "pathology_result",
                                "journey_clinical.pathology_result", "clinical"),
        _event_index_projection(_genomic_test_canonical(), "genomic_test",
                                "journey_clinical.genomic_test", "clinical"),
        _event_index_projection(_genomic_result_canonical(), "genomic_result",
                                "journey_clinical.genomic_result", "clinical"),
        _event_index_projection(_indication_canonical(), "indication",
                                "journey_clinical.indication", "clinical"),
        _event_index_projection(_micro_isolate_canonical(), "microbiology_isolate",
                                "journey_clinical.microbiology_isolate", "clinical"),
        _event_index_projection(_susceptibility_canonical(), "susceptibility_result",
                                "journey_clinical.susceptibility_result", "clinical"),
        _event_index_projection(_form_canonical(), "form",
                                "journey_clinical.form", "clinical"),
        _event_index_projection(_vital_sign_canonical(), "vital_sign",
                                "journey_clinical.vital_sign", "clinical"),
        _event_index_projection(_clinical_score_canonical(), "clinical_score",
                                "journey_clinical.clinical_score", "clinical"),
        _event_index_projection(_medication_admin_canonical(), "medication_admin",
                                "journey_clinical.medication_admin", "clinical"),
        _event_index_projection(_medication_order_canonical(), "medication_order",
                                "journey_clinical.medication_order", "clinical"),
        _event_index_projection(_medication_dispense_canonical(), "medication_dispense",
                                "journey_clinical.medication_dispense", "clinical"),
        _event_index_projection(_coded_finding_typed(), "clinical_finding",
                                "journey_clinical.clinical_finding", "clinical"),
        _event_index_projection(_nomen_finding_typed(), "clinical_finding",
                                "journey_clinical.clinical_finding", "clinical"),
        _event_index_projection(_date_finding_typed(), "clinical_finding",
                                "journey_clinical.clinical_finding", "clinical"),
        _event_index_projection(_text_finding_typed(), "clinical_finding",
                                "journey_clinical.clinical_finding", "clinical"),
        _event_index_projection(_imaging_exam_canonical(), "imaging_exam",
                                "journey_clinical.imaging_exam", "clinical"),
        _event_index_projection(_artifact_asset_canonical(), "artifact",
                                "journey_artifact.asset", "clinical"),
        _event_index_projection(_document_canonical(), "document",
                                "journey_text.document", "clinical"),
        _event_index_projection(_appointment_canonical(), "appointment",
                                "journey_clinical.appointment", "administrative"),
        _event_index_projection(_referral_canonical(), "referral",
                                "journey_clinical.referral", "administrative"),
        _event_index_projection(_rtt_pathway_canonical(), "rtt_pathway",
                                "journey_clinical.rtt_pathway", "administrative"),
        _event_index_projection(_rtt_activity_canonical(), "rtt_activity",
                                "journey_clinical.rtt_activity", "administrative"),
        _event_index_projection(_waiting_list_index_representative(), "waiting_list_entry",
                                "journey_clinical.waiting_list_entry", "administrative"),
        _event_index_projection(_community_care_activity_canonical(), "community_care_activity",
                                "journey_clinical.community_care_activity", "clinical"),
        _event_index_projection(_community_care_contact_canonical(), "community_care_contact",
                                "journey_clinical.community_care_contact", "administrative"),
        _event_index_projection(_hrg_grouping_canonical(), "hrg_grouping",
                                "journey_clinical.hrg_grouping", "administrative"),
        _event_index_projection(_costed_activity_canonical(), "costed_activity",
                                "journey_clinical.costed_activity", "administrative"),
        _event_index_projection(_drug_expenditure_canonical(), "drug_expenditure",
                                "journey_clinical.drug_expenditure", "administrative"),
        _event_index_projection(_medication_supply_canonical(), "medication_supply",
                                "journey_clinical.medication_supply", "clinical"),
        _event_index_projection(_elective_access_entry_canonical(), "elective_access_entry",
                                "journey_clinical.elective_access_entry", "administrative"),
        _event_index_projection(_pathway_tracking_canonical(), "pathway_tracking",
                                "journey_clinical.pathway_tracking", "administrative"),
        _event_index_projection(_critical_care_period_canonical(), "critical_care_period",
                                "journey_clinical.critical_care_period", "administrative"),
        _event_index_projection(_critical_care_activity_canonical(), "critical_care_activity",
                                "journey_clinical.critical_care_activity", "clinical"),
        _event_index_projection(_cc_procedure_canonical(), "procedure",
                                "journey_clinical.procedure", "clinical"),
        _event_index_projection(_critical_care_admission_canonical(), "critical_care_admission",
                                "journey_clinical.critical_care_admission", "administrative"),
        _event_index_projection(_cc_daily_score_canonical(), "critical_care_daily_score",
                                "journey_clinical.critical_care_daily_score", "clinical"),
        _event_index_projection(_neonatal_episode_canonical(), "neonatal_episode",
                                "journey_clinical.neonatal_episode", "administrative"),
        _event_index_projection(_neonatal_care_day_canonical(), "neonatal_care_day",
                                "journey_clinical.neonatal_care_day", "clinical"),
        _event_index_projection(_neonatal_examination_canonical(), "neonatal_examination",
                                "journey_clinical.neonatal_examination", "clinical"),
        _event_index_projection(_baby_delivery_canonical(), "baby_delivery",
                                "journey_clinical.baby_delivery", "clinical"),
        _event_index_projection(_labour_delivery_canonical(), "labour_delivery",
                                "journey_clinical.labour_delivery", "clinical"),
        _event_index_projection(_maternity_care_contact_canonical(), "maternity_care_contact",
                                "journey_clinical.maternity_care_contact", "administrative"),
        _event_index_projection(_maternity_diagnosis_canonical(), "condition",
                                "journey_clinical.condition", "clinical"),
        _event_index_projection(_research_enrollment_canonical(), "research_enrollment",
                                "journey_clinical.research_enrollment", "administrative"),
        _event_index_projection(_mill_radiology_exam_canonical(), "imaging_exam",
                                "journey_clinical.imaging_exam", "clinical"),
    ]
    out = lanes[0]
    for lane in lanes[1:]:
        out = out.unionByName(lane)
    return out

In [0]:
def _iweb_target_system(vocabulary):
    return (
        F.when(vocabulary == "SNOMED", F.lit("http://snomed.info/sct"))
        .when(vocabulary == "OPCS4", F.lit("http://fhir.hl7.org.uk/CodeSystem/OPCS-4"))
        .when(vocabulary == "ICD10", F.lit("http://hl7.org/fhir/sid/icd-10"))
        .when(vocabulary == "LOINC", F.lit("http://loinc.org"))
        .otherwise(F.concat(F.lit("urn:iweb:"),
                            F.lower(F.coalesce(vocabulary, F.lit("unmapped")))))
    )

In [0]:
SRC_CODED_EVENT_FINDING_MAPPING = "4_prod.bronze.map_coded_events_omop_bridge"
SRC_IWEB_MULTISELECT = "4_prod.bronze.iweb_multiselect_value"

def _map_projection(df, code_col, display_col, system, target_domain, map_source):
    filtered = df.where(F.col(code_col).isNotNull())
    return filtered.select(
        stable_id(
            "patient_event_map",
            F.col("patient_event_id"), F.lit(system), F.col(code_col),
            F.lit(target_domain), F.lit(map_source), F.lit(None),
        ).alias("patient_event_map_id"),
        "patient_event_id",
        F.lit(system).alias("mapped_coding_system"),
        F.col(code_col).cast("string").alias("mapped_code"),
        F.col(display_col).alias("mapped_display"),
        F.lit(target_domain).alias("target_domain"),
        F.lit(map_source).alias("map_source"),
        F.lit(None).cast("string").alias("map_version"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        "loaded_at",
    )

def _transfusion_blood_group_event_map():
    """Publish unit mapping plus a distinct patient mapping at event-map grain."""
    transfusion = _transfusion_canonical()
    unit = _map_projection(
        transfusion, "unit_group_concept_id", "_unit_group_concept_name",
        "http://snomed.info/sct", "observation",
        "lookup.bloodtrack_blood_group_map:applied",
    )
    distinct_patient = transfusion.where(
        F.col("patient_group_concept_id").isNotNull()
        & ~F.col("patient_group_concept_id").eqNullSafe(F.col("unit_group_concept_id"))
    )
    patient = _map_projection(
        distinct_patient, "patient_group_concept_id", "_patient_group_concept_name",
        "http://snomed.info/sct", "observation",
        "lookup.bloodtrack_blood_group_map:applied",
    )
    return unit.unionByName(patient)

@materialized_view(
    name=_n("journey_events._event_map"),
    private=True,
    comment="INTERNAL mapping-lane union (one row per event x mapping) behind patient_event.",
    refresh_policy="incremental",
)
def _event_map():
    family = _family_history_canonical()
    return (
        _map_projection(family, "_snomed_code", "_snomed_display",
                        "http://snomed.info/sct", "family_history", "bronze.map_family_history")
        .unionByName(
            _map_projection(family, "_icd10_code", "_icd10_display",
                            "http://hl7.org/fhir/sid/icd-10", "family_history", "bronze.map_family_history")
        )
        .unionByName(
            _map_projection(family, "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "family_history", "bronze.map_family_history")
        )
        .unionByName(
            _map_projection(_medication_dispense_canonical(), "_dmd_code", "_dmd_display",
                            "http://snomed.info/sct", "drug", "bronze.map_pharmacy_issue")
        )
        .unionByName(
            _map_projection(_condition_diagnosis_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "condition", "bronze.map_diagnosis")
        )
        .unionByName(
            _map_projection(_condition_diagnosis_canonical(), "_icd10_code", "_icd10_display",
                            "http://hl7.org/fhir/sid/icd-10", "condition", "bronze.map_diagnosis")
        )
        .unionByName(
            _map_projection(_condition_diagnosis_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "condition", "bronze.map_diagnosis")
        )
        .unionByName(
            _map_projection(_condition_problem_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "condition", "bronze.map_problem")
        )
        .unionByName(
            _map_projection(_condition_problem_canonical(), "_icd10_code", "_icd10_display",
                            "http://hl7.org/fhir/sid/icd-10", "condition", "bronze.map_problem")
        )
        .unionByName(
            _map_projection(_condition_problem_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "condition", "bronze.map_problem")
        )
        .unionByName(
            _map_projection(_procedure_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "procedure", "bronze.map_procedure")
        )
        .unionByName(
            _map_projection(_procedure_canonical(), "_opcs4_code", "_opcs4_display",
                            "http://fhir.hl7.org.uk/CodeSystem/OPCS-4", "procedure", "bronze.map_procedure")
        )
        .unionByName(
            _map_projection(_cc_procedure_canonical(), "_opcs4_code", "_opcs4_display",
                            "http://fhir.hl7.org.uk/CodeSystem/OPCS-4", "procedure",
                            "bronze.map_critical_care_procedure")
        )
        .unionByName(
            _map_projection(_maternity_diagnosis_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "condition",
                            "bronze.map_maternity_diagnosis")
        )
        .unionByName(
            _map_projection(_procedure_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "procedure", "bronze.map_procedure")
        )
        .unionByName(
            _map_projection(_implant_procedure_canonical(), "_device_snomed_code", "_device_snomed_display",
                            "http://snomed.info/sct", "device", "bronze.map_implant_details")
        )
        .unionByName(
            _map_projection(_pathology_requested_test_canonical(), "_test_snomed_code", "_test_snomed_display",
                            "http://snomed.info/sct", "measurement", "bronze.map_pathology_requested_test")
        )
        .unionByName(
            _map_projection(_pathology_requested_test_canonical(), "_test_omop_code", "_test_omop_display",
                            "urn:omop:concept_id", "measurement", "bronze.map_pathology_requested_test")
        )
        .unionByName(
            _map_projection(_genomic_test_canonical(), "_gt_snomed_code", "_gt_snomed_display",
                            "http://snomed.info/sct", "measurement", "bronze.map_pathology_genetic_test")
        )
        .unionByName(
            _map_projection(_genomic_test_canonical(), "_gt_loinc_code", "_gt_loinc_display",
                            "http://loinc.org", "measurement", "bronze.map_pathology_genetic_test")
        )
        .unionByName(
            _map_projection(_genomic_test_canonical(), "_gt_omop_code", "_gt_omop_display",
                            "urn:omop:concept_id", "measurement", "bronze.map_pathology_genetic_test")
        )
        .unionByName(
            _map_projection(_genomic_result_canonical(), "_gr_snomed_code", "_gr_snomed_display",
                            "http://snomed.info/sct", "measurement", "bronze.map_pathology_genetic_result")
        )
        .unionByName(
            _map_projection(_genomic_result_canonical(), "_gr_clinvar_code", "_gr_clinvar_display",
                            "urn:clinvar", "measurement", "bronze.map_pathology_genetic_result:clinvar")
        )
        .unionByName(
            _map_projection(_genomic_result_canonical(), "_gr_omop_code", "_gr_omop_display",
                            "urn:omop:concept_id", "measurement", "bronze.map_pathology_genetic_result:omop-genomic")
        )
        .unionByName(
            _map_projection(_indication_canonical(), "_ind_snomed_code", "_ind_snomed_display",
                            "http://snomed.info/sct", "condition", "bronze.map_pathology_indication")
        )
        .unionByName(
            _map_projection(_indication_canonical(), "_ind_omop_code", "_ind_omop_display",
                            "urn:omop:concept_id", "condition", "bronze.map_pathology_indication")
        )
        .unionByName(
            _map_projection(_micro_isolate_canonical(), "_iso_snomed_code", "_iso_snomed_display",
                            "http://snomed.info/sct", "measurement_value", "bronze.map_pathology_microbiology_isolate")
        )
        .unionByName(
            _map_projection(_micro_isolate_canonical(), "_iso_omop_code", "_iso_omop_display",
                            "urn:omop:concept_id", "measurement_value", "bronze.map_pathology_microbiology_isolate")
        )
        .unionByName(
            _map_projection(_susceptibility_canonical(), "_sus_code_code", "_sus_code_display",
                            "urn:barts:pathology:antimicrobial", "drug", "bronze.map_pathology_antimicrobial_susceptibility")
        )
        .unionByName(
            _map_projection(_susceptibility_canonical(), "_sus_omop_code", "_sus_omop_display",
                            "urn:omop:concept_id", "drug", "bronze.map_pathology_antimicrobial_susceptibility")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_test_snomed_code", "_test_snomed_display",
                            "http://snomed.info/sct", "measurement", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_test_loinc_code", "_test_loinc_display",
                            "http://loinc.org", "measurement", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_test_omop_code", "_test_omop_display",
                            "urn:omop:concept_id", "measurement", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_result_snomed_code", "_result_snomed_display",
                            "http://snomed.info/sct", "measurement_value", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_result_loinc_code", "_result_loinc_display",
                            "http://loinc.org", "measurement_value", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_pathology_result_canonical(), "_result_omop_code", "_result_omop_display",
                            "urn:omop:concept_id", "measurement_value", "bronze.map_pathology")
        )
        .unionByName(
            _map_projection(_vital_sign_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "measurement", "bronze.map_numeric_events_or_form")
        )
        .unionByName(
            _map_projection(_clinical_score_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "observation", "bronze.map_numeric_events_or_form")
        )
        .unionByName(
            _map_projection(_medication_admin_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "drug", "bronze.map_med_admin")
        )
        .unionByName(
            _map_projection(_medication_admin_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "drug", "bronze.map_med_admin")
        )
        .unionByName(
            _map_projection(_medication_admin_canonical(), "_rxnorm_code", "_rxnorm_display",
                            "http://www.nlm.nih.gov/research/umls/rxnorm", "drug",
                            "bronze.map_med_admin")
        )
        .unionByName(_coded_bridge_event_map())
        .unionByName(
            _dynamic_map_projection(
                _coded_finding_typed(), "_omop_code", "_omop_display",
                "urn:omop:concept_id", "_omop_domain", "bronze.map_coded_events:manual",
            )
        )
        .unionByName(
            _dynamic_map_projection(
                _nomen_finding_typed(), "_omop_code", "_omop_display",
                "urn:omop:concept_id", "_omop_domain", "bronze.map_nomen_events",
            )
        )
        .unionByName(
            _map_projection(
                _nomen_finding_typed(), "_snomed_code", "_snomed_display",
                "http://snomed.info/sct", "observation", "bronze.map_nomen_events",
            )
        )
        .unionByName(
            _dynamic_map_projection(
                _text_finding_typed(), "_omop_code", "_omop_display",
                "urn:omop:concept_id", "_omop_domain", "bronze.map_text_events:concept",
            )
        )
        .unionByName(
            _dynamic_map_projection(
                _text_finding_typed(), "_omop_value_code", "_omop_value_display",
                "urn:omop:concept_id", "_omop_value_domain", "bronze.map_text_events:value",
            )
        )
        .unionByName(_luna_code_event_map(
            _referral_canonical(), "treatment_function_code",
            "urn:barts:luna:treatment-function"))
        .unionByName(_luna_code_event_map(
            _rtt_pathway_canonical(), "treatment_function_code",
            "urn:barts:luna:treatment-function"))
        .unionByName(_luna_code_event_map(
            _rtt_activity_canonical(), "treatment_function_code",
            "urn:barts:luna:treatment-function"))
        .unionByName(_luna_code_event_map(
            _rtt_pathway_canonical(), "current_status_code",
            "urn:barts:luna:rtt-status"))
        .unionByName(_luna_code_event_map(
            _rtt_activity_canonical(), "status_code",
            "urn:barts:luna:rtt-status"))
        .unionByName(_seed_code_event_map(
            _waiting_list_index_representative(), "status_code",
            "urn:cerner:code_value", "wl_status_cdf_meaning"))
        .unionByName(
            _map_projection(_allergy_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "condition", "bronze.map_allergy")
        )
        .unionByName(
            _map_projection(_transfusion_canonical(), "product_concept_id", "product_concept_name",
                            "http://snomed.info/sct", "device",
                            "lookup.bloodtrack_product_group_map:applied")
        )
        .unionByName(
            _map_projection(_transfusion_canonical(), "_product_proc_concept_id",
                            "_product_proc_concept_name", "http://snomed.info/sct", "procedure",
                            "lookup.bloodtrack_product_group_map:applied")
        )
        .unionByName(_transfusion_blood_group_event_map())
        .unionByName(
            _map_projection(_cancer_treatment_canonical(), "_drug_omop_code", "_drug_omop_display",
                            "urn:omop:concept_id", "drug",
                            "lookup.cancer_treatment_term_map:applied")
        )
        .unionByName(
            _map_projection(_cancer_treatment_canonical(), "_procurement_omop_code",
                            "_procurement_omop_display", "urn:omop:concept_id", "procedure",
                            "bronze.map_cancer_treatment:procurement")
        )
        .unionByName(
            _map_projection(_cancer_treatment_canonical(), "_procurement_opcs4_code",
                            "_procurement_opcs4_display",
                            "http://fhir.hl7.org.uk/CodeSystem/OPCS-4", "procedure",
                            "bronze.map_cancer_treatment:procurement")
        )
        .unionByName(
            _map_projection(_cancer_treatment_canonical(), "_delivery_omop_code",
                            "_delivery_omop_display", "urn:omop:concept_id", "procedure",
                            "bronze.map_cancer_treatment:delivery")
        )
        .unionByName(
            _map_projection(_cancer_treatment_canonical(), "_delivery_opcs4_code",
                            "_delivery_opcs4_display",
                            "http://fhir.hl7.org.uk/CodeSystem/OPCS-4", "procedure",
                            "bronze.map_cancer_treatment:delivery")
        )
        .unionByName(
            _map_projection(_condition_stage_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "condition",
                            "bronze.map_aria_diagnosis_staging")
        )
        .unionByName(
            _map_projection(_endoscopy_finding_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "observation",
                            "bronze.map_endobase_exam_term:c4")
        )
        .unionByName(
            _map_projection(_endoscopy_finding_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "observation",
                            "bronze.map_endobase_exam_term:c4")
        )
        .unionByName(
            _map_projection(_device_canonical(), "_snomed_code", "_snomed_display",
                            "http://snomed.info/sct", "device",
                            "lookup.mediconnect_device_type_map:applied")
        )
        .unionByName(
            _map_projection(_medication_supply_canonical(), "_dmd_code", "_dmd_display",
                            "http://snomed.info/sct", "drug", "bronze.map_homecare_request")
        )
        .unionByName(
            _map_projection(_drug_expenditure_canonical(), "_dmd_code", "_dmd_display",
                            "http://snomed.info/sct", "drug",
                            "bronze.map_finance_hcd_expenditure")
        )
        .unionByName(
            _map_projection(_drug_expenditure_canonical(), "_omop_code", "_omop_display",
                            "urn:omop:concept_id", "drug",
                            "bronze.map_finance_hcd_expenditure")
        )
        .unionByName(
            _map_projection(_community_care_activity_canonical(),
                            "_snomed_candidate_code", "_snomed_candidate_name",
                            "http://snomed.info/sct", "community_activity",
                            "bronze.map_community_care_activity")
        )
        .unionByName(
            _map_projection(_community_care_activity_canonical(),
                            "_snomed_candidate_omop_code", "_snomed_candidate_omop_display",
                            "urn:omop:concept_id", "community_activity",
                            "bronze.map_community_care_activity")
        )
        .unionByName(
            _map_projection(_elective_access_entry_canonical(), "_opcs4_code", "_opcs4_display",
                            "http://fhir.hl7.org.uk/CodeSystem/OPCS-4", "procedure",
                            "bronze.map_elective_access_list_procedure")
        )
        .unionByName(_iweb_multiselect_event_map())
    )

def _dynamic_map_projection(df, code_col, display_col, system, domain_col, map_source):
    filtered = df.where(F.col(code_col).isNotNull())
    domain = F.lower(F.coalesce(F.col(domain_col), F.lit("observation")))
    return filtered.select(
        stable_id(
            "patient_event_map", F.col("patient_event_id"), F.lit(system), F.col(code_col),
            domain, F.lit(map_source), F.lit(None),
        ).alias("patient_event_map_id"),
        "patient_event_id", F.lit(system).alias("mapped_coding_system"),
        F.col(code_col).cast("string").alias("mapped_code"),
        F.col(display_col).alias("mapped_display"), domain.alias("target_domain"),
        F.lit(map_source).alias("map_source"), F.lit(None).cast("string").alias("map_version"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"), "loaded_at",
    )

def _luna_code_event_map(fact_df, code_col, source_system_urn):
    m = (
        read_source(SRC_LUNA_CODE_MAPPING)
        .where(F.col("source_coding_system") == F.lit(source_system_urn))
        .select(
            F.col("source_code").alias("_m_source_code"),
            F.col("target_coding_system").alias("_m_target_system"),
            F.col("target_code").alias("_m_target_code"),
            F.col("target_display").alias("_m_target_display"),
            F.col("target_domain").alias("_m_target_domain"),
            F.col("mapping_rule_id").alias("_m_rule"),
        )
    )
    facts = fact_df.where(F.col(code_col).isNotNull()).select(
        "patient_event_id", F.col(code_col).alias("_f_code"),
        "loaded_at",
    )
    joined = facts.join(m, facts["_f_code"] == m["_m_source_code"], "inner")
    map_source = F.concat(F.lit("lookup.luna_code_map:"), F.col("_m_rule"))
    return joined.select(
        stable_id(
            "patient_event_map", F.col("patient_event_id"), F.col("_m_target_system"),
            F.col("_m_target_code"), F.col("_m_target_domain"), map_source,
            F.lit(None),
        ).alias("patient_event_map_id"),
        "patient_event_id",
        F.col("_m_target_system").alias("mapped_coding_system"),
        F.col("_m_target_code").cast("string").alias("mapped_code"),
        F.col("_m_target_display").alias("mapped_display"),
        F.col("_m_target_domain").alias("target_domain"),
        map_source.alias("map_source"),
        F.lit(None).cast("string").alias("map_version"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        "loaded_at",
    )

def _seed_code_event_map(fact_df, code_col, source_system_urn, rule_id):
    m = (
        read_source(SRC_SEED_CODE_MAPPING)
        .where(
            (F.col("source_coding_system") == F.lit(source_system_urn))
            & (F.col("mapping_rule_id") == F.lit(rule_id))
        )
        .select(
            F.col("source_code").alias("_m_source_code"),
            F.col("target_coding_system").alias("_m_target_system"),
            F.col("target_code").alias("_m_target_code"),
            F.col("target_display").alias("_m_target_display"),
            F.col("target_domain").alias("_m_target_domain"),
            F.col("mapping_rule_id").alias("_m_rule"),
        )
    )
    facts = fact_df.where(F.col(code_col).isNotNull()).select(
        "patient_event_id", F.col(code_col).alias("_f_code"),
        "loaded_at",
    )
    joined = facts.join(m, facts["_f_code"] == m["_m_source_code"], "inner")
    map_source = F.concat(F.lit("lookup.seed_code_map:"), F.col("_m_rule"))
    return joined.select(
        stable_id(
            "patient_event_map", F.col("patient_event_id"), F.col("_m_target_system"),
            F.col("_m_target_code"), F.col("_m_target_domain"), map_source,
            F.lit(None),
        ).alias("patient_event_map_id"),
        "patient_event_id",
        F.col("_m_target_system").alias("mapped_coding_system"),
        F.col("_m_target_code").cast("string").alias("mapped_code"),
        F.col("_m_target_display").alias("mapped_display"),
        F.col("_m_target_domain").alias("target_domain"),
        map_source.alias("map_source"),
        F.lit(None).cast("string").alias("map_version"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        "loaded_at",
    )

def _coded_bridge_event_map():
    b = read_source(SRC_CODED_EVENT_FINDING_MAPPING).alias("b")
    event_id = stable_id("coded_event:mill", b.SOURCE_ROW_KEY)
    domain = F.lower(F.coalesce(b.OMOP_CONCEPT_DOMAIN, F.lit("observation")))
    return b.where(b.OMOP_CONCEPT_ID.isNotNull()).select(
        stable_id(
            "patient_event_map:coded_bridge", event_id, b.OMOP_CONCEPT_ID,
            b.MAPPING_RULE_ID, b.MAPPING_RANK, b.SOURCE_FIELD, F.lit(None),
        ).alias("patient_event_map_id"),
        event_id.alias("patient_event_id"),
        F.lit("urn:omop:concept_id").alias("mapped_coding_system"),
        b.OMOP_CONCEPT_ID.cast("string").alias("mapped_code"),
        b.OMOP_CONCEPT_NAME.alias("mapped_display"), domain.alias("target_domain"),
        F.lit("bronze.map_coded_events_omop_bridge").alias("map_source"),
        F.lit(None).cast("string").alias("map_version"),
        b.OMOP_VALID_START_DATE.cast("timestamp").alias("valid_from"),
        b.OMOP_VALID_END_DATE.cast("timestamp").alias("valid_to"),
        F.coalesce(b.PIPELINE_UPDT_DT_TM, b.MAPPING_ADC_UPDT).alias("loaded_at"),
    )

def _iweb_multiselect_event_map():
    v = read_source(SRC_IWEB_VALUE_MAP).where(
        F.col("TARGET_CODE").isNotNull() | F.col("TARGET_CONCEPT_ID").isNotNull()
    ).select(
        F.col("SOURCE_TABLE").alias("_v_table"),
        F.col("FIELD_NAME").alias("_v_field"),
        F.col("CODE").alias("_v_code"),
        F.col("TARGET_VOCABULARY").alias("_v_vocabulary"),
        F.col("TARGET_CODE").alias("_v_target_code"),
        F.col("TARGET_CONCEPT_ID").alias("_v_target_concept_id"),
        F.col("TARGET_CONCEPT_NAME").alias("_v_target_name"),
        F.col("MAPPING_STATUS").alias("_v_mapping_status"),
    )
    m = read_source(SRC_IWEB_MULTISELECT).join(
        v,
        (F.col("SOURCE_TABLE") == F.col("_v_table"))
        & (F.col("FIELD_NAME") == F.col("_v_field"))
        & (F.col("CODE") == F.col("_v_code")),
        "inner",
    )
    family = (
        F.when(F.col("SOURCE_TABLE") == "reg_coronary_subprocedure", F.lit("coronary_lesion"))
        .when(F.col("SOURCE_TABLE").isin("reg_cs2010g_pre1", "reg_cs2010g_pre2",
                                         "reg_cs2010g_post1", "reg_cs2010g_post2"),
              F.lit("surgery_episode"))
        .when(F.col("SOURCE_TABLE") == "reg_cs2010g_subprocedure", F.lit("surgery_procedure"))
        .when(F.col("SOURCE_TABLE") == "reg_cs2010g_followup", F.lit("surgery_followup"))
        .when(F.col("SOURCE_TABLE") == "reg_dghminap", F.lit("acs_transfer"))
        .when(F.col("SOURCE_TABLE") == "reg_eracs", F.lit("eracs_episode"))
        .when(F.col("SOURCE_TABLE") == "reg_mort", F.lit("mortality_review"))
        .when(F.col("SOURCE_TABLE") == "reg_noncoronary", F.lit("noncoronary_procedure"))
        .when(F.col("SOURCE_TABLE").isin("reg_mdt", "reg_ctmdt"), F.lit("cardiac_mdt"))
    )
    registry_type = (
        F.when(F.col("SOURCE_TABLE") == "reg_mdt", F.lit("CORONARY_REVASC"))
        .when(F.col("SOURCE_TABLE") == "reg_ctmdt", F.lit("CT_AORTIC"))
        .otherwise(F.lit("~"))
    )
    m = m.withColumn("_family", family).withColumn("_registry_type", registry_type) \
        .where(F.col("_family").isNotNull())
    target_system = _iweb_target_system(F.col("_v_vocabulary"))
    target_code = F.coalesce(F.col("_v_target_code"),
                             F.col("_v_target_concept_id").cast("string"))
    parent_event_id = stable_id("registry_entry:iweb", F.col("_family"),
                                F.col("_registry_type"), F.col("ENTRY_ID"))
    map_source = F.concat(F.lit("lookup.iweb_value_to_concept:"),
                          F.coalesce(F.col("_v_mapping_status"), F.lit("UNKNOWN")))
    return m.select(
        stable_id("patient_event_map", parent_event_id, target_system, target_code,
                  F.lit("registry"), map_source, F.lit(None))
        .alias("patient_event_map_id"),
        parent_event_id.alias("patient_event_id"), target_system.alias("mapped_coding_system"),
        target_code.alias("mapped_code"), F.col("_v_target_name").alias("mapped_display"),
        F.lit("registry").alias("target_domain"), map_source.alias("map_source"),
        F.lit(None).cast("string").alias("map_version"),
        F.lit(None).cast("timestamp").alias("valid_from"),
        F.lit(None).cast("timestamp").alias("valid_to"),
        F.col("ADC_UPDT").alias("loaded_at"),
    ).dropDuplicates(["patient_event_map_id"])

In [0]:
PATIENT_EVENT_COLUMN_COMMENTS = {
    "patient_event_row_id": "Deterministic row key (the mapping id for mapped rows; a namespaced unmapped id otherwise).",
    "patient_event_id": "Immutable fact-grain event identifier shared across a mapped event's N rows.",
    "subject_key": "deterministic subject join key.",
    "subject_id_system": "Identifier system behind subject_key.",
    "person_id": "Resolved Millennium person id when available.",
    "identity_status": "Resolution state.",
    "encounter_id": "Encounter reference when the source supplies one.",
    "event_datetime": "Clinical or administrative event time.",
    "event_end_datetime": "Event end when the source supplies one.",
    "event_type": "Fact kind (condition",
    "fact_category": "Clinical versus administrative fact.",
    "fact_table": "Typed fact table holding the event's values.",
    "fact_row_id": "Row key inside fact_table.",
    "source_system": "Plain-language source system (Millennium",
    "source_object": "Native source entity in plain language",
    "source_row_key": "Native source record key.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source code as recorded.",
    "source_display": "Source display as recorded.",
    "record_status": "Normalized lifecycle status carried as data.",
    "confidentiality_code": "Source confidentiality label.",
    "vip_ind": "Source VIP indicator carried as data.",
    "withheld_identity_ind": "Source withheld-identity indicator.",
    "mapped_coding_system": "Standard coding system for this mapping row; null on the unmapped row.",
    "mapped_code": "Standard code.",
    "mapped_display": "Standard display or definition.",
    "target_domain": "Downstream routing domain for this mapping.",
    "map_source": "Governed mapping product that produced the row.",
    "map_version": "Mapping content version.",
    "load_batch_id": "Deterministic batch token from bronze load time.",
    "loaded_at": "Bronze load or update timestamp; never pipeline wall-clock time.",
}

# Physical person/time clustering is applied by _jm_pipeline_admin post-update, exactly as for
# the v1 UNION-backed index (the join-backed MV keeps the same repair pathway).
@materialized_view(
    name=_n("journey_events.patient_event"),
    comment="One row per admitted event x standard mapping (unmapped events carry one row with "
            "a null mapping block). Who/when/what, plain-language source, and standard-terms "
            "meaning in a single read. Timeline counts use COUNT(DISTINCT patient_event_id).",
    refresh_policy="incremental",
    column_comments=PATIENT_EVENT_COLUMN_COMMENTS,
)
def patient_event():
    e = spark.read.table(_n("journey_events._event_index")).alias("e")
    m = spark.read.table(_n("journey_events._event_map")).alias("m")
    return (
        e.join(m, F.col("e.patient_event_id") == F.col("m.patient_event_id"), "left")
        .select(
            F.coalesce(
                F.col("m.patient_event_map_id"),
                stable_id("patient_event:unmapped", F.col("e.patient_event_id")),
            ).alias("patient_event_row_id"),
            F.col("e.patient_event_id").alias("patient_event_id"),
            F.col("e.subject_key").alias("subject_key"),
            F.col("e.subject_id_system").alias("subject_id_system"),
            F.col("e.person_id").alias("person_id"),
            F.col("e.identity_status").alias("identity_status"),
            F.col("e.encounter_id").alias("encounter_id"),
            F.col("e.event_datetime").alias("event_datetime"),
            F.col("e.event_end_datetime").alias("event_end_datetime"),
            F.col("e.event_type").alias("event_type"),
            F.col("e.fact_category").alias("fact_category"),
            F.col("e.fact_table").alias("fact_table"),
            F.col("e.fact_row_id").alias("fact_row_id"),
            F.col("e.source_system").alias("source_system"),
            F.col("e.source_object").alias("source_object"),
            F.col("e.source_row_key").alias("source_row_key"),
            F.col("e.source_coding_system").alias("source_coding_system"),
            F.col("e.source_code").alias("source_code"),
            F.col("e.source_display").alias("source_display"),
            F.col("e.record_status").alias("record_status"),
            F.col("e.confidentiality_code").alias("confidentiality_code"),
            F.col("e.vip_ind").alias("vip_ind"),
            F.col("e.withheld_identity_ind").alias("withheld_identity_ind"),
            F.col("m.mapped_coding_system").alias("mapped_coding_system"),
            F.col("m.mapped_code").alias("mapped_code"),
            F.col("m.mapped_display").alias("mapped_display"),
            F.col("m.target_domain").alias("target_domain"),
            F.col("m.map_source").alias("map_source"),
            F.col("m.map_version").alias("map_version"),
            F.col("e.load_batch_id").alias("load_batch_id"),
            F.col("e.loaded_at").alias("loaded_at"),
        )
    )

In [0]:
SRC_PHARMACY_ISSUE = "4_prod.bronze.map_pharmacy_issue"
SRC_PATHOLOGY_REPORT_VERSIONS = "4_prod.bronze.map_pathology_report"

def _endobase_document_canonical():
    # One clinically meaningful report per exam. Template and free-text terms are
    # interleaved in source display order; DGVS tab names are not landed, so section
    # titles deliberately expose the raw section/subsection ids instead of inventing
    # a decode. CREATED_TS is authoring provenance and is only a clinical-time fallback
    # when the parent exam is absent.
    t = (
        read_source(SRC_ENDOBASE_EXAM_TERM)
        .where(F.coalesce(F.col("SOURCE_PRESENT_IND"), F.lit(True)))
        .where(_present(F.col("TERM_TEXT")))
    )
    ordered_term = F.struct(
        F.coalesce(F.col("SECTION_TAB_ID").cast("long"), F.lit(2147483647)).alias("section_sort"),
        F.coalesce(F.col("SUBSECTION_TAB_ID").cast("long"), F.lit(2147483647)).alias("subsection_sort"),
        F.coalesce(F.col("DISPLAY_ORDER").cast("long"), F.lit(2147483647)).alias("display_sort"),
        F.col("ENDOBASE_EXAM_TERM_ID").cast("long").alias("term_sort"),
        F.col("SECTION_TAB_ID").cast("string").alias("section_tab_id"),
        F.col("SUBSECTION_TAB_ID").cast("string").alias("subsection_tab_id"),
        F.col("TERM_TEXT").alias("term_text"),
        
    )
    assembled = (
        t.groupBy("ENDOBASE_EXAM_ID")
        .agg(
            F.sort_array(F.collect_list(ordered_term)).alias("_ordered_terms"),
            F.min("CREATED_TS").alias("_first_authored_ts"),
            F.max("ADC_UPDT").alias("_term_loaded_at"),
            F.count(F.lit(1)).cast("long").alias("_term_count"),
            F.countDistinct("PERSON_ID").alias("_term_person_count"),
            F.max("PERSON_ID").alias("_term_person_id"),
        )
        .withColumn(
            "_document_text",
            F.expr("concat_ws('\\n', transform(_ordered_terms, x -> x.term_text))"),
        )
        .withColumn("_anon_document_text", F.lit(None).cast("string"))
        .withColumn(
            "_sections_json",
            F.expr("""
              to_json(transform(
                _ordered_terms,
                (x, i) -> named_struct(
                  'sequence', i + 1,
                  'section_title', concat(
                    'section_tab_id=', coalesce(x.section_tab_id, 'null'),
                    ';subsection_tab_id=', coalesce(x.subsection_tab_id, 'null')
                  ),
                  'section_text', x.term_text
                )
              ))
            """),
        )
        .alias("a")
    )
    x = read_source(SRC_ENDOBASE_EXAM).select(
        F.col("ENDOBASE_EXAM_ID").alias("_x_exam_id"),
        F.col("PERSON_ID").alias("_x_person_id"),
        F.col("MILL_ENCNTR_ID").alias("_x_encntr_id"),
        F.col("MILL_ORDER_ID").alias("_x_order_id"),
        F.col("PERFORMED_TS_CLEAN").alias("_x_performed_ts"),
        F.col("EXAM_TS_CLEAN").alias("_x_exam_ts"),
        F.col("TRUE_START_TS_CLEAN").alias("_x_true_start_ts"),
        F.col("START_TS_CLEAN").alias("_x_start_ts"),
        F.col("TRUE_END_TS_CLEAN").alias("_x_true_end_ts"),
        F.col("END_TS_CLEAN").alias("_x_end_ts"),
        F.col("SOURCE_CREATE_TS").alias("_x_source_create_ts"),
        F.col("EXAM_TYPE_DESC").alias("_x_exam_type_desc"),
        F.col("SIGNER_ID").alias("_x_signer_id"),
        F.col("EXAMINER_ID").alias("_x_examiner_id"),
        F.col("SOURCE_PRESENT_IND").alias("_x_source_present_ind"),
        F.col("PIPELINE_UPDT_DT_TM").alias("_x_pipeline_updt"),
        F.col("ADC_UPDT").alias("_x_loaded_at"),
    ).alias("x")
    s = assembled.join(x, F.col("a.ENDOBASE_EXAM_ID") == F.col("x._x_exam_id"), "left")
    e = read_source(SRC_ENCOUNTER).select(
        F.col("ENCNTR_ID").alias("_e_encntr_id"),
        F.col("PERSON_ID").alias("_e_person_id"),
        F.col("ADC_UPDT").alias("_e_loaded_at"),
    ).alias("e")
    s = s.join(e, F.col("x._x_encntr_id") == F.col("e._e_encntr_id"), "left")
    term_person = F.when(F.col("a._term_person_count") == 1, F.col("a._term_person_id"))
    resolved_person = F.coalesce(F.col("x._x_person_id"), F.col("e._e_person_id"), term_person)
    natural = F.col("a.ENDOBASE_EXAM_ID").cast("string")
    event_id = stable_id("document:endobase_exam", F.col("a.ENDOBASE_EXAM_ID"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", resolved_person)], SRC_ENDOBASE_EXAM, natural
    )
    event_time = F.coalesce(
        F.col("x._x_performed_ts"), F.col("x._x_exam_ts"),
        F.col("x._x_true_start_ts"), F.col("x._x_start_ts"),
        F.when(F.col("x._x_exam_id").isNull(), F.col("a._first_authored_ts")),
    )
    event_end = F.coalesce(F.col("x._x_true_end_ts"), F.col("x._x_end_ts"))
    loaded_at = F.greatest(
        F.col("a._term_loaded_at"), F.col("x._x_loaded_at"), F.col("e._e_loaded_at")
    )
    inactive = F.col("x._x_exam_id").isNotNull() & ~F.coalesce(
        F.col("x._x_source_present_ind"), F.lit(True)
    )
    author_source_id = F.coalesce(
        F.col("x._x_signer_id"), F.col("x._x_examiner_id")
    ).cast("string")
    source_code = F.lit("ENDOSCOPY_REPORT")
    source_display = F.lit("Endoscopy report")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        resolved_person.cast("string").alias("person_id"),
        F.when(resolved_person.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("x._x_encntr_id").isNotNull(),
               stable_id("encounter:mill", F.col("x._x_encntr_id"))).alias("encounter_id"),
        event_time.alias("event_datetime"), event_end.alias("event_end_datetime"),
        F.lit("urn:endobase:document-type").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:endobase:document-type"), source_code,
                       source_display, True)
        ).alias("_document_type_json"),
        F.coalesce(F.col("x._x_exam_type_desc"), source_display).alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.when(author_source_id.isNotNull(), F.lit("report_author")).alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.when(F.col("x._x_exam_id").isNull(), F.lit("orphan_exam"))
         .otherwise(F.lit("assembled")).alias("status_code"),
        F.concat_ws(
            ":", natural, F.col("a._term_count").cast("string"),
            F.date_format(F.col("a._term_loaded_at"), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"),
        ).alias("version_id"),
        F.col("a._document_text").alias("document_text"),
        F.col("a._sections_json").alias("_sections_json"),
        F.lit("endobase-exam-assembly-v1").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"), F.lit("UTF-8").alias("encoding"),
        F.sha2(F.col("a._document_text"), 256).alias("text_sha256"),
        F.length(F.col("a._document_text")).cast("long").alias("text_length"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        F.coalesce(F.col("x._x_source_create_ts"), F.col("a._first_authored_ts"))
         .alias("record_status_effective_from"),
        F.when(inactive, loaded_at).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit(None).cast("string").alias("document_class"),
        F.lit(None).cast("string").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        F.lit(None).cast("string").alias("source_parent_event_id"),
        F.lit(None).cast("string").alias("parent_relation"),
        F.lit(None).cast("string").alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.col("a._anon_document_text").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:barts:endobase").alias("_label_system"),
        F.lit("endoscopy_report").alias("_label_key"),
        F.lit("endobase_exam").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.greatest(F.col("a._term_loaded_at"), F.col("x._x_pipeline_updt"))
         .alias("source_update_timestamp"),
        loaded_at.alias("loaded_at"),
        F.lit("endobase").alias("_source_system"),
        F.lit(SRC_ENDOBASE_EXAM).alias("_source_table"), natural.alias("_source_row_id"),
        F.lit(None).cast("string").alias("_raw_content_sha256"),
        F.when(F.col("x._x_person_id").isNotNull() | term_person.isNotNull(), F.lit("direct"))
         .when(F.col("e._e_person_id").isNotNull(), F.lit("encounter_join"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
        F.when(author_source_id.isNotNull(), F.lit("urn:barts:endobase:staff-id"))
         .alias("_author_id_system"),
        author_source_id.alias("_author_source_id"),
        F.lit("reassembled").alias("_assembly_status"),
        F.col("a._term_count").cast("long").alias("_chunk_count"),
    )

def _neonatal_narrative_document_canonical():
    s = read_source(SRC_NEO_NARRATIVE)
    narrative_source = s.select(
        "EntityID", "BABY_PERSON_ID", "ADC_UPDT", "SOURCE_PRESENT_IND",
        "DischargeSummaryCompletedBy", "DischargeSummaryCompletedBy_Grade",
        *_NEO_NARRATIVE_FIELDS,
    )
    stacked = None
    for narrative_field in _NEO_NARRATIVE_FIELDS:
        field_rows = narrative_source.select(
            "EntityID", "BABY_PERSON_ID", "ADC_UPDT", "SOURCE_PRESENT_IND",
            "DischargeSummaryCompletedBy", "DischargeSummaryCompletedBy_Grade",
            F.lit(narrative_field).alias("narrative_field"),
            F.col(narrative_field).alias("narrative_text"),
            F.lit(None).cast("string").alias("anon_narrative_text"),
        ).where(
            F.col("narrative_text").isNotNull()
            & (F.trim(F.col("narrative_text")) != "")
        )
        stacked = field_rows if stacked is None else stacked.unionByName(field_rows)
    ep = read_source(SRC_NEO_EPISODE).select(
        F.col("EntityID").alias("_ep_entity_id"),
        F.col("DischTime_CLEAN").alias("_ep_disch"),
        F.col("AdmitTime_CLEAN").alias("_ep_admit"),
    )
    d = stacked.join(ep, stacked.EntityID == ep._ep_entity_id, "left")
    event_id = stable_id("document:neonatal_narrative", d.EntityID, d.narrative_field)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", d.BABY_PERSON_ID)],
        SRC_NEO_NARRATIVE,
        F.concat_ws("|", d.EntityID, d.narrative_field),
    )
    retracted = ~F.coalesce(d.SOURCE_PRESENT_IND, F.lit(True))
    event_time = F.coalesce(d._ep_disch, d._ep_admit)
    author_source_id = F.when(_present(d.DischargeSummaryCompletedBy),
                              F.trim(d.DischargeSummaryCompletedBy))
    author_grade = F.when(_present(d.DischargeSummaryCompletedBy_Grade),
                          F.trim(d.DischargeSummaryCompletedBy_Grade))
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        d.BABY_PERSON_ID.cast("string").alias("person_id"),
        F.when(d.BABY_PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"), event_time.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:badgernet:narrative-field").alias("source_coding_system"),
        d.narrative_field.alias("source_code"), d.narrative_field.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:badgernet:narrative-field"),
                       d.narrative_field, d.narrative_field, True)
        ).alias("_document_type_json"),
        F.concat_ws(" ", F.lit("Neonatal"), d.narrative_field).alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.when(author_source_id.isNotNull(),
               F.coalesce(author_grade, F.lit("badgernet_care_team")))
         .alias("author_role"),
        F.lit(None).cast("string").alias("service_id"), F.lit("final").alias("status_code"),
        F.concat_ws(":", d.EntityID, d.narrative_field).alias("version_id"),
        d.narrative_text.alias("document_text"), F.lit("[]").alias("_sections_json"),
        F.lit("badgernet-narrative-v1").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"), F.lit("UTF-8").alias("encoding"),
        F.sha2(d.narrative_text, 256).alias("text_sha256"),
        F.length(d.narrative_text).cast("long").alias("text_length"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.when(retracted, d.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit('["IG"]').alias("_sensitivity_labels_json"),
        F.lit(None).cast("string").alias("document_class"),
        F.lit(None).cast("string").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        F.lit(None).cast("string").alias("source_parent_event_id"),
        F.lit(None).cast("string").alias("parent_relation"),
        F.lit(None).cast("string").alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:badgernet:narrative-field").alias("_label_system"),
        F.lower(F.trim(d.narrative_field)).alias("_label_key"),
        F.lit("neonatal_episode_narrative").alias("source_feed"),
        F.date_format(d.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        d.ADC_UPDT.alias("source_update_timestamp"), d.ADC_UPDT.alias("loaded_at"),
        F.lit("badgernet").alias("_source_system"),
        F.lit(SRC_NEO_NARRATIVE).alias("_source_table"),
        F.concat_ws("|", d.EntityID, d.narrative_field).alias("_source_row_id"),
        F.lit(None).cast("string").alias("_raw_content_sha256"),
        F.when(author_source_id.isNotNull(), F.lit("urn:badgernet:staff-name"))
         .alias("_author_id_system"),
        author_source_id.alias("_author_source_id"),
        F.when(d.BABY_PERSON_ID.isNotNull(), F.lit("direct"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
    )

In [0]:
SRC_NEO_NARRATIVE = "4_prod.bronze.map_neonatal_episode_narrative"

def _pharmacy_issue_canonical():
    s = read_source(SRC_PHARMACY_ISSUE)
    event_id = stable_id("pharmacy_issue:jac", s.PHARMACY_ISSUE_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:jac:lnkpid", s.LNKPID)],
        SRC_PHARMACY_ISSUE,
        s.PHARMACY_ISSUE_ID,
    )
    retracted = F.coalesce(~s.SOURCE_PRESENT_IND, F.lit(False))
    promoted = F.upper(F.trim(F.coalesce(s.ISSUE_CATEGORY, F.lit("")))) == F.lit("ISSUE")
    payload = F.parse_json(F.to_json(F.struct(*[s[c] for c in s.columns])))
    return s.select(
        event_id.alias("patient_event_id"),
        event_id.alias("fact_row_id"),
        skey.alias("subject_key"),
        ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("long").cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .when(s.LNKPID.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.ISSUE_DTTM.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:jac:issue_type").alias("source_coding_system"),
        s.ISSUE_TYPE.alias("source_code"),
        s.ISSUE_CATEGORY.alias("source_display"),
        s.DRUG_FULL.alias("value_text"),
        s.TOTAL_UNITS.cast("decimal(38,4)").alias("value_number"),
        F.lit(None).cast("timestamp").alias("value_datetime"),
        s.DRUG_DOSEUNIT.alias("unit"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.SOURCE_RECORD_UPDATED_DT.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.parse_json(F.lit("null")).alias("sensitivity_labels"),
        F.lit("pharmacy_issue").alias("feed_id"),
        payload.alias("payload"),
        F.when(promoted, F.lit("journey_clinical.medication_dispense"))
        .alias("promoted_to_table"),
        F.when(promoted, event_id).alias("promoted_to_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_RECORD_UPDATED_DT.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("jac").alias("_source_system"),
        F.lit(SRC_PHARMACY_ISSUE).alias("_source_table"),
        s.PHARMACY_ISSUE_ID.alias("_source_row_id"),
        s.DMD_VTM_CODE.alias("_dmd_code"),
        s.DMD_VTM_NAME.alias("_dmd_display"),
    )

ADMISSION_METRICS_COLUMN_COMMENTS = {
    "feed_id": "Registered feed.",
    "route_id": "Route or exclusion reason.",
    "admitted": "Whether these rows entered silver.",
    "load_date": "Bronze load date bucket for per-load accounting.",
    "row_count": "Rows in this bucket.",
    "latest_loaded_at": "Latest bronze load time observed in the bucket.",
}

@materialized_view(
    name=_n("journey_reference.admission_metrics"),
    comment="Admission-gate accounting: rows seen per registered route with admitted flag. "
            "route_id doubles as the exclusion reason for admitted=false rows. Product-wide "
            "usable-code gate: all source_code-bearing fact lanes report "
            "excluded_unusable_code; document (narrative route) and the structured admin facts "
            "without source_code are exempt; registry_entry is exempt as a "
            "code-free class. Exclusions are counted here and "
            "retained nowhere.",
    refresh_policy="incremental",
    column_comments=ADMISSION_METRICS_COLUMN_COMMENTS,
)
def admission_metrics():
    def lane(df, feed_id, route_col, admitted_col):
        return df.select(
            F.lit(feed_id).alias("feed_id"),
            route_col.cast("string").alias("route_id"),
            admitted_col.alias("admitted"),
            F.to_date(F.col("loaded_at")).alias("load_date"),
            F.col("loaded_at"),
        )

    def typed_route(df):
        gated = F.col("_typed_route") & _usable_code(F.col("source_code"))
        route = (
            F.when(gated, F.lit("clinical_finding"))
             .when(F.col("_typed_route"), F.lit("excluded_unusable_code"))
             .otherwise(F.lit("excluded_untyped"))
        )
        return df.withColumn("_metric_route", route), gated

    def fact_lane(df, feed_id, fact_route):
        ok = _usable_code(F.col("source_code"))
        route = F.when(ok, F.lit(fact_route)).otherwise(F.lit("excluded_unusable_code"))
        return lane(df, feed_id, route, ok)

    coded_df, coded_ok = typed_route(_coded_finding_canonical())
    nomen_df, nomen_ok = typed_route(_nomen_finding_canonical())
    date_df, date_ok = typed_route(_date_finding_canonical())
    text_gated = _usable_code(F.col("source_code"))
    text_route = (
        F.when(F.col("_route") == "annex_catch_all", F.lit("excluded_untyped"))
         .when(~text_gated, F.lit("excluded_unusable_code"))
         .otherwise(F.col("_route"))
    )
    rows = (
        lane(coded_df, "coded_events", F.col("_metric_route"), coded_ok)
        .unionByName(lane(nomen_df, "nomen_events", F.col("_metric_route"), nomen_ok))
        .unionByName(lane(date_df, "date_events", F.col("_metric_route"), date_ok))
        .unionByName(lane(
            _text_finding_canonical(), "text_events", text_route,
            (F.col("_route") != "annex_catch_all") & text_gated,
        ))
        .unionByName(lane(
            _numeric_vital_canonical(), "numeric_events",
            F.when(_usable_code(F.col("source_code")), F.lit("vital_sign"))
             .otherwise(F.lit("excluded_unusable_code")),
            _usable_code(F.col("source_code")),
        ))
        .unionByName(lane(
            _numeric_score_canonical(), "numeric_events",
            F.when(_usable_code(F.col("source_code")), F.lit("clinical_score"))
             .otherwise(F.lit("excluded_unusable_code")),
            _usable_code(F.col("source_code")),
        ))
        .unionByName(fact_lane(
            _promoted_vital_from_stage(), "form_promotion", "vital_sign"
        ))
        .unionByName(fact_lane(
            _promoted_score_from_stage(), "form_promotion", "clinical_score"
        ))
        .unionByName(lane(
            _numeric_excluded_canonical(), "numeric_events",
            F.lit("excluded_residual"), F.lit(False),
        ))
        .unionByName(lane(
            _pharmacy_issue_canonical().where(
                F.upper(F.trim(F.coalesce(F.col("source_display"), F.lit("")))) != "ISSUE"
            ),
            "pharmacy_issue", F.lit("excluded_non_issue_audit"), F.lit(False),
        ))
        .unionByName(fact_lane(
            _medication_dispense_canonical_pregate(), "pharmacy_issue", "medication_dispense"
        ))
        .unionByName(fact_lane(
            _family_history_canonical_pregate(), "family_history", "family_history"
        ))
        .unionByName(fact_lane(
            _allergy_canonical_pregate(), "allergy", "allergy_intolerance"
        ))
        .unionByName(fact_lane(
            _transfusion_canonical_pregate(), "bloodtrack", "transfusion"
        ))
        .unionByName(fact_lane(
            _cancer_treatment_canonical_pregate(), "cancer_treatment", "cancer_treatment"
        ))
        .unionByName(fact_lane(
            _condition_stage_canonical_pregate(), "aria_staging", "condition_stage"
        ))
        .unionByName(fact_lane(
            _endobase_procedure_canonical_pregate(), "endobase_exam", "procedure"
        ))
        .unionByName(fact_lane(
            _endoscopy_finding_canonical_pregate(), "endobase_exam_term", "endoscopy_finding"
        ))
        .unionByName(lane(
            _endobase_document_canonical(), "endobase_exam",
            F.lit("document"), F.lit(True),
        ))
        .unionByName(lane(
            _order_comment_document_canonical(), "order_comment",
            F.lit("document"), F.lit(True),
        ))
        .unionByName(lane(
            _elective_access_comment_document_canonical(), "elective_access_comment",
            F.lit("document"), F.lit(True),
        ))
        .unionByName(lane(
            _registry_entry_canonical(), "iweb_registry",
            F.col("registry_family"), F.lit(True),
        ))
        .unionByName(fact_lane(
            _device_canonical_pregate(), "mediconnect_device", "device"
        ))
        .unionByName(fact_lane(
            _condition_diagnosis_canonical_pregate(), "diagnosis", "condition"
        ))
        .unionByName(fact_lane(
            _condition_problem_canonical_pregate(), "problem", "condition"
        ))
        .unionByName(fact_lane(
            _procedure_canonical_pregate(), "procedure", "procedure"
        ))
        .unionByName(fact_lane(
            _implant_procedure_canonical_pregate(), "implant_details", "procedure"
        ))
        .unionByName(fact_lane(
            _theatre_procedure_canonical_pregate(), "theatre_case", "procedure"
        ))
        .unionByName(fact_lane(
            _pathology_result_canonical_pregate(), "pathology", "pathology_result"
        ))
        .unionByName(fact_lane(
            _pathology_requested_test_canonical_pregate(),
            "pathology_requested_test", "pathology_order"
        ))
        .unionByName(fact_lane(
            _genomic_test_canonical_pregate(), "pathology_genetic_test", "genomic_test"
        ))
        .unionByName(fact_lane(
            _genomic_result_canonical_pregate(), "pathology_genetic_result", "genomic_result"
        ))
        .unionByName(fact_lane(
            _indication_canonical_pregate(), "pathology_indication", "indication"
        ))
        .unionByName(fact_lane(
            _micro_isolate_canonical_pregate(), "pathology_microbiology_isolate", "microbiology_isolate"
        ))
        .unionByName(fact_lane(
            _susceptibility_canonical_pregate(), "pathology_antimicrobial_susceptibility", "susceptibility_result"
        ))
        .unionByName(lane(
            _pathology_specimen_canonical(), "pathology_accession",
            F.lit("specimen"), F.lit(True)
        ))
        .unionByName(lane(
            _pathology_report_series_canonical(), "pathology_report",
            F.lit("pathology_report"), F.lit(True)
        ))
        .unionByName(lane(
            _pathology_report_document_canonical(), "pathology_report",
            F.lit("document"), F.lit(True)
        ))
        .unionByName(lane(
            _pathology_report_document_history_canonical(), "pathology_report",
            F.lit("document_history"), F.lit(True)
        ))
        .unionByName(lane(
            _pathology_report_document_base(
                read_source(SRC_PATHOLOGY_REPORT_VERSIONS).where(
                    F.col("report_text").isNull() | (F.trim(F.col("report_text")) == "")
                )
            ),
            "pathology_report", F.lit("excluded_no_text"), F.lit(False)
        ))
        .unionByName(fact_lane(
            _form_canonical_pregate(), "form_activity", "form"
        ))
        .unionByName(fact_lane(
            _medication_admin_canonical_pregate(), "med_admin", "medication_admin"
        ))
        .unionByName(fact_lane(
            _medication_order_canonical_pregate(), "medication_order", "medication_order"
        ))
        .unionByName(fact_lane(
            _imaging_exam_canonical_pregate(), "pacs_examination", "imaging_exam"
        ))
        .unionByName(fact_lane(
            _community_care_activity_canonical_pregate(),
            "community_care_activity", "community_care_activity"
        ))
        .unionByName(lane(
            _community_care_contact_canonical(), "community_care_contact",
            F.lit("community_care_contact"), F.lit(True)
        ))
        .unionByName(lane(
            _hrg_grouping_canonical(), "slam_apc_hrg", F.col("source_feed"), F.lit(True)
        ).where(F.col("route_id") == "slam_apc_hrg"))
        .unionByName(lane(
            _hrg_grouping_canonical(), "slam_op_hrg", F.col("source_feed"), F.lit(True)
        ).where(F.col("route_id") == "slam_op_hrg"))
        .unionByName(lane(
            _costed_activity_canonical(), "slam_costed_activity",
            F.lit("costed_activity"), F.lit(True)
        ))
        .unionByName(lane(
            _drug_expenditure_canonical(), "finance_hcd_expenditure",
            F.lit("drug_expenditure"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _medication_supply_canonical_pregate(), "homecare_request", "medication_supply"
        ))
        .unionByName(lane(
            _elective_access_entry_canonical(), "elective_access_list",
            F.lit("elective_access_entry"), F.lit(True)
        ))
        .unionByName(lane(
            _pathway_tracking_canonical(), "cancer_ptl",
            F.lit("pathway_tracking"), F.lit(True)
        ))
        .unionByName(lane(
            _critical_care_period_canonical(), "critical_care_period",
            F.lit("critical_care_period"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _critical_care_activity_canonical_pregate(),
            "critical_care_activity", "critical_care_activity"
        ))
        .unionByName(fact_lane(
            _cc_procedure_canonical_pregate(),
            "critical_care_procedure", "procedure"
        ))
        .unionByName(lane(
            _critical_care_admission_canonical(), "critical_care_admission",
            F.lit("critical_care_admission"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _cc_daily_score_canonical_pregate(),
            "critical_care_daily_score", "critical_care_daily_score"
        ))
        .unionByName(lane(
            _neonatal_episode_canonical(), "neonatal_episode",
            F.lit("neonatal_episode"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _neonatal_care_day_canonical_pregate(),
            "neonatal_critical_care", "neonatal_care_day"
        ))
        .unionByName(fact_lane(
            _neonatal_examination_canonical_pregate(),
            "neonatal_examination", "neonatal_examination"
        ))
        .unionByName(lane(
            _neonatal_narrative_document_canonical(), "neonatal_episode_narrative",
            F.lit("document"), F.lit(True)
        ))
        .unionByName(lane(
            _baby_delivery_canonical(), "maternity_baby_delivery",
            F.lit("baby_delivery"), F.lit(True)
        ))
        .unionByName(lane(
            _labour_delivery_canonical(), "maternity_labour_delivery",
            F.lit("labour_delivery"), F.lit(True)
        ))
        .unionByName(lane(
            _maternity_care_contact_canonical(), "maternity_care_contact",
            F.lit("maternity_care_contact"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _maternity_diagnosis_canonical_pregate(),
            "maternity_diagnosis", "condition"
        ))
        .unionByName(lane(
            _research_enrollment_canonical(), "research_subject",
            F.lit("research_enrollment"), F.lit(True)
        ))
        .unionByName(fact_lane(
            _mill_radiology_exam_canonical_pregate(),
            "radiology_event", "imaging_exam"
        ))
        .unionByName(lane(
            _pregnancy_reconciliation_source(), "mat_pregnancy_msds_unmatched",
            F.lit("reconciliation"), F.lit(True)
        ))
    )
    return rows.groupBy("feed_id", "route_id", "admitted", "load_date").agg(
        F.count(F.lit(1)).alias("row_count"),
        F.max("loaded_at").alias("latest_loaded_at"),
    )

def _numeric_excluded_canonical():
    s = read_source(SRC_NUMERIC_EVENTS)
    s = s.where(_numeric_route_expr(s) == "excluded")
    event_id = stable_id("numeric_event:mill", s.EVENT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_NUMERIC_EVENTS, s.EVENT_ID
    )
    payload = F.parse_json(F.to_json(F.struct(*[s[c] for c in s.columns])))
    deleted = F.coalesce(s.SOURCE_DELETED_IND, F.lit(False))
    ended = s.CLINICAL_EVENT_VALID_UNTIL_DT_TM.isNotNull() & (
        s.CLINICAL_EVENT_VALID_UNTIL_DT_TM < F.lit("2100-01-01").cast("timestamp")
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.PERFORMED_DT_TM, s.EVENT_START_DT_TM).alias("event_datetime"),
        s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:event_cd").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"),
        F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY).alias("source_display"),
        s.RESULT_TEXT_EFFECTIVE.alias("value_text"),
        s.NUMERIC_RESULT.cast("decimal(38,10)").alias("value_number"),
        F.lit(None).cast("timestamp").alias("value_datetime"),
        F.coalesce(s.UNIT_OF_MEASURE_DISPLAY, s.RESULT_UNITS_DISPLAY).alias("unit"),
        F.when(deleted, F.lit("retracted")).when(ended, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.CLINICAL_EVENT_VALID_FROM_DT_TM.alias("record_status_effective_from"),
        F.when(deleted | ended, s.CLINICAL_EVENT_VALID_UNTIL_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"), F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.parse_json(F.lit("null")).alias("sensitivity_labels"),
        F.lit("numeric_event").alias("feed_id"), payload.alias("payload"),
        F.lit(None).cast("string").alias("promoted_to_table"),
        F.lit(None).cast("string").alias("promoted_to_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.greatest(s.STRING_RESULT_UPDT_DT_TM, s.CLINICAL_EVENT_UPDT_DT_TM, s.ADC_UPDT)
         .alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_NUMERIC_EVENTS).alias("_source_table"),
        s.EVENT_ID.cast("string").alias("_source_row_id"),
    )

_NEO_NARRATIVE_FIELDS = [
    "FinalSummaryText", "BirthSummary", "EpisodeSummary",
    "DiagnosisDuringStay", "DrugsDuringStay", "MaternalMedicalNotes",
]

In [0]:
def _encounter_canonical():
    s = read_source(SRC_ENCOUNTER)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)],
        SRC_ENCOUNTER,
        s.ENCNTR_ID,
    )
    type_class = F.lower(F.coalesce(s.encntr_type_class_desc, s.encntr_class_desc, F.lit("")))
    ended = F.coalesce(
        s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"),
        F.lit(False),
    )
    return s.select(
        stable_id("encounter:mill", s.ENCNTR_ID).alias("encounter_id"),
        skey.alias("subject_key"),
        ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.lit(None).cast("string").alias("parent_encounter_id"),
        F.lit("source_parent_unavailable").alias("parentage_status"),
        F.when(type_class.contains("inpatient"), F.lit("spell"))
         .when(type_class.contains("emergency"), F.lit("emergency_visit"))
         .when(type_class.contains("outpatient"), F.lit("outpatient_attendance"))
         .when(type_class.contains("recurring"), F.lit("recurring_contact"))
         .when(type_class.contains("preadmit"), F.lit("preadmission"))
         .when(type_class.contains("wait list"), F.lit("waiting_list_placeholder"))
         .when(type_class.contains("results only"), F.lit("results_only"))
         .otherwise(F.lit("other")).alias("encounter_level"),
        s.ENCNTR_CLASS_CD.cast("string").alias("class_code"),
        s.encntr_class_desc.alias("class_display"),
        s.ENCNTR_TYPE_CD.cast("string").alias("type_code"),
        s.encntr_type_desc.alias("type_display"),
        s.ENCNTR_TYPE_CLASS_CD.cast("string").alias("type_class_code"),
        s.encntr_type_class_desc.alias("type_class_display"),
        s.ENCNTR_STATUS_CD.cast("string").alias("status_code"),
        s.encntr_status_desc.alias("status_display"),
        s.ARRIVAL_DT_TM_BEST.alias("period_start"),
        s.DEPARTURE_DT_TM_BEST.alias("period_end"),
        s.ARRIVAL_METHOD.alias("arrival_method"),
        s.ARRIVAL_CONFIDENCE.alias("arrival_confidence"),
        s.DEPARTURE_METHOD.alias("departure_method"),
        s.DEPARTURE_CONFIDENCE.alias("departure_confidence"),
        s.LENGTH_OF_STAY_MINUTES.cast("long").alias("length_of_stay_minutes"),
        s.SCHEDULED_ARRIVAL_DT_TM.alias("scheduled_start"),
        s.SCHEDULED_DEPARTURE_DT_TM.alias("scheduled_end"),
        s.REG_DT_TM.alias("registration_datetime"),
        s.INPATIENT_ADMIT_DT_TM.alias("inpatient_admit_datetime"),
        s.DISCH_DT_TM.alias("discharge_datetime"),
        s.ENCOUNTER_COMPLETE_DT_TM_EFFECTIVE.alias("workflow_complete_datetime"),
        s.ARRIVE_DT_TM.alias("raw_arrival_datetime"),
        s.DEPART_DT_TM.alias("raw_departure_datetime"),
        s.ADMIT_SRC_CD.cast("string").alias("admission_source_code"),
        s.admit_src_desc.alias("admission_source_display"),
        s.DISCH_TO_LOCTN_CD.cast("string").alias("discharge_destination_code"),
        s.disch_loctn_desc.alias("discharge_destination_display"),
        s.MED_SERVICE_CD.cast("string").alias("responsible_service_code"),
        s.med_service_desc.alias("responsible_service_display"),
        s.SPECIALTY_UNIT_CD.cast("string").alias("specialty_code"),
        s.specialty_unit_desc.alias("specialty_display"),
        F.when(s.LOC_NURSE_UNIT_CD.isNotNull(),
               stable_id("location:mill:nurse_unit", s.LOC_NURSE_UNIT_CD)).alias("current_location_id"),
        F.when(s.ORGANIZATION_ID.isNotNull(),
               stable_id("organization:mill", s.ORGANIZATION_ID)).alias("organization_id"),
        F.when(s.SERVICE_PROVIDER_ORG_ID.isNotNull(),
               stable_id("organization:mill", s.SERVICE_PROVIDER_ORG_ID))
         .alias("service_provider_organization_id"),
        s.REASON_FOR_VISIT.alias("reason_for_visit"),
        s.ATTENDANCE_EVIDENCE.alias("attendance_evidence"),
        s.ATTENDANCE_WITNESS_COUNT.cast("int").alias("attendance_witness_count"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(ended, s.END_EFFECTIVE_DT_TM).alias("record_status_effective_to"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDT_DT_TM.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        s.ENCNTR_ID.alias("_source_encounter_id"),
        s.REG_PRSNL_ID.alias("_registration_practitioner_id"),
        s.DISCH_PRSNL_ID.alias("_discharge_practitioner_id"),
        s.CREATE_PRSNL_ID.alias("_creator_practitioner_id"),
        s.REG_DT_TM.alias("_registration_role_start"),
        s.DISCH_DT_TM.alias("_discharge_role_start"),
        s.CREATE_DT_TM.alias("_creator_role_start"),
    )

In [0]:
# ==== Encounter spine, location stays, relationships and care participation ====

ENCOUNTER_PUBLIC_COLUMNS = [
    "encounter_id", "subject_key", "subject_id_system", "person_id",
    "parent_encounter_id", "parentage_status", "encounter_level",
    "class_code", "class_display", "type_code", "type_display",
    "type_class_code", "type_class_display", "status_code", "status_display",
    "period_start", "period_end", "arrival_method", "arrival_confidence",
    "departure_method", "departure_confidence", "length_of_stay_minutes",
    "scheduled_start", "scheduled_end", "registration_datetime",
    "inpatient_admit_datetime", "discharge_datetime", "workflow_complete_datetime",
    "raw_arrival_datetime", "raw_departure_datetime",
    "admission_source_code", "admission_source_display",
    "discharge_destination_code", "discharge_destination_display",
    "responsible_service_code", "responsible_service_display",
    "specialty_code", "specialty_display", "current_location_id",
    "organization_id", "service_provider_organization_id", "reason_for_visit",
    "attendance_evidence", "attendance_witness_count",
    "confidentiality_code", "vip_ind", "record_status",
    "record_status_effective_from", "record_status_effective_to",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

ENCOUNTER_COLUMN_COMMENTS = {
    "encounter_id": "Deterministic encounter primary key.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "parent_encounter_id": "Governed containment parent when supplied by source evidence.",
    "parentage_status": "Provenance state for encounter containment.",
    "encounter_level": "Rules-light source-classified encounter level.",
    "class_code": "Source encounter class code.",
    "class_display": "Source encounter class display.",
    "type_code": "Source encounter type code.",
    "type_display": "Source encounter type display.",
    "type_class_code": "Source encounter type-class code.",
    "type_class_display": "Source encounter type-class display.",
    "status_code": "Native encounter status code.",
    "status_display": "Native encounter status display.",
    "period_start": "Best observed encounter start.",
    "period_end": "Best observed encounter end.",
    "arrival_method": "Method selecting period_start.",
    "arrival_confidence": "Source-derived confidence for period_start.",
    "departure_method": "Method selecting period_end.",
    "departure_confidence": "Source-derived confidence for period_end.",
    "length_of_stay_minutes": "Source-productised encounter duration.",
    "scheduled_start": "Scheduled arrival timestamp.",
    "scheduled_end": "Scheduled departure timestamp.",
    "registration_datetime": "Registration timestamp retained as source evidence.",
    "inpatient_admit_datetime": "Inpatient admission timestamp.",
    "discharge_datetime": "Discharge timestamp.",
    "workflow_complete_datetime": "Administrative workflow completion timestamp.",
    "raw_arrival_datetime": "Raw ARRIVE_DT_TM retained without asserting observability.",
    "raw_departure_datetime": "Raw DEPART_DT_TM retained without asserting clinical meaning.",
    "admission_source_code": "Admission source code.",
    "admission_source_display": "Admission source display.",
    "discharge_destination_code": "Discharge destination code.",
    "discharge_destination_display": "Discharge destination display.",
    "responsible_service_code": "Responsible service code.",
    "responsible_service_display": "Responsible service display.",
    "specialty_code": "Source specialty-unit code.",
    "specialty_display": "Source specialty-unit display.",
    "current_location_id": "Current source nurse-unit location reference.",
    "organization_id": "Source encounter organization reference.",
    "service_provider_organization_id": "Source service-provider organization reference.",
    "reason_for_visit": "Verbatim source reason for visit.",
    "attendance_evidence": "Productised attendance evidence classification.",
    "attendance_witness_count": "Number of attendance witnesses in the source product.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source active-status timestamp.",
    "record_status_effective_to": "Source effective end when superseded.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_spine.encounter"),
    comment="One source encounter with evidence-based timestamps. Parent remains null where bronze supplies no governed containment key.",
    cluster_by=["person_id", "period_start"],
    refresh_policy="incremental",
    column_comments=ENCOUNTER_COLUMN_COMMENTS,
)
def encounter():
    return _encounter_canonical().select(*ENCOUNTER_PUBLIC_COLUMNS)

In [0]:
SRC_LOCATION_HISTORY = "4_prod.bronze.map_patient_journey"

def _location_stay_canonical():
    s = read_source(SRC_LOCATION_HISTORY)
    staged = s.withColumn(
        "_stop_discriminator",
        F.when(s.LOCATION_STOP_SEQUENCE.isNotNull(),
               F.concat(F.lit("stop:"), s.LOCATION_STOP_SEQUENCE.cast("string")))
         .otherwise(F.concat(F.lit("event:"), s.ENCNTR_LOC_HIST_ID.cast("string"))),
    )
    grouped = (
        staged.groupBy("ENCNTR_ID", "_stop_discriminator")
        .agg(
            F.max("PERSON_ID").alias("PERSON_ID"),
            F.min("ENCNTR_LOC_HIST_ID").alias("source_row_id_min"),
            F.max("ENCNTR_LOC_HIST_ID").alias("source_row_id_max"),
            F.count(F.lit(1)).alias("source_history_row_count"),
            F.min("HISTORY_EVENT_SEQUENCE").alias("first_history_event_sequence"),
            F.max("HISTORY_EVENT_SEQUENCE").alias("last_history_event_sequence"),
            F.min("LOCATION_STOP_START_DT_TM").alias("stay_start"),
            F.max("LOCATION_STOP_END_DT_TM").alias("stay_end"),
            F.max("LOC_NURSE_UNIT_CD").alias("nurse_unit_cd"),
            F.max("NURSE_UNIT_DESC").alias("nurse_unit_display"),
            F.max("LOC_BUILDING_CD").alias("building_cd"),
            F.max("BUILDING_DESC").alias("building_display"),
            F.max("LOC_FACILITY_CD").alias("facility_cd"),
            F.max("FACILITY_DESC").alias("facility_display"),
            F.max("LOC_ROOM_CD").alias("room_cd"),
            F.max("ROOM_DESC").alias("room_display"),
            F.max("LOC_BED_CD").alias("bed_cd"),
            F.max("BED_DESC").alias("bed_display"),
            F.max("MED_SERVICE_CD").alias("service_code"),
            F.max("MED_SERVICE_DESC").alias("service_display"),
            F.max("TRANSFER_REASON_CD").alias("transfer_reason_code"),
            F.max("TRANSFER_REASON_DESC").alias("transfer_reason_display"),
            F.max("CONFID_LEVEL_CD").alias("confidentiality_code"),
            F.max("VIP_CD").alias("vip_code"),
            F.max("ADC_UPDT").alias("loaded_at"),
        )
    )
    source_row_id = F.concat_ws(
        ":",
        F.col("ENCNTR_ID").cast("string"),
        F.col("_stop_discriminator"),
        F.col("source_row_id_min").cast("string"),
        F.col("source_row_id_max").cast("string"),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("PERSON_ID"))],
        SRC_LOCATION_HISTORY,
        source_row_id,
    )
    ended = F.coalesce(
        F.col("stay_end") < F.lit("2100-01-01").cast("timestamp"),
        F.lit(False),
    )
    projected = grouped.select(
        stable_id("location_stay:mill", F.col("ENCNTR_ID"), F.col("_stop_discriminator"))
          .alias("location_stay_id"),
        stable_id("encounter:mill", F.col("ENCNTR_ID")).alias("encounter_id"),
        skey.alias("subject_key"),
        ssys.alias("subject_id_system"),
        F.col("PERSON_ID").cast("string").alias("person_id"),
        F.when(F.col("nurse_unit_cd").isNotNull(),
               stable_id("location:mill:nurse_unit", F.col("nurse_unit_cd")))
         .when(F.col("building_cd").isNotNull(),
               stable_id("location:mill:building", F.col("building_cd")))
         .when(F.col("facility_cd").isNotNull(),
               stable_id("location:mill:facility", F.col("facility_cd")))
         .alias("_candidate_location_id"),
        F.col("stay_start").alias("period_start"),
        F.col("stay_end").alias("period_end"),
        F.col("nurse_unit_cd").cast("string").alias("nurse_unit_code"),
        F.col("nurse_unit_display"),
        F.col("building_cd").cast("string").alias("building_code"),
        F.col("building_display"),
        F.col("facility_cd").cast("string").alias("facility_code"),
        F.col("facility_display"),
        F.col("room_cd").cast("string").alias("room_code"),
        F.col("room_display"),
        F.col("bed_cd").cast("string").alias("bed_code"),
        F.col("bed_display"),
        F.col("service_code").cast("string").alias("service_code"),
        F.col("service_display"),
        F.col("transfer_reason_code").cast("string").alias("transfer_reason_code"),
        F.col("transfer_reason_display"),
        F.col("source_history_row_count").cast("long").alias("source_history_row_count"),
        F.col("first_history_event_sequence").cast("int").alias("first_history_event_sequence"),
        F.col("last_history_event_sequence").cast("int").alias("last_history_event_sequence"),
        F.col("confidentiality_code").cast("string").alias("confidentiality_code"),
        (F.coalesce(F.col("vip_code"), F.lit(0)) != 0).alias("vip_ind"),
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.when(ended, F.col("stay_end")).alias("record_status_effective_to"),
        F.lit(SRC_LOCATION_HISTORY).alias("source_table"),
        source_row_id.alias("source_row_id"),
        F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("loaded_at"),
    )
    valid_locations = _location_projection().select(
        F.col("location_id").alias("_valid_location_id")
    )
    joined = projected.join(
        valid_locations,
        projected._candidate_location_id == valid_locations._valid_location_id,
        "left",
    )
    return joined.select(*[
        F.col("_valid_location_id").alias("location_id")
        if c == "_candidate_location_id" else F.col(c)
        for c in projected.columns
    ])

LOCATION_STAY_COLUMN_COMMENTS = {
    "location_stay_id": "Deterministic location-stop primary key.",
    "encounter_id": "Encounter reference for this stop.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "location_id": "Governed current location-dimension FK when resolvable.",
    "period_start": "Location-stop start.",
    "period_end": "Location-stop end.",
    "nurse_unit_code": "Historical nurse-unit code.",
    "nurse_unit_display": "Historical nurse-unit display.",
    "building_code": "Historical building code.",
    "building_display": "Historical building display.",
    "facility_code": "Historical facility code.",
    "facility_display": "Historical facility display.",
    "room_code": "Historical room code.",
    "room_display": "Historical room display.",
    "bed_code": "Historical bed code.",
    "bed_display": "Historical bed display.",
    "service_code": "Service code during the stop.",
    "service_display": "Service display during the stop.",
    "transfer_reason_code": "Source transfer-reason code.",
    "transfer_reason_display": "Source transfer-reason display.",
    "source_history_row_count": "Number of history rows grouped into the stop.",
    "first_history_event_sequence": "First source history sequence in the stop.",
    "last_history_event_sequence": "Last source history sequence in the stop.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_to": "Stop end used as lifecycle end when present.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Deterministic grouped source-row identity.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Latest bronze load timestamp in the grouped stop.",
}

@materialized_view(
    name=_n("journey_spine.location_stay"),
    comment="One physical location stop per encounter, anchored on the source encounter and preserving bed/room detail.",
    cluster_by=["person_id", "period_start"],
    refresh_policy="incremental",
    column_comments=LOCATION_STAY_COLUMN_COMMENTS,
)
def location_stay():
    return _location_stay_canonical()

In [0]:
def _care_participation_projection(df, practitioner_col, role, start_col):
    return (
        df.where(F.col(practitioner_col).isNotNull())
          .select(
              stable_id(
                  "care_participation:mill",
                  F.col("_source_encounter_id"), F.col(practitioner_col), F.lit(role),
              ).alias("care_participation_id"),
              "subject_key", "subject_id_system", "person_id",
              F.col("encounter_id"),
              F.lit(None).cast("string").alias("journey_id"),
              F.lit(None).cast("string").alias("service_id"),
              stable_id("practitioner:mill", F.col(practitioner_col)).alias("practitioner_id"),
              F.lit(role).alias("role"),
              F.col(start_col).alias("valid_from"),
              F.lit(None).cast("timestamp").alias("valid_to"),
              F.lit("source_encounter_personnel_field").alias("construction_rule"),
              F.lit("0.4.0").alias("construction_version"),
              F.lit(SRC_ENCOUNTER).alias("source_table"),
              F.col("_source_encounter_id").cast("string").alias("source_row_id"),
              "load_batch_id", "loaded_at",
          )
    )

CARE_PARTICIPATION_COLUMN_COMMENTS = {
    "care_participation_id": "Deterministic participation primary key.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "encounter_id": "Encounter context.",
    "journey_id": "Journey context when available.",
    "service_id": "Service-registration context when available.",
    "practitioner_id": "Practitioner reference.",
    "role": "Source-field-derived participation role.",
    "valid_from": "Participation validity start.",
    "valid_to": "Participation validity end.",
    "construction_rule": "Governed derivation rule.",
    "construction_version": "Governed derivation-rule version.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Source encounter row identity.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_spine.care_participation"),
    comment="Encounter-scoped practitioner participation from registration, creation, and discharge source fields.",
    refresh_policy="incremental",
    column_comments=CARE_PARTICIPATION_COLUMN_COMMENTS,
)
def care_participation():
    e = _encounter_canonical()
    return (
        _care_participation_projection(e, "_registration_practitioner_id", "registrar", "_registration_role_start")
        .unionByName(
            _care_participation_projection(e, "_creator_practitioner_id", "recorder", "_creator_role_start")
        )
        .unionByName(
            _care_participation_projection(e, "_discharge_practitioner_id", "discharger", "_discharge_role_start")
        )
    )

In [0]:
# ==== Terminology and mapping plane ====

SRC_LUNA_CODE_MAPPING = "3_lookup.omop.luna_code_map"
SRC_SEED_CODE_MAPPING = "3_lookup.omop.seed_code_map"
SRC_IWEB_VALUE_MAP = "3_lookup.omop.iweb_value_to_concept"
SRC_MEDICONNECT_TYPE_MAP = "3_lookup.omop.mediconnect_device_type_map"
SRC_DIAGNOSIS_MAPPING = "4_prod.bronze.map_diagnosis"
SRC_CODED_EVENT_MAPPING = "4_prod.bronze.map_coded_events_omop_bridge"
SRC_PATHOLOGY_MAPPING = "4_prod.bronze.map_pathology"
SRC_PATHOLOGY_CROSS_ARM_TEST_MAP = "3_lookup.omop.pathology_cross_arm_test_map"
SRC_PATHOLOGY_HGNC_GENE = "3_lookup.omop.pathology_hgnc_gene"
SRC_PATHOLOGY_HGNC_ALIAS = "3_lookup.omop.pathology_hgnc_alias"
SRC_PATHOLOGY_ANTIMICROBIAL_MAP = "3_lookup.omop.pathology_antimicrobial_map"
SRC_PATHOLOGY_PANEL_DEFINITION = "3_lookup.omop.pathology_panel_definition"
SRC_PATHOLOGY_PANEL_GENE = "3_lookup.omop.pathology_panel_gene"
SRC_CANCER_DRUG_MAP = "3_lookup.omop.cancer_treatment_term_map"
SRC_BLOODTRACK_BLOOD_GROUP_MAP = "3_lookup.omop.bloodtrack_blood_group_map"
SRC_BLOODTRACK_PRODUCT_GROUP_MAP = "3_lookup.omop.bloodtrack_product_group_map"

def _concept_map_projection(
    df,
    source_system,
    source_code,
    source_display,
    target_system,
    target_code,
    target_display,
    target_domain,
    map_source,
    source_table,
    mapping_rule_id,
    mapping_rank,
    valid_from,
    valid_to,
    loaded_at,
    *,
    review_status_col=None,
):
    staged = df.select(
        source_system.cast("string").alias("source_coding_system"),
        source_code.cast("string").alias("source_code"),
        source_display.cast("string").alias("source_display"),
        target_system.cast("string").alias("target_coding_system"),
        target_code.cast("string").alias("target_code"),
        target_display.cast("string").alias("target_display"),
        target_domain.cast("string").alias("target_domain"),
        map_source.cast("string").alias("map_source"),
        source_table.cast("string").alias("source_table"),
        F.lit(None).cast("string").alias("map_version"),
        (F.lit("source_carried") if review_status_col is None else review_status_col)
        .cast("string").alias("review_status"),
        mapping_rule_id.cast("string").alias("mapping_rule_id"),
        mapping_rank.cast("int").alias("mapping_rank"),
        valid_from.cast("date").alias("valid_from"),
        valid_to.cast("date").alias("valid_to"),
        loaded_at.cast("timestamp").alias("loaded_at"),
    ).where(
        F.col("source_coding_system").isNotNull()
        & F.col("source_code").isNotNull()
        & F.col("target_coding_system").isNotNull()
        & F.col("target_code").isNotNull()
    )
    grouped = (
        staged.groupBy(
            "source_coding_system", "source_code",
            "target_coding_system", "target_code", "target_domain",
            "map_source", "source_table", "map_version", "mapping_rule_id", "mapping_rank",
            "review_status", "valid_from", "valid_to",
        )
        .agg(
            F.max("source_display").alias("source_display"),
            F.max("target_display").alias("target_display"),
            F.count(F.lit(1)).cast("long").alias("source_row_count"),
            F.max("loaded_at").alias("loaded_at"),
        )
    )
    return grouped.select(
        stable_id(
            "terminology-map",
            F.col("source_coding_system"), F.col("source_code"),
            F.col("target_coding_system"), F.col("target_code"),
            F.col("target_domain"), F.col("map_source"),
            F.col("map_version"), F.col("mapping_rule_id"), F.col("mapping_rank"),
            F.col("valid_from"), F.col("valid_to"),
        ).alias("concept_map_id"),
        "source_coding_system", "source_code", "source_display",
        "target_coding_system", "target_code", "target_display", "target_domain",
        F.lit(None).cast("string").alias("equivalence"),
        "map_source", "map_version", "mapping_rule_id", "mapping_rank",
        "review_status",
        "valid_from", "valid_to",
        "source_table",
        "source_row_count", "loaded_at",
        )

def _union_frames(frames):
    result = frames[0]
    for frame in frames[1:]:
        result = result.unionByName(frame)
    return result

def _concept_map_projection_all():
    d = read_source(SRC_DIAGNOSIS_MAPPING)
    diagnosis_source_system = F.coalesce(
        d.source_vocabulary_desc,
        d.CONCEPT_CKI_SOURCE,
        F.lit("urn:cerner:nomenclature"),
    )
    diagnosis_source_code = F.coalesce(
        d.SOURCE_IDENTIFIER,
        d.CONCEPT_CKI_IDENTIFIER,
        d.NOMENCLATURE_ID.cast("string"),
    )
    diagnosis_source_display = F.coalesce(
        d.SOURCE_STRING, d.DIAGNOSIS_DISPLAY, d.DIAGNOSIS_TEXT
    )
    no_date = F.lit(None).cast("date")
    no_rank = F.lit(None).cast("int")
    mappings = [
        _concept_map_projection(
            d, diagnosis_source_system, diagnosis_source_code, diagnosis_source_display,
            F.lit("http://snomed.info/sct"), d.SNOMED_CODE, d.SNOMED_TERM,
            F.lit("condition"), F.lit("bronze.map_diagnosis:snomed"),
            F.lit(SRC_DIAGNOSIS_MAPPING),
            F.lit("diagnosis_snomed"), d.SNOMED_MATCH_NUMBER, no_date, no_date, d.ADC_UPDT,
        ),
        _concept_map_projection(
            d, diagnosis_source_system, diagnosis_source_code, diagnosis_source_display,
            F.lit("http://hl7.org/fhir/sid/icd-10"), d.ICD10_CODE, d.ICD10_TERM,
            F.lit("condition"), F.lit("bronze.map_diagnosis:icd10"),
            F.lit(SRC_DIAGNOSIS_MAPPING),
            F.lit("diagnosis_icd10"), d.ICD10_MATCH_NUMBER, no_date, no_date, d.ADC_UPDT,
        ),
        _concept_map_projection(
            d, diagnosis_source_system, diagnosis_source_code, diagnosis_source_display,
            F.lit("urn:omop:concept_id"), d.OMOP_CONCEPT_ID, d.OMOP_CONCEPT_NAME,
            F.coalesce(F.lower(d.OMOP_CONCEPT_DOMAIN), F.lit("condition")),
            F.lit("bronze.map_diagnosis:omop"), F.lit(SRC_DIAGNOSIS_MAPPING),
            F.lit("diagnosis_omop"),
            d.OMOP_MATCH_NUMBER, no_date, no_date, d.ADC_UPDT,
        ),
    ]

    b = read_source(SRC_CODED_EVENT_MAPPING)
    mappings.append(
        _concept_map_projection(
            b,
            F.concat(F.lit("urn:cerner:coded-event:"),
                     F.lower(F.coalesce(b.SOURCE_FIELD, F.lit("unknown")))),
            b.SOURCE_VALUE,
            b.MATCHED_SOURCE_VARIANT,
            F.lit("urn:omop:concept_id"),
            b.OMOP_CONCEPT_ID,
            b.OMOP_CONCEPT_NAME,
            F.coalesce(F.lower(b.OMOP_CONCEPT_DOMAIN), F.lower(b.OMOP_TABLE), F.lit("unknown")),
            F.lit("bronze.map_coded_events_omop_bridge:omop"),
            F.lit(SRC_CODED_EVENT_MAPPING),
            b.MAPPING_RULE_ID,
            b.MAPPING_RANK,
            b.OMOP_VALID_START_DATE,
            b.OMOP_VALID_END_DATE,
            F.coalesce(b.MAPPING_ADC_UPDT, b.PIPELINE_UPDT_DT_TM),
        )
    )

    p = read_source(SRC_PATHOLOGY_MAPPING)
    pathology_source_system = F.coalesce(
        p.code_system, F.lit("urn:barts:pathology:event-code")
    )
    pathology_source_code = F.coalesce(p.code, p.EVENT_CD.cast("string"))
    pathology_source_display = F.coalesce(p.description, p.EVENT_CD_DISPLAY)
    result_source_system = F.concat(
        F.lit("urn:barts:pathology:result-value:"), pathology_source_code
    )
    pathology_loaded_at = F.coalesce(p.mapping_updated_at, p.ADC_UPDT, p.source_adc_updt)
    mappings.extend([
        _concept_map_projection(
            p, pathology_source_system, pathology_source_code, pathology_source_display,
            F.lit("http://snomed.info/sct"), p.test_snomed_code, p.description,
            F.lit("measurement"), F.lit("bronze.map_pathology:test-snomed"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.test_confidence_tier, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, pathology_source_system, pathology_source_code, pathology_source_display,
            F.lit("http://loinc.org"), p.test_loinc_code, p.description,
            F.lit("measurement"), F.lit("bronze.map_pathology:test-loinc"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.test_confidence_tier, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, pathology_source_system, pathology_source_code, pathology_source_display,
            F.lit("urn:omop:concept_id"), p.test_omop_concept_id, p.measurement_concept_name,
            F.lit("measurement"), F.lit("bronze.map_pathology:test-omop"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.test_confidence_tier, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, result_source_system, p.value_source_value, p.value_source_value,
            F.lit("http://snomed.info/sct"), p.result_snomed_code, p.result_concept_name,
            F.lit("measurement_value"), F.lit("bronze.map_pathology:result-snomed"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.result_mapping_match_type, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, result_source_system, p.value_source_value, p.value_source_value,
            F.lit("http://loinc.org"), p.result_loinc_code, p.result_concept_name,
            F.lit("measurement_value"), F.lit("bronze.map_pathology:result-loinc"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.result_mapping_match_type, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, result_source_system, p.value_source_value, p.value_source_value,
            F.lit("urn:omop:concept_id"), p.result_omop_concept_id, p.result_concept_name,
            F.lit("measurement_value"), F.lit("bronze.map_pathology:result-omop"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.result_mapping_match_type, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, F.lit("urn:barts:pathology:unit"), p.unit_source_value, p.unit_source_value,
            F.lit("http://unitsofmeasure.org"), p.ucum_code, p.ucum_code,
            F.lit("unit"), F.lit("bronze.map_pathology:unit-ucum"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.unit_mapping_match_type, no_rank, no_date, no_date, pathology_loaded_at,
        ),
        _concept_map_projection(
            p, F.lit("urn:barts:pathology:unit"), p.unit_source_value, p.unit_source_value,
            F.lit("urn:omop:concept_id"), p.unit_concept_id, p.ucum_code,
            F.lit("unit"), F.lit("bronze.map_pathology:unit-omop"),
            F.lit(SRC_PATHOLOGY_MAPPING),
            p.unit_mapping_match_type, no_rank, no_date, no_date, pathology_loaded_at,
        ),
    ])
    xa = read_source(SRC_PATHOLOGY_CROSS_ARM_TEST_MAP)
    mappings.append(
        _concept_map_projection(
            xa, F.lit("urn:barts:pathology:wkg-tlc"), xa.wkg_code, xa.wkg_code,
            F.lit("urn:cerner:order-mnemonic"), xa.order_mnemonic, xa.order_mnemonic,
            F.lit("measurement"), F.lit("lookup.pathology_cross_arm_test_map"),
            F.lit(SRC_PATHOLOGY_CROSS_ARM_TEST_MAP),
            F.concat(F.lit("tfc:"), F.coalesce(xa.tfc_code.cast("string"), F.lit("~"))),
            no_rank, xa.valid_from, xa.valid_to, xa.ADC_UPDT,
            review_status_col=xa.status,
        )
    )
    hg = read_source(SRC_PATHOLOGY_HGNC_GENE)
    mappings.append(
        _concept_map_projection(
            hg, F.lit("urn:barts:pathology:gene-symbol"), hg.approved_symbol,
            hg.approved_name, F.lit("urn:hgnc"), hg.hgnc_id, hg.approved_symbol,
            F.lit("gene"), F.lit("lookup.pathology_hgnc_gene"),
            F.lit(SRC_PATHOLOGY_HGNC_GENE),
            hg.reference_release, no_rank, no_date, no_date, hg.ADC_UPDT,
        )
    )
    ha = read_source(SRC_PATHOLOGY_HGNC_ALIAS)
    mappings.append(
        _concept_map_projection(
            ha, F.lit("urn:barts:pathology:gene-symbol"), ha.alias_symbol,
            ha.alias_symbol, F.lit("urn:hgnc"), ha.hgnc_id, ha.approved_symbol,
            F.lit("gene"), F.lit("lookup.pathology_hgnc_alias"),
            F.lit(SRC_PATHOLOGY_HGNC_ALIAS),
            ha.alias_type, no_rank, no_date, no_date, ha.ADC_UPDT,
            review_status_col=F.when(F.coalesce(ha.ambiguous_ind, F.lit(False)),
                                     F.lit("ambiguous")).otherwise(F.lit("source_carried")),
        )
    )
    am = read_source(SRC_PATHOLOGY_ANTIMICROBIAL_MAP)
    mappings.append(
        _concept_map_projection(
            am, am.code_system, am.code, am.antimicrobial_text,
            F.lit("urn:omop:concept_id"), am.antimicrobial_omop_concept_id,
            am.antimicrobial_text, F.lit("drug"),
            F.lit("lookup.pathology_antimicrobial_map"),
            F.lit(SRC_PATHOLOGY_ANTIMICROBIAL_MAP),
            am.method, no_rank, no_date, no_date, am.ADC_UPDT,
            review_status_col=am.status,
        )
    )
    pdef = read_source(SRC_PATHOLOGY_PANEL_DEFINITION)
    mappings.append(
        _concept_map_projection(
            pdef, F.lit("urn:barts:pathology:panel-code"),
            F.concat_ws("@", pdef.panel_code, pdef.panel_version), pdef.panel_name,
            F.lit("urn:barts:pathology:assay-code"), pdef.source_assay_code,
            pdef.panel_name, F.lit("measurement"),
            F.lit("lookup.pathology_panel_definition"),
            F.lit(SRC_PATHOLOGY_PANEL_DEFINITION),
            pdef.analysis_context, no_rank, pdef.effective_from, pdef.effective_to, pdef.ADC_UPDT,
        )
    )
    pgn = read_source(SRC_PATHOLOGY_PANEL_GENE)
    mappings.append(
        _concept_map_projection(
            pgn, F.lit("urn:barts:pathology:panel-code"),
            F.concat_ws("@", pgn.panel_code, pgn.panel_version), pgn.panel_code,
            F.lit("urn:hgnc"), pgn.hgnc_id, pgn.gene_symbol,
            F.lit("gene"), F.lit("lookup.pathology_panel_gene"),
            F.lit(SRC_PATHOLOGY_PANEL_GENE),
            pgn.test_scope, no_rank, no_date, no_date, pgn.ADC_UPDT,
        )
    )
    l = read_source(SRC_LUNA_CODE_MAPPING)
    mappings.append(
        _concept_map_projection(
            l, l.source_coding_system, l.source_code, l.source_display,
            l.target_coding_system, l.target_code, l.target_display, l.target_domain,
            F.concat(F.lit("lookup.luna_code_map:"), l.mapping_rule_id),
            F.lit(SRC_LUNA_CODE_MAPPING),
            l.mapping_rule_id, l.mapping_rank, no_date, no_date, l.updated_at,
        )
    )
    sm = read_source(SRC_SEED_CODE_MAPPING)
    mappings.append(
        _concept_map_projection(
            sm, sm.source_coding_system, sm.source_code, sm.source_display,
            sm.target_coding_system, sm.target_code, sm.target_display, sm.target_domain,
            F.concat(F.lit("lookup.seed_code_map:"), sm.mapping_rule_id),
            F.lit(SRC_SEED_CODE_MAPPING),
            sm.mapping_rule_id, sm.mapping_rank, no_date, no_date, sm.updated_at,
            review_status_col=sm.review_status,
        )
    )
    iw = read_source(SRC_IWEB_VALUE_MAP)
    mappings.append(
        _concept_map_projection(
            iw,
            F.concat(F.lit("urn:iweb:"), F.lower(iw.SOURCE_TABLE), F.lit(":"),
                     F.lower(iw.FIELD_NAME)),
            iw.CODE, iw.LABEL,
            _iweb_target_system(iw.TARGET_VOCABULARY),
            F.coalesce(iw.TARGET_CODE, iw.TARGET_CONCEPT_ID.cast("string")),
            iw.TARGET_CONCEPT_NAME, F.lit("registry"),
            F.concat(F.lit("lookup.iweb_value_to_concept:"), iw.MAPPING_STATUS),
            F.lit(SRC_IWEB_VALUE_MAP), iw.MAPPING_METHOD, no_rank,
            no_date, no_date, iw.CURATED_AT,
            review_status_col=iw.MAPPING_STATUS,
        )
    )
    ct = read_source(SRC_CANCER_DRUG_MAP)
    mappings.append(
        _concept_map_projection(
            ct, F.lit("urn:barts:sact:drug-token"), ct.drug_token, ct.drug_token,
            F.lit("urn:omop:concept_id"), ct.drug_concept_id, ct.drug_concept_name,
            F.lit("drug"),
            F.concat(F.lit("lookup.cancer_treatment_term_map:"), ct.mapping_status),
            F.lit(SRC_CANCER_DRUG_MAP), ct.mapping_status, no_rank,
            no_date, no_date, ct.ADC_UPDT,
            review_status_col=ct.mapping_status,
        )
    )
    bg = read_source(SRC_BLOODTRACK_BLOOD_GROUP_MAP)
    mappings.append(
        _concept_map_projection(
            bg, F.lit("urn:bloodtrack:blood-group"), bg.BLOOD_GROUP_SOURCE_VALUE,
            bg.BLOOD_GROUP_SOURCE_VALUE, F.lit("http://snomed.info/sct"),
            bg.SNOMED_CONCEPT_ID, bg.SNOMED_CONCEPT_NAME, F.lit("observation"),
            F.concat(F.lit("lookup.bloodtrack_blood_group_map:"), bg.MAPPING_STATUS),
            F.lit(SRC_BLOODTRACK_BLOOD_GROUP_MAP), bg.MAPPING_METHOD, no_rank,
            no_date, no_date, bg.CURATED_AT,
            review_status_col=bg.MAPPING_STATUS,
        )
    )
    pg = read_source(SRC_BLOODTRACK_PRODUCT_GROUP_MAP)
    mappings.extend([
        _concept_map_projection(
            pg, F.lit("urn:bloodtrack:product-group"), pg.BLOOD_PRODUCT_GROUP,
            pg.BLOOD_PRODUCT_GROUP, F.lit("http://snomed.info/sct"),
            pg.PRODUCT_CONCEPT_ID, pg.PRODUCT_CONCEPT_NAME, F.lit("device"),
            F.concat(F.lit("lookup.bloodtrack_product_group_map:"), pg.MAPPING_STATUS),
            F.lit(SRC_BLOODTRACK_PRODUCT_GROUP_MAP), pg.MAPPING_METHOD, no_rank,
            no_date, no_date, pg.CURATED_AT,
            review_status_col=pg.MAPPING_STATUS,
        ),
        _concept_map_projection(
            pg, F.lit("urn:bloodtrack:product-group"), pg.BLOOD_PRODUCT_GROUP,
            pg.BLOOD_PRODUCT_GROUP, F.lit("http://snomed.info/sct"),
            pg.PROC_CONCEPT_ID, pg.PROC_CONCEPT_NAME, F.lit("procedure"),
            F.concat(F.lit("lookup.bloodtrack_product_group_map:"), pg.MAPPING_STATUS),
            F.lit(SRC_BLOODTRACK_PRODUCT_GROUP_MAP), pg.MAPPING_METHOD, no_rank,
            no_date, no_date, pg.CURATED_AT,
            review_status_col=pg.MAPPING_STATUS,
        ),
    ])
    md = read_source(SRC_MEDICONNECT_TYPE_MAP)
    mappings.append(
        _concept_map_projection(
            md, F.lit("urn:barts:mediconnect:device-type"), md.DEVICE_TYPE,
            md.SOURCE_LABEL, F.lit("http://snomed.info/sct"),
            md.SNOMED_CONCEPT_ID, md.SNOMED_CONCEPT_NAME, F.lit("device"),
            F.concat(F.lit("lookup.mediconnect_device_type_map:"), md.MAPPING_STATUS),
            F.lit(SRC_MEDICONNECT_TYPE_MAP), md.MAPPING_METHOD, no_rank,
            no_date, no_date, md.CURATED_AT,
            review_status_col=md.MAPPING_STATUS,
        )
    )
    return _union_frames(mappings)

CONCEPT_MAP_COLUMN_COMMENTS = {
    "concept_map_id": "Deterministic versioned mapping primary key.",
    "source_coding_system": "Source code-system namespace.",
    "source_code": "Source code or mapped source value.",
    "source_display": "Source display carried from bronze.",
    "target_coding_system": "Target code-system namespace.",
    "target_code": "Target code or concept identifier.",
    "target_display": "Target display carried from bronze.",
    "target_domain": "Intended downstream semantic domain.",
    "equivalence": "FHIR equivalence when explicitly supplied; null rather than inferred.",
    "map_source": "Governed mapping product and route.",
    "map_version": "Pinned mapping release.",
    "mapping_rule_id": "Source mapping rule or confidence discriminator.",
    "mapping_rank": "Source-supplied candidate rank where available.",
    "review_status": "Review state carried verbatim from the mapping source; source_carried rows are bronze-carried without review",
    "valid_from": "Target mapping validity start where supplied.",
    "valid_to": "Target mapping validity end where supplied.",
    "source_table": "Configured bronze source or explicitly authorized development fixture.",
    "source_row_count": "Number of bronze mapping observations represented by the row.",
    "loaded_at": "Latest bronze mapping load timestamp represented.",
}

@materialized_view(
    name=_n("journey_reference.concept_map"),
    comment="Versioned one-to-many source-to-target mappings observed in governed bronze products; no winner is selected.",
    refresh_policy="incremental",
    column_comments=CONCEPT_MAP_COLUMN_COMMENTS,
)
def concept_map():
    return _concept_map_projection_all()

In [0]:
CONCEPT_REGISTRY_COLUMN_COMMENTS = {
    "concept_registry_id": "Deterministic concept-registry primary key.",
    "coding_system": "Coding-system namespace.",
    "code": "Observed code.",
    "preferred_display": "Deterministically selected observed display.",
    "status": "Registry observation status.",
    "source_use_count": "Source-side event and mapping observation count.",
    "target_use_count": "Target-side mapping observation count.",
    "first_observed_at": "Earliest represented bronze observation timestamp.",
    "last_observed_at": "Latest represented bronze observation timestamp.",
}

@materialized_view(
    name=_n("journey_reference.concept_registry"),
    comment="Observed source and target coding-system/code pairs across the event index and concept map.",
    refresh_policy="incremental",
    column_comments=CONCEPT_REGISTRY_COLUMN_COMMENTS,
)
def concept_registry():
    mappings = spark.read.table(_n("journey_reference.concept_map"))
    events = spark.read.table(_n("journey_events._event_index"))
    event_mappings = spark.read.table(_n("journey_events._event_map"))
    event_codes = events.where(events.source_code.isNotNull()).select(
        F.coalesce(events.source_coding_system, F.lit("urn:unknown")).alias("coding_system"),
        events.source_code.alias("code"),
        events.source_display.alias("display"),
        F.lit(1).cast("long").alias("source_use_count"),
        F.lit(0).cast("long").alias("target_use_count"),
        events.loaded_at.alias("observed_at"),
    )
    map_sources = mappings.select(
        F.col("source_coding_system").alias("coding_system"),
        F.col("source_code").alias("code"),
        F.col("source_display").alias("display"),
        F.col("source_row_count").cast("long").alias("source_use_count"),
        F.lit(0).cast("long").alias("target_use_count"),
        F.col("loaded_at").alias("observed_at"),
    )
    map_targets = mappings.select(
        F.col("target_coding_system").alias("coding_system"),
        F.col("target_code").alias("code"),
        F.col("target_display").alias("display"),
        F.lit(0).cast("long").alias("source_use_count"),
        F.col("source_row_count").cast("long").alias("target_use_count"),
        F.col("loaded_at").alias("observed_at"),
    )
    event_map_targets = event_mappings.select(
        F.col("mapped_coding_system").alias("coding_system"),
        F.col("mapped_code").alias("code"),
        F.col("mapped_display").alias("display"),
        F.lit(0).cast("long").alias("source_use_count"),
        F.lit(1).cast("long").alias("target_use_count"),
        F.col("loaded_at").alias("observed_at"),
    )
    grouped = (
        event_codes.unionByName(map_sources).unionByName(map_targets).unionByName(event_map_targets)
        .groupBy("coding_system", "code")
        .agg(
            F.max("display").alias("preferred_display"),
            F.sum("source_use_count").cast("long").alias("source_use_count"),
            F.sum("target_use_count").cast("long").alias("target_use_count"),
            F.min("observed_at").alias("first_observed_at"),
            F.max("observed_at").alias("last_observed_at"),
        )
    )
    return grouped.select(
        stable_id("terminology-concept", F.col("coding_system"), F.col("code"))
          .alias("concept_registry_id"),
        "coding_system", "code", "preferred_display",
        F.lit("observed").alias("status"),
        "source_use_count", "target_use_count",
        "first_observed_at", "last_observed_at",
        )

In [0]:
SRC_VALUE_SET_RELEASE = "3_lookup.omop.value_set_release"

VALUE_SET_COLUMN_COMMENTS = {
    "value_set_id": "Deterministic value-set membership primary key.",
    "canonical_url": "Canonical value-set URL.",
    "version": "Pinned value-set version.",
    "member_system": "Member coding system.",
    "member_code": "Member code.",
    "member_display": "Member display.",
    "status": "Value-set membership status.",
    "source_table": "Governed bronze value-set source.",
    "source_row_id": "Stable source membership identity.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_reference.value_set"),
    comment="Pinned value-set membership from the governed value_set_release lookup asset; "
            "v3-ServiceDeliveryLocationRoleType is a recorded v1 exclusion.",
    refresh_policy="incremental",
    column_comments=VALUE_SET_COLUMN_COMMENTS,
)
def value_set():
    r = read_source(SRC_VALUE_SET_RELEASE)
    return r.select(
        stable_id(
            "value-set-member", r.canonical_url, r.value_set_version,
            r.member_system, r.member_code,
        ).alias("value_set_id"),
        r.canonical_url.alias("canonical_url"),
        r.value_set_version.alias("version"),
        r.member_system.alias("member_system"),
        r.member_code.alias("member_code"),
        r.member_display.alias("member_display"),
        r.membership_status.alias("status"),
        F.lit(SRC_VALUE_SET_RELEASE).alias("source_table"),
        F.concat_ws("|", r.canonical_url, r.member_system, r.member_code)
         .alias("source_row_id"),
        r.release_id.alias("load_batch_id"),
        r.updated_at.alias("loaded_at"),
    )

In [0]:
# ==== Typed clinical facts — condition and procedure ====

SRC_PROBLEM_HISTORY = "4_prod.bronze.map_problem_history"

def _problem_revision_grouped_query():
    s = read_source(SRC_PROBLEM_HISTORY)
    current = read_source(SRC_PROBLEM).select("PROBLEM_ID").distinct()
    s = s.join(current, "PROBLEM_ID", "inner")
    revision = F.struct(
        F.coalesce(s.PROBLEM_REVISION_RANK, F.lit(0)).cast("long").alias("sequence"),
        s.PROBLEM_REVISION_RANK.cast("long").alias("revision_rank"),
        s.IS_CURRENT_PROBLEM_REVISION.alias("is_current_revision"),
        s.LIFE_CYCLE_STATUS_CD.cast("string").alias("clinical_status_code"),
        s.life_cycle_status_desc.alias("clinical_status_display"),
        s.LIFE_CYCLE_DT_TM.alias("clinical_status_datetime"),
        s.CONFIRMATION_STATUS_CD.cast("string").alias("verification_status_code"),
        s.confirmation_status_desc.alias("verification_status_display"),
        F.coalesce(s.ANNOTATED_DISPLAY, s.PROBLEM_DISPLAY, s.SOURCE_STRING).alias("display"),
        s.SEVERITY_CD.cast("string").alias("severity_code"),
        s.severity_desc.alias("severity_display"),
        s.ONSET_DT_TM.alias("onset_datetime"),
        s.ASSERTED_DT_TM.alias("asserted_datetime"),
        s.BEG_EFFECTIVE_DT_TM.alias("effective_start"),
        s.END_EFFECTIVE_DT_TM.alias("effective_end"),
        F.coalesce(s.SOURCE_DELETED_IND, F.lit(False)).alias("source_tombstone_ind"),
        s.UPDT_DT_TM.alias("source_update_timestamp"),
    )
    history = (
        s.groupBy("PROBLEM_ID")
        .agg(
            F.to_json(F.sort_array(F.collect_list(revision))).alias("_revision_history_json"),
            F.count(F.lit(1)).cast("long").alias("_revision_history_count"),
            F.max(s.ADC_UPDT).alias("_evidence_loaded_at"),
        )
        .select(
            stable_id("condition:mill:problem", F.col("PROBLEM_ID"))
            .alias("_revision_condition_event_id"),
            F.col("_revision_history_json"),
            F.col("_revision_history_count"),
            F.col("_evidence_loaded_at"),
        )
    )
    problem = (
        _condition_problem_canonical()
        .join(
            history,
            F.col("fact_row_id") == history["_revision_condition_event_id"],
            "left",
        )
        .withColumn(
            "_revision_history_count",
            F.coalesce(F.col("_revision_history_count"), F.lit(0).cast("long")),
        )
        .withColumn(
            "loaded_at", F.greatest(F.col("loaded_at"), F.col("_evidence_loaded_at"))
        )
    )
    diagnosis = (
        _condition_diagnosis_canonical()
        .withColumn("_revision_history_json", F.lit(None).cast("string"))
        .withColumn("_revision_history_count", F.lit(0).cast("long"))
    )
    maternity = (
        _maternity_diagnosis_canonical()
        .withColumn("_revision_history_json", F.lit(None).cast("string"))
        .withColumn("_revision_history_count", F.lit(0).cast("long"))
    )
    return (
        diagnosis.select(*CONDITION_PRIMITIVE_COLUMNS)
        .unionByName(problem.select(*CONDITION_PRIMITIVE_COLUMNS))
        .unionByName(maternity.select(*CONDITION_PRIMITIVE_COLUMNS))
    )

@materialized_view(
    name=_n("journey_clinical._problem_revision_grouped"),
    private=True,
    comment="Internal condition primitive with ordered problem-revision history and deterministic JSON boundaries.",
    refresh_policy="incremental",
)
def _problem_revision_grouped():
    return _problem_revision_grouped_query()

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_condition"),
    comment="Private JSON bridge and Gold cross-rule flags for condition.",
    refresh_policy="incremental",
)
def _qc_condition():
    source = (
        spark.read.table(_n("journey_clinical._problem_revision_grouped"))
        .withColumn("revision_history", F.col("_revision_history_json"))
        .withColumn("revision_history_count", F.col("_revision_history_count"))
    )
    return _cross_qc_primitive(
        source,
        "condition",
        {"condition_code": "_condition_code_json"},
    )

In [0]:
CONDITION_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system",
    "person_id", "identity_status", "encounter_id",
    "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "condition_code",
    "category_code", "category_display", "clinical_status_code", "clinical_status_display",
    "verification_status_code", "verification_status_display",
    "onset_datetime", "abatement_datetime", "body_site_code", "body_site_display",
    "severity_code", "severity_display", "laterality_code", "laterality_display",
    "asserted_datetime", "asserter_practitioner_id", "recorder_practitioner_id",
    "revision_history", "revision_history_count",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

CONDITION_PRIMITIVE_COLUMNS = [
    {
        "condition_code": "_condition_code_json",
        "revision_history": "_revision_history_json",
        "revision_history_count": "_revision_history_count",
    }.get(name, name)
    for name in CONDITION_PUBLIC_COLUMNS
]

In [0]:
CONDITION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide condition identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime": "Primary assertion timestamp.",
    "event_end_datetime": "Source effective end where supplied.",
    "source_coding_system": "Verbatim source coding system.",
    "source_code": "Source condition code",
    "source_display": "Verbatim source condition display.",
    "condition_code": "Source and mapped condition codings.",
    "category_code": "Source condition category code.",
    "category_display": "Source condition category display.",
    "clinical_status_code": "Source clinical-status code.",
    "clinical_status_display": "Source clinical-status display.",
    "verification_status_code": "Source verification-status code.",
    "verification_status_display": "Source verification-status display.",
    "onset_datetime": "Source onset timestamp.",
    "abatement_datetime": "Source abatement or effective-end timestamp.",
    "body_site_code": "Body-site code where supplied.",
    "body_site_display": "Body-site display where supplied.",
    "severity_code": "Source severity code.",
    "severity_display": "Source severity display.",
    "laterality_code": "Source laterality code.",
    "laterality_display": "Source laterality display.",
    "asserted_datetime": "Source assertion timestamp.",
    "asserter_practitioner_id": "Asserting practitioner reference.",
    "recorder_practitioner_id": "Recording or source-status practitioner reference.",
    "revision_history": "Ordered JSON revision history from map_problem_history; tombstoned revisions carry source_tombstone_ind.",
    "revision_history_count": "Number of retained problem revision rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered source feed owning the assertion.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Latest bronze load time across the fact row and its folded evidence rows.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.condition"),
    comment="Diagnosis and problem assertions from registered bronze feeds; no cross-feed deduplication or active-only filtering.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CONDITION_COLUMN_COMMENTS,
)
def condition():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_condition")),
        "condition",
        {"condition_code": "_condition_code_json"},
        CONDITION_PUBLIC_COLUMNS,
    )

In [0]:
SRC_IMPLANT_EVENTS = "4_prod.bronze.map_implant_detail_events"

def _implant_attribute_grouped_query():
    e = read_source(SRC_IMPLANT_EVENTS)
    implants = _implant_procedure_canonical().select(
        "patient_event_id", "_base_clinical_event_id", "_implant_form_event_id"
    )
    joined = e.join(
        implants,
        (e.CLINICAL_EVENT_ID == implants._base_clinical_event_id)
        | (e.PARENT_EVENT_ID == implants._implant_form_event_id),
        "inner",
    )
    attribute = F.struct(
        F.coalesce(e.PERFORMED_DT_TM, e.EVENT_START_DT_TM).cast("long").alias("sequence"),
        e.CLINICAL_EVENT_ID.cast("string").alias("clinical_event_id"),
        e.ATTRIBUTE_NAME.alias("attribute_name"),
        e.ATTRIBUTE_VALUE.alias("attribute_value"),
        e.EVENT_TAG.alias("event_tag"),
        e.EVENT_TITLE_TEXT.alias("event_title"),
        e.RESULT_VAL.alias("result_value"),
        e.VALUE_SOURCE.alias("value_source"),
        e.VALUE_CONFLICT_IND.alias("value_conflict_ind"),
        e.PERFORMED_DT_TM.alias("performed_datetime"),
        e.VALID_FROM_DT_TM.alias("valid_from"),
    )
    return (
        joined.groupBy("patient_event_id")
        .agg(
            F.to_json(F.sort_array(F.collect_list(attribute)))
            .alias("_implant_attribute_json"),
            F.count(F.lit(1)).cast("long").alias("_implant_attribute_count"),
            F.max(e.ADC_UPDT).alias("_evidence_loaded_at"),
        )
        .select(
            F.col("patient_event_id").alias("_implant_attribute_event_id"),
            F.col("_implant_attribute_json"),
            F.col("_implant_attribute_count"),
            F.col("_evidence_loaded_at"),
        )
    )

@materialized_view(
    name=_n("journey_clinical._implant_attribute_grouped"),
    private=True,
    comment="Internal ordered implant-attribute aggregate keyed by implant procedure event.",
    refresh_policy="incremental",
)
def _implant_attribute_grouped():
    return _implant_attribute_grouped_query()

In [0]:
PROCEDURE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system",
    "person_id", "identity_status", "encounter_id",
    "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "procedure_code",
    "status_code", "status_display", "performed_start", "performed_end",
    "body_site_code", "body_site_display", "laterality_code", "laterality_display",
    "performer_practitioner_id", "procedure_location_code", "procedure_location_display",
    "procedure_note", "implant_description", "device_code", "device_display",
    "manufacturer", "serial_number", "batch_number", "udi_di", "udi_standard",
    "quantity", "implant_attribute_history", "implant_attribute_count",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _procedure_canonical_pregate():
    s = read_source(SRC_PROCEDURE)
    event_id = stable_id("procedure:mill:procedure", s.PROCEDURE_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_PROCEDURE, s.PROCEDURE_ID
    )
    raw_source_code = F.coalesce(
        s.SOURCE_IDENTIFIER, s.CONCEPT_CKI_IDENTIFIER, s.NOMENCLATURE_ID.cast("string")
    )
    source_display = F.coalesce(s.SOURCE_STRING, s.PROCEDURE_DISPLAY, s.PROC_FTDESC)
    source_code = _code_or_display(raw_source_code, source_display)
    inactive = F.lower(F.coalesce(s.active_status_desc, F.lit(""))).contains("inactive")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.PROCEDURE_DT_TM_EFFECTIVE, s.PROC_START_DT_TM, s.PROC_DT_TM).alias("event_datetime"),
        s.PROC_END_DT_TM.alias("event_end_datetime"),
        F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                   F.lit("urn:cerner:nomenclature")).alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.coalesce(s.source_vocabulary_desc, s.CONCEPT_CKI_SOURCE,
                                  F.lit("urn:cerner:nomenclature")), source_code, source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), s.SNOMED_CODE, s.SNOMED_TERM, False,
                       "bronze.map_procedure", None),
            coding_obj(F.lit("http://fhir.hl7.org.uk/CodeSystem/OPCS-4"), s.OPCS4_CODE, s.OPCS4_TERM, False,
                       "bronze.map_procedure", None),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_CONCEPT_ID, s.OMOP_CONCEPT_NAME, False,
                       "bronze.map_procedure", None),
        ).alias("_procedure_code_json"),
        F.when(inactive, F.lit("stopped")).otherwise(F.lit("completed")).alias("status_code"),
        s.active_status_desc.alias("status_display"),
        F.coalesce(s.PROC_START_DT_TM, s.PROCEDURE_DT_TM_EFFECTIVE).alias("performed_start"),
        s.PROC_END_DT_TM.alias("performed_end"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        s.LATERALITY_CD.cast("string").alias("laterality_code"), s.laterality_desc.alias("laterality_display"),
        F.lit(None).cast("string").alias("performer_practitioner_id"),
        s.PROC_LOC_CD.cast("string").alias("procedure_location_code"),
        s.proc_location_desc.alias("procedure_location_display"),
        s.PROCEDURE_NOTE.alias("procedure_note"),
        F.lit(None).cast("string").alias("implant_description"),
        F.lit(None).cast("string").alias("device_code"), F.lit(None).cast("string").alias("device_display"),
        F.lit(None).cast("string").alias("manufacturer"), F.lit(None).cast("string").alias("serial_number"),
        F.lit(None).cast("string").alias("batch_number"), F.lit(None).cast("string").alias("udi_di"),
        F.lit(None).cast("string").alias("udi_standard"), F.lit(None).cast("decimal(38,6)").alias("quantity"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVE_STATUS_DT_TM.alias("record_status_effective_from"),
        F.when(inactive, s.ACTIVE_STATUS_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("procedure").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_PROCEDURE).alias("_source_table"),
        s.PROCEDURE_ID.cast("string").alias("_source_row_id"),
        s.SNOMED_CODE.cast("string").alias("_snomed_code"), s.SNOMED_TERM.alias("_snomed_display"),
        s.OPCS4_CODE.cast("string").alias("_opcs4_code"), s.OPCS4_TERM.alias("_opcs4_display"),
        s.OMOP_CONCEPT_ID.cast("string").alias("_omop_code"), s.OMOP_CONCEPT_NAME.alias("_omop_display"),
        F.lit(None).cast("string").alias("_device_snomed_code"),
        F.lit(None).cast("string").alias("_device_snomed_display"),
    )

def _procedure_canonical():
    return _procedure_canonical_pregate().where(_usable_code(F.col("source_code")))

def _implant_procedure_canonical_pregate():
    s = read_source(SRC_IMPLANT_DETAILS)
    source_row_id = F.concat_ws(":", s.EVENT_ID.cast("string"), s.IMPLANT_SEQUENCE.cast("string"))
    event_id = stable_id("procedure:mill:implant", s.EVENT_ID, s.IMPLANT_SEQUENCE)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_IMPLANT_DETAILS, source_row_id
    )
    loaded = F.coalesce(s.SOURCE_MAX_ADC_UPDT, s.ADC_UPDT, s.BASE_EVENT_ADC_UPDT)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.IMPLANT_DT_TM, s.EVENT_START_DT_TM, s.CLINSIG_UPDT_DT_TM).alias("event_datetime"),
        s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:barts:implant:primary-procedure").alias("source_coding_system"),
        s.PRIMARY_PROCEDURE.alias("source_code"), s.PRIMARY_PROCEDURE.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:barts:implant:primary-procedure"),
                       s.PRIMARY_PROCEDURE, s.PRIMARY_PROCEDURE, True)
        ).alias("_procedure_code_json"),
        F.lit("completed").alias("status_code"), F.lit(None).cast("string").alias("status_display"),
        F.coalesce(s.IMPLANT_DT_TM, s.EVENT_START_DT_TM).alias("performed_start"),
        s.EVENT_END_DT_TM.alias("performed_end"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("laterality_code"), s.SIDE_OF_PROCEDURE.alias("laterality_display"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(), stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.lit(None).cast("string").alias("procedure_location_code"),
        F.lit(None).cast("string").alias("procedure_location_display"),
        F.lit(None).cast("string").alias("procedure_note"),
        s.IMPLANT_DESCRIPTION.alias("implant_description"),
        F.coalesce(s.SNOMED_DEVICE_CONCEPT_ID.cast("string"), s.GMDN_CODE.cast("string"))
         .alias("device_code"),
        F.coalesce(s.SNOMED_DEVICE_CONCEPT_NAME, s.GMDN_NAME, s.DEVICE_TYPE).alias("device_display"),
        s.MANUFACTURER.alias("manufacturer"), F.coalesce(s.GS1_SERIAL_NUMBER, s.SERIAL_NUMBER).alias("serial_number"),
        s.GS1_BATCH_NUMBER.alias("batch_number"), s.UDI_DI.alias("udi_di"), s.UDI_STANDARD.alias("udi_standard"),
        s.QUANTITY_NUMERIC.cast("decimal(38,6)").alias("quantity"),
        F.lit("active").alias("record_status"),
        s.VALID_FROM_DT_TM.alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("implant_details").alias("source_feed"),
        F.date_format(loaded, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.CLINSIG_UPDT_DT_TM.alias("source_update_timestamp"), loaded.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_IMPLANT_DETAILS).alias("_source_table"),
        source_row_id.alias("_source_row_id"),
        F.lit(None).cast("string").alias("_snomed_code"), F.lit(None).cast("string").alias("_snomed_display"),
        F.lit(None).cast("string").alias("_opcs4_code"), F.lit(None).cast("string").alias("_opcs4_display"),
        F.lit(None).cast("string").alias("_omop_code"), F.lit(None).cast("string").alias("_omop_display"),
        s.SNOMED_DEVICE_CONCEPT_ID.cast("string").alias("_device_snomed_code"),
        s.SNOMED_DEVICE_CONCEPT_NAME.alias("_device_snomed_display"),
        s.BASE_CLINICAL_EVENT_ID.alias("_base_clinical_event_id"),
        s.IMPLANT_FORM_EVENT_ID.alias("_implant_form_event_id"),
    )

def _implant_procedure_canonical():
    return _implant_procedure_canonical_pregate().where(_usable_code(F.col("source_code")))

def _theatre_procedure_canonical_pregate():
    p = read_source(SRC_THEATRE_CASE_PROCEDURE).alias("p")
    c = read_source(SRC_THEATRE_CASE).alias("c")
    s = p.join(c, p.SURG_CASE_ID == c.SURG_CASE_ID, "left")
    source_row_id = p.SURG_CASE_PROC_ID.cast("string")
    event_id = stable_id("procedure:surginet_case", p.SURG_CASE_PROC_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", p.PERSON_ID)], SRC_THEATRE_CASE_PROCEDURE, source_row_id
    )
    raw_source_code = F.coalesce(p.SURG_PROC_CD.cast("string"), p.PROC_TEXT)
    source_display = F.coalesce(p.SURG_PROC_DESCRIPTION, p.PROC_TEXT)
    source_code = _code_or_display(
        raw_source_code,
        source_display,
        excluded_displays=(".", "N/A", "SEE ADDITIONAL COMMENTS"),
    )
    case_status = F.upper(F.coalesce(c.CASE_STATUS, F.lit("SCHEDULED_ONLY")))
    status = (
        F.when(case_status == "CANCELLED", F.lit("stopped"))
        .when(case_status == "PERFORMED", F.lit("completed"))
        .otherwise(F.lit("preparation"))
    )
    inactive = (
        (F.coalesce(p.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True)) == F.lit(False))
        | (F.coalesce(p.ACTIVE_IND.cast("long"), F.lit(1)) == 0)
    )
    performed_start = F.coalesce(
        p.PROC_START_DT_TM, c.SURG_START_DT_TM, c.FIRST_PERFORMED_MILESTONE_DT_TM,
        c.SCHED_START_DT_TM,
    )
    loaded_at = F.greatest(p.ADC_UPDT, c.ADC_UPDT)
    source_update = F.greatest(p.SOURCE_ADC_UPDT, c.SOURCE_ADC_UPDT)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        p.PERSON_ID.cast("string").alias("person_id"),
        F.when(p.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.when(p.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", p.ENCNTR_ID))
        .alias("encounter_id"),
        performed_start.alias("event_datetime"), p.PROC_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:surginet:procedure").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:cerner:surginet:procedure"), source_code,
                       source_display, True)
        ).alias("_procedure_code_json"),
        status.alias("status_code"), c.CASE_STATUS.alias("status_display"),
        performed_start.alias("performed_start"), p.PROC_END_DT_TM.alias("performed_end"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("laterality_code"),
        F.lit(None).cast("string").alias("laterality_display"),
        F.when(p.PRIMARY_SURGEON_ID.isNotNull(),
               stable_id("practitioner:mill", p.PRIMARY_SURGEON_ID))
        .alias("performer_practitioner_id"),
        c.SURG_OP_LOC_CD.cast("string").alias("procedure_location_code"),
        c.SURG_OP_LOCATION_DESCRIPTION.alias("procedure_location_display"),
        F.concat_ws(
            " | ", p.PROC_TEXT,
            F.when(p.MODIFIER_DESCRIPTIONS.isNotNull(), F.to_json(p.MODIFIER_DESCRIPTIONS)),
        ).alias("procedure_note"),
        F.lit(None).cast("string").alias("implant_description"),
        F.lit(None).cast("string").alias("device_code"),
        F.lit(None).cast("string").alias("device_display"),
        F.lit(None).cast("string").alias("manufacturer"),
        F.lit(None).cast("string").alias("serial_number"),
        F.lit(None).cast("string").alias("batch_number"),
        F.lit(None).cast("string").alias("udi_di"),
        F.lit(None).cast("string").alias("udi_standard"),
        F.lit(None).cast("decimal(38,6)").alias("quantity"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        F.coalesce(p.PROC_START_DT_TM, c.SCHED_START_DT_TM).alias("record_status_effective_from"),
        F.when(inactive, F.coalesce(p.SOURCE_ABSENT_DETECTED_TS, p.ADC_UPDT))
        .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("theatre_case_procedure").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), loaded_at.alias("loaded_at"),
        F.lit("surginet").alias("_source_system"),
        F.lit(SRC_THEATRE_CASE_PROCEDURE).alias("_source_table"),
        source_row_id.alias("_source_row_id"),
        F.lit(None).cast("string").alias("_snomed_code"),
        F.lit(None).cast("string").alias("_snomed_display"),
        F.lit(None).cast("string").alias("_opcs4_code"),
        F.lit(None).cast("string").alias("_opcs4_display"),
        F.lit(None).cast("string").alias("_omop_code"),
        F.lit(None).cast("string").alias("_omop_display"),
        F.lit(None).cast("string").alias("_device_snomed_code"),
        F.lit(None).cast("string").alias("_device_snomed_display"),
        p.SURG_CASE_ID.cast("string").alias("_theatre_case_id"),
    )

def _theatre_procedure_canonical():
    return _theatre_procedure_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_PROCEDURE = "4_prod.bronze.map_procedure"

PROCEDURE_PRIMITIVE_COLUMNS = [
    {
        "procedure_code": "_procedure_code_json",
        "implant_attribute_history": "_implant_attribute_json",
        "implant_attribute_count": "_implant_attribute_count",
    }.get(name, name)
    for name in PROCEDURE_PUBLIC_COLUMNS
]

def _procedure_primitive_query():
    attributes = spark.read.table(_n("journey_clinical._implant_attribute_grouped")).alias("a")
    procedure_rows = (
        _procedure_canonical()
        .withColumn("_implant_attribute_json", F.lit(None).cast("string"))
        .withColumn("_implant_attribute_count", F.lit(0).cast("long"))
    )
    implant_rows = (
        _implant_procedure_canonical()
        .join(
            attributes,
            F.col("patient_event_id") == F.col("a._implant_attribute_event_id"),
            "left",
        )
        .withColumn("_implant_attribute_json", F.col("a._implant_attribute_json"))
        .withColumn(
            "_implant_attribute_count",
            F.coalesce(F.col("a._implant_attribute_count"), F.lit(0).cast("long")),
        )
        .withColumn(
            "loaded_at", F.greatest(F.col("loaded_at"), F.col("a._evidence_loaded_at"))
        )
    )
    theatre_rows = (
        _theatre_procedure_canonical()
        .withColumn("_implant_attribute_json", F.lit(None).cast("string"))
        .withColumn("_implant_attribute_count", F.lit(0).cast("long"))
    )
    cc_rows = (
        _cc_procedure_canonical()
        .withColumn("_implant_attribute_json", F.lit(None).cast("string"))
        .withColumn("_implant_attribute_count", F.lit(0).cast("long"))
    )
    endobase_rows = (
        _endobase_procedure_canonical()
        .withColumn("_implant_attribute_json", F.lit(None).cast("string"))
        .withColumn("_implant_attribute_count", F.lit(0).cast("long"))
    )
    return (
        procedure_rows.select(*PROCEDURE_PRIMITIVE_COLUMNS)
        .unionByName(implant_rows.select(*PROCEDURE_PRIMITIVE_COLUMNS))
        .unionByName(theatre_rows.select(*PROCEDURE_PRIMITIVE_COLUMNS))
        .unionByName(cc_rows.select(*PROCEDURE_PRIMITIVE_COLUMNS))
        .unionByName(endobase_rows.select(*PROCEDURE_PRIMITIVE_COLUMNS))
    )

@materialized_view(
    name=_n("journey_clinical._procedure_primitive"),
    private=True,
    comment="Internal orderable procedure union; CodeableConcept JSON crosses the incremental boundary.",
    refresh_policy="incremental",
)
def _procedure_primitive():
    return _procedure_primitive_query()

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_procedure"),
    comment="Private JSON bridge and Gold cross-rule flags for procedure.",
    refresh_policy="incremental",
)
def _qc_procedure():
    source = (
        spark.read.table(_n("journey_clinical._procedure_primitive"))
        .withColumn("implant_attribute_history", F.col("_implant_attribute_json"))
        .withColumn("implant_attribute_count", F.col("_implant_attribute_count"))
    )
    return _cross_qc_primitive(
        source,
        "procedure",
        {"procedure_code": "_procedure_code_json"},
    )

In [0]:
PROCEDURE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide procedure identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime": "Primary performed timestamp.",
    "event_end_datetime": "Procedure end timestamp.",
    "source_coding_system": "Verbatim source coding system.",
    "source_code": "Source procedure code or implant label",
    "source_display": "Verbatim source procedure display.",
    "procedure_code": "Source and mapped procedure codings.",
    "status_code": "Source-derived FHIR procedure status.",
    "status_display": "Source status display.",
    "performed_start": "Procedure performed start.",
    "performed_end": "Procedure performed end.",
    "body_site_code": "Body-site code where supplied.",
    "body_site_display": "Body-site display where supplied.",
    "laterality_code": "Source laterality code.",
    "laterality_display": "Source laterality display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "procedure_location_code": "Source procedure-location code retained without asserting a location-dimension FK.",
    "procedure_location_display": "Source procedure-location display.",
    "procedure_note": "Source procedure note.",
    "implant_description": "Implant description for implant-placement facts.",
    "device_code": "Device concept identifier for implant-placement facts.",
    "device_display": "Device concept display.",
    "manufacturer": "Implant manufacturer.",
    "serial_number": "Implant serial number.",
    "batch_number": "Implant batch number.",
    "udi_di": "Unique device identifier device identifier.",
    "udi_standard": "UDI issuing standard.",
    "quantity": "Parsed implant quantity.",
    "implant_attribute_history": "Ordered JSON implant attribute evidence from map_implant_detail_events.",
    "implant_attribute_count": "Number of retained implant attribute rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered source feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Latest bronze load time across the fact row and its folded evidence rows.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.procedure"),
    comment="Performed procedure and implant-placement evidence from registered bronze feeds; source facts remain separate.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=PROCEDURE_COLUMN_COMMENTS,
)
def procedure():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_procedure")),
        "procedure",
        {"procedure_code": "_procedure_code_json"},
        PROCEDURE_PUBLIC_COLUMNS,
    )

In [0]:
# ==== Pathology, reports and specimens ====

SRC_PATHOLOGY_ACCESSION_SOURCE = "4_prod.bronze.map_pathology_accession_source"

def _alias_resolved_accession(df, id_col="pathology_accession_id"):
    """Resolve retired accessions to survivors before specimen:accession minting."""
    a = read_source(SRC_PATHOLOGY_ACCESSION_ALIAS).select(
        F.col("retired_pathology_accession_id").alias("_alias_retired"),
        F.col("survivor_pathology_accession_id").alias("_alias_survivor"),
    )
    return (
        df.join(a, df[id_col] == a["_alias_retired"], "left")
        .withColumn(
            "_resolved_accession_id",
            F.coalesce(F.col("_alias_survivor"), F.col(id_col)),
        )
        .drop("_alias_retired", "_alias_survivor")
    )

def _pathology_parent_link_canonical():
    src = _alias_resolved_accession(read_source(SRC_PATHOLOGY_ACCESSION_SOURCE))
    ver = read_source(SRC_PATHOLOGY_REPORT_VERSIONS).select(
        F.col("source_record_key").alias("source_parent_key"),
        F.col("report_series_id"),
    )
    return (
        src.select(
            "source_parent_key", "source_system",
            F.col("_resolved_accession_id").alias("pathology_accession_id"),
            F.col("encounter_id").alias("source_encounter_id"),
            F.col("ADC_UPDT").alias("loaded_at"),
        )
        .join(ver, "source_parent_key", "left")
    )

In [0]:
SRC_PATHOLOGY_ACCESSION_ALIAS = "4_prod.bronze.map_pathology_accession_alias"

@materialized_view(
    name=_n("journey_events._pathology_parent_link"),
    private=True,
    comment="INTERNAL source-parent linkage: source_parent_key -> alias-resolved accession, "
            "report series (1:1 parent->version->series, preflight-verified), and parent encounter.",
    refresh_policy="incremental",
)
def _pathology_parent_link():
    return _pathology_parent_link_canonical()

In [0]:
SRC_PATHOLOGY_ACCESSION = "4_prod.bronze.map_pathology_accession"

@materialized_view(
    name=_n("journey_events._pathology_accession_identity"),
    private=True,
    comment="INTERNAL accession identity: status-honest registry person columns plus MRN/NHS "
            "evidence aggregated from accession_source. Survivor accessions only; aliases resolve "
            "before grouping so retired evidence follows the survivor.",
    refresh_policy="incremental",
)
def _pathology_accession_identity():
    acc = _alias_resolved_accession(read_source(SRC_PATHOLOGY_ACCESSION))
    survivors = acc.where(F.col("pathology_accession_id") == F.col("_resolved_accession_id"))
    ev = (
        _alias_resolved_accession(read_source(SRC_PATHOLOGY_ACCESSION_SOURCE))
        .groupBy(F.col("_resolved_accession_id").alias("pathology_accession_id"))
        .agg(
            F.max("mrn").alias("evidence_mrn"),
            F.max("nhs_number").alias("evidence_nhs"),
            F.count(F.lit(1)).cast("long").alias("source_row_count"),
            F.max("ADC_UPDT").alias("evidence_loaded_at"),
        )
    )
    return survivors.select(
        "pathology_accession_id", "primary_source_accession_id", "canonical_accession_status",
        "canonical_person_id", "person_resolution_status", "normalized_lab_no",
        "lab_series", "discipline", "request_dt", "sample_dt", "report_dt",
        "clinical_details", "tlcs_requested", "conditions", "reason", "urgent_flag",
        "body_site_code", "body_site_snomed_code", "specimen_type_code",
        "specimen_type_snomed_code", "source_site_code", "lifecycle_status",
        "research_qi_only", "created_at", "ADC_UPDT",
    ).join(ev, "pathology_accession_id", "left")

In [0]:
def _pathology_specimen_canonical():
    i = spark.read.table(_n("journey_events._pathology_accession_identity"))
    event_id = stable_id("specimen:accession", F.col("pathology_accession_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_ACCESSION, F.col("pathology_accession_id"),
    )
    return i.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("canonical_person_id").cast("string").alias("person_id"),
        F.when(F.col("person_resolution_status") == "eligible", F.lit("resolved"))
         .when(F.col("person_resolution_status") == "conflicting", F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce("sample_dt", "request_dt", "report_dt")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:specimen-type").alias("source_coding_system"),
        F.col("specimen_type_code").cast("string").alias("source_code"),
        F.col("specimen_type_code").cast("string").alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:barts:pathology:specimen-type"),
                       F.col("specimen_type_code"), F.col("specimen_type_code"), True),
            coding_obj(F.lit("http://snomed.info/sct"), F.col("specimen_type_snomed_code"),
                       F.col("specimen_type_code"), False,
                       "bronze.map_pathology_accession", None),
        ).alias("specimen_type"),
        F.col("pathology_accession_id").alias("accession_identifier"),
        F.col("primary_source_accession_id"), F.col("normalized_lab_no"),
        F.col("canonical_accession_status"), F.col("person_resolution_status"),
        F.col("lab_series"), F.col("discipline"), F.col("urgent_flag"),
        F.col("research_qi_only"), F.col("clinical_details"), F.col("tlcs_requested"),
        F.col("conditions"), F.col("reason"), F.col("body_site_code"),
        F.col("body_site_snomed_code"), F.col("specimen_type_code"),
        F.col("specimen_type_snomed_code"),
        _clamped_ts(F.col("sample_dt")).alias("sample_datetime"),
        _clamped_ts(F.col("request_dt")).alias("request_datetime"),
        _clamped_ts(F.col("report_dt")).alias("report_datetime"),
        F.coalesce(F.col("source_row_count"), F.lit(0).cast("long"))
         .alias("source_history_row_count"),
        F.lit("active").alias("record_status"),
        F.col("created_at").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_accession").alias("source_feed"),
        F.date_format("ADC_UPDT", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("created_at").alias("source_update_timestamp"),
        F.col("ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_ACCESSION).alias("_source_table"),
        F.col("pathology_accession_id").alias("_source_row_id"),
    )

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_specimen"),
    comment="Private JSON bridge and Gold cross-rule flags for specimen.",
    refresh_policy="incremental",
)
def _qc_specimen():
    return _cross_qc_primitive(
        _pathology_specimen_canonical(),
        "specimen",
        {"specimen_type": "_qc_specimen_type_json"},
    )

In [0]:
SPECIMEN_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "specimen_type",
    "accession_identifier", "primary_source_accession_id", "normalized_lab_no",
    "canonical_accession_status", "person_resolution_status", "lab_series", "discipline",
    "urgent_flag", "research_qi_only", "clinical_details", "tlcs_requested", "conditions",
    "reason", "body_site_code", "body_site_snomed_code", "specimen_type_code",
    "specimen_type_snomed_code", "sample_datetime", "request_datetime", "report_datetime",
    "source_history_row_count",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

SPECIMEN_COLUMN_COMMENTS = {
    "patient_event_id": "Stable specimen event identifier minted from pathology_accession_id.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime": "Collection or receipt timestamp.",
    "event_end_datetime": "Event end when supplied.",
    "source_coding_system": "Source specimen type system.",
    "source_code": "Source specimen type code.",
    "source_display": "Source specimen type display.",
    "specimen_type": "Source specimen type CodeableConcept.",
    "accession_identifier": "Canonical accession identifier (pathology_accession_id); never LabNo-derived.",
    "primary_source_accession_id": "Primary source accession identifier retained as evidence.",
    "normalized_lab_no": "Matching evidence only — reused lab numbers reach 3",
    "canonical_accession_status": "Canonical accession-link state.",
    "person_resolution_status": "Person-projection eligibility state.",
    "lab_series": "Source laboratory series.",
    "discipline": "Source pathology discipline.",
    "urgent_flag": "Source urgent-request indicator code (Y",
    "research_qi_only": "Registry doctrine flag; true on 100% of rows today — describe-only",
    "clinical_details": "Source clinical details. Identifiable free text; ig_risk 4",
    "tlcs_requested": "Requested TLC context. Identifiable free text; ig_risk 4",
    "conditions": "Source request conditions. Identifiable free text; ig_risk 4",
    "reason": "Source request reason. Identifiable free text; ig_risk 4",
    "body_site_code": "Source collection body-site code.",
    "body_site_snomed_code": "Source-provided SNOMED body-site code.",
    "specimen_type_code": "Native specimen-type code.",
    "specimen_type_snomed_code": "Source-provided SNOMED specimen-type code.",
    "sample_datetime": "Clamped source sample timestamp.",
    "request_datetime": "Clamped source request timestamp.",
    "report_datetime": "Clamped source report timestamp.",
    "source_history_row_count": "accession_source rows behind this accession.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.specimen"),
    comment="One canonical pathology accession with collection and request evidence.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=SPECIMEN_COLUMN_COMMENTS,
)
def specimen():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_specimen")),
        "specimen",
        {"specimen_type": "_qc_specimen_type_json"},
        SPECIMEN_PUBLIC_COLUMNS,
    )

In [0]:
def _pathology_report_series_canonical():
    # The report feed already carries the canonical accession. Resolve
    # aliases on that governed key directly; source_record_key identifies the
    # source result used to assemble a version and is not an accession key.
    r = _alias_resolved_accession(
        read_source(SRC_PATHOLOGY_REPORT_VERSIONS)
    ).alias("r")
    text_present = r.report_text.isNotNull() & (F.trim(r.report_text) != "")
    link = spark.read.table(_n("journey_events._pathology_parent_link")).select(
        F.col("source_parent_key").alias("_link_parent_key"),
        F.col("pathology_accession_id").alias("_link_accession_id"),
        F.col("source_encounter_id").alias("_link_encounter_id"),
    ).alias("l")
    i = spark.read.table(_n("journey_events._pathology_accession_identity")).select(
        F.col("pathology_accession_id").alias("_identity_accession_id"),
        "canonical_person_id", "person_resolution_status", "evidence_mrn", "evidence_nhs",
    ).alias("i")
    enriched = (
        r.join(link, r.source_record_key == F.col("l._link_parent_key"), "left")
        .join(
            i,
            F.col("r._resolved_accession_id") == F.col("i._identity_accession_id"),
            "left",
        )
    )
    latest = (
        enriched.groupBy(r.report_series_id.alias("report_series_id"))
        .agg(
            F.max(F.struct(
                r.version_ordinal, r.report_version_id, r.source_record_key,
                r.report_role, r.discipline, r.report_code, r.report_section,
                r.lifecycle_status, r.supersedes_report_version_id, r.issued_dt,
                r.report_text_hash, r.research_qi_only, r.valid_from,
                F.col("r._resolved_accession_id").alias("pathology_accession_id"),
                F.col("l._link_encounter_id").alias("source_encounter_id"),
                F.col("i.canonical_person_id").alias("canonical_person_id"),
                F.col("i.person_resolution_status").alias("person_resolution_status"),
                F.col("i.evidence_mrn").alias("evidence_mrn"),
                F.col("i.evidence_nhs").alias("evidence_nhs"),
            )).alias("v"),
            F.count(F.lit(1)).cast("long").alias("version_count"),
            F.max(F.when(r.is_current & text_present, r.report_version_id))
             .alias("current_text_version_id"),
            F.max(F.when(r.is_current, F.lit(True)).otherwise(F.lit(False)))
             .alias("is_current_present"),
            F.max("ADC_UPDT").alias("loaded_at"),
        )
    )
    event_id = stable_id("pathology_report:series", F.col("report_series_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("v.canonical_person_id")),
         ("urn:barts:mrn", F.col("v.evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("v.evidence_nhs"))],
        SRC_PATHOLOGY_REPORT_VERSIONS, F.col("report_series_id"),
    )
    retracted = F.col("v.lifecycle_status").isin("cancelled", "entered_in_error")
    return latest.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("v.canonical_person_id").cast("string").alias("person_id"),
        F.when(F.col("v.person_resolution_status") == "eligible", F.lit("resolved"))
         .when(F.col("v.person_resolution_status") == "conflicting", F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("v.source_encounter_id").isNotNull(),
               stable_id("encounter:mill", F.col("v.source_encounter_id")))
         .alias("encounter_id"),
        _clamped_ts(F.col("v.issued_dt")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:report-code").alias("source_coding_system"),
        F.col("v.report_code").cast("string").alias("source_code"),
        F.coalesce(F.col("v.report_role"), F.col("v.report_code")).cast("string")
         .alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:barts:pathology:report-code"),
                       F.col("v.report_code"), F.col("v.report_role"), True),
        ).alias("report_code"),
        F.col("v.report_role").alias("report_role"),
        F.col("v.discipline").alias("discipline"),
        F.col("v.report_section").alias("report_section"),
        F.col("v.lifecycle_status").alias("lifecycle_status"),
        F.col("v.version_ordinal").cast("long").alias("version_ordinal"),
        F.col("version_count"),
        F.col("v.report_version_id").alias("report_version_id"),
        F.col("v.supersedes_report_version_id").alias("supersedes_report_version_id"),
        F.col("is_current_present"),
        F.when(F.col("current_text_version_id").isNotNull(),
               stable_id("document:pathology_report", F.col("current_text_version_id")))
         .alias("document_id"),
        _clamped_ts(F.col("v.issued_dt")).alias("issued_datetime"),
        F.when(F.col("v.pathology_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("v.pathology_accession_id")))
         .alias("specimen_id"),
        F.col("v.pathology_accession_id").alias("accession_identifier"),
        F.col("v.report_text_hash").alias("report_text_hash"),
        F.col("v.research_qi_only").alias("research_qi_only"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        F.col("v.valid_from").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_report").alias("source_feed"),
        F.date_format("loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("v.valid_from").alias("source_update_timestamp"), F.col("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_REPORT_VERSIONS).alias("_source_table"),
        F.col("report_series_id").alias("_source_row_id"),
    )

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_pathology_report"),
    comment="Private JSON bridge and Gold cross-rule flags for pathology_report.",
    refresh_policy="incremental",
)
def _qc_pathology_report():
    return _cross_qc_primitive(
        _pathology_report_series_canonical(),
        "pathology_report",
        {"report_code": "_qc_report_code_json"},
    )

In [0]:
REPORT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "report_code",
    "report_role", "discipline", "report_section", "lifecycle_status", "version_ordinal",
    "version_count", "report_version_id", "supersedes_report_version_id",
    "is_current_present", "document_id", "issued_datetime", "specimen_id",
    "accession_identifier", "report_text_hash", "research_qi_only",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

PATHOLOGY_REPORT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable pathology report identifier minted from report_series_id.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Report issue or result timestamp.",
    "event_end_datetime": "Report event end.",
    "source_coding_system": "Source test coding system.",
    "source_code": "Source test code.",
    "source_display": "Source test display.",
    "report_code": "Source and mapped report codings.",
    "report_role": "Latest-version report role.",
    "discipline": "Latest-version pathology discipline.",
    "report_section": "Latest-version report section.",
    "lifecycle_status": "Latest-version lifecycle status.",
    "version_ordinal": "Latest version ordinal in the report series.",
    "version_count": "Number of report versions in the series.",
    "report_version_id": "Latest report-version identifier.",
    "supersedes_report_version_id": "Prior report-version identifier superseded by the latest version.",
    "is_current_present": "True when the series has a current version row.",
    "document_id": "Current text-bearing report-version document reference; nullable.",
    "issued_datetime": "Report issue timestamp.",
    "specimen_id": "Linked specimen identifier.",
    "accession_identifier": "Canonical pathology_accession_id.",
    "report_text_hash": "Bronze hash for the latest report version",
    "research_qi_only": "Registry doctrine flag; true on 100% of rows today — describe-only",
    "record_status": "Latest-version lifecycle normalized to active or retracted.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.pathology_report"),
    comment="One pathology report series; latest-version projection; versions in journey_text.document.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=PATHOLOGY_REPORT_COLUMN_COMMENTS,
)
def pathology_report():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_pathology_report")),
        "pathology_report",
        {"report_code": "_qc_report_code_json"},
        REPORT_PUBLIC_COLUMNS,
    )

In [0]:
RESULT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "result_code",
    "pathology_report_id", "specimen_id", "equivalence_group", "representation_role",
    "preferred_result_ind", "person_projection_status",
    "value_number", "value_text", "value_datetime",
    "value_concept_id", "value_concept_display", "operator_concept_id",
    "unit_source_value", "ucum_code", "unit_concept_id",
    "reference_range_low", "reference_range_high", "interpretation_code",
    "polarity", "finding_axis", "result_status", "body_site_code", "clinician_code",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _pathology_result_canonical_pregate(source_df=None, include_parent_links=False):
    s = read_source(SRC_PATHOLOGY) if source_df is None else source_df
    source_row_id = F.coalesce(
        s.source_record_key,
        F.concat_ws(":", s.source_table, s.source_event_id.cast("string"), s.source_sequence_start.cast("string")),
    )
    event_id = stable_id("pathology_result:pathology", source_row_id)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:mrn", s.MRN),
         ("https://fhir.nhs.uk/Id/nhs-number", s.NHS_Number)],
        SRC_PATHOLOGY, source_row_id,
    )
    ended = s.valid_until_dt_tm.isNotNull() & (
        s.valid_until_dt_tm < F.lit("2100-01-01").cast("timestamp")
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved"))
         .when(_present(s.MRN) | _present(s.NHS_Number), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        _clamped_ts(s.measurement_datetime).alias("event_datetime"),
        _clamped_ts(s.event_end_dt_tm).alias("event_end_datetime"),
        F.coalesce(s.code_system, F.lit("urn:barts:pathology:test")).alias("source_coding_system"),
        F.coalesce(s.code, s.EVENT_CD.cast("string")).alias("source_code"),
        F.coalesce(s.description, s.EVENT_CD_DISPLAY).alias("source_display"),
        codeable_concept_json(
            coding_obj(F.coalesce(s.code_system, F.lit("urn:barts:pathology:test")),
                       F.coalesce(s.code, s.EVENT_CD.cast("string")),
                       F.coalesce(s.description, s.EVENT_CD_DISPLAY), True),
            coding_obj(F.lit("http://snomed.info/sct"), s.test_snomed_code, s.description, False,
                       "bronze.map_pathology", None),
            coding_obj(F.lit("http://loinc.org"), s.test_loinc_code, s.description, False,
                       "bronze.map_pathology", None),
            coding_obj(F.lit("urn:omop:concept_id"), s.test_omop_concept_id,
                       s.measurement_concept_name, False, "bronze.map_pathology", None),
        ).alias("_result_code_json"),
        *(
            [
                F.when(
                    F.col("_link_series_id").isNotNull(),
                    stable_id("pathology_report:series", F.col("_link_series_id")),
                ).alias("pathology_report_id"),
                F.when(
                    F.col("_link_accession_id").isNotNull(),
                    stable_id("specimen:accession", F.col("_link_accession_id")),
                ).alias("specimen_id"),
                F.col("_eq_group").alias("equivalence_group"),
                F.col("_eq_role").alias("representation_role"),
                F.col("_eq_preferred").alias("preferred_result_ind"),
                F.col("_eq_person_projection").alias("person_projection_status"),
            ]
            if include_parent_links else []
        ),
        s.value_as_number.cast("decimal(38,10)").alias("value_number"),
        s.value_source_value.alias("value_text"), s.value_as_datetime.alias("value_datetime"),
        s.value_as_concept_id.cast("string").alias("value_concept_id"),
        s.result_concept_name.alias("value_concept_display"),
        s.operator_concept_id.cast("string").alias("operator_concept_id"),
        s.unit_source_value, s.ucum_code, s.unit_concept_id.cast("string").alias("unit_concept_id"),
        s.range_low.cast("decimal(38,10)").alias("reference_range_low"),
        s.range_high.cast("decimal(38,10)").alias("reference_range_high"),
        s.normalcy.alias("interpretation_code"), s.result_growth_grade.alias("polarity"),
        s.master_result_type.alias("finding_axis"), s.result_status,
        s.body_site_code, s.clinician_code,
        F.when(ended, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.valid_from_dt_tm.alias("record_status_effective_from"),
        F.when(ended, s.valid_until_dt_tm).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"), F.lit("pathology").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.source_adc_updt.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"), F.lit(SRC_PATHOLOGY).alias("_source_table"),
        source_row_id.alias("_source_row_id"), s.source_parent_key.alias("_source_parent_key"),
        s.test_snomed_code.cast("string").alias("_test_snomed_code"), s.description.alias("_test_snomed_display"),
        s.test_loinc_code.cast("string").alias("_test_loinc_code"), s.description.alias("_test_loinc_display"),
        s.test_omop_concept_id.cast("string").alias("_test_omop_code"),
        s.measurement_concept_name.alias("_test_omop_display"),
        s.result_snomed_code.cast("string").alias("_result_snomed_code"),
        s.result_concept_name.alias("_result_snomed_display"),
        s.result_loinc_code.cast("string").alias("_result_loinc_code"),
        s.result_concept_name.alias("_result_loinc_display"),
        s.result_omop_concept_id.cast("string").alias("_result_omop_code"),
        s.result_concept_name.alias("_result_omop_display"),
    )

def _pathology_result_canonical(source_df=None, include_parent_links=False):
    return _pathology_result_canonical_pregate(
        source_df, include_parent_links
    ).where(_usable_code(F.col("source_code")))

In [0]:
SRC_PATHOLOGY = "4_prod.bronze.map_pathology"
SRC_PATHOLOGY_RESULT_EQUIVALENCE = "4_prod.bronze.map_pathology_result_equivalence"

RESULT_PRIMITIVE_COLUMNS = [
    "_result_code_json" if c == "result_code" else c for c in RESULT_PUBLIC_COLUMNS
]

@materialized_view(
    name=_n("journey_clinical._pathology_result_primitive"),
    private=True,
    comment="Internal repointed result rows; CodeableConcept JSON crosses the incremental "
            "boundary — a VARIANT output column on a join-shaped flow crashes Enzyme planning "
            "(GROUP_EXPRESSION_TYPE_IS_NOT_ORDERABLE) and forces complete recompute every update.",
    refresh_policy="incremental",
)
def _pathology_result_primitive():
    raw = read_source(SRC_PATHOLOGY).alias("s")
    link = _pathology_parent_link_canonical().select(
        F.col("source_parent_key").alias("_source_parent_key"),
        F.col("pathology_accession_id").alias("_link_accession_id"),
        F.col("report_series_id").alias("_link_series_id"),
    ).alias("l")
    eq = read_source(SRC_PATHOLOGY_RESULT_EQUIVALENCE).select(
        F.col("source_record_key").alias("_eq_key"),
        F.col("canonical_result_id").alias("_eq_group"),
        F.col("representation_role").alias("_eq_role"),
        F.col("preferred_result_ind").alias("_eq_preferred"),
        F.col("person_projection_status").alias("_eq_person_projection"),
    ).alias("e")
    joined = (
        raw.join(link, raw.source_parent_key == F.col("l._source_parent_key"), "left")
        .join(eq, raw.source_record_key == F.col("e._eq_key"), "left")
        .select("s.*", "l._link_accession_id", "l._link_series_id",
                "e._eq_group", "e._eq_role", "e._eq_preferred", "e._eq_person_projection")
    )
    return _pathology_result_canonical(
        joined, include_parent_links=True
    ).select(*RESULT_PRIMITIVE_COLUMNS)

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_pathology_result"),
    comment="Private JSON bridge and Gold cross-rule flags for pathology_result.",
    refresh_policy="incremental",
)
def _qc_pathology_result():
    return _cross_qc_primitive(
        spark.read.table(_n("journey_clinical._pathology_result_primitive")),
        "pathology_result",
        {"result_code": "_result_code_json"},
    )

In [0]:
PATHOLOGY_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable pathology result identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Measurement timestamp.",
    "event_end_datetime": "Measurement end timestamp.",
    "source_coding_system": "Source test coding system.",
    "source_code": "Source test code.",
    "source_display": "Source test display.",
    "result_code": "Source and mapped test codings.",
    "pathology_report_id": "Report SERIES reference via the accession parent link; NULL when the parent has no textual report (70% of accessions).",
    "specimen_id": "Accession-minted specimen reference (specimen:accession)",
    "equivalence_group": "canonical_result_id from map_pathology_result_equivalence; currently all singleton groups under unique_source_result_v1. NULL means the spine key is not covered by the equivalence layer or source_record_key is NULL.",
    "representation_role": "Result representation role within the equivalence group.",
    "preferred_result_ind": "Whether this is the preferred representation.",
    "person_projection_status": "Equivalence-layer person projection state.",
    "value_number": "Numeric result value.",
    "value_text": "Verbatim result value.",
    "value_datetime": "Datetime result value.",
    "value_concept_id": "Coded result concept identifier.",
    "value_concept_display": "Coded result concept display.",
    "operator_concept_id": "Result comparison operator concept.",
    "unit_source_value": "Verbatim source unit.",
    "ucum_code": "UCUM unit code.",
    "unit_concept_id": "OMOP unit concept identifier.",
    "reference_range_low": "Reference-range lower bound.",
    "reference_range_high": "Reference-range upper bound.",
    "interpretation_code": "Source interpretation or normalcy.",
    "polarity": "Source result polarity or growth grade.",
    "finding_axis": "Source finding/result axis.",
    "result_status": "Source result status.",
    "body_site_code": "Source body-site code.",
    "clinician_code": "Source clinician code.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.pathology_result"),
    comment="One pathology result with typed values, units, ranges and retained mapping evidence.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=PATHOLOGY_RESULT_COLUMN_COMMENTS,
)
def pathology_result():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_pathology_result")),
        "pathology_result",
        {"result_code": "_result_code_json"},
        RESULT_PUBLIC_COLUMNS,
    )

In [0]:
SRC_PATHOLOGY_REQUESTED_TEST = "4_prod.bronze.map_pathology_requested_test"

def _pathology_requested_test_canonical_pregate():
    t = _alias_resolved_accession(read_source(SRC_PATHOLOGY_REQUESTED_TEST))
    i = spark.read.table(_n("journey_events._pathology_accession_identity")).select(
        F.col("pathology_accession_id").alias("_resolved_accession_id"),
        "canonical_person_id", "person_resolution_status", "evidence_mrn", "evidence_nhs",
        F.col("request_dt").alias("_acc_request_dt"),
        F.col("sample_dt").alias("_acc_sample_dt"),
    )
    j = t.join(i, "_resolved_accession_id", "left")
    event_id = stable_id(
        "pathology_order:requested_test", F.col("requested_test_occurrence_id")
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_REQUESTED_TEST, F.col("requested_test_occurrence_id"),
    )
    source_code = F.coalesce(F.col("wkg_code"), F.col("order_mnemonic"))
    system_uri = (
        F.when(F.col("source_system") == "TFC_LIMS", F.lit("urn:barts:pathology:wkg-tlc"))
        .otherwise(F.lit("urn:cerner:order-mnemonic"))
    )
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("canonical_person_id").cast("string").alias("person_id"),
        F.when(F.col("person_resolution_status") == "eligible", F.lit("resolved"))
         .when(F.col("person_resolution_status") == "conflicting", F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce("_acc_request_dt", "_acc_sample_dt")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        system_uri.alias("source_coding_system"),
        source_code.cast("string").alias("source_code"),
        F.coalesce(F.col("test_description"), source_code).cast("string").alias("source_display"),
        F.col("source_system").alias("source_arm"),
        F.col("wkg_code"), F.col("tlc_code"),
        F.col("order_id").cast("string").alias("order_id"), F.col("order_mnemonic"),
        F.col("raw_request_text"), F.col("test_description"),
        F.col("test_snomed_code").cast("string").alias("test_snomed_code"),
        F.col("test_omop_concept_id").cast("string").alias("test_omop_concept_id"),
        F.col("mapping_status"), F.col("request_ordinal").cast("long").alias("request_ordinal"),
        stable_id("specimen:accession", F.col("_resolved_accession_id")).alias("specimen_id"),
        F.lit("active").alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format("ADC_UPDT", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        F.col("ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_REQUESTED_TEST).alias("_source_table"),
        F.col("requested_test_occurrence_id").alias("_source_row_id"),
        F.col("test_snomed_code").cast("string").alias("_test_snomed_code"),
        F.coalesce(F.col("test_description"), source_code).alias("_test_snomed_display"),
        F.col("test_omop_concept_id").cast("string").alias("_test_omop_code"),
        F.coalesce(F.col("test_description"), source_code).alias("_test_omop_display"),
    )

def _pathology_requested_test_canonical():
    return _pathology_requested_test_canonical_pregate().where(
        _usable_code(F.col("source_code"))
    )

In [0]:
PATHOLOGY_ORDER_COLUMN_COMMENTS = {
    "patient_event_id": "Stable pathology order event identifier minted from requested_test_occurrence_id.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime": "Requested or source-validity timestamp.",
    "event_end_datetime": "Order event end when supplied.",
    "source_coding_system": "Native order catalogue coding system.",
    "source_code": "Native order catalogue code.",
    "source_display": "Native order description.",
    "source_arm": "Source arm.",
    "wkg_code": "TFC work-group code.",
    "tlc_code": "TFC test-level code.",
    "order_id": "CERNER native order identifier when present.",
    "order_mnemonic": "CERNER order mnemonic.",
    "raw_request_text": "Native request wording. Identifiable free text; ig_risk 4",
    "test_description": "Native requested-test description.",
    "test_snomed_code": "Source-provided requested-test SNOMED code.",
    "test_omop_concept_id": "Source-provided requested-test OMOP concept identifier.",
    "mapping_status": "'mapped' means a rule ran, not that codes landed — CERNER arm carries zero baked codes.",
    "request_ordinal": "Source request ordinal within the accession.",
    "specimen_id": "Alias-resolved accession-minted specimen reference.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.pathology_order"),
    comment="One requested-test occurrence across TFC_LIMS and CERNER; anchors accession-scoped request threads.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=PATHOLOGY_ORDER_COLUMN_COMMENTS,
)
def pathology_order():
    s = _pathology_requested_test_canonical()
    return s.select(
        "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
        "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
        "source_coding_system", "source_code", "source_display",
        "source_arm", "wkg_code", "tlc_code", "order_id", "order_mnemonic",
        "raw_request_text", "test_description", "test_snomed_code", "test_omop_concept_id",
        "mapping_status", "request_ordinal", "specimen_id",
        "record_status", "record_status_effective_from", "record_status_effective_to",
        "confidentiality_code", "vip_ind", "withheld_identity_ind",
        "load_batch_id", "source_update_timestamp", "loaded_at",
    )

In [0]:
SRC_PATHOLOGY_GENETIC_TEST = "4_prod.bronze.map_pathology_genetic_test"

def _genomic_identity_stage():
    return spark.read.table(_n("journey_events._pathology_accession_identity")).select(
        F.col("pathology_accession_id").alias("_identity_accession_id"),
        "canonical_person_id", "person_resolution_status", "evidence_mrn", "evidence_nhs",
        F.col("request_dt").alias("_acc_request_dt"),
        F.col("sample_dt").alias("_acc_sample_dt"),
        F.col("report_dt").alias("_acc_report_dt"),
    )

def _identity_mapped_columns():
    return [
        F.col("canonical_person_id").cast("string").alias("person_id"),
        F.when(F.col("person_resolution_status") == "eligible", F.lit("resolved"))
         .when(F.col("person_resolution_status") == "conflicting", F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
    ]

def _genomic_test_canonical_pregate():
    t = _alias_resolved_accession(read_source(SRC_PATHOLOGY_GENETIC_TEST)).alias("t")
    i = _genomic_identity_stage().alias("i")
    r = read_source(SRC_PATHOLOGY_REPORT_VERSIONS).select(
        F.col("report_version_id").alias("_rv_id"),
        F.col("report_series_id").alias("_rv_series_id"),
        F.col("issued_dt").alias("_rv_issued_dt"),
    ).alias("r")
    j = (
        t.join(i, F.col("t._resolved_accession_id") == F.col("i._identity_accession_id"), "left")
        .join(r, F.col("t.report_version_id") == F.col("r._rv_id"), "left")
    )
    event_id = stable_id("genomic_test:assay", F.col("t.genetic_test_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_GENETIC_TEST, F.col("t.genetic_test_id"),
    )
    superseded = ~F.coalesce(F.col("t.is_current"), F.lit(True))
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        *_identity_mapped_columns(),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce(F.col("r._rv_issued_dt"), F.col("_acc_sample_dt"),
                               F.col("_acc_request_dt"))).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:assay-code").alias("source_coding_system"),
        F.col("t.assay_code").cast("string").alias("source_code"),
        F.coalesce(F.col("t.assay_name"), F.col("t.assay_code")).alias("source_display"),
        F.col("t.assay_code"), F.col("t.assay_name"), F.col("t.method"),
        F.col("t.analysis_context"), F.col("t.overall_result_status"),
        F.col("t.panel_code"), F.col("t.panel_version"), F.col("t.panel_version_inferred"),
        F.col("t.parser_profile_id"), F.col("t.report_version_id"),
        F.when(F.col("r._rv_series_id").isNotNull(),
               stable_id("pathology_report:series", F.col("r._rv_series_id")))
         .alias("pathology_report_id"),
        F.when(F.col("t._resolved_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("t._resolved_accession_id")))
         .alias("specimen_id"),
        F.col("t._resolved_accession_id").alias("accession_identifier"),
        F.col("t.test_snomed_code").cast("string").alias("test_snomed_code"),
        F.col("t.test_loinc_code").cast("string").alias("test_loinc_code"),
        F.col("t.test_omop_concept_id").cast("string").alias("test_omop_concept_id"),
        F.col("t.is_current"), F.col("t.research_qi_only"),
        F.when(superseded, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_genetic_test").alias("source_feed"),
        F.date_format(F.col("t.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("t.ADC_UPDT").alias("source_update_timestamp"),
        F.col("t.ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_GENETIC_TEST).alias("_source_table"),
        F.col("t.genetic_test_id").alias("_source_row_id"),
        F.col("t.test_snomed_code").cast("string").alias("_gt_snomed_code"),
        F.coalesce(F.col("t.assay_name"), F.col("t.assay_code")).alias("_gt_snomed_display"),
        F.col("t.test_loinc_code").cast("string").alias("_gt_loinc_code"),
        F.coalesce(F.col("t.assay_name"), F.col("t.assay_code")).alias("_gt_loinc_display"),
        F.col("t.test_omop_concept_id").cast("string").alias("_gt_omop_code"),
        F.coalesce(F.col("t.assay_name"), F.col("t.assay_code")).alias("_gt_omop_display"),
    )

def _genomic_test_canonical():
    return _genomic_test_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
# ==== Genomics, indication and microbiology/AMR ====

GENOMIC_TEST_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "assay_code", "assay_name", "method", "analysis_context", "overall_result_status",
    "panel_code", "panel_version", "panel_version_inferred", "parser_profile_id",
    "report_version_id", "pathology_report_id", "specimen_id", "accession_identifier",
    "test_snomed_code", "test_loinc_code", "test_omop_concept_id",
    "is_current", "research_qi_only",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

GENOMIC_TEST_COLUMN_COMMENTS = {
    "patient_event_id": "Stable genomic-test event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Report issue time with accession fallback.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Assay coding namespace.",
    "source_code": "Source assay code.",
    "source_display": "Source assay display.",
    "assay_code": "Source assay or report code.",
    "assay_name": "Resolved assay name.",
    "method": "Reported assay method.",
    "analysis_context": "Somatic or germline analysis context.",
    "overall_result_status": "Source assay-level result status.",
    "panel_code": "Governed panel code.",
    "panel_version": "Explicit or inferred panel version.",
    "panel_version_inferred": "Whether panel version was inferred.",
    "parser_profile_id": "Deterministic parser profile.",
    "report_version_id": "Parent report-version identifier.",
    "pathology_report_id": "Report-series fact reference.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "test_snomed_code": "Approved SNOMED assay code.",
    "test_loinc_code": "Approved LOINC assay code.",
    "test_omop_concept_id": "Standard OMOP assay concept.",
    "is_current": "Whether the source report version is current.",
    "research_qi_only": "Research/QI release flag.",
    "record_status": "Normalized report-version status.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.genomic_test"),
    comment="One molecular/cytogenetic assay per report version (all versions published; "
            "record_status supersedes on is_current). Vocab and panel columns are all-NULL "
            "in the current build and lanes are pre-wired.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=GENOMIC_TEST_COLUMN_COMMENTS,
)
def genomic_test():
    return _genomic_test_canonical().select(*GENOMIC_TEST_PUBLIC_COLUMNS)

In [0]:
def _genomic_result_canonical_pregate():
    g = read_source(SRC_PATHOLOGY_GENETIC_RESULT).alias("g")
    parent = _alias_resolved_accession(
        read_source(SRC_PATHOLOGY_GENETIC_TEST)
    ).select(
        F.col("genetic_test_id").alias("_gt_id"),
        F.col("_resolved_accession_id"),
    ).alias("p")
    i = _genomic_identity_stage().alias("i")
    r = read_source(SRC_PATHOLOGY_REPORT_VERSIONS).select(
        F.col("report_version_id").alias("_rv_id"),
        F.col("issued_dt").alias("_rv_issued_dt"),
    ).alias("r")
    j = (
        g.join(parent, F.col("g.genetic_test_id") == F.col("p._gt_id"), "left")
        .join(i, F.col("p._resolved_accession_id") == F.col("i._identity_accession_id"), "left")
        .join(r, F.col("g.report_version_id") == F.col("r._rv_id"), "left")
    )
    event_id = stable_id("genomic_result:finding", F.col("g.genetic_result_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_GENETIC_RESULT, F.col("g.genetic_result_id"),
    )
    retracted = F.col("g.lifecycle_status").isin("cancelled", "entered_in_error")
    superseded = ~F.coalesce(F.col("g.is_current"), F.lit(True))
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        *_identity_mapped_columns(),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce(F.col("r._rv_issued_dt"), F.col("_acc_sample_dt"),
                               F.col("_acc_request_dt"))).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.when(F.coalesce(F.col("g.normalized_gene_symbol"),
                          F.col("g.reported_gene_symbol")).isNotNull(),
               F.lit("urn:barts:pathology:gene-symbol"))
         .otherwise(F.lit("urn:barts:pathology:alteration-type"))
         .alias("source_coding_system"),
        F.coalesce(F.col("g.normalized_gene_symbol"), F.col("g.reported_gene_symbol"),
                   F.col("g.alteration_type"), F.col("g.detection_status"))
         .cast("string").alias("source_code"),
        F.coalesce(F.col("g.reported_gene_symbol"), F.col("g.normalized_gene_symbol"),
                   F.col("g.alteration_type"), F.col("g.detection_status"))
         .alias("source_display"),
        F.when(F.col("g.genetic_test_id").isNotNull(),
               stable_id("genomic_test:assay", F.col("g.genetic_test_id")))
         .alias("genomic_test_id"),
        F.col("g.report_version_id"),
        F.col("g.hgnc_id"), F.col("g.reported_gene_symbol"), F.col("g.normalized_gene_symbol"),
        F.col("g.partner_hgnc_id"), F.col("g.partner_gene_symbol"),
        F.col("g.alteration_type"), F.col("g.detection_status"),
        F.col("g.hgvs_c_raw"), F.col("g.hgvs_c_parsed"),
        F.col("g.hgvs_p_raw"), F.col("g.hgvs_p_parsed"),
        F.col("g.transcript"), F.col("g.hgvs_validation_status"),
        F.col("g.genome_build"), F.col("g.chromosome"),
        F.col("g.position_start"), F.col("g.position_end"),
        F.col("g.vaf_raw"), F.col("g.vaf"), F.col("g.zygosity"),
        F.col("g.reported_classification"), F.col("g.reported_tier"),
        F.col("g.copy_number"), F.col("g.ratio_raw"), F.col("g.iscn_raw"),
        F.col("g.clinvar_concept_id").cast("string").alias("clinvar_concept_id"),
        F.col("g.omop_genomic_concept_id").cast("string").alias("omop_genomic_concept_id"),
        F.col("g.snomed_code").cast("string").alias("snomed_code"),
        F.col("g.evidence_text"), F.col("g.evidence_start"), F.col("g.evidence_end"),
        F.col("g.parser_profile_id"), F.col("g.parser_version"),
        F.col("g.review_status"), F.col("g.lifecycle_status"),
        F.col("g.is_current"), F.col("g.research_qi_only"),
        F.when(F.col("p._resolved_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("p._resolved_accession_id")))
         .alias("specimen_id"),
        F.col("p._resolved_accession_id").alias("accession_identifier"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_genetic_result").alias("source_feed"),
        F.date_format(F.col("g.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("g.ADC_UPDT").alias("source_update_timestamp"),
        F.col("g.ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_GENETIC_RESULT).alias("_source_table"),
        F.col("g.genetic_result_id").alias("_source_row_id"),
        F.col("g.snomed_code").cast("string").alias("_gr_snomed_code"),
        F.coalesce(F.col("g.reported_classification"), F.col("g.alteration_type"))
         .alias("_gr_snomed_display"),
        F.col("g.clinvar_concept_id").cast("string").alias("_gr_clinvar_code"),
        F.col("g.reported_classification").alias("_gr_clinvar_display"),
        F.col("g.omop_genomic_concept_id").cast("string").alias("_gr_omop_code"),
        F.col("g.reported_classification").alias("_gr_omop_display"),
    )

def _genomic_result_canonical():
    return _genomic_result_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_PATHOLOGY_GENETIC_RESULT = "4_prod.bronze.map_pathology_genetic_result"

GENOMIC_RESULT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "genomic_test_id", "report_version_id", "hgnc_id", "reported_gene_symbol",
    "normalized_gene_symbol", "partner_hgnc_id", "partner_gene_symbol",
    "alteration_type", "detection_status", "hgvs_c_raw", "hgvs_c_parsed",
    "hgvs_p_raw", "hgvs_p_parsed", "transcript", "hgvs_validation_status",
    "genome_build", "chromosome", "position_start", "position_end",
    "vaf_raw", "vaf", "zygosity", "reported_classification", "reported_tier",
    "copy_number", "ratio_raw", "iscn_raw",
    "clinvar_concept_id", "omop_genomic_concept_id", "snomed_code",
    "evidence_text", "evidence_start", "evidence_end",
    "parser_profile_id", "parser_version", "review_status", "lifecycle_status",
    "is_current", "research_qi_only", "specimen_id", "accession_identifier",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

GENOMIC_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable genomic-result event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Report issue time with accession fallback.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Gene-symbol or alteration coding namespace.",
    "source_code": "Gene",
    "source_display": "Finding display.",
    "genomic_test_id": "Parent genomic-test fact reference.",
    "report_version_id": "Source report-version identifier.",
    "hgnc_id": "Primary HGNC identifier.",
    "reported_gene_symbol": "Gene symbol as reported.",
    "normalized_gene_symbol": "Approved HGNC symbol.",
    "partner_hgnc_id": "Partner HGNC identifier.",
    "partner_gene_symbol": "Partner gene symbol.",
    "alteration_type": "Reported alteration type.",
    "detection_status": "Finding detection status.",
    "hgvs_c_raw": "Raw coding HGVS.",
    "hgvs_c_parsed": "Parsed coding HGVS.",
    "hgvs_p_raw": "Raw protein HGVS.",
    "hgvs_p_parsed": "Parsed protein HGVS.",
    "transcript": "Reported transcript.",
    "hgvs_validation_status": "HGVS validation state.",
    "genome_build": "Explicit genome build.",
    "chromosome": "Explicit chromosome.",
    "position_start": "Genomic start coordinate.",
    "position_end": "Genomic end coordinate.",
    "vaf_raw": "Raw variant allele frequency.",
    "vaf": "Parsed variant allele fraction.",
    "zygosity": "Reported zygosity.",
    "reported_classification": "Reported classification.",
    "reported_tier": "Reported tier.",
    "copy_number": "Reported copy number.",
    "ratio_raw": "Raw molecular ratio.",
    "iscn_raw": "Raw ISCN notation.",
    "clinvar_concept_id": "ClinVar concept identifier.",
    "omop_genomic_concept_id": "OMOP Genomic concept identifier.",
    "snomed_code": "SNOMED finding code.",
    "evidence_text": "Minimal narrative evidence span.",
    "evidence_start": "Evidence start offset.",
    "evidence_end": "Evidence end offset.",
    "parser_profile_id": "Parser profile.",
    "parser_version": "Parser implementation version.",
    "review_status": "Source finding review status.",
    "lifecycle_status": "Inherited report lifecycle.",
    "is_current": "Whether the source report version is current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized finding lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.genomic_result"),
    comment="One reportable molecular/cytogenetic finding. Schema-first: zero rows in both "
            "catalogs as of 2026-08-18 (empty_by_design — no detected assays upstream).",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=GENOMIC_RESULT_COLUMN_COMMENTS,
)
def genomic_result():
    return _genomic_result_canonical().select(*GENOMIC_RESULT_PUBLIC_COLUMNS)

In [0]:
SRC_PATHOLOGY_GENE_TESTED = "4_prod.bronze.map_pathology_gene_tested"

GENE_TESTED_COLUMN_COMMENTS = {
    "gene_tested_row_id": "Stable assay-gene reference row identifier.",
    "source_gene_tested_id": "Bronze gene-tested identifier.",
    "genomic_test_id": "Parent genomic-test fact identifier.",
    "source_genetic_test_id": "Bronze parent genetic-test identifier.",
    "hgnc_id": "Governed HGNC identifier.",
    "reported_gene_symbol": "Gene symbol as reported or configured.",
    "normalized_gene_symbol": "Current approved HGNC symbol.",
    "alias_match_type": "HGNC symbol-resolution route.",
    "evidence_type": "Source of assay-gene membership.",
    "test_scope": "Reported exon",
    "panel_version_inferred": "Whether membership used an inferred panel version.",
    "confidence": "Deterministic normalization confidence.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_reference.gene_tested"),
    comment="Assay-gene denominator, HGNC-normalized; reference surface, no event index. "
            "Covers 42.6% of assays in the current build (BCRL/FLT3 complete; "
            "MNGS 10.5%, TNGS 16.6%) — recorded, not gated.",
    refresh_policy="incremental",
    column_comments=GENE_TESTED_COLUMN_COMMENTS,
)
def gene_tested():
    g = read_source(SRC_PATHOLOGY_GENE_TESTED)
    return g.select(
        stable_id("gene_tested:assay-gene", g.gene_tested_id).alias("gene_tested_row_id"),
        g.gene_tested_id.alias("source_gene_tested_id"),
        stable_id("genomic_test:assay", g.genetic_test_id).alias("genomic_test_id"),
        g.genetic_test_id.alias("source_genetic_test_id"),
        g.hgnc_id, g.reported_gene_symbol, g.normalized_gene_symbol,
        g.alias_match_type, g.evidence_type, g.test_scope,
        g.panel_version_inferred, g.confidence,
        g.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
def _indication_canonical_pregate():
    x = _alias_resolved_accession(read_source(SRC_PATHOLOGY_INDICATION)).alias("x")
    i = _genomic_identity_stage().alias("i")
    j = x.join(i, F.col("x._resolved_accession_id") == F.col("i._identity_accession_id"), "left")
    event_id = stable_id("indication:evidence", F.col("x.indication_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_INDICATION, F.col("x.indication_id"),
    )
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        *_identity_mapped_columns(),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce(F.col("_acc_report_dt"), F.col("_acc_sample_dt"),
                               F.col("_acc_request_dt"))).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:indication-text").alias("source_coding_system"),
        F.col("x.source_text").cast("string").alias("source_code"),
        F.col("x.source_text").alias("source_display"),
        F.col("x.relation_type"), F.col("x.source_field"), F.col("x.source_text"),
        F.col("x.evidence_text"), F.col("x.evidence_start"), F.col("x.evidence_end"),
        F.col("x.snomed_code").cast("string").alias("snomed_code"),
        F.col("x.snomed_term"),
        F.col("x.omop_concept_id").cast("string").alias("omop_concept_id"),
        F.col("x.assertion"), F.col("x.temporality"), F.col("x.experiencer"),
        F.col("x.rule_id"), F.col("x.rule_version"), F.col("x.confidence"),
        F.col("x.mapping_status"), F.col("x.ig_release_status"),
        F.col("x.is_current"), F.col("x.research_qi_only"),
        F.when(F.col("x._resolved_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("x._resolved_accession_id")))
         .alias("specimen_id"),
        F.col("x._resolved_accession_id").alias("accession_identifier"),
        F.lit("active").alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_indication").alias("source_feed"),
        F.date_format(F.col("x.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("x.ADC_UPDT").alias("source_update_timestamp"),
        F.col("x.ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_INDICATION).alias("_source_table"),
        F.col("x.indication_id").alias("_source_row_id"),
        F.col("x.snomed_code").cast("string").alias("_ind_snomed_code"),
        F.coalesce(F.col("x.snomed_term"), F.col("x.source_text")).alias("_ind_snomed_display"),
        F.col("x.omop_concept_id").cast("string").alias("_ind_omop_code"),
        F.coalesce(F.col("x.snomed_term"), F.col("x.source_text")).alias("_ind_omop_display"),
    )

def _indication_canonical():
    return _indication_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_PATHOLOGY_INDICATION = "4_prod.bronze.map_pathology_indication"

INDICATION_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "relation_type", "source_field", "source_text", "evidence_text",
    "evidence_start", "evidence_end", "snomed_code", "snomed_term", "omop_concept_id",
    "assertion", "temporality", "experiencer", "rule_id", "rule_version", "confidence",
    "mapping_status", "ig_release_status", "is_current", "research_qi_only",
    "specimen_id", "accession_identifier",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

INDICATION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable indication event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession report/sample/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Indication text namespace.",
    "source_code": "Coded diagnosis display string.",
    "source_display": "Coded diagnosis display string.",
    "relation_type": "Evidence relation type.",
    "source_field": "Source evidence field.",
    "source_text": "Unmodified coded diagnosis display string.",
    "evidence_text": "Exact evidence span.",
    "evidence_start": "Evidence start offset.",
    "evidence_end": "Evidence end offset.",
    "snomed_code": "SNOMED condition code.",
    "snomed_term": "SNOMED display term.",
    "omop_concept_id": "Standard OMOP condition concept.",
    "assertion": "Evidence assertion.",
    "temporality": "Evidence temporality.",
    "experiencer": "Evidence experiencer.",
    "rule_id": "Deterministic rule identifier.",
    "rule_version": "Deterministic rule version.",
    "confidence": "Rule-specific confidence.",
    "mapping_status": "Source mapping status.",
    "ig_release_status": "IG release status published verbatim.",
    "is_current": "Whether evidence remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized evidence lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.indication"),
    comment="Accession-scoped diagnosis-context evidence (single deterministic lane "
            "diagnosis_context_window_v1). Source/evidence text are coded diagnosis display "
            "strings; ig_release_status published verbatim; prod activation is gated.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=INDICATION_COLUMN_COMMENTS,
)
def indication():
    return _indication_canonical().select(*INDICATION_PUBLIC_COLUMNS)

In [0]:
def _micro_isolate_canonical_pregate():
    m = _alias_resolved_accession(read_source(SRC_PATHOLOGY_MICRO_ISOLATE)).alias("m")
    i = _genomic_identity_stage().alias("i")
    j = m.join(i, F.col("m._resolved_accession_id") == F.col("i._identity_accession_id"), "left")
    event_id = stable_id("microbiology_isolate:isolate", F.col("m.microbiology_isolate_id"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_MICRO_ISOLATE, F.col("m.microbiology_isolate_id"),
    )
    retracted = F.col("m.lifecycle_status").isin("cancelled", "entered_in_error")
    superseded = ~F.coalesce(F.col("m.is_current"), F.lit(True))
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        *_identity_mapped_columns(),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce(F.col("_acc_sample_dt"), F.col("_acc_report_dt"),
                               F.col("_acc_request_dt"))).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.when(F.col("m.organism_text").isNotNull(),
               F.lit("urn:barts:pathology:organism-text"))
         .when(F.col("m.organism_snomed_code").isNotNull(),
               F.lit("http://snomed.info/sct"))
         .otherwise(F.lit("urn:omop:concept_id"))
         .alias("source_coding_system"),
        F.coalesce(F.col("m.organism_text"),
                   F.col("m.organism_snomed_code").cast("string"),
                   F.col("m.organism_omop_concept_id").cast("string"))
         .alias("source_code"),
        F.coalesce(F.col("m.organism_text"),
                   F.col("m.organism_snomed_code").cast("string"))
         .alias("source_display"),
        F.when(F.col("m.source_record_key").isNotNull(),
               stable_id("pathology_result:pathology", F.col("m.source_record_key")))
         .alias("pathology_result_id"),
        F.col("m.report_version_id"), F.col("m.specimen_type_code"), F.col("m.organism_text"),
        F.col("m.organism_snomed_code").cast("string").alias("organism_snomed_code"),
        F.col("m.organism_omop_concept_id").cast("string").alias("organism_omop_concept_id"),
        F.col("m.suspected_ind"), F.col("m.growth_grade"),
        F.col("m.lifecycle_status"), F.col("m.is_current"), F.col("m.research_qi_only"),
        F.when(F.col("m._resolved_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("m._resolved_accession_id")))
         .alias("specimen_id"),
        F.col("m._resolved_accession_id").alias("accession_identifier"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_microbiology_isolate").alias("source_feed"),
        F.date_format(F.col("m.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("m.ADC_UPDT").alias("source_update_timestamp"),
        F.col("m.ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_MICRO_ISOLATE).alias("_source_table"),
        F.col("m.microbiology_isolate_id").alias("_source_row_id"),
        F.col("m.organism_snomed_code").cast("string").alias("_iso_snomed_code"),
        F.col("m.organism_text").alias("_iso_snomed_display"),
        F.col("m.organism_omop_concept_id").cast("string").alias("_iso_omop_code"),
        F.col("m.organism_text").alias("_iso_omop_display"),
    )

def _micro_isolate_canonical():
    return _micro_isolate_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_PATHOLOGY_MICRO_ISOLATE = "4_prod.bronze.map_pathology_microbiology_isolate"

MICRO_ISOLATE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "pathology_result_id", "report_version_id", "specimen_type_code", "organism_text",
    "organism_snomed_code", "organism_omop_concept_id", "suspected_ind", "growth_grade",
    "lifecycle_status", "is_current", "research_qi_only",
    "specimen_id", "accession_identifier",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

MICROBIOLOGY_ISOLATE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable isolate event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession sample/report/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Organism coding namespace.",
    "source_code": "Organism text or code.",
    "source_display": "Organism display.",
    "pathology_result_id": "Linked pathology-result fact.",
    "report_version_id": "Related report version.",
    "specimen_type_code": "Source specimen type.",
    "organism_text": "Organism as reported.",
    "organism_snomed_code": "SNOMED organism code.",
    "organism_omop_concept_id": "Standard OMOP organism concept.",
    "suspected_ind": "Whether the organism is hedged or suspected.",
    "growth_grade": "Reported growth grade.",
    "lifecycle_status": "Inherited source lifecycle.",
    "is_current": "Whether the isolate remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized isolate lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.microbiology_isolate"),
    comment="One organism/isolate finding per accession. Schema-first: empty_by_design "
            "(0 rows in both catalogs as of 2026-08-18).",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=MICROBIOLOGY_ISOLATE_COLUMN_COMMENTS,
)
def microbiology_isolate():
    return _micro_isolate_canonical().select(*MICRO_ISOLATE_PUBLIC_COLUMNS)

In [0]:
def _susceptibility_canonical_pregate():
    a = _alias_resolved_accession(
        read_source(SRC_PATHOLOGY_ANTIMICROBIAL_SUSCEPTIBILITY)
    ).alias("a")
    i = _genomic_identity_stage().alias("i")
    j = a.join(i, F.col("a._resolved_accession_id") == F.col("i._identity_accession_id"), "left")
    event_id = stable_id(
        "susceptibility_result:observation", F.col("a.susceptibility_result_id")
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("canonical_person_id")),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_ANTIMICROBIAL_SUSCEPTIBILITY,
        F.col("a.susceptibility_result_id"),
    )
    retracted = F.col("a.lifecycle_status").isin("cancelled", "entered_in_error")
    superseded = ~F.coalesce(F.col("a.is_current"), F.lit(True))
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        *_identity_mapped_columns(),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(F.coalesce(F.col("_acc_sample_dt"), F.col("_acc_report_dt"),
                               F.col("_acc_request_dt"))).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:antimicrobial").alias("source_coding_system"),
        F.coalesce(F.col("a.antimicrobial_code"), F.col("a.antimicrobial_text"))
         .cast("string").alias("source_code"),
        F.coalesce(F.col("a.antimicrobial_text"), F.col("a.antimicrobial_code"))
         .alias("source_display"),
        F.when(F.col("a.microbiology_isolate_id").isNotNull(),
               stable_id("microbiology_isolate:isolate", F.col("a.microbiology_isolate_id")))
         .alias("microbiology_isolate_id"),
        F.when(F.col("a.source_record_key").isNotNull(),
               stable_id("pathology_result:pathology", F.col("a.source_record_key")))
         .alias("pathology_result_id"),
        F.col("a.link_status"), F.col("a.antimicrobial_text"),
        F.col("a.antimicrobial_code"),
        F.col("a.antimicrobial_omop_concept_id").cast("string")
         .alias("antimicrobial_omop_concept_id"),
        F.col("a.interpretation_raw"), F.col("a.interpretation"),
        F.col("a.mic_raw"), F.col("a.mic"), F.col("a.unit_source_value"), F.col("a.method"),
        F.col("a.lifecycle_status"), F.col("a.is_current"), F.col("a.research_qi_only"),
        F.when(F.col("a._resolved_accession_id").isNotNull(),
               stable_id("specimen:accession", F.col("a._resolved_accession_id")))
         .alias("specimen_id"),
        F.col("a._resolved_accession_id").alias("accession_identifier"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pathology_antimicrobial_susceptibility").alias("source_feed"),
        F.date_format(F.col("a.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("a.ADC_UPDT").alias("source_update_timestamp"),
        F.col("a.ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_ANTIMICROBIAL_SUSCEPTIBILITY).alias("_source_table"),
        F.col("a.susceptibility_result_id").alias("_source_row_id"),
        F.col("a.antimicrobial_code").cast("string").alias("_sus_code_code"),
        F.col("a.antimicrobial_text").alias("_sus_code_display"),
        F.col("a.antimicrobial_omop_concept_id").cast("string").alias("_sus_omop_code"),
        F.col("a.antimicrobial_text").alias("_sus_omop_display"),
    )

def _susceptibility_canonical():
    return _susceptibility_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_PATHOLOGY_ANTIMICROBIAL_SUSCEPTIBILITY = "4_prod.bronze.map_pathology_antimicrobial_susceptibility"

SUSCEPTIBILITY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "microbiology_isolate_id", "pathology_result_id", "link_status",
    "antimicrobial_text", "antimicrobial_code", "antimicrobial_omop_concept_id",
    "interpretation_raw", "interpretation", "mic_raw", "mic", "unit_source_value", "method",
    "lifecycle_status", "is_current", "research_qi_only",
    "specimen_id", "accession_identifier",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

SUSCEPTIBILITY_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable susceptibility event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession sample/report/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Antimicrobial coding namespace.",
    "source_code": "Antimicrobial code or text.",
    "source_display": "Antimicrobial display.",
    "microbiology_isolate_id": "Linked isolate fact when unique.",
    "pathology_result_id": "Linked pathology-result fact.",
    "link_status": "Isolate-link resolution state.",
    "antimicrobial_text": "Antimicrobial as reported.",
    "antimicrobial_code": "Source antimicrobial code.",
    "antimicrobial_omop_concept_id": "Standard OMOP antimicrobial concept.",
    "interpretation_raw": "Raw susceptibility result.",
    "interpretation": "Normalized susceptibility interpretation.",
    "mic_raw": "Raw MIC text.",
    "mic": "Parsed MIC value.",
    "unit_source_value": "Raw MIC unit.",
    "method": "Susceptibility method.",
    "lifecycle_status": "Inherited source lifecycle.",
    "is_current": "Whether the result remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized susceptibility lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.susceptibility_result"),
    comment="One isolate-antimicrobial susceptibility observation. Schema-first: "
            "empty_by_design (0 rows in both catalogs as of 2026-08-18).",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=SUSCEPTIBILITY_RESULT_COLUMN_COMMENTS,
)
def susceptibility_result():
    return _susceptibility_canonical().select(*SUSCEPTIBILITY_PUBLIC_COLUMNS)

In [0]:
REQUEST_THREAD_PUBLIC_COLUMNS = [
    "request_thread_id", "source_patient_event_id", "target_patient_event_id",
    "link_type_code", "subject_key", "person_id", "encounter_id",
    "request_identifier", "response_identifier", "requested_datetime", "responded_datetime",
    "source_history_row_count", "record_status", "record_status_effective_from",
    "record_status_effective_to", "source_system", "source_table", "source_row_id",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _pathology_request_thread_canonical():
    t = _pathology_requested_test_canonical().select(
        F.col("patient_event_id").alias("source_patient_event_id"),
        F.col("_source_row_id").alias("request_identifier"),
        "subject_key", "person_id", "specimen_id",
        F.col("event_datetime").alias("requested_datetime"),
        "load_batch_id", "loaded_at",
    )
    r = _pathology_report_series_canonical().select(
        F.col("patient_event_id").alias("target_patient_event_id"),
        F.col("_source_row_id").alias("response_identifier"),
        F.col("specimen_id").alias("_r_specimen_id"),
        F.col("event_datetime").alias("responded_datetime"),
        F.col("loaded_at").alias("_r_loaded_at"),
    )
    j = t.join(r, t["specimen_id"] == r["_r_specimen_id"], "inner")
    return j.select(
        stable_id(
            "request_thread:pathology", F.col("source_patient_event_id"),
            F.col("target_patient_event_id"), F.lit("order_to_report"),
        ).alias("request_thread_id"),
        "source_patient_event_id", "target_patient_event_id",
        F.lit("order_to_report").alias("link_type_code"),
        "subject_key", "person_id", F.lit(None).cast("string").alias("encounter_id"),
        "request_identifier", "response_identifier", "requested_datetime",
        "responded_datetime", F.lit(1).cast("long").alias("source_history_row_count"),
        F.lit("active").alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit("laboratory").alias("source_system"),
        F.lit(SRC_PATHOLOGY_REQUESTED_TEST).alias("source_table"),
        F.concat_ws("|", "request_identifier", "response_identifier").alias("source_row_id"),
        "load_batch_id", F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        F.greatest("loaded_at", "_r_loaded_at").alias("loaded_at"),
    )

REQUEST_THREAD_COLUMN_COMMENTS = {
    "request_thread_id": "Deterministic request-edge primary key.",
    "source_patient_event_id": "Request-side patient event identifier.",
    "target_patient_event_id": "Response-side patient event identifier.",
    "link_type_code": "Controlled directed relationship type.",
    "subject_key": "Always-populated subject key copied from request evidence.",
    "person_id": "Resolved person identifier when available.",
    "encounter_id": "Encounter context when supplied by source evidence.",
    "request_identifier": "Verbatim source order identifier.",
    "response_identifier": "Verbatim source report-parent identifier.",
    "requested_datetime": "Source request timestamp.",
    "responded_datetime": "Source report or measurement timestamp.",
    "source_history_row_count": "Number of source rows supporting the edge.",
    "record_status": "Normalized edge lifecycle from source validity.",
    "record_status_effective_from": "Earliest supporting source validity start.",
    "record_status_effective_to": "Source validity end when superseded.",
    "source_system": "Source system identifier.",
    "source_table": "Fully qualified configured source table.",
    "source_row_id": "Representative stable source row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp supporting the edge.",
}

@materialized_view(
    name=_n("journey_spine.request_thread"),
    comment="Directed request-to-response edges: pathology order-to-report, medication order-to-administration, waiting-list entry-to-appointment.",
    refresh_policy="incremental",
    column_comments=REQUEST_THREAD_COLUMN_COMMENTS,
)
def request_thread():
    pathology = _pathology_request_thread_canonical().select(*REQUEST_THREAD_PUBLIC_COLUMNS)
    medication = _medication_request_thread_canonical().select(*REQUEST_THREAD_PUBLIC_COLUMNS)
    waiting_list = _waiting_list_appointment_thread_canonical().select(*REQUEST_THREAD_PUBLIC_COLUMNS)
    return pathology.unionByName(medication).unionByName(waiting_list)

def _medication_request_thread_canonical():
    a = read_source(SRC_MED_ADMIN).alias("a")
    o = read_source(SRC_MEDICATION_ORDER).alias("o")
    joined = a.where(F.col("a.ORDER_ID").isNotNull()).join(
        o, F.col("a.ORDER_ID") == F.col("o.ORDER_ID"), "inner"
    )
    source_event_id = stable_id("medication_order:mill", F.col("o.ORDER_ID"))
    target_event_id = stable_id("medication_admin:mill", F.col("a.EVENT_ID"))
    skey, _ = subject_key_with_system(
        [("urn:cerner:person_id", F.coalesce(F.col("a.PERSON_ID"), F.col("o.PERSON_ID")))],
        SRC_MED_ADMIN, F.col("a.EVENT_ID"),
    )
    retracted = F.coalesce(F.col("o.SOURCE_PRESENT_IND"), F.lit(True)) == F.lit(False)
    ended = F.col("a.CE_VALID_UNTIL_DT_TM").isNotNull() & (
        F.col("a.CE_VALID_UNTIL_DT_TM") < F.lit("2100-01-01").cast("timestamp")
    )
    loaded_at = F.greatest(F.col("a.ADC_UPDT"), F.col("o.ADC_UPDT"))
    source_update = F.greatest(F.col("a.ADC_UPDT"), F.col("o.SOURCE_ADC_UPDT"),
                               F.col("o.ADC_UPDT"))
    return joined.select(
        stable_id("request_thread:medication", source_event_id, target_event_id,
                  F.lit("order_to_administration")).alias("request_thread_id"),
        source_event_id.alias("source_patient_event_id"),
        target_event_id.alias("target_patient_event_id"),
        F.lit("order_to_administration").alias("link_type_code"), skey.alias("subject_key"),
        F.coalesce(F.col("a.PERSON_ID"), F.col("o.PERSON_ID")).cast("string").alias("person_id"),
        F.when(F.coalesce(F.col("a.ENCNTR_ID"), F.col("o.ENCNTR_ID")).isNotNull(),
               stable_id("encounter:mill", F.coalesce(F.col("a.ENCNTR_ID"), F.col("o.ENCNTR_ID"))))
         .alias("encounter_id"),
        F.col("o.ORDER_ID").cast("string").alias("request_identifier"),
        F.col("a.EVENT_ID").cast("string").alias("response_identifier"),
        F.col("o.ORIG_ORDER_DT_TM").alias("requested_datetime"),
        F.coalesce(F.col("a.ADMIN_START_DT_TM"), F.col("a.PERFORMED_DT_TM"))
         .alias("responded_datetime"),
        F.lit(2).cast("long").alias("source_history_row_count"),
        F.when(retracted, F.lit("retracted")).when(ended, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.col("o.ORIG_ORDER_DT_TM").alias("record_status_effective_from"),
        F.when(retracted, F.col("o.SOURCE_ABSENT_DETECTED_TS"))
         .when(ended, F.col("a.CE_VALID_UNTIL_DT_TM")).alias("record_status_effective_to"),
        F.lit("millennium").alias("source_system"),
        F.concat_ws("|", F.lit(SRC_MEDICATION_ORDER), F.lit(SRC_MED_ADMIN)).alias("source_table"),
        F.col("a.EVENT_ID").cast("string").alias("source_row_id"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"),
        loaded_at.alias("loaded_at"),
    )

def _waiting_list_appointment_thread_canonical():
    # One edge per entry: representative (CURRENT wins) with a non-zero SCH_EVENT_ID,
    # existence-gated against the appointment source so no dangling target id is minted,
    # AND person-gated so an edge never links two different patients (the edge copies the
    # waiting-list subject; an id-only join would silently cross patients).
    # Full-volume 2026-08-12: 14,312,122/14,333,454 distinct ids resolve (99.85%);
    # of 14,324,720 matched pairs 14,271,836 are person-concordant (99.63%),
    # 52,841 appointment-person NULL and 43 discordant are excluded here.
    rep = _waiting_list_index_representative().where(
        F.col("sch_event_id").isNotNull() & (F.col("sch_event_id") != 0)
    ).alias("rep")
    appt = (
        read_source(SRC_APPOINTMENT)
        .where(F.col("SCH_EVENT_ID").isNotNull() & F.col("PERSON_ID").isNotNull())
        .select(
            F.col("SCH_EVENT_ID").cast("long").alias("_appt_sch_event_id"),
            F.col("PERSON_ID").cast("string").alias("_appt_person_id"),
        )
        .distinct()
        .alias("appt")
    )
    joined = rep.join(
        appt,
        (F.col("rep.sch_event_id") == F.col("_appt_sch_event_id"))
        & (F.col("rep.person_id") == F.col("_appt_person_id")),
        "inner",
    )
    target_id = stable_id("appointment:mill_scheduling", F.col("_appt_sch_event_id"))
    return joined.select(
        stable_id(
            "request_thread:waiting_list", F.col("rep.patient_event_id"), target_id,
            F.lit("waiting_list_to_appointment"),
        ).alias("request_thread_id"),
        F.col("rep.patient_event_id").alias("source_patient_event_id"),
        target_id.alias("target_patient_event_id"),
        F.lit("waiting_list_to_appointment").alias("link_type_code"),
        F.col("rep.subject_key").alias("subject_key"),
        F.col("rep.person_id").alias("person_id"),
        F.col("rep.encounter_id").alias("encounter_id"),
        F.col("rep.pm_wait_list_id").cast("string").alias("request_identifier"),
        F.col("rep.sch_event_id").cast("string").alias("response_identifier"),
        F.col("rep.event_datetime").alias("requested_datetime"),
        F.col("rep.scheduled_datetime").alias("responded_datetime"),
        F.lit(1).cast("long").alias("source_history_row_count"),
        F.col("rep.record_status").alias("record_status"),
        F.col("rep.record_status_effective_from").alias("record_status_effective_from"),
        F.col("rep.record_status_effective_to").alias("record_status_effective_to"),
        F.lit("millennium-pm").alias("source_system"),
        F.lit(SRC_WAITING_LIST).alias("source_table"),
        F.concat_ws(
            ":", F.col("rep.pm_wait_list_id").cast("string"), F.col("rep.row_source"),
            F.col("rep.source_version_id").cast("string"),
        ).alias("source_row_id"),
        F.col("rep.load_batch_id").alias("load_batch_id"),
        F.col("rep.source_update_timestamp").alias("source_update_timestamp"),
        F.col("rep.loaded_at").alias("loaded_at"),
    )

In [0]:
SRC_FORM_ITEM = "4_prod.bronze.map_powerform_assessment_item"
SRC_FORM_ASSESSMENT = "4_prod.bronze.map_powerform_assessment"
SRC_FORM_VTE = "4_prod.bronze.map_mat_vte_assessment"

def _form_grouped_query():
    a = read_source(SRC_FORM_ACTIVITY).alias("a")
    i = read_source(SRC_FORM_ITEM).alias("i")
    g = read_source(SRC_FORM_ASSESSMENT).alias("g")
    v = read_source(SRC_FORM_VTE).select(
        "DOC_RESPONSE_KEY", "Pregnancy_ID", "PregnancyMatchMethod", "Source_ADC_UPDT"
    ).alias("v")
    joined = (
        a.join(i, a.DOC_RESPONSE_KEY == i.DOC_RESPONSE_KEY, "left")
         .join(g, a.DCP_FORMS_ACTIVITY_ID == g.DCP_FORMS_ACTIVITY_ID, "left")
         .join(v, a.DOC_RESPONSE_KEY == v.DOC_RESPONSE_KEY, "left")
    )
    response = F.struct(
        F.coalesce(a.RESPONSE_SEQUENCE_NBR, F.lit(0)).cast("long").alias("sequence"),
        a.DOC_RESPONSE_KEY.alias("response_id"),
        a.SECTION_DESC_TXT.alias("section"),
        a.SECTION_REF_ID.cast("string").alias("section_id"),
        a.DCP_INPUT_REF_ID.cast("string").alias("element_id"),
        a.ELEMENT_LABEL_TXT.alias("element_label"),
        a.GRID_NAME_TXT.alias("grid_name"),
        a.GRID_ROW_DESC_TXT.alias("grid_row"),
        i.RESPONSE_KIND.alias("response_kind"),
        F.coalesce(i.CANONICAL_TEXT_RESULT, i.RESPONSE_TEXT_TXT,
                   a.RESPONSE_VALUE_TXT, a.STRING_RESPONSE_TXT).alias("response_value_text"),
        F.coalesce(i.CANONICAL_NUMERIC_RESULT, a.NUMERIC_RESPONSE_NBR)
         .cast("decimal(38,10)").alias("response_value_number"),
        F.coalesce(i.CANONICAL_DATE_RESULT, i.RESPONSE_DT_TM).alias("response_value_datetime"),
        F.when(i.RESPONSE_NOMENCLATURE_ID.isNotNull(), F.lit("urn:cerner:nomenclature"))
         .when(i.RESPONSE_CODE_VALUE_ID.isNotNull(), F.lit("urn:cerner:code_value"))
         .alias("response_coding_system"),
        F.coalesce(i.RESPONSE_NOMENCLATURE_ID, i.RESPONSE_CODE_VALUE_ID)
         .cast("string").alias("response_coding_code"),
        i.RESPONSE_TEXT_TXT.alias("response_coding_display"),
        i.QUESTION_CONCEPT_ID.cast("string").alias("question_concept_id"),
        i.VALUE_CONCEPT_ID.cast("string").alias("value_concept_id"),
        i.UNIT_CONCEPT_ID.cast("string").alias("unit_concept_id"),
        v.Pregnancy_ID.cast("string").alias("pregnancy_id"),
        v.PregnancyMatchMethod.alias("pregnancy_match_method"),
        (a.ACTIVE_IND == 1).alias("active_ind"),
        i.SOURCE_PRESENT_IND.alias("source_present_ind"),
        i.CANONICAL_SOURCE_DELETED_IND.alias("source_deleted_ind"),
        i.CANONICAL_MATCH_STATUS.alias("canonical_match_status"),
    )
    empty_response = (
        a.RESPONSE_VALUE_TXT.isNull() & a.STRING_RESPONSE_TXT.isNull()
        & a.NUMERIC_RESPONSE_NBR.isNull() & a.RESPONSE_NOMENCLATURE_ID.isNull()
        & a.RESPONSE_CODE_VALUE_ID.isNull()
    )
    grouped = joined.groupBy(a.DCP_FORMS_ACTIVITY_ID).agg(
        F.max(a.PERSON_ID_LONG).alias("PERSON_ID"),
        F.max(a.ENCNTR_ID_LONG).alias("ENCNTR_ID"),
        F.max(a.ORGANIZATION_ID).alias("ORGANIZATION_ID"),
        F.max(a.FORM_REF_ID).alias("FORM_REF_ID"),
        F.max(a.FORM_STATUS_CD).alias("FORM_STATUS_CD"),
        F.max(a.STATUS).alias("STATUS"),
        F.max(a.FORM_DESC_TXT).alias("FORM_DESC_TXT"),
        F.min(a.DOCUMENTATION_DT_TM).alias("authored_datetime"),
        F.max(F.coalesce(a.LAST_DOCUMENTED_DT_TM, a.PERFORMED_DT_TM))
         .alias("completed_datetime"),
        F.max(a.PERFORMED_PRSNL_ID_LONG).alias("PERFORMED_PRSNL_ID"),
        F.parse_json(F.to_json(F.sort_array(F.collect_list(response)))).alias("responses"),
        F.count(F.lit(1)).cast("long").alias("response_row_count"),
        F.sum(F.when(a.ACTIVE_IND == 1, F.lit(1)).otherwise(F.lit(0)))
         .cast("long").alias("active_response_row_count"),
        F.sum(F.when(empty_response, F.lit(1)).otherwise(F.lit(0)))
         .cast("long").alias("empty_response_row_count"),
        F.max(g.INVALID_ROW_COUNT).cast("long").alias("invalid_response_row_count"),
        F.max(g.CANONICAL_MATCHED_ROW_COUNT).cast("long").alias("matched_response_row_count"),
        F.max(g.CANONICAL_UNMATCHED_ROW_COUNT).cast("long").alias("unmatched_response_row_count"),
        F.max(g.CONTEXT_CONFLICT_IND.cast("int")).alias("context_conflict_int"),
        F.max(g.CONTEXT_QUARANTINED_IND.cast("int")).alias("context_quarantined_int"),
        F.max(g.ASSESSMENT_ACTIVE_IND.cast("int")).alias("assessment_active_int"),
        F.max(g.SOURCE_PRESENT_IND.cast("int")).alias("source_present_int"),
        F.max(F.greatest(a.ADC_UPDT, a.SOURCE_ACTIVITY_ADC_UPDT,
                         a.SOURCE_EVENT_ADC_UPDT, a.SOURCE_LABEL_ADC_UPDT))
         .alias("source_update_timestamp"),
        F.max(a.ADC_UPDT).alias("activity_loaded_at"),
        F.max(i.ADC_UPDT).alias("item_loaded_at"),
        F.max(g.ADC_UPDT).alias("assessment_loaded_at"),
        F.max(v.Source_ADC_UPDT).alias("vte_loaded_at"),
    )
    return grouped

@materialized_view(
    name=_n("journey_clinical._form_grouped"),
    private=True,
    comment="Internal top-level aggregate for incrementally maintained ordered PowerForm responses.",
    refresh_policy="incremental",
)
def _form_grouped():
    return _form_grouped_query()

In [0]:
def _form_canonical_pregate():
    grouped = spark.read.table(_n("journey_clinical._form_grouped"))
    event_id = stable_id("form:mill:powerform", F.col("DCP_FORMS_ACTIVITY_ID"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("PERSON_ID"))],
        SRC_FORM_ACTIVITY, F.col("DCP_FORMS_ACTIVITY_ID"),
    )
    retracted = F.coalesce(F.col("source_present_int") == 0, F.lit(False))
    superseded = F.coalesce(F.col("assessment_active_int") == 0, F.lit(False)) | (
        F.col("active_response_row_count") == 0
    )
    loaded_at = F.greatest(
        "activity_loaded_at", "item_loaded_at", "assessment_loaded_at", "vte_loaded_at"
    )
    return grouped.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("PERSON_ID").cast("string").alias("person_id"),
        F.when(_present(F.col("PERSON_ID")), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("ENCNTR_ID").isNotNull(), stable_id("encounter:mill", F.col("ENCNTR_ID")))
         .alias("encounter_id"),
        F.col("authored_datetime").alias("event_datetime"),
        F.col("completed_datetime").alias("event_end_datetime"),
        F.lit("urn:cerner:form_ref_id").alias("source_coding_system"),
        F.col("FORM_REF_ID").cast("string").alias("source_code"),
        F.col("FORM_DESC_TXT").alias("source_display"),
        F.col("FORM_REF_ID").cast("string").alias("form_type_code"),
        F.col("FORM_DESC_TXT").alias("form_type_display"),
        F.col("FORM_STATUS_CD").cast("string").alias("form_status_code"),
        F.col("STATUS").alias("form_status_display"),
        F.col("authored_datetime"), F.col("completed_datetime"),
        F.when(F.col("PERFORMED_PRSNL_ID").isNotNull(),
               stable_id("practitioner:mill", F.col("PERFORMED_PRSNL_ID")))
         .alias("performed_practitioner_id"),
        F.when(F.col("ORGANIZATION_ID").isNotNull(),
               stable_id("organization:mill", F.col("ORGANIZATION_ID"))).alias("organization_id"),
        F.col("responses"),
        F.col("response_row_count"), F.col("active_response_row_count"),
        F.col("empty_response_row_count"), F.coalesce("invalid_response_row_count", F.lit(0)).alias("invalid_response_row_count"),
        F.coalesce("matched_response_row_count", F.lit(0)).alias("matched_response_row_count"),
        F.coalesce("unmatched_response_row_count", F.lit(0)).alias("unmatched_response_row_count"),
        (F.coalesce("context_conflict_int", F.lit(0)) != 0).alias("context_conflict_ind"),
        (F.coalesce("context_quarantined_int", F.lit(0)) != 0).alias("context_quarantined_ind"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.col("authored_datetime").alias("record_status_effective_from"),
        F.when(retracted | superseded, F.col("completed_datetime")).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("powerform").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("source_update_timestamp"), loaded_at.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_FORM_ACTIVITY).alias("_source_table"),
        F.col("DCP_FORMS_ACTIVITY_ID").cast("string").alias("_source_row_id"),
    )

def _form_canonical():
    return _form_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_form"),
    comment="Private JSON bridge and Gold cross-rule flags for form.",
    refresh_policy="incremental",
)
def _qc_form():
    return _cross_qc_primitive(
        _form_canonical(),
        "form",
        {"responses": "_qc_responses_json"},
    )

In [0]:
# ==== Forms, vital signs, clinical scores and form-element promotion ====

FORM_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "form_type_code", "form_type_display", "form_status_code", "form_status_display",
    "authored_datetime", "completed_datetime", "performed_practitioner_id", "organization_id",
    "responses", "response_row_count", "active_response_row_count", "empty_response_row_count",
    "invalid_response_row_count", "matched_response_row_count", "unmatched_response_row_count",
    "context_conflict_ind", "context_quarantined_ind",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

FORM_COLUMN_COMMENTS = {
    "patient_event_id": "Stable form event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "First documented timestamp.",
    "event_end_datetime": "Last documented or performed timestamp.",
    "source_coding_system": "Source form coding system.",
    "source_code": "Source form reference identifier.",
    "source_display": "Source form description.",
    "form_type_code": "Source form type code.",
    "form_type_display": "Source form type display.",
    "form_status_code": "Source form status code.",
    "form_status_display": "Source form status display.",
    "authored_datetime": "Earliest form documentation timestamp.",
    "completed_datetime": "Latest documentation or performed timestamp.",
    "performed_practitioner_id": "Performing practitioner reference.",
    "organization_id": "Source organization reference.",
    "responses": "Ordered lossless form responses with typed values and mapping evidence.",
    "response_row_count": "Number of response rows grouped into the form.",
    "active_response_row_count": "Number of active response rows.",
    "empty_response_row_count": "Number of retained empty response rows.",
    "invalid_response_row_count": "Number of source-classified invalid response rows.",
    "matched_response_row_count": "Number of canonical matched response rows.",
    "unmatched_response_row_count": "Number of canonical unmatched response rows.",
    "context_conflict_ind": "Whether source form context conflicts were detected.",
    "context_quarantined_ind": "Whether source form context was quarantined.",
    "record_status": "Normalized form lifecycle.",
    "record_status_effective_from": "Form lifecycle start.",
    "record_status_effective_to": "Form lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest contributing bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.form"),
    comment="One PowerForm instance with ordered lossless responses and reconciliation evidence.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=FORM_COLUMN_COMMENTS,
)
def form():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_form")),
        "form",
        {"responses": "_qc_responses_json"},
        FORM_PUBLIC_COLUMNS,
    )

In [0]:
def _numeric_route_expr(s):
    label = F.lower(F.coalesce(
        s.EVENT_LABEL, s.EVENT_CD_DISPLAY, s.OMOP_MANUAL_CONCEPT_NAME, F.lit("")
    ))
    score = label.rlike("(news|ews|score|scale|glasgow|gcs|braden|waterlow|morse|frailty|risk)")
    vital = label.rlike(
        "^(systolic blood pressure|diastolic blood pressure|mean arterial pressure.*|"
        "respiratory rate|heart rate.*|peripheral pulse rate|pulse rate|spo2|"
        "oxygen saturation.*|temperature .*|weight|height|body mass index|bmi|"
        "capillary refill time actual)$"
    )
    return F.when(score, F.lit("clinical_score")).when(vital, F.lit("vital_sign")).otherwise(F.lit("excluded"))

VITAL_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "vital_code",
    "value_number", "value_text", "unit_source_value", "unit_concept_id",
    "reference_range_low", "reference_range_high", "method_code", "method_display",
    "body_site_code", "body_site_display", "interpretation_code", "interpretation_display",
    "result_status_code", "result_status_display", "performer_practitioner_id",
    "source_form_id", "promotion_rule_id",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

VITAL_STAGE_COLUMNS = VITAL_PUBLIC_COLUMNS + [
    "_source_system", "_source_table", "_source_row_id", "_omop_code", "_omop_display",
]

def _numeric_vital_canonical():
    s = read_source(SRC_NUMERIC_EVENTS)
    s = s.where(_numeric_route_expr(s) == "vital_sign")
    event_id = stable_id("numeric_event:mill", s.EVENT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_NUMERIC_EVENTS, s.EVENT_ID
    )
    deleted = F.coalesce(s.SOURCE_DELETED_IND, F.lit(False))
    ended = s.CLINICAL_EVENT_VALID_UNTIL_DT_TM.isNotNull() & (
        s.CLINICAL_EVENT_VALID_UNTIL_DT_TM < F.lit("2100-01-01").cast("timestamp")
    )
    source_update = F.greatest(
        s.STRING_RESULT_UPDT_DT_TM, s.CLINICAL_EVENT_UPDT_DT_TM, s.ADC_UPDT
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.PERFORMED_DT_TM, s.EVENT_START_DT_TM).alias("event_datetime"),
        s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:event_cd").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"),
        F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY).alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:event_cd"), s.EVENT_CD,
                       F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY), True),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_MANUAL_CONCEPT_ID,
                       s.OMOP_MANUAL_CONCEPT_NAME, False, "bronze.map_numeric_events", None),
        ).alias("vital_code"),
        s.NUMERIC_RESULT.cast("decimal(38,10)").alias("value_number"),
        s.RESULT_TEXT_EFFECTIVE.alias("value_text"),
        F.coalesce(s.UNIT_OF_MEASURE_DISPLAY, s.RESULT_UNITS_DISPLAY).alias("unit_source_value"),
        s.OMOP_MANUAL_UNITS.cast("string").alias("unit_concept_id"),
        s.NORMAL_LOW.cast("decimal(38,10)").alias("reference_range_low"),
        s.NORMAL_HIGH.cast("decimal(38,10)").alias("reference_range_high"),
        s.ENTRY_MODE_CD.cast("string").alias("method_code"), s.ENTRY_MODE_DISPLAY.alias("method_display"),
        F.lit(None).cast("string").alias("body_site_code"),
        F.lit(None).cast("string").alias("body_site_display"),
        s.NORMALCY_CD.cast("string").alias("interpretation_code"),
        s.NORMALCY_DISPLAY.alias("interpretation_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        s.RESULT_STATUS_DISPLAY.alias("result_status_display"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(), stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.lit(None).cast("string").alias("source_form_id"),
        F.lit(None).cast("string").alias("promotion_rule_id"),
        F.when(deleted, F.lit("retracted")).when(ended, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.CLINICAL_EVENT_VALID_FROM_DT_TM.alias("record_status_effective_from"),
        F.when(deleted | ended, s.CLINICAL_EVENT_VALID_UNTIL_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("numeric_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_NUMERIC_EVENTS).alias("_source_table"),
        s.EVENT_ID.cast("string").alias("_source_row_id"),
        s.OMOP_MANUAL_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.OMOP_MANUAL_CONCEPT_NAME.alias("_omop_display"),
    )

In [0]:
@materialized_view(
    name=_n("journey_clinical._numeric_vital_sign"),
    private=True,
    comment="Internal incrementally maintained native numeric-event vital route.",
    refresh_policy="incremental",
)
def _numeric_vital_sign():
    return _numeric_vital_canonical().select(*VITAL_STAGE_COLUMNS)

In [0]:
def _form_promotion_source(target_fact_type):
    i = read_source(SRC_FORM_ITEM).alias("i")
    c = read_source(SRC_FORM_PROMOTION_CONFIG).where(
        F.col("active_ind") & (F.col("target_fact_type") == target_fact_type)
    ).alias("c")
    return i.join(c, i.DCP_INPUT_REF_ID == c.dcp_input_ref_id, "inner")

In [0]:
SRC_FORM_PROMOTION_CONFIG = "3_lookup.omop.form_promotion_config"
PROMOTED_VITAL_STAGE_COLUMNS = [c for c in VITAL_STAGE_COLUMNS if c != "vital_code"]

def _promoted_vital_canonical():
    s = _form_promotion_source("vital_sign")
    event_id = stable_id("form_promotion:vital_sign", F.col("i.DOC_RESPONSE_KEY"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("i.PERSON_ID_LONG"))],
        SRC_FORM_ITEM, F.col("i.DOC_RESPONSE_KEY"),
    )
    deleted = F.coalesce(F.col("i.CANONICAL_SOURCE_DELETED_IND"), F.lit(False))
    absent = ~F.coalesce(F.col("i.SOURCE_PRESENT_IND"), F.lit(False))
    inactive = F.coalesce(F.col("i.ACTIVE_IND"), F.lit(0)) == 0
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("i.PERSON_ID_LONG").cast("string").alias("person_id"),
        F.when(_present(F.col("i.PERSON_ID_LONG")), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(F.col("i.ENCNTR_ID_LONG").isNotNull(),
               stable_id("encounter:mill", F.col("i.ENCNTR_ID_LONG"))).alias("encounter_id"),
        F.coalesce(F.col("i.RESPONSE_DT_TM"), F.col("i.PERFORMED_DT_TM"),
                   F.col("i.DOCUMENTATION_DT_TM")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.col("c.source_coding_system").alias("source_coding_system"),
        F.col("c.source_code").alias("source_code"), F.col("c.source_display").alias("source_display"),
        codeable_concept(
            coding_obj(F.col("c.source_coding_system"), F.col("c.source_code"),
                       F.col("c.source_display"), True)
        ).alias("vital_code"),
        F.coalesce(F.col("i.CANONICAL_NUMERIC_RESULT"), F.col("i.NUMERIC_RESPONSE_NBR"))
         .cast("decimal(38,10)").alias("value_number"),
        F.coalesce(F.col("i.CANONICAL_TEXT_RESULT"), F.col("i.RESPONSE_TEXT_TXT"))
         .alias("value_text"),
        F.lit(None).cast("string").alias("unit_source_value"),
        F.col("i.UNIT_CONCEPT_ID").cast("string").alias("unit_concept_id"),
        F.lit(None).cast("decimal(38,10)").alias("reference_range_low"),
        F.lit(None).cast("decimal(38,10)").alias("reference_range_high"),
        F.lit(None).cast("string").alias("method_code"), F.lit(None).cast("string").alias("method_display"),
        F.lit(None).cast("string").alias("body_site_code"), F.lit(None).cast("string").alias("body_site_display"),
        F.lit(None).cast("string").alias("interpretation_code"),
        F.lit(None).cast("string").alias("interpretation_display"),
        F.col("i.FORM_STATUS_CD").cast("string").alias("result_status_code"),
        F.col("i.STATUS").alias("result_status_display"),
        F.when(F.col("i.PERFORMED_PRSNL_ID_LONG").isNotNull(),
               stable_id("practitioner:mill", F.col("i.PERFORMED_PRSNL_ID_LONG")))
         .alias("performer_practitioner_id"),
        stable_id("form:mill:powerform", F.col("i.DCP_FORMS_ACTIVITY_ID")).alias("source_form_id"),
        F.col("c.promotion_rule_id"),
        F.when(deleted | absent, F.lit("retracted")).when(inactive, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.col("i.FIRST_DOCUMENTED_DT_TM").alias("record_status_effective_from"),
        F.when(deleted | absent | inactive, F.col("i.LAST_DOCUMENTED_DT_TM"))
         .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"), F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("form_promotion").alias("source_feed"),
        F.date_format(F.col("i.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("i.ADC_UPDT").alias("source_update_timestamp"),
        F.col("i.ADC_UPDT").alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_FORM_ITEM).alias("_source_table"),
        F.col("i.DOC_RESPONSE_KEY").alias("_source_row_id"),
        F.col("i.QUESTION_CONCEPT_ID").cast("string").alias("_omop_code"),
        F.col("c.source_display").alias("_omop_display"),
    )

@materialized_view(
    name=_n("journey_clinical._promoted_vital_sign_primitive"),
    private=True,
    comment="Internal incrementally maintained governed form-promotion vital route.",
    refresh_policy="incremental",
)
def _promoted_vital_sign_primitive():
    return _promoted_vital_canonical().select(*PROMOTED_VITAL_STAGE_COLUMNS)

In [0]:
def _promoted_vital_from_stage():
    s = spark.read.table(_n("journey_clinical._promoted_vital_sign_primitive"))
    return s.withColumn(
        "vital_code",
        codeable_concept(
            coding_obj(
                F.col("source_coding_system"), F.col("source_code"),
                F.col("source_display"), True,
            )
        ),
    ).select(*VITAL_STAGE_COLUMNS)

def _vital_sign_canonical():
    return _vital_sign_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
def _vital_sign_canonical_pregate():
    numeric = spark.read.table(_n("journey_clinical._numeric_vital_sign"))
    promoted = _promoted_vital_from_stage()
    return numeric.unionByName(promoted)

@materialized_view(
    name=_n("journey_clinical._qc_vital_sign"),
    comment="Private JSON bridge and Gold cross-rule flags for vital_sign.",
    refresh_policy="incremental",
)
def _qc_vital_sign():
    return _cross_qc_primitive(
        _vital_sign_canonical(),
        "vital_sign",
        {"vital_code": "_qc_vital_code_json"},
    )

In [0]:
VITAL_SIGN_COLUMN_COMMENTS = {
    "patient_event_id": "Stable vital event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Measurement timestamp.",
    "event_end_datetime": "Measurement end timestamp.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source vital code.",
    "source_display": "Source vital display.",
    "vital_code": "Source and mapped vital CodeableConcept.",
    "value_number": "Numeric vital value.",
    "value_text": "Verbatim result text.",
    "unit_source_value": "Source result unit.",
    "unit_concept_id": "Mapped unit concept identifier.",
    "reference_range_low": "Reference-range lower bound.",
    "reference_range_high": "Reference-range upper bound.",
    "method_code": "Source measurement method code.",
    "method_display": "Source measurement method display.",
    "body_site_code": "Body-site code when supplied.",
    "body_site_display": "Body-site display when supplied.",
    "interpretation_code": "Source interpretation code.",
    "interpretation_display": "Source interpretation display.",
    "result_status_code": "Source result status code.",
    "result_status_display": "Source result status display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "source_form_id": "Source form event for promoted measurements.",
    "promotion_rule_id": "Governed promotion-rule identifier.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source route.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.vital_sign"),
    comment="Native numeric-event and governed form-promoted vital measurements.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=VITAL_SIGN_COLUMN_COMMENTS,
)
def vital_sign():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_vital_sign")),
        "vital_sign",
        {"vital_code": "_qc_vital_code_json"},
        VITAL_PUBLIC_COLUMNS,
    )

In [0]:
SCORE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "score_code",
    "score_name", "value_number", "value_text", "unit_source_value", "unit_concept_id",
    "components", "component_count", "interpretation_code", "interpretation_display",
    "result_status_code", "result_status_display", "performer_practitioner_id",
    "source_form_id", "promotion_rule_id",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

SCORE_STAGE_COLUMNS = SCORE_PUBLIC_COLUMNS + [
    "_source_system", "_source_table", "_source_row_id", "_omop_code", "_omop_display",
]

def _numeric_score_canonical():
    s = read_source(SRC_NUMERIC_EVENTS)
    s = s.where(_numeric_route_expr(s) == "clinical_score")
    event_id = stable_id("numeric_event:mill", s.EVENT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_NUMERIC_EVENTS, s.EVENT_ID
    )
    deleted = F.coalesce(s.SOURCE_DELETED_IND, F.lit(False))
    ended = s.CLINICAL_EVENT_VALID_UNTIL_DT_TM.isNotNull() & (
        s.CLINICAL_EVENT_VALID_UNTIL_DT_TM < F.lit("2100-01-01").cast("timestamp")
    )
    component = F.struct(
        F.lit(1).cast("long").alias("sequence"), s.EVENT_ID.cast("string").alias("component_id"),
        F.lit("urn:cerner:event_cd").alias("coding_system"),
        s.EVENT_CD.cast("string").alias("coding_code"),
        F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY).alias("coding_display"),
        s.NUMERIC_RESULT.cast("decimal(38,10)").alias("value_number"),
        s.RESULT_TEXT_EFFECTIVE.alias("value_text"),
        F.coalesce(s.UNIT_OF_MEASURE_DISPLAY, s.RESULT_UNITS_DISPLAY).alias("unit"),
    )
    source_update = F.greatest(
        s.STRING_RESULT_UPDT_DT_TM, s.CLINICAL_EVENT_UPDT_DT_TM, s.ADC_UPDT
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        F.coalesce(s.PERFORMED_DT_TM, s.EVENT_START_DT_TM).alias("event_datetime"),
        s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:event_cd").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"),
        F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY).alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:event_cd"), s.EVENT_CD,
                       F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY), True),
            coding_obj(F.lit("urn:omop:concept_id"), s.OMOP_MANUAL_CONCEPT_ID,
                       s.OMOP_MANUAL_CONCEPT_NAME, False, "bronze.map_numeric_events", None),
        ).alias("score_code"),
        F.coalesce(s.OMOP_MANUAL_CONCEPT_NAME, s.EVENT_LABEL, s.EVENT_CD_DISPLAY).alias("score_name"),
        s.NUMERIC_RESULT.cast("decimal(38,10)").alias("value_number"),
        s.RESULT_TEXT_EFFECTIVE.alias("value_text"),
        F.coalesce(s.UNIT_OF_MEASURE_DISPLAY, s.RESULT_UNITS_DISPLAY).alias("unit_source_value"),
        s.OMOP_MANUAL_UNITS.cast("string").alias("unit_concept_id"),
        F.parse_json(F.to_json(F.array(component))).alias("components"),
        F.lit(1).cast("long").alias("component_count"),
        s.NORMALCY_CD.cast("string").alias("interpretation_code"),
        s.NORMALCY_DISPLAY.alias("interpretation_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        s.RESULT_STATUS_DISPLAY.alias("result_status_display"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(), stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.lit(None).cast("string").alias("source_form_id"),
        F.lit(None).cast("string").alias("promotion_rule_id"),
        F.when(deleted, F.lit("retracted")).when(ended, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.CLINICAL_EVENT_VALID_FROM_DT_TM.alias("record_status_effective_from"),
        F.when(deleted | ended, s.CLINICAL_EVENT_VALID_UNTIL_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"), F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("numeric_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_NUMERIC_EVENTS).alias("_source_table"),
        s.EVENT_ID.cast("string").alias("_source_row_id"),
        s.OMOP_MANUAL_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.OMOP_MANUAL_CONCEPT_NAME.alias("_omop_display"),
    )

In [0]:
@materialized_view(
    name=_n("journey_clinical._numeric_clinical_score"),
    private=True,
    comment="Internal incrementally maintained native numeric-event clinical-score route.",
    refresh_policy="incremental",
)
def _numeric_clinical_score():
    return _numeric_score_canonical().select(*SCORE_STAGE_COLUMNS)

In [0]:
PROMOTED_SCORE_STAGE_COLUMNS = [
    c for c in SCORE_STAGE_COLUMNS if c not in ("score_code", "components")
]

def _promoted_score_canonical():
    s = _form_promotion_source("clinical_score")
    event_id = stable_id("form_promotion:clinical_score", F.col("i.DOC_RESPONSE_KEY"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("i.PERSON_ID_LONG"))],
        SRC_FORM_ITEM, F.col("i.DOC_RESPONSE_KEY"),
    )
    deleted = F.coalesce(F.col("i.CANONICAL_SOURCE_DELETED_IND"), F.lit(False))
    absent = ~F.coalesce(F.col("i.SOURCE_PRESENT_IND"), F.lit(False))
    inactive = F.coalesce(F.col("i.ACTIVE_IND"), F.lit(0)) == 0
    component = F.struct(
        F.lit(1).cast("long").alias("sequence"),
        F.col("i.DOC_RESPONSE_KEY").alias("component_id"),
        F.col("c.source_coding_system").alias("coding_system"),
        F.col("c.source_code").alias("coding_code"), F.col("c.source_display").alias("coding_display"),
        F.coalesce(F.col("i.CANONICAL_NUMERIC_RESULT"), F.col("i.NUMERIC_RESPONSE_NBR"))
         .cast("decimal(38,10)").alias("value_number"),
        F.coalesce(F.col("i.CANONICAL_TEXT_RESULT"), F.col("i.RESPONSE_TEXT_TXT")).alias("value_text"),
        F.lit(None).cast("string").alias("unit"),
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("i.PERSON_ID_LONG").cast("string").alias("person_id"),
        F.when(_present(F.col("i.PERSON_ID_LONG")), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(F.col("i.ENCNTR_ID_LONG").isNotNull(),
               stable_id("encounter:mill", F.col("i.ENCNTR_ID_LONG"))).alias("encounter_id"),
        F.coalesce(F.col("i.RESPONSE_DT_TM"), F.col("i.PERFORMED_DT_TM"),
                   F.col("i.DOCUMENTATION_DT_TM")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.col("c.source_coding_system").alias("source_coding_system"),
        F.col("c.source_code").alias("source_code"), F.col("c.source_display").alias("source_display"),
        codeable_concept(
            coding_obj(F.col("c.source_coding_system"), F.col("c.source_code"),
                       F.col("c.source_display"), True)
        ).alias("score_code"),
        F.col("c.source_display").alias("score_name"),
        F.coalesce(F.col("i.CANONICAL_NUMERIC_RESULT"), F.col("i.NUMERIC_RESPONSE_NBR"))
         .cast("decimal(38,10)").alias("value_number"),
        F.coalesce(F.col("i.CANONICAL_TEXT_RESULT"), F.col("i.RESPONSE_TEXT_TXT")).alias("value_text"),
        F.lit(None).cast("string").alias("unit_source_value"),
        F.col("i.UNIT_CONCEPT_ID").cast("string").alias("unit_concept_id"),
        F.parse_json(F.to_json(F.array(component))).alias("components"),
        F.lit(1).cast("long").alias("component_count"),
        F.lit(None).cast("string").alias("interpretation_code"),
        F.lit(None).cast("string").alias("interpretation_display"),
        F.col("i.FORM_STATUS_CD").cast("string").alias("result_status_code"),
        F.col("i.STATUS").alias("result_status_display"),
        F.when(F.col("i.PERFORMED_PRSNL_ID_LONG").isNotNull(),
               stable_id("practitioner:mill", F.col("i.PERFORMED_PRSNL_ID_LONG")))
         .alias("performer_practitioner_id"),
        stable_id("form:mill:powerform", F.col("i.DCP_FORMS_ACTIVITY_ID")).alias("source_form_id"),
        F.col("c.promotion_rule_id"),
        F.when(deleted | absent, F.lit("retracted")).when(inactive, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.col("i.FIRST_DOCUMENTED_DT_TM").alias("record_status_effective_from"),
        F.when(deleted | absent | inactive, F.col("i.LAST_DOCUMENTED_DT_TM"))
         .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"), F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("form_promotion").alias("source_feed"),
        F.date_format(F.col("i.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("i.ADC_UPDT").alias("source_update_timestamp"),
        F.col("i.ADC_UPDT").alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_FORM_ITEM).alias("_source_table"),
        F.col("i.DOC_RESPONSE_KEY").alias("_source_row_id"),
        F.col("i.QUESTION_CONCEPT_ID").cast("string").alias("_omop_code"),
        F.col("c.source_display").alias("_omop_display"),
    )

@materialized_view(
    name=_n("journey_clinical._promoted_clinical_score_primitive"),
    private=True,
    comment="Internal incrementally maintained governed form-promotion clinical-score route.",
    refresh_policy="incremental",
)
def _promoted_clinical_score_primitive():
    return _promoted_score_canonical().select(*PROMOTED_SCORE_STAGE_COLUMNS)

In [0]:
def _promoted_score_from_stage():
    s = spark.read.table(_n("journey_clinical._promoted_clinical_score_primitive"))
    component = F.struct(
        F.lit(1).cast("long").alias("sequence"),
        F.col("_source_row_id").alias("component_id"),
        F.col("source_coding_system").alias("coding_system"),
        F.col("source_code").alias("coding_code"),
        F.col("source_display").alias("coding_display"),
        F.col("value_number").alias("value_number"),
        F.col("value_text").alias("value_text"),
        F.col("unit_source_value").alias("unit"),
    )
    return (
        s.withColumn(
            "score_code",
            codeable_concept(
                coding_obj(
                    F.col("source_coding_system"), F.col("source_code"),
                    F.col("source_display"), True,
                )
            ),
        )
        .withColumn("components", F.parse_json(F.to_json(F.array(component))))
        .select(*SCORE_STAGE_COLUMNS)
    )

def _clinical_score_canonical():
    return _clinical_score_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
def _clinical_score_canonical_pregate():
    numeric = spark.read.table(_n("journey_clinical._numeric_clinical_score"))
    promoted = _promoted_score_from_stage()
    return numeric.unionByName(promoted)

@materialized_view(
    name=_n("journey_clinical._qc_clinical_score"),
    comment="Private JSON bridge and Gold cross-rule flags for clinical_score.",
    refresh_policy="incremental",
)
def _qc_clinical_score():
    return _cross_qc_primitive(
        _clinical_score_canonical(),
        "clinical_score",
        {
            "components": "_qc_components_json",
            "score_code": "_qc_score_code_json",
        },
    )

In [0]:
CLINICAL_SCORE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable clinical-score event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Score timestamp.",
    "event_end_datetime": "Score end timestamp.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source score code.",
    "source_display": "Source score display.",
    "score_code": "Source and mapped score CodeableConcept.",
    "score_name": "Human-readable score or scale name.",
    "value_number": "Numeric score value.",
    "value_text": "Verbatim score text.",
    "unit_source_value": "Source unit when supplied.",
    "unit_concept_id": "Mapped unit concept identifier.",
    "components": "Ordered component values retained for the score result.",
    "component_count": "Number of ordered component objects.",
    "interpretation_code": "Source interpretation code.",
    "interpretation_display": "Source interpretation display.",
    "result_status_code": "Source result status code.",
    "result_status_display": "Source result status display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "source_form_id": "Source form event for promoted scores.",
    "promotion_rule_id": "Governed promotion-rule identifier.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source route.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.clinical_score"),
    comment="Native numeric-event and governed form-promoted clinical score results.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CLINICAL_SCORE_COLUMN_COMMENTS,
)
def clinical_score():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_clinical_score")),
        "clinical_score",
        {
            "components": "_qc_components_json",
            "score_code": "_qc_score_code_json",
        },
        SCORE_PUBLIC_COLUMNS,
    )

In [0]:
SRC_MED_ADMIN_INGREDIENT = "4_prod.bronze.map_med_admin_ingredient"

def _med_admin_ingredient_grouped_query():
    s = read_source(SRC_MED_ADMIN_INGREDIENT)
    ingredient = F.struct(
        F.coalesce(s.ACTION_SEQUENCE, F.lit(0)).cast("long").alias("action_sequence"),
        F.coalesce(s.COMP_SEQUENCE, F.lit(0)).cast("long").alias("component_sequence"),
        F.concat_ws(":", s.INGREDIENT_ORDER_ID.cast("string"),
                    s.ACTION_SEQUENCE.cast("string"), s.COMP_SEQUENCE.cast("string"))
         .alias("component_id"),
        F.lit("urn:cerner:synonym_id").alias("coding_system"),
        s.SYNONYM_ID.cast("string").alias("coding_code"),
        F.coalesce(s.ORDERED_AS_MNEMONIC, s.SUPPLIED_AS_MNEMONIC,
                   s.ORDER_MNEMONIC, s.HNA_ORDER_MNEMONIC).alias("coding_display"),
        s.STRENGTH.cast("decimal(38,10)").alias("strength_value"),
        s.STRENGTH_UNIT_CD.cast("string").alias("strength_unit"),
        s.VOLUME.cast("decimal(38,10)").alias("volume_value"),
        s.VOLUME_UNIT_CD.cast("string").alias("volume_unit"),
        F.coalesce(s.ORDERED_DOSE, s.DOSE_QUANTITY).cast("decimal(38,10)").alias("dose_value"),
        F.coalesce(s.ORDERED_DOSE_UNIT_CD, s.DOSE_QUANTITY_UNIT_CD).cast("string").alias("dose_unit"),
        s.NORMALIZED_RATE.cast("decimal(38,10)").alias("rate_value"),
        s.NORMALIZED_RATE_UNIT_CD.cast("string").alias("rate_unit"),
        s.CONCENTRATION.cast("decimal(38,10)").alias("concentration_value"),
        s.CONCENTRATION_UNIT_CD.cast("string").alias("concentration_unit"),
        s.INGREDIENT_TYPE_FLAG.cast("string").alias("ingredient_type_code"),
        (s.CLINICALLY_SIGNIFICANT_FLAG == 1).alias("clinically_significant_ind"),
        (s.INCLUDE_IN_TOTAL_VOLUME_FLAG == 1).alias("include_in_total_volume_ind"),
        s.FREETEXT_DOSE.alias("freetext_dose"),
        F.lit(True).alias("source_present_ind"),
    )
    return s.groupBy("EVENT_ID").agg(
        F.to_json(F.sort_array(F.collect_list(ingredient))).alias("ingredients_json"),
        F.count(F.lit(1)).cast("long").alias("ingredient_count"),
        F.max(F.greatest(s.UPDT_DT_TM, s.ADC_UPDT)).alias("ingredient_source_update_timestamp"),
        F.max(s.ADC_UPDT).alias("ingredient_loaded_at"),
    )

@materialized_view(
    name=_n("journey_clinical._med_admin_ingredient_grouped"),
    private=True,
    comment="Internal ordered ingredient aggregate for medication administrations.",
    refresh_policy="incremental",
)
def _med_admin_ingredient_grouped():
    return _med_admin_ingredient_grouped_query()

In [0]:
def _med_order_action_grouped_query():
    s = read_source(SRC_MEDICATION_ORDER_ACTION)
    action = F.struct(
        F.coalesce(s.ACTION_SEQUENCE, F.lit(0)).cast("long").alias("sequence"),
        F.coalesce(s.ACTION_TYPE_DESCRIPTION, s.ACTION_TYPE_CD.cast("string")).alias("status_kind"),
        s.ORDER_STATUS_CD.cast("string").alias("status_code"),
        F.coalesce(s.ORDER_STATUS_DESCRIPTION, s.ACTION_TYPE_DESCRIPTION).alias("status_display"),
        F.coalesce(s.ACTION_DT_TM, s.EFFECTIVE_DT_TM, s.ORDER_DT_TM).alias("effective_datetime"),
        F.when(s.ACTION_PERSONNEL_ID.isNotNull(),
               stable_id("practitioner:mill", s.ACTION_PERSONNEL_ID)).alias("practitioner_id"),
        s.ACTION_SEQUENCE.cast("long").alias("source_action_sequence"),
        s.ACTION_TYPE_CD.cast("string").alias("action_type_code"),
        s.ACTION_TYPE_DESCRIPTION.alias("action_type_display"),
        s.ACTION_QUALIFIER_CD.cast("string").alias("action_qualifier_code"),
        s.ACTION_QUALIFIER_DESCRIPTION.alias("action_qualifier_display"),
        (s.ACTION_REJECTED_IND == 1).alias("action_rejected_ind"),
        s.HISTORICAL_FEED_IND.alias("historical_feed_ind"),
        s.SOURCE_PRESENT_IND.alias("source_present_ind"),
    )
    return s.groupBy("ORDER_ID").agg(
        F.to_json(F.sort_array(F.collect_list(action))).alias("status_history_json"),
        F.count(F.lit(1)).cast("long").alias("status_history_count"),
        F.max(F.greatest(s.SOURCE_ADC_UPDT, s.ORDER_SOURCE_ADC_UPDT, s.ADC_UPDT))
         .alias("action_source_update_timestamp"),
        F.max(s.ADC_UPDT).alias("action_loaded_at"),
    )

@materialized_view(
    name=_n("journey_clinical._med_order_action_grouped"),
    private=True,
    comment="Internal ordered status-history aggregate for medication orders.",
    refresh_policy="incremental",
)
def _med_order_action_grouped():
    return _med_order_action_grouped_query()

In [0]:
SRC_MEDICATION_ORDER_INGREDIENT = "4_prod.bronze.map_medication_order_ingredient"

def _med_order_ingredient_grouped_query():
    s = read_source(SRC_MEDICATION_ORDER_INGREDIENT)
    ingredient = F.struct(
        F.coalesce(s.ACTION_SEQUENCE, F.lit(0)).cast("long").alias("action_sequence"),
        F.coalesce(s.COMP_SEQUENCE, F.lit(0)).cast("long").alias("component_sequence"),
        F.concat_ws(":", s.ORDER_ID.cast("string"), s.ACTION_SEQUENCE.cast("string"),
                    s.COMP_SEQUENCE.cast("string")).alias("component_id"),
        F.lit("urn:cerner:synonym_id").alias("coding_system"),
        s.SYNONYM_ID.cast("string").alias("coding_code"),
        F.coalesce(s.ORDERED_AS_MNEMONIC, s.SUPPLIED_AS_MNEMONIC,
                   s.ORDER_MNEMONIC, s.HNA_ORDER_MNEMONIC).alias("coding_display"),
        s.STRENGTH.cast("decimal(38,10)").alias("strength_value"),
        s.STRENGTH_UNIT_DESCRIPTION.alias("strength_unit"),
        s.VOLUME.cast("decimal(38,10)").alias("volume_value"),
        s.VOLUME_UNIT_DESCRIPTION.alias("volume_unit"),
        F.coalesce(s.ORDERED_DOSE, s.DOSE_QUANTITY).cast("decimal(38,10)").alias("dose_value"),
        F.coalesce(s.ORDERED_DOSE_UNIT_DESCRIPTION,
                   s.DOSE_QUANTITY_UNIT_DESCRIPTION).alias("dose_unit"),
        s.NORMALIZED_RATE.cast("decimal(38,10)").alias("rate_value"),
        s.NORMALIZED_RATE_UNIT_DESCRIPTION.alias("rate_unit"),
        s.CONCENTRATION.cast("decimal(38,10)").alias("concentration_value"),
        s.CONCENTRATION_UNIT_DESCRIPTION.alias("concentration_unit"),
        s.INGREDIENT_TYPE_FLAG.cast("string").alias("ingredient_type_code"),
        (s.CLINICALLY_SIGNIFICANT_FLAG == 1).alias("clinically_significant_ind"),
        (s.INCLUDE_IN_TOTAL_VOLUME_FLAG == 1).alias("include_in_total_volume_ind"),
        s.FREETEXT_DOSE.alias("freetext_dose"),
        s.SOURCE_PRESENT_IND.alias("source_present_ind"),
    )
    return s.groupBy("ORDER_ID").agg(
        F.to_json(F.sort_array(F.collect_list(ingredient))).alias("ingredients_json"),
        F.count(F.lit(1)).cast("long").alias("ingredient_count"),
        F.max(F.greatest(s.SOURCE_ADC_UPDT, s.ORDER_SOURCE_ADC_UPDT, s.ADC_UPDT))
         .alias("ingredient_source_update_timestamp"),
        F.max(s.ADC_UPDT).alias("ingredient_loaded_at"),
    )

@materialized_view(
    name=_n("journey_clinical._med_order_ingredient_grouped"),
    private=True,
    comment="Internal ordered ingredient aggregate for medication orders.",
    refresh_policy="incremental",
)
def _med_order_ingredient_grouped():
    return _med_order_ingredient_grouped_query()

In [0]:
SRC_MEDICATION_ORDER_DETAIL = "4_prod.bronze.map_medication_order_detail"

def _med_order_detail_grouped_query():
    s = read_source(SRC_MEDICATION_ORDER_DETAIL)
    detail = F.struct(
        F.coalesce(s.ACTION_SEQUENCE, F.lit(0)).cast("long").alias("action_sequence"),
        F.coalesce(s.DETAIL_SEQUENCE, F.lit(0)).cast("long").alias("detail_sequence"),
        s.OE_FIELD_ID.cast("string").alias("field_id"),
        s.OE_FIELD_MEANING.alias("field_meaning"),
        F.coalesce(s.OE_FIELD_DISPLAY_VALUE_EXTEND,
                   s.OE_FIELD_DISPLAY_VALUE).alias("value_text"),
        s.OE_FIELD_VALUE.cast("decimal(38,10)").alias("value_number"),
        s.OE_FIELD_DT_TM_VALUE.alias("value_datetime"),
        s.DETAIL_HISTORY_CONTRACT.alias("history_contract"),
        s.PARENT_ACTION_SEQUENCE.cast("long").alias("parent_action_sequence"),
        s.LAST_ACTION_SEQUENCE.cast("long").alias("last_action_sequence"),
        s.SOURCE_PRESENT_IND.alias("source_present_ind"),
    )
    return s.groupBy("ORDER_ID").agg(
        F.to_json(F.sort_array(F.collect_list(detail))).alias("order_details_json"),
        F.count(F.lit(1)).cast("long").alias("order_detail_count"),
        F.max(F.greatest(s.SOURCE_ADC_UPDT, s.ORDER_SOURCE_ADC_UPDT, s.ADC_UPDT))
         .alias("detail_source_update_timestamp"),
        F.max(s.ADC_UPDT).alias("detail_loaded_at"),
    )

@materialized_view(
    name=_n("journey_clinical._med_order_detail_grouped"),
    private=True,
    comment="Internal ordered latest-action-detail aggregate for medication orders.",
    refresh_policy="incremental",
)
def _med_order_detail_grouped():
    return _med_order_detail_grouped_query()

In [0]:
def _med_admin_status_entry(sequence, status_kind, status_code, status_display,
                            effective_datetime, practitioner_id):
    return F.struct(
        F.lit(sequence).cast("long").alias("sequence"),
        status_kind.cast("string").alias("status_kind"),
        status_code.cast("string").alias("status_code"),
        status_display.cast("string").alias("status_display"),
        effective_datetime.cast("timestamp").alias("effective_datetime"),
        practitioner_id.cast("string").alias("practitioner_id"),
        F.lit(None).cast("long").alias("source_action_sequence"),
        F.lit(True).alias("source_present_ind"),
    )

def _medication_admin_primitive_query():
    a = read_source(SRC_MED_ADMIN).alias("a")
    ingredients = spark.read.table(_n("journey_clinical._med_admin_ingredient_grouped")).alias("i")
    orders = read_source(SRC_MEDICATION_ORDER).select(
        F.col("ORDER_ID").alias("_linked_order_id")
    ).alias("o")
    joined = (
        a.join(ingredients, F.col("a.EVENT_ID") == F.col("i.EVENT_ID"), "left")
         .join(orders, F.col("a.ORDER_ID") == F.col("o._linked_order_id"), "left")
    )
    event_id = stable_id("medication_admin:mill", F.col("a.EVENT_ID"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("a.PERSON_ID"))], SRC_MED_ADMIN, F.col("a.EVENT_ID")
    )
    performer = F.coalesce(F.col("a.PERFORMED_PRSNL_ID"), F.col("a.PRSNL_ID"))
    verifier = F.coalesce(F.col("a.MAE_VERIFIED_PRSNL_ID"), F.col("a.CE_VERIFIED_PRSNL_ID"))
    status_entries = F.filter(F.array(
        _med_admin_status_entry(
            10, F.lit("scheduled"),
            F.when(F.col("a.SCHEDULED_DT_TM").isNotNull(), F.lit("scheduled")),
            F.when(F.col("a.SCHEDULED_DT_TM").isNotNull(), F.lit("Scheduled")),
            F.col("a.SCHEDULED_DT_TM"), F.lit(None).cast("string"),
        ),
        _med_admin_status_entry(
            20, F.lit("administration_event"), F.col("a.EVENT_TYPE_CD"),
            F.col("a.EVENT_TYPE_DISPLAY"),
            F.coalesce(F.col("a.PERFORMED_DT_TM"), F.col("a.ADMIN_START_DT_TM")),
            F.when(performer.isNotNull(), stable_id("practitioner:mill", performer)),
        ),
        _med_admin_status_entry(
            30, F.lit("result"), F.col("a.RESULT_STATUS_CD"),
            F.col("a.RESULT_STATUS_DISPLAY"),
            F.coalesce(F.col("a.ADMIN_END_DT_TM"), F.col("a.ADMIN_START_DT_TM")),
            F.when(verifier.isNotNull(), stable_id("practitioner:mill", verifier)),
        ),
        _med_admin_status_entry(
            40, F.lit("verified"),
            F.when(F.coalesce(F.col("a.VERIFICATION_DT_TM"),
                              F.col("a.VERIFIED_DT_TM")).isNotNull(), F.lit("verified")),
            F.when(F.coalesce(F.col("a.VERIFICATION_DT_TM"),
                              F.col("a.VERIFIED_DT_TM")).isNotNull(), F.lit("Verified")),
            F.coalesce(F.col("a.VERIFICATION_DT_TM"), F.col("a.VERIFIED_DT_TM")),
            F.when(verifier.isNotNull(), stable_id("practitioner:mill", verifier)),
        ),
        _med_admin_status_entry(
            50, F.lit("order_status"), F.col("a.ORDER_STATUS_CD"),
            F.col("a.ORDER_STATUS_DISPLAY"), F.col("a.ORDER_STATUS_DT_TM"),
            F.lit(None).cast("string"),
        ),
        _med_admin_status_entry(
            60, F.lit("suspended"),
            F.when((F.col("a.SUSPEND_IND") == 1) | F.col("a.SUSPEND_EFFECTIVE_DT_TM").isNotNull(),
                   F.lit("suspended")),
            F.when((F.col("a.SUSPEND_IND") == 1) | F.col("a.SUSPEND_EFFECTIVE_DT_TM").isNotNull(),
                   F.lit("Suspended")),
            F.col("a.SUSPEND_EFFECTIVE_DT_TM"), F.lit(None).cast("string"),
        ),
        _med_admin_status_entry(
            70, F.lit("resumed"),
            F.when((F.col("a.RESUME_IND") == 1) | F.col("a.RESUME_EFFECTIVE_DT_TM").isNotNull(),
                   F.lit("resumed")),
            F.when((F.col("a.RESUME_IND") == 1) | F.col("a.RESUME_EFFECTIVE_DT_TM").isNotNull(),
                   F.lit("Resumed")),
            F.col("a.RESUME_EFFECTIVE_DT_TM"), F.lit(None).cast("string"),
        ),
        _med_admin_status_entry(
            80, F.lit("discontinued"),
            F.when((F.col("a.DISCONTINUE_IND") == 1)
                   | F.col("a.DISCONTINUE_EFFECTIVE_DT_TM").isNotNull(), F.lit("discontinued")),
            F.when((F.col("a.DISCONTINUE_IND") == 1)
                   | F.col("a.DISCONTINUE_EFFECTIVE_DT_TM").isNotNull(), F.lit("Discontinued")),
            F.col("a.DISCONTINUE_EFFECTIVE_DT_TM"), F.lit(None).cast("string"),
        ),
    ), lambda x: x["status_code"].isNotNull() | x["effective_datetime"].isNotNull())
    ended = F.col("a.CE_VALID_UNTIL_DT_TM").isNotNull() & (
        F.col("a.CE_VALID_UNTIL_DT_TM") < F.lit("2100-01-01").cast("timestamp")
    )
    superseded = ended | (F.coalesce(F.col("a.MR_IS_CURRENT_IND"), F.lit(True)) == F.lit(False))
    loaded_at = F.greatest(F.col("a.ADC_UPDT"), F.col("i.ingredient_loaded_at"))
    source_update = F.greatest(
        F.col("a.MAE_ADC_UPDT"), F.col("a.MR_ADC_UPDT"), F.col("a.ORDERS_ADC_UPDT"),
        F.col("a.OI_ADC_UPDT"), F.col("a.SYNONYM_ADC_UPDT"), F.col("a.LOOKUP_ADC_UPDT"),
        F.col("a.ADC_UPDT"), F.col("i.ingredient_source_update_timestamp"),
    )
    with_status = joined.withColumn("_status_history_array", status_entries)
    return with_status.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("a.PERSON_ID").cast("string").alias("person_id"),
        F.when(_present(F.col("a.PERSON_ID")), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("a.ENCNTR_ID").isNotNull(), stable_id("encounter:mill", F.col("a.ENCNTR_ID")))
         .alias("encounter_id"),
        F.coalesce(F.col("a.ADMIN_START_DT_TM"), F.col("a.PERFORMED_DT_TM"),
                   F.col("a.SCHEDULED_DT_TM")).alias("event_datetime"),
        F.col("a.ADMIN_END_DT_TM").alias("event_end_datetime"),
        F.lit("urn:cerner:order_synonym_id").alias("source_coding_system"),
        F.col("a.ORDER_SYNONYM_ID").cast("string").alias("source_code"),
        F.coalesce(F.col("a.ORDER_MNEMONIC"), F.col("a.ORDERED_AS_MNEMONIC"),
                   F.col("a.HNA_ORDER_MNEMONIC")).alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:cerner:order_synonym_id"), F.col("a.ORDER_SYNONYM_ID"),
                       F.coalesce(F.col("a.ORDER_MNEMONIC"), F.col("a.ORDERED_AS_MNEMONIC")), True),
            coding_obj(F.lit("http://www.nlm.nih.gov/research/umls/rxnorm"), F.col("a.RXNORM_CUI"),
                       F.col("a.RXNORM_STR"), False, "bronze.map_med_admin", None),
            coding_obj(F.lit("http://snomed.info/sct"),
                       F.coalesce(F.col("a.SNOMED_VALIDATED_CODE"), F.col("a.SNOMED_CODE"),
                                  F.col("a.LOOKUP_SNOMED_CODE")),
                       F.coalesce(F.col("a.SNOMED_VALIDATED_STR"), F.col("a.SNOMED_STR"),
                                  F.col("a.LOOKUP_SNOMED_FROM_OMOP")),
                       False, "bronze.map_med_admin", None),
            coding_obj(F.lit("urn:omop:concept_id"),
                       F.coalesce(F.col("a.OMOP_STANDARD_CONCEPT_ID"), F.col("a.OMOP_CONCEPT_ID")),
                       F.coalesce(F.col("a.OMOP_STANDARD_CONCEPT_NAME"), F.col("a.OMOP_CONCEPT_NAME")),
                       False, "bronze.map_med_admin", None),
        ).alias("_medication_code_json"),
        F.when(F.col("o._linked_order_id").isNotNull(),
               stable_id("medication_order:mill", F.col("a.ORDER_ID"))).alias("medication_order_id"),
        F.col("a.RESULT_STATUS_CD").cast("string").alias("administration_status_code"),
        F.col("a.RESULT_STATUS_DISPLAY").alias("administration_status_display"),
        F.col("a.EVENT_TYPE_CD").cast("string").alias("source_event_type_code"),
        F.col("a.EVENT_TYPE_DISPLAY").alias("source_event_type_display"),
        F.to_json(F.col("_status_history_array")).alias("_status_history_json"),
        F.size(F.col("_status_history_array")).cast("long").alias("status_history_count"),
        F.coalesce(F.col("a.ADMIN_DOSAGE"), F.col("a.DOSE_VALUE_EFFECTIVE"))
         .cast("decimal(38,10)").alias("dose_value"),
        F.coalesce(F.col("a.ADMIN_DOSAGE_UNIT_DISPLAY"), F.col("a.DOSE_UNIT_NORMALIZED"))
         .alias("dose_unit"),
        F.col("a.INITIAL_DOSAGE").cast("decimal(38,10)").alias("initial_dose_value"),
        F.col("a.INITIAL_DOSAGE_UNIT_DISPLAY").alias("initial_dose_unit"),
        F.col("a.DOSE_IN_MG").cast("decimal(38,10)").alias("dose_in_mg"),
        F.col("a.DOSE_IN_ML").cast("decimal(38,10)").alias("dose_in_ml"),
        F.col("a.DOSE_STANDARDIZATION_STATUS").alias("dose_standardization_status"),
        F.col("a.ADMIN_ROUTE_CD").cast("string").alias("route_code"),
        F.col("a.ADMIN_ROUTE_DISPLAY").alias("route_display"),
        F.col("a.ADMIN_SITE_CD").cast("string").alias("site_code"),
        F.col("a.ADMIN_SITE_DISPLAY").alias("site_display"),
        F.col("a.INFUSED_VOLUME").cast("decimal(38,10)").alias("infused_volume"),
        F.col("a.INFUSED_VOLUME_UNIT_DISPLAY").alias("infused_volume_unit"),
        F.col("a.INFUSION_RATE").cast("decimal(38,10)").alias("infusion_rate"),
        F.col("a.INFUSION_UNIT_DISPLAY").alias("infusion_rate_unit"),
        F.col("i.ingredients_json").alias("_ingredients_json"),
        F.coalesce(F.col("i.ingredient_count"), F.lit(0)).cast("long").alias("ingredient_count"),
        F.when(performer.isNotNull(), stable_id("practitioner:mill", performer))
         .alias("performer_practitioner_id"),
        F.when(verifier.isNotNull(), stable_id("practitioner:mill", verifier))
         .alias("verifier_practitioner_id"),
        F.when(F.col("a.NURSE_UNIT_CD").isNotNull(),
               stable_id("location:mill:nurse_unit", F.col("a.NURSE_UNIT_CD"))).alias("location_id"),
        F.when(F.col("a.ORGANIZATION_ID").isNotNull(),
               stable_id("organization:mill", F.col("a.ORGANIZATION_ID"))).alias("organization_id"),
        F.col("a.SCHEDULED_DT_TM").alias("scheduled_datetime"),
        F.coalesce(F.col("a.PERFORMED_DT_TM"), F.col("a.ADMIN_START_DT_TM"))
         .alias("performed_datetime"),
        F.coalesce(F.col("a.VERIFICATION_DT_TM"), F.col("a.VERIFIED_DT_TM"))
         .alias("verified_datetime"),
        F.col("a.ORDER_STATUS_CD").cast("string").alias("order_status_code"),
        F.col("a.ORDER_STATUS_DISPLAY").alias("order_status_display"),
        (F.col("a.PRN_IND") == 1).alias("prn_ind"), (F.col("a.IV_IND") == 1).alias("iv_ind"),
        F.when(superseded, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.coalesce(F.col("a.CE_VALID_FROM_DT_TM"), F.col("a.MR_VALID_FROM_DT_TM"),
                   F.col("a.ADMIN_START_DT_TM")).alias("record_status_effective_from"),
        F.when(superseded, F.coalesce(F.col("a.CE_VALID_UNTIL_DT_TM"),
                                     F.col("a.MR_VALID_UNTIL_DT_TM")))
         .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("med_admin").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"),
        loaded_at.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_MED_ADMIN).alias("_source_table"),
        F.col("a.EVENT_ID").cast("string").alias("_source_row_id"),
        F.coalesce(F.col("a.SNOMED_VALIDATED_CODE"), F.col("a.SNOMED_CODE"),
                   F.col("a.LOOKUP_SNOMED_CODE")).cast("string").alias("_snomed_code"),
        F.coalesce(F.col("a.SNOMED_VALIDATED_STR"), F.col("a.SNOMED_STR"),
                   F.col("a.LOOKUP_SNOMED_FROM_OMOP")).alias("_snomed_display"),
        F.coalesce(F.col("a.OMOP_STANDARD_CONCEPT_ID"), F.col("a.OMOP_CONCEPT_ID"))
         .cast("string").alias("_omop_code"),
        F.coalesce(F.col("a.OMOP_STANDARD_CONCEPT_NAME"), F.col("a.OMOP_CONCEPT_NAME"))
         .alias("_omop_display"),
        F.col("a.RXNORM_CUI").cast("string").alias("_rxnorm_code"),
        F.col("a.RXNORM_STR").alias("_rxnorm_display"),
    )

@materialized_view(
    name=_n("journey_clinical._medication_admin_primitive"),
    private=True,
    comment="Internal primitive medication-administration join with deterministic JSON boundaries.",
    refresh_policy="incremental",
)
def _medication_admin_primitive():
    return _medication_admin_primitive_query()

In [0]:
def _medication_order_primitive_query():
    o = read_source(SRC_MEDICATION_ORDER).alias("o")
    actions = spark.read.table(_n("journey_clinical._med_order_action_grouped")).alias("a")
    ingredients = spark.read.table(_n("journey_clinical._med_order_ingredient_grouped")).alias("i")
    details = spark.read.table(_n("journey_clinical._med_order_detail_grouped")).alias("d")
    joined = (
        o.join(actions, F.col("o.ORDER_ID") == F.col("a.ORDER_ID"), "left")
         .join(ingredients, F.col("o.ORDER_ID") == F.col("i.ORDER_ID"), "left")
         .join(details, F.col("o.ORDER_ID") == F.col("d.ORDER_ID"), "left")
    )
    event_id = stable_id("medication_order:mill", F.col("o.ORDER_ID"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", F.col("o.PERSON_ID"))], SRC_MEDICATION_ORDER, F.col("o.ORDER_ID")
    )
    retracted = F.coalesce(F.col("o.SOURCE_PRESENT_IND"), F.lit(True)) == F.lit(False)
    superseded = (F.coalesce(F.col("o.INACTIVE_ORDER_FLAG"), F.lit(0)) == 1) | (
        F.coalesce(F.col("o.ACTIVE_IND"), F.lit(1)) == 0
    )
    loaded_at = F.greatest(F.col("o.ADC_UPDT"), F.col("a.action_loaded_at"),
                           F.col("i.ingredient_loaded_at"), F.col("d.detail_loaded_at"))
    source_update = F.greatest(
        F.col("o.SOURCE_ADC_UPDT"), F.col("o.ADC_UPDT"),
        F.col("a.action_source_update_timestamp"),
        F.col("i.ingredient_source_update_timestamp"),
        F.col("d.detail_source_update_timestamp"),
    )
    return joined.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.col("o.PERSON_ID").cast("string").alias("person_id"),
        F.when(_present(F.col("o.PERSON_ID")), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("o.ENCNTR_ID").isNotNull(), stable_id("encounter:mill", F.col("o.ENCNTR_ID")))
         .alias("encounter_id"),
        F.coalesce(F.col("o.ORIG_ORDER_DT_TM"), F.col("o.CURRENT_START_DT_TM"))
         .alias("event_datetime"),
        F.coalesce(F.col("o.PROJECTED_STOP_DT_TM"), F.col("o.SOFT_STOP_DT_TM"),
                   F.col("o.DISCONTINUE_EFFECTIVE_DT_TM")).alias("event_end_datetime"),
        F.lit("urn:cerner:synonym_id").alias("source_coding_system"),
        F.col("o.SYNONYM_ID").cast("string").alias("source_code"),
        F.coalesce(F.col("o.ORDER_MNEMONIC"), F.col("o.ORDERED_AS_MNEMONIC"),
                   F.col("o.HNA_ORDER_MNEMONIC")).alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:cerner:synonym_id"), F.col("o.SYNONYM_ID"),
                       F.coalesce(F.col("o.ORDER_MNEMONIC"), F.col("o.ORDERED_AS_MNEMONIC"),
                                  F.col("o.HNA_ORDER_MNEMONIC")), True)
        ).alias("_medication_code_json"),
        F.col("o.ORDER_STATUS_CD").cast("string").alias("order_status_code"),
        F.col("o.ORDER_STATUS_DESCRIPTION").alias("order_status_display"),
        F.col("o.DEPT_STATUS_CD").cast("string").alias("department_status_code"),
        F.col("o.DEPT_STATUS_DESCRIPTION").alias("department_status_display"),
        F.col("o.ACTIVE_STATUS_CD").cast("string").alias("active_status_code"),
        F.col("o.ACTIVE_STATUS_DESCRIPTION").alias("active_status_display"),
        F.lit("order").alias("intent_code"),
        F.col("o.MED_ORDER_TYPE_CD").cast("string").alias("medication_order_type_code"),
        F.col("o.MED_ORDER_TYPE_DESCRIPTION").alias("medication_order_type_display"),
        F.col("o.ORIG_ORDER_DT_TM").alias("authored_datetime"),
        F.col("o.CURRENT_START_DT_TM").alias("effective_start_datetime"),
        F.col("o.PROJECTED_STOP_DT_TM").alias("projected_stop_datetime"),
        F.col("o.DISCONTINUE_EFFECTIVE_DT_TM").alias("discontinued_datetime"),
        F.col("o.FREQUENCY_ID").cast("string").alias("frequency_id"),
        (F.col("o.PRN_IND") == 1).alias("prn_ind"), (F.col("o.IV_IND") == 1).alias("iv_ind"),
        (F.col("o.SUSPEND_IND") == 1).alias("suspend_ind"),
        (F.col("o.RESUME_IND") == 1).alias("resume_ind"),
        (F.col("o.DISCONTINUE_IND") == 1).alias("discontinue_ind"),
        F.when(F.col("o.LAST_UPDATE_PROVIDER_ID").isNotNull(),
               stable_id("practitioner:mill", F.col("o.LAST_UPDATE_PROVIDER_ID")))
         .alias("requester_practitioner_id"),
        F.when(F.col("o.ORGANIZATION_ID").isNotNull(),
               stable_id("organization:mill", F.col("o.ORGANIZATION_ID"))).alias("organization_id"),
        F.col("o.CLINICAL_DISPLAY_LINE").alias("clinical_display_line"),
        F.col("o.ORDER_DETAIL_DISPLAY_LINE").alias("order_detail_display_line"),
        F.col("a.status_history_json").alias("_status_history_json"),
        F.coalesce(F.col("a.status_history_count"), F.lit(0)).cast("long")
         .alias("status_history_count"),
        F.col("i.ingredients_json").alias("_ingredients_json"),
        F.coalesce(F.col("i.ingredient_count"), F.lit(0)).cast("long").alias("ingredient_count"),
        F.col("d.order_details_json").alias("_order_details_json"),
        F.coalesce(F.col("d.order_detail_count"), F.lit(0)).cast("long")
         .alias("order_detail_count"),
        F.when(retracted, F.lit("retracted")).when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        F.col("o.ORIG_ORDER_DT_TM").alias("record_status_effective_from"),
        F.when(retracted, F.col("o.SOURCE_ABSENT_DETECTED_TS"))
         .when(superseded, F.coalesce(F.col("o.DISCONTINUE_EFFECTIVE_DT_TM"),
                                     F.col("o.STATUS_DT_TM"))).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("medication_order").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"),
        loaded_at.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_MEDICATION_ORDER).alias("_source_table"),
        F.col("o.ORDER_ID").cast("string").alias("_source_row_id"),
    )

@materialized_view(
    name=_n("journey_clinical._medication_order_primitive"),
    private=True,
    comment="Internal primitive medication-order join with deterministic JSON boundaries.",
    refresh_policy="incremental",
)
def _medication_order_primitive():
    return _medication_order_primitive_query()

In [0]:
def _medication_admin_canonical_pregate():
    p = spark.read.table(_n("journey_clinical._medication_admin_primitive"))
    return (
        p.withColumn("medication_code", F.parse_json(F.col("_medication_code_json")))
         .withColumn("status_history", F.parse_json(F.col("_status_history_json")))
         .withColumn("ingredients", F.parse_json(F.col("_ingredients_json")))
         .drop("_medication_code_json", "_status_history_json", "_ingredients_json")
    )

def _medication_admin_canonical():
    return _medication_admin_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_medication_admin"),
    comment="Private JSON bridge and Gold cross-rule flags for medication_admin.",
    refresh_policy="incremental",
)
def _qc_medication_admin():
    return _cross_qc_primitive(
        _medication_admin_canonical(),
        "medication_admin",
        {
            "ingredients": "_qc_ingredients_json",
            "medication_code": "_qc_medication_code_json",
            "status_history": "_qc_status_history_json",
        },
    )

In [0]:
# ==== Medication administration, prescribing and order-to-administration threads ====

MEDICATION_ADMIN_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "medication_code",
    "medication_order_id", "administration_status_code", "administration_status_display",
    "source_event_type_code", "source_event_type_display", "status_history",
    "status_history_count", "dose_value", "dose_unit", "initial_dose_value",
    "initial_dose_unit", "dose_in_mg", "dose_in_ml", "dose_standardization_status",
    "route_code", "route_display", "site_code", "site_display", "infused_volume",
    "infused_volume_unit", "infusion_rate", "infusion_rate_unit", "ingredients",
    "ingredient_count", "performer_practitioner_id", "verifier_practitioner_id",
    "location_id", "organization_id", "scheduled_datetime", "performed_datetime",
    "verified_datetime", "order_status_code", "order_status_display", "prn_ind", "iv_ind",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

MEDICATION_ADMIN_COLUMN_COMMENTS = {
    "patient_event_id": "Stable administration event identifier.",
    "fact_row_id": "Storage row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Subject-key identifier system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Administration start or performed time.",
    "event_end_datetime": "Administration end time.",
    "source_coding_system": "Source medication coding system.",
    "source_code": "Source order-synonym identifier.",
    "source_display": "Source medication display.",
    "medication_code": "Source and mapped medication codings.",
    "medication_order_id": "Linked governed medication order when present.",
    "administration_status_code": "Current source result status code.",
    "administration_status_display": "Current source result status display.",
    "source_event_type_code": "Source medication event type code.",
    "source_event_type_display": "Source medication event type display.",
    "status_history": "Ordered administration and order-status milestones.",
    "status_history_count": "Number of retained status milestones.",
    "dose_value": "Effective administered dose.",
    "dose_unit": "Administered dose unit.",
    "initial_dose_value": "Initially documented dose.",
    "initial_dose_unit": "Initially documented dose unit.",
    "dose_in_mg": "Bronze-standardized milligram dose.",
    "dose_in_ml": "Bronze-standardized millilitre dose.",
    "dose_standardization_status": "Bronze dose-standardization status.",
    "route_code": "Administration route code.",
    "route_display": "Administration route display.",
    "site_code": "Administration site code.",
    "site_display": "Administration site display.",
    "infused_volume": "Infused volume.",
    "infused_volume_unit": "Infused volume unit.",
    "infusion_rate": "Infusion rate.",
    "infusion_rate_unit": "Infusion rate unit.",
    "ingredients": "Ordered ingredient-component evidence.",
    "ingredient_count": "Number of retained ingredient rows.",
    "performer_practitioner_id": "Administration performer.",
    "verifier_practitioner_id": "Administration verifier.",
    "location_id": "Administering nurse-unit location.",
    "organization_id": "Source organization reference.",
    "scheduled_datetime": "Scheduled administration time.",
    "performed_datetime": "Performed administration time.",
    "verified_datetime": "Verification time.",
    "order_status_code": "Linked source order status code.",
    "order_status_display": "Linked source order status display.",
    "prn_ind": "As-needed indicator.",
    "iv_ind": "Intravenous indicator.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.medication_admin"),
    comment="One Millennium medication-administration event with current state and ordered history.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=MEDICATION_ADMIN_COLUMN_COMMENTS,
)
def medication_admin():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_medication_admin")),
        "medication_admin",
        {
            "ingredients": "_qc_ingredients_json",
            "medication_code": "_qc_medication_code_json",
            "status_history": "_qc_status_history_json",
        },
        MEDICATION_ADMIN_PUBLIC_COLUMNS,
    )

In [0]:
def _medication_order_canonical_pregate():
    p = spark.read.table(_n("journey_clinical._medication_order_primitive"))
    return (
        p.withColumn("medication_code", F.parse_json(F.col("_medication_code_json")))
         .withColumn("status_history", F.parse_json(F.col("_status_history_json")))
         .withColumn("ingredients", F.parse_json(F.col("_ingredients_json")))
         .withColumn("order_details", F.parse_json(F.col("_order_details_json")))
         .drop(
             "_medication_code_json", "_status_history_json",
             "_ingredients_json", "_order_details_json",
         )
    )

def _medication_order_canonical():
    return _medication_order_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_medication_order"),
    comment="Private JSON bridge and Gold cross-rule flags for medication_order.",
    refresh_policy="incremental",
)
def _qc_medication_order():
    return _cross_qc_primitive(
        _medication_order_canonical(),
        "medication_order",
        {
            "ingredients": "_qc_ingredients_json",
            "medication_code": "_qc_medication_code_json",
            "order_details": "_qc_order_details_json",
            "status_history": "_qc_status_history_json",
        },
    )

In [0]:
MEDICATION_ORDER_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "medication_code",
    "order_status_code", "order_status_display", "department_status_code",
    "department_status_display", "active_status_code", "active_status_display",
    "intent_code", "medication_order_type_code", "medication_order_type_display",
    "authored_datetime", "effective_start_datetime", "projected_stop_datetime",
    "discontinued_datetime", "frequency_id", "prn_ind", "iv_ind", "suspend_ind",
    "resume_ind", "discontinue_ind", "requester_practitioner_id", "organization_id",
    "clinical_display_line", "order_detail_display_line", "status_history",
    "status_history_count", "ingredients", "ingredient_count", "order_details",
    "order_detail_count", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

MEDICATION_ORDER_COLUMN_COMMENTS = {
    "patient_event_id": "Stable medication-order event identifier.",
    "fact_row_id": "Storage row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Subject-key identifier system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Order authored or start time.",
    "event_end_datetime": "Projected stop or discontinue time.",
    "source_coding_system": "Source medication coding system.",
    "source_code": "Source medication synonym identifier.",
    "source_display": "Source medication display.",
    "medication_code": "Source medication CodeableConcept.",
    "order_status_code": "Current source order status code.",
    "order_status_display": "Current source order status display.",
    "department_status_code": "Department workflow status code.",
    "department_status_display": "Department workflow status display.",
    "active_status_code": "Source active-status code.",
    "active_status_display": "Source active-status display.",
    "intent_code": "Medication request intent.",
    "medication_order_type_code": "Source medication-order type code.",
    "medication_order_type_display": "Source medication-order type display.",
    "authored_datetime": "Original order time.",
    "effective_start_datetime": "Current effective start.",
    "projected_stop_datetime": "Projected stop time.",
    "discontinued_datetime": "Discontinue effective time.",
    "frequency_id": "Source frequency identifier.",
    "prn_ind": "As-needed indicator.",
    "iv_ind": "Intravenous indicator.",
    "suspend_ind": "Source suspended indicator.",
    "resume_ind": "Source resumed indicator.",
    "discontinue_ind": "Source discontinued indicator.",
    "requester_practitioner_id": "Last updating provider reference.",
    "organization_id": "Source organization reference.",
    "clinical_display_line": "Source clinical display line.",
    "order_detail_display_line": "Source order-detail display line.",
    "status_history": "Ordered source action/status history.",
    "status_history_count": "Number of retained action rows.",
    "ingredients": "Ordered medication-order ingredients.",
    "ingredient_count": "Number of retained ingredient rows.",
    "order_details": "Ordered latest-action detail rows.",
    "order_detail_count": "Number of retained detail rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.medication_order"),
    comment="One Millennium medication order with current state and ordered action, ingredient, and detail evidence.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=MEDICATION_ORDER_COLUMN_COMMENTS,
)
def medication_order():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_medication_order")),
        "medication_order",
        {
            "ingredients": "_qc_ingredients_json",
            "medication_code": "_qc_medication_code_json",
            "order_details": "_qc_order_details_json",
            "status_history": "_qc_status_history_json",
        },
        MEDICATION_ORDER_PUBLIC_COLUMNS,
    )

In [0]:
def _medication_dispense_canonical_pregate():
    s = read_source(SRC_PHARMACY_ISSUE).where(
        F.upper(F.trim(F.coalesce(F.col("ISSUE_CATEGORY"), F.lit("")))) == F.lit("ISSUE")
    )
    event_id = stable_id("pharmacy_issue:jac", s.PHARMACY_ISSUE_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:jac:lnkpid", s.LNKPID)],
        SRC_PHARMACY_ISSUE,
        s.PHARMACY_ISSUE_ID,
    )
    retracted = F.coalesce(~s.SOURCE_PRESENT_IND, F.lit(False))
    source_code = F.coalesce(s.JAC_DRUG_ID, s.BNF_CODE_RAW)
    source_display = F.coalesce(s.DRUG_FULL, s.DRUG_NAME)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("long").cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.LNKPID), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.ISSUE_DTTM.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:jac:drug-id").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:jac:drug-id"), source_code, source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), s.DMD_VTM_CODE, s.DMD_VTM_NAME,
                       False, "bronze.map_pharmacy_issue", None),
        ).alias("medication_code"),
        F.when(retracted, F.lit("stopped")).otherwise(F.lit("completed"))
        .alias("status_code"),
        s.ISSUE_TYPE.alias("issue_type"), s.ISSUE_CATEGORY.alias("issue_category"),
        s.TOTAL_UNITS.cast("decimal(38,18)").alias("quantity"),
        s.DRUG_DOSEUNIT.alias("quantity_unit"),
        s.ISSUED_CONTAINERS.cast("decimal(38,18)").alias("issued_containers"),
        s.UNITS_PER_CONTAINER.cast("decimal(38,18)").alias("units_per_container"),
        s.DRUG_FORM.alias("drug_form"), s.DRUG_STRENGTH.alias("drug_strength"),
        F.when(s.CARE_SITE_CD.isNotNull(),
               stable_id("location:mill:nurse_unit", s.CARE_SITE_CD)).alias("location_id"),
        s.ISSUE_VALUE_GBP.cast("decimal(38,18)").alias("issue_value_gbp"),
        s.DAILYISSUES_KEY.alias("source_transaction_identifier"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.SOURCE_RECORD_UPDATED_DT.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pharmacy_issue").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_RECORD_UPDATED_DT.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("jac").alias("_source_system"),
        F.lit(SRC_PHARMACY_ISSUE).alias("_source_table"),
        s.PHARMACY_ISSUE_ID.alias("_source_row_id"),
        s.DMD_VTM_CODE.alias("_dmd_code"), s.DMD_VTM_NAME.alias("_dmd_display"),
    )

def _medication_dispense_canonical():
    return _medication_dispense_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_medication_dispense"),
    comment="Private JSON bridge and Gold cross-rule flags for medication_dispense.",
    refresh_policy="incremental",
)
def _qc_medication_dispense():
    return _cross_qc_primitive(
        _medication_dispense_canonical(),
        "medication_dispense",
        {"medication_code": "_qc_medication_code_json"},
    )

In [0]:
MEDICATION_DISPENSE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "medication_code",
    "status_code", "issue_type", "issue_category", "quantity", "quantity_unit",
    "issued_containers", "units_per_container", "drug_form", "drug_strength",
    "location_id", "issue_value_gbp", "source_transaction_identifier", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

MEDICATION_DISPENSE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable promotion-safe event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context when a source supplies one.",
    "event_datetime": "Source issue timestamp.",
    "event_end_datetime": "Dispense end timestamp when supplied.",
    "source_coding_system": "Verbatim JAC drug coding system.",
    "source_code": "Verbatim JAC drug identifier.",
    "source_display": "Verbatim JAC drug description.",
    "medication_code": "Source JAC and mapped dm+d medication codings.",
    "status_code": "Source-presence-derived dispense status.",
    "issue_type": "Raw JAC issue type.",
    "issue_category": "Governed JAC transaction category; only ISSUE is typed here.",
    "quantity": "Best-effort parsed total units.",
    "quantity_unit": "Source dose-unit description.",
    "issued_containers": "Parsed source container count.",
    "units_per_container": "Parsed source units per container.",
    "drug_form": "Source drug form.",
    "drug_strength": "Source drug strength.",
    "location_id": "Resolved care-site reference when a unique source location match exists.",
    "issue_value_gbp": "Source-recorded issue value including legitimate negative values outside this route.",
    "source_transaction_identifier": "Raw JAC dailyissues traceability key.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Upstream source update timestamp.",
    "record_status_effective_to": "Retraction timestamp when source presence is false.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "source_feed": "Registered source feed owning the typed fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.medication_dispense"),
    comment="JAC ISSUE transactions admitted directly as typed dispense facts.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=MEDICATION_DISPENSE_COLUMN_COMMENTS,
)
def medication_dispense():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_medication_dispense")),
        "medication_dispense",
        {"medication_code": "_qc_medication_code_json"},
        MEDICATION_DISPENSE_PUBLIC_COLUMNS,
    )

In [0]:
# ==== Assertions and generic event routing ====

CLINICAL_FINDING_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "finding_code", "finding_kind",
    "value_text", "value_datetime", "value_code", "value_display", "normalcy_code",
    "normalcy_display", "result_status_code", "result_status_display", "order_id",
    "parent_event_id", "performer_practitioner_id", "verifier_practitioner_id",
    "organization_id", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _closed_timestamp(value):
    return value.isNotNull() & (value < F.lit("2100-01-01").cast("timestamp"))

def _coded_finding_canonical():
    s = read_source(SRC_CODED_EVENTS).alias("s")
    natural = F.coalesce(
        s.SOURCE_ROW_KEY,
        F.concat_ws(":", s.EVENT_ID.cast("string"), s.SEQUENCE_NBR.cast("string")),
        s.ROW_HASH.cast("string"),
    )
    event_id = stable_id("coded_event:mill", natural)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_CODED_EVENTS, natural
    )
    event_time = F.coalesce(s.PERFORMED_DT_TM, s.EVENT_END_DT_TM, s.EVENT_START_DT_TM)
    event_display = F.coalesce(s.EVENT_LABEL, s.EVENT_CD_LABEL, s.EVENT_TITLE_TEXT, s.EVENT_TAG)
    value_code = F.when(s.NOMENCLATURE_ID > 0, s.NOMENCLATURE_ID.cast("string")) \
        .otherwise(s.RESULT_CD.cast("string"))
    value_display = F.coalesce(s.RESULT_LABEL, s.DESCRIPTOR, s.RESULT_VAL)
    ended = _closed_timestamp(F.least(s.CR_VALID_UNTIL_DT_TM, s.CE_VALID_UNTIL_DT_TM))
    status = _generic_record_status(F.lit(False), ended)
    source_update = F.greatest(s.SOURCE_ADC_UPDT, s.CR_UPDT_DT_TM, s.CE_UPDT_DT_TM, s.ADC_UPDT)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        event_time.alias("event_datetime"), s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:code_value").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"), event_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:code_value"), s.EVENT_CD, event_display, True)
        ).alias("finding_code"),
        F.lit("coded").alias("finding_kind"),
        F.coalesce(s.DESCRIPTOR, s.RESULT_VAL).alias("value_text"),
        F.lit(None).cast("timestamp").alias("value_datetime"),
        value_code.alias("value_code"), value_display.alias("value_display"),
        s.NORMALCY_CD.cast("string").alias("normalcy_code"),
        s.NORMALCY_DISPLAY.alias("normalcy_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        s.RESULT_STATUS_LABEL.alias("result_status_display"),
        s.ORDER_ID.cast("string").alias("order_id"),
        s.PARENT_EVENT_ID.cast("string").alias("parent_event_id"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.when(s.VERIFIED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.VERIFIED_PRSNL_ID))
         .alias("verifier_practitioner_id"),
        F.when(s.ORGANIZATION_ID.isNotNull(), stable_id("organization:mill", s.ORGANIZATION_ID))
         .alias("organization_id"),
        status.alias("record_status"),
        F.coalesce(s.CR_VALID_FROM_DT_TM, s.CE_VALID_FROM_DT_TM, event_time)
         .alias("record_status_effective_from"),
        F.when(status == "superseded", F.least(s.CR_VALID_UNTIL_DT_TM, s.CE_VALID_UNTIL_DT_TM))
         .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("coded_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_CODED_EVENTS).alias("_source_table"), natural.alias("_source_row_id"),
        s.SOURCE_ROW_KEY.alias("_mapping_source_row_key"),
        s.OMOP_MANUAL_CONCEPT.cast("string").alias("_omop_code"),
        s.OMOP_MANUAL_CONCEPT_NAME.alias("_omop_display"),
        s.OMOP_MANUAL_CONCEPT_DOMAIN.alias("_omop_domain"),
        _generic_payload(s).alias("_payload"),
        s.SOURCE_ROW_KEY.isNotNull().alias("_typed_route"),
    )

def _nomen_finding_canonical():
    s = read_source(SRC_NOMEN_EVENTS).alias("s")
    natural = F.coalesce(
        s.SOURCE_ROW_KEY,
        F.concat_ws(":", s.EVENT_ID.cast("string"), s.SEQUENCE_NBR.cast("string")),
        s.ROW_HASH.cast("string"),
    )
    event_id = stable_id("nomen_event:mill", natural)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_NOMEN_EVENTS, natural
    )
    event_time = F.coalesce(s.CLINICAL_EVENT_DT_TM, s.EVENT_END_DT_TM,
                            s.PERFORMED_DT_TM, s.EVENT_START_DT_TM)
    event_display = F.coalesce(s.EVENT_NAME, s.EVENT_CD_DISPLAY, s.EVENT_TAG)
    value_code = F.when(_present(s.SOURCE_IDENTIFIER), F.trim(s.SOURCE_IDENTIFIER)) \
        .otherwise(F.coalesce(s.NOMENCLATURE_CODE, s.NOMENCLATURE_ID.cast("string")))
    value_display = F.coalesce(s.SOURCE_STRING, s.SNOMED_TERM,
                               s.RESOLVED_OMOP_CONCEPT_NAME, s.EVENT_NAME)
    ended_at = F.least(s.CR_VALID_UNTIL_DT_TM, s.CE_VALID_UNTIL_DT_TM)
    ended = _closed_timestamp(ended_at)
    status = _generic_record_status(s.SOURCE_DELETED_IND, ended, s.AUTHENTIC_FLAG)
    source_update = F.greatest(s.SOURCE_CHANGE_TS, s.NOMENCLATURE_ADC_UPDT, s.ADC_UPDT)
    organization = F.coalesce(s.CE_ORGANIZATION_ID, s.CR_ORGANIZATION_ID)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        event_time.alias("event_datetime"), s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:code_value").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"), event_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:code_value"), s.EVENT_CD, event_display, True)
        ).alias("finding_code"),
        F.lit("nomenclature").alias("finding_kind"),
        s.SOURCE_STRING.alias("value_text"), F.lit(None).cast("timestamp").alias("value_datetime"),
        value_code.alias("value_code"), value_display.alias("value_display"),
        s.NORMALCY_CD.cast("string").alias("normalcy_code"),
        s.NORMALCY_DISPLAY.alias("normalcy_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        s.RESULT_STATUS_DISPLAY.alias("result_status_display"),
        s.ORDER_ID.cast("string").alias("order_id"),
        s.PARENT_EVENT_ID.cast("string").alias("parent_event_id"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.when(s.VERIFIED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.VERIFIED_PRSNL_ID))
         .alias("verifier_practitioner_id"),
        F.when(organization.isNotNull(), stable_id("organization:mill", organization))
         .alias("organization_id"),
        status.alias("record_status"),
        F.coalesce(s.CR_VALID_FROM_DT_TM, s.CE_VALID_FROM_DT_TM, event_time)
         .alias("record_status_effective_from"),
        F.when(status == "retracted", source_update)
         .when(status == "superseded", ended_at).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("nomen_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_NOMEN_EVENTS).alias("_source_table"), natural.alias("_source_row_id"),
        s.RESOLVED_OMOP_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.RESOLVED_OMOP_CONCEPT_NAME.alias("_omop_display"),
        s.RESOLVED_OMOP_CONCEPT_DOMAIN.alias("_omop_domain"),
        s.SNOMED_CODE.cast("string").alias("_snomed_code"),
        s.SNOMED_TERM.alias("_snomed_display"),
        _generic_payload(s).alias("_payload"),
        s.SOURCE_ROW_KEY.isNotNull().alias("_typed_route"),
    )

def _date_finding_canonical():
    s = read_source(SRC_DATE_EVENTS).alias("s")
    natural = F.coalesce(s.EVENT_ID.cast("string"), s.ROW_HASH.cast("string"))
    event_id = stable_id("date_event:mill", natural)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_DATE_EVENTS, natural
    )
    event_time = F.coalesce(s.RESULT_DT_TM, s.PERFORMED_DT_TM,
                            s.EVENT_END_DT_TM, s.EVENT_START_DT_TM)
    event_display = F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY, s.EVENT_TITLE_TEXT, s.EVENT_TAG)
    ended_at = F.least(s.DATE_RESULT_VALID_UNTIL_DT_TM, s.CLINICAL_EVENT_VALID_UNTIL_DT_TM)
    ended = _closed_timestamp(ended_at)
    status = _generic_record_status(s.SOURCE_DELETED_IND, ended, s.AUTHENTIC_FLAG)
    source_update = F.greatest(s.DATE_RESULT_EFFECTIVE_UPDT_DT_TM,
                               s.CLINICAL_EVENT_ADC_UPDT, s.LOOKUP_ADC_UPDT, s.ADC_UPDT)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        event_time.alias("event_datetime"), s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:code_value").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"), event_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:code_value"), s.EVENT_CD, event_display, True)
        ).alias("finding_code"),
        F.lit("date").alias("finding_kind"), F.lit(None).cast("string").alias("value_text"),
        s.RESULT_DT_TM.alias("value_datetime"), F.lit(None).cast("string").alias("value_code"),
        F.lit(None).cast("string").alias("value_display"),
        s.NORMALCY_CD.cast("string").alias("normalcy_code"),
        s.NORMALCY_DISPLAY.alias("normalcy_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        F.lit(None).cast("string").alias("result_status_display"),
        s.ORDER_ID.cast("string").alias("order_id"),
        s.PARENT_EVENT_ID.cast("string").alias("parent_event_id"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.when(s.VERIFIED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.VERIFIED_PRSNL_ID))
         .alias("verifier_practitioner_id"),
        F.when(s.ORGANIZATION_ID.isNotNull(), stable_id("organization:mill", s.ORGANIZATION_ID))
         .alias("organization_id"),
        status.alias("record_status"),
        F.coalesce(s.DATE_RESULT_VALID_FROM_DT_TM, s.CLINICAL_EVENT_VALID_FROM_DT_TM, event_time)
         .alias("record_status_effective_from"),
        F.when(status == "retracted", source_update)
         .when(status == "superseded", ended_at).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("date_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_DATE_EVENTS).alias("_source_table"), natural.alias("_source_row_id"),
        _generic_payload(s).alias("_payload"),
        (s.EVENT_ID.isNotNull() & (s.RESULT_DT_TM.isNotNull() | s.EVENT_CD.isNotNull()))
         .alias("_typed_route"),
    )

def _text_finding_canonical():
    s = read_source(SRC_TEXT_EVENTS).alias("s")
    natural = F.coalesce(s.EVENT_ID.cast("string"), s.ROW_HASH.cast("string"))
    event_id = stable_id("text_event:mill", natural)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_TEXT_EVENTS, natural
    )
    event_time = F.coalesce(s.RESULT_DT_TM, s.PERFORMED_DT_TM,
                            s.EVENT_END_DT_TM, s.EVENT_START_DT_TM)
    event_display = F.coalesce(s.EVENT_LABEL, s.EVENT_CD_DISPLAY, s.EVENT_TITLE_TEXT, s.EVENT_TAG)
    ended_at = F.least(s.STRING_RESULT_VALID_UNTIL_DT_TM, s.CLINICAL_EVENT_VALID_UNTIL_DT_TM)
    ended = _closed_timestamp(ended_at)
    status = _generic_record_status(s.SOURCE_DELETED_IND, ended, s.AUTHENTIC_FLAG)
    source_update = F.greatest(s.STRING_RESULT_EFFECTIVE_UPDT_DT_TM,
                               s.CLINICAL_EVENT_ADC_UPDT, s.LONG_TEXT_ADC_UPDT,
                               s.LOOKUP_ADC_UPDT, s.ADC_UPDT)
    document_candidate = _text_document_candidate(s)
    route = F.when(s.EVENT_ID.isNull(), F.lit("annex_catch_all")) \
        .when(document_candidate, F.lit("document_candidate")) \
        .otherwise(F.lit("clinical_finding"))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(_present(s.PERSON_ID), F.lit("resolved")).otherwise(F.lit("unresolved"))
         .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        event_time.alias("event_datetime"), s.EVENT_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:code_value").alias("source_coding_system"),
        s.EVENT_CD.cast("string").alias("source_code"), event_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:cerner:code_value"), s.EVENT_CD, event_display, True)
        ).alias("finding_code"),
        F.lit("text").alias("finding_kind"), s.TEXT_RESULT.alias("value_text"),
        F.lit(None).cast("timestamp").alias("value_datetime"),
        F.lit(None).cast("string").alias("value_code"),
        F.lit(None).cast("string").alias("value_display"),
        s.NORMALCY_CD.cast("string").alias("normalcy_code"),
        s.NORMALCY_DISPLAY.alias("normalcy_display"),
        s.RESULT_STATUS_CD.cast("string").alias("result_status_code"),
        s.RESULT_STATUS_DISPLAY.alias("result_status_display"),
        s.ORDER_ID.cast("string").alias("order_id"),
        s.PARENT_EVENT_ID.cast("string").alias("parent_event_id"),
        s.PARENT_EVENT_CD.cast("string").alias("_parent_event_cd"),
        F.when(s.PERFORMED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.PERFORMED_PRSNL_ID))
         .alias("performer_practitioner_id"),
        F.when(s.VERIFIED_PRSNL_ID.isNotNull(),
               stable_id("practitioner:mill", s.VERIFIED_PRSNL_ID))
         .alias("verifier_practitioner_id"),
        F.when(s.ORGANIZATION_ID.isNotNull(), stable_id("organization:mill", s.ORGANIZATION_ID))
         .alias("organization_id"),
        status.alias("record_status"),
        F.coalesce(s.STRING_RESULT_VALID_FROM_DT_TM, s.CLINICAL_EVENT_VALID_FROM_DT_TM, event_time)
         .alias("record_status_effective_from"),
        F.when(status == "retracted", source_update)
         .when(status == "superseded", ended_at).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("text_event").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_TEXT_EVENTS).alias("_source_table"), natural.alias("_source_row_id"),
        s.OMOP_MANUAL_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.OMOP_MANUAL_CONCEPT_NAME.alias("_omop_display"),
        s.OMOP_MANUAL_CONCEPT_DOMAIN.alias("_omop_domain"),
        s.OMOP_MANUAL_VALUE_CONCEPT_ID.cast("string").alias("_omop_value_code"),
        s.OMOP_MANUAL_VALUE_CONCEPT_NAME.alias("_omop_value_display"),
        s.OMOP_MANUAL_VALUE_CONCEPT_DOMAIN.alias("_omop_value_domain"),
        F.lit(None).cast("string").alias("_anon_document_text"),
        F.lit(None).cast("string").alias("_anon_document_status"),
        _generic_payload(s).alias("_payload"), route.alias("_route"),
    )

def _coded_finding_typed():
    return _coded_finding_canonical().where(
        F.col("_typed_route") & _usable_code(F.col("source_code"))
    )

def _nomen_finding_typed():
    return _nomen_finding_canonical().where(
        F.col("_typed_route") & _usable_code(F.col("source_code"))
    )

def _date_finding_typed():
    return _date_finding_canonical().where(
        F.col("_typed_route") & _usable_code(F.col("source_code"))
    )

def _text_finding_typed():
    return _text_finding_canonical().where(
        (F.col("_route") == "clinical_finding") & _usable_code(F.col("source_code"))
    )

In [0]:
def _generic_record_status(deleted, ended, authentic=None):
    status = F.when(F.coalesce(deleted.cast("boolean"), F.lit(False)), F.lit("retracted"))
    superseded = ended
    if authentic is not None:
        superseded = superseded | (F.coalesce(authentic.cast("long"), F.lit(1)) == 0)
    return status.when(superseded, F.lit("superseded")).otherwise(F.lit("active"))

def _generic_payload(source):
    return F.parse_json(F.to_json(F.struct(*[source[c] for c in source.columns])))

TEXT_EVENT_DOCUMENT_CANDIDATE_PREDICATE = (
    "EVENT_ID IS NOT NULL "
    "AND coalesce(TEXT_RESULT_LENGTH, length(TEXT_RESULT), 0) > 100 "
    "AND lower(coalesce(nullif(trim(EVENT_LABEL), ''), nullif(trim(EVENT_CD_DISPLAY), ''), "
    "nullif(trim(EVENT_TITLE_TEXT), ''), '')) NOT IN ("
    "'25 vit d comment','25-hydroxy vitamin d3 serum','ana comment','ana pattern',"
    "'cardiolipin antibody comment','cardiolipin antibody screen','egfr comment',"
    "'estimated gfr','fbc comments','gastric parietal cell antibody','haemoglobin',"
    "'haemoglobinopathy screen conclusion','haemostasis comments','hav igm qualitative',"
    "'hb core antibody qualitative (anti hbc)','hb surface antigen qualitative (hbsag)',"
    "'hba1c comments','hba1c diabetic control ranges:','hcv igg qualitative',"
    "'hiv 1 and 2 antibody qualitative','kleihauer','lithium comments',"
    "'lupus anticoagulant screen','malaria rdt antigen result','oestradiol ref. range',"
    "'pcr comment','tacrolimus comments','tb gamma interferon assay',"
    "'tissue transglutaminase antibody comment','troponin comments',"
    "'urine albumin comments','urine protein comments')"
)

def _text_document_candidate(source):
    return F.expr(TEXT_EVENT_DOCUMENT_CANDIDATE_PREDICATE)

@materialized_view(
    name=_n("journey_clinical._qc_clinical_finding"),
    comment="Private JSON bridge and Gold cross-rule flags for clinical_finding.",
    refresh_policy="incremental",
)
def _qc_clinical_finding():
    source = (
        _coded_finding_typed().select(*CLINICAL_FINDING_PUBLIC_COLUMNS)
        .unionByName(_nomen_finding_typed().select(*CLINICAL_FINDING_PUBLIC_COLUMNS))
        .unionByName(_date_finding_typed().select(*CLINICAL_FINDING_PUBLIC_COLUMNS))
        .unionByName(_text_finding_typed().select(*CLINICAL_FINDING_PUBLIC_COLUMNS))
    )
    return _cross_qc_primitive(
        source,
        "clinical_finding",
        {"finding_code": "_qc_finding_code_json"},
    )

In [0]:
CLINICAL_FINDING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide clinical-finding identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime": "Clinically relevant finding timestamp.",
    "event_end_datetime": "Source effective end timestamp.",
    "source_coding_system": "Verbatim source event coding system.",
    "source_code": "Verbatim source event code.",
    "source_display": "Verbatim source event display.",
    "finding_code": "Source finding CodeableConcept.",
    "finding_kind": "Generic result representation carried by the source feed.",
    "value_text": "Verbatim text result or descriptor.",
    "value_datetime": "Date or date-time result value.",
    "value_code": "Source coded result value.",
    "value_display": "Source coded result display.",
    "normalcy_code": "Source normalcy or interpretation code.",
    "normalcy_display": "Source normalcy or interpretation display.",
    "result_status_code": "Source result-status code.",
    "result_status_display": "Source result-status display.",
    "order_id": "Source order identifier when supplied.",
    "parent_event_id": "Source parent-event identifier.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "verifier_practitioner_id": "Verifying practitioner reference.",
    "organization_id": "Source organization reference.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered generic-event feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.clinical_finding"),
    comment="Governed residual clinical observations from exhaustive generic-event routes.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CLINICAL_FINDING_COLUMN_COMMENTS,
)
def clinical_finding():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_clinical_finding")),
        "clinical_finding",
        {"finding_code": "_qc_finding_code_json"},
        CLINICAL_FINDING_PUBLIC_COLUMNS,
    )

In [0]:
SRC_PACS_EXAMINATION = "4_prod.bronze.map_pacs_examination"

IMAGING_EXAM_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "exam_code", "status_code",
    "accession_identifier", "study_instance_uid", "modality_code", "body_site_code",
    "report_patient_event_id", "requester_practitioner_id", "performer_practitioner_id",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _imaging_exam_canonical_pregate():
    s = read_source(SRC_PACS_EXAMINATION)
    event_id = stable_id("imaging_exam:pacs", s.PACS_EXAMINATION_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID),
         ("urn:sectra:pacs-patient-id", s.PACS_PATIENT_ID)],
        SRC_PACS_EXAMINATION,
        s.PACS_EXAMINATION_ID,
    )
    retracted = F.coalesce(~s.SOURCE_PRESENT_IND, F.lit(False))
    source_display = s.EXAMINATION_DESCRIPTION
    source_code = _code_or_display(s.EXAMINATION_CODE, source_display)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(s.PACS_PATIENT_ID.isNotNull(), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        F.coalesce(s.EXAMINATION_DT_TM, s.ARRIVAL_DT_TM).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:sectra:examination-code").alias("source_coding_system"),
        source_code.alias("source_code"),
        source_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:sectra:examination-code"), source_code,
                       source_display, True)
        ).alias("exam_code"),
        F.when(retracted, F.lit("cancelled")).otherwise(F.lit("available"))
        .alias("status_code"),
        s.MILL_LINK_REF.alias("accession_identifier"),
        s.STUDY_INSTANCE_UID.alias("study_instance_uid"), s.MODALITY.alias("modality_code"),
        s.BODY_PART.alias("body_site_code"),
        F.when(s.LATEST_REPORT_ID.isNotNull(),
               stable_id("document:pacs_report", s.LATEST_REPORT_ID))
        .alias("report_patient_event_id"),
        F.lit(None).cast("string").alias("requester_practitioner_id"),
        F.lit(None).cast("string").alias("performer_practitioner_id"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.SRC_ADC_UPDT.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("pacs_examination").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SRC_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("sectra-pacs").alias("_source_system"),
        F.lit(SRC_PACS_EXAMINATION).alias("_source_table"),
        s.PACS_EXAMINATION_ID.cast("string").alias("_source_row_id"),
    )

def _imaging_exam_canonical():
    return _imaging_exam_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_imaging_exam"),
    comment="Private JSON bridge and Gold cross-rule flags for imaging_exam.",
    refresh_policy="incremental",
)
def _qc_imaging_exam():
    source = (
        _imaging_exam_canonical().select(*IMAGING_EXAM_PUBLIC_COLUMNS)
        .unionByName(_mill_radiology_exam_canonical().select(*IMAGING_EXAM_PUBLIC_COLUMNS))
    )
    return _cross_qc_primitive(
        source,
        "imaging_exam",
        {"exam_code": "_qc_exam_code_json"},
    )

In [0]:
IMAGING_EXAM_COLUMN_COMMENTS = {
    "patient_event_id": "Stable imaging-exam identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated deterministic subject key when populated.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person reference.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Imaging study start timestamp.",
    "event_end_datetime": "Imaging study end timestamp.",
    "source_coding_system": "Source exam coding system.",
    "source_code": "Source exam code",
    "source_display": "Source exam display.",
    "exam_code": "Source and mapped imaging-exam CodeableConcept.",
    "status_code": "Imaging study status.",
    "accession_identifier": "Imaging accession identifier.",
    "study_instance_uid": "DICOM study instance UID.",
    "modality_code": "Imaging modality code.",
    "body_site_code": "Body-site code.",
    "report_patient_event_id": "Linked diagnostic-report event when available.",
    "requester_practitioner_id": "Requesting practitioner reference.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered PACS or Millennium radiology source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.imaging_exam"),
    comment="One PACS examination/study from the governed bronze landing with report linkage.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=IMAGING_EXAM_COLUMN_COMMENTS,
)
def imaging_exam():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_imaging_exam")),
        "imaging_exam",
        {"exam_code": "_qc_exam_code_json"},
        IMAGING_EXAM_PUBLIC_COLUMNS,
    )

In [0]:
def _pacs_study_artifact_canonical():
    """One traceable DICOM-study asset per governed PACS examination.

    The study UID is a source locator, not a promise that Silver can retrieve the
    underlying pixels. storage_uri remains null until a governed serving manifest exists.
    """
    s = read_source(SRC_PACS_EXAMINATION)
    artifact_id = stable_id("artifact:pacs_study", s.PACS_EXAMINATION_ID)
    imaging_event_id = stable_id("imaging_exam:pacs", s.PACS_EXAMINATION_ID)
    skey, ssys = subject_key_with_system(
        [
            ("urn:cerner:person_id", s.PERSON_ID),
            ("urn:sectra:pacs-patient-id", s.PACS_PATIENT_ID),
        ],
        SRC_PACS_EXAMINATION,
        s.PACS_EXAMINATION_ID,
    )
    retracted = F.coalesce(~s.SOURCE_PRESENT_IND, F.lit(False))
    event_time = F.coalesce(s.EXAMINATION_DT_TM, s.ARRIVAL_DT_TM)
    source_code = _code_or_display(s.EXAMINATION_CODE, s.EXAMINATION_DESCRIPTION)
    return (
        s.where(s.STUDY_INSTANCE_UID.isNotNull())
        .select(
            artifact_id.alias("patient_event_id"),
            artifact_id.alias("fact_row_id"),
            artifact_id.alias("artifact_id"),
            F.lit(None).cast("string").alias("parent_artifact_id"),
            skey.alias("subject_key"),
            ssys.alias("subject_id_system"),
            s.PERSON_ID.cast("string").alias("person_id"),
            F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
            .when(s.PACS_PATIENT_ID.isNotNull(), F.lit("provisional"))
            .otherwise(F.lit("unresolved")).alias("identity_status"),
            F.lit(None).cast("string").alias("encounter_id"),
            event_time.alias("event_datetime"),
            F.lit(None).cast("timestamp").alias("event_end_datetime"),
            F.lit("urn:sectra:examination-code").alias("source_coding_system"),
            source_code.alias("source_code"),
            s.EXAMINATION_DESCRIPTION.alias("source_display"),
            codeable_concept(
                coding_obj(
                    F.lit("urn:sectra:examination-code"),
                    source_code,
                    s.EXAMINATION_DESCRIPTION,
                    True,
                )
            ).alias("artifact_type"),
            F.lit("image").alias("artifact_class"),
            F.lit("study").alias("artifact_level"),
            event_time.alias("acquisition_datetime"),
            s.MODALITY.alias("modality_code"),
            s.BODY_PART.alias("body_site_code"),
            F.lit("urn:dicom:study-instance-uid").alias("locator_system"),
            s.STUDY_INSTANCE_UID.alias("locator_value"),
            F.lit(None).cast("string").alias("storage_uri"),
            F.when(retracted, F.lit("withdrawn"))
            .when(s.SERIES_PIXEL_DATA_IND == F.lit(False), F.lit("unavailable"))
            .otherwise(F.lit("metadata_only")).alias("availability_status"),
            s.STUDY_INSTANCE_UID.alias("study_instance_uid"),
            F.lit(None).cast("string").alias("series_instance_uid"),
            F.lit(None).cast("string").alias("sop_instance_uid"),
            F.coalesce(s.SERIES_COUNT_MEASURED, s.SERIES_COUNT).cast("long")
            .alias("series_count"),
            F.coalesce(s.SERIES_OBJECT_COUNT, s.IMAGE_COUNT).cast("long")
            .alias("object_count"),
            s.FOLDER_COUNT.cast("long").alias("folder_count"),
            s.SERIES_PIXEL_DATA_IND.cast("boolean").alias("payload_present_ind"),
            F.lit(None).cast("string").alias("file_name"),
            F.lit("application/dicom").alias("content_type"),
            s.STORED_SIZE_BYTES.cast("long").alias("byte_size"),
            F.lit(None).cast("string").alias("sha256"),
            s.PACS_EXAMINATION_ID.cast("string").alias("source_artifact_id"),
            F.lit(None).cast("string").alias("ingest_run_id"),
            s.LAST_ACCESSED_DT_TM.alias("last_accessed_datetime"),
            s.ARCHIVE_STATE_CD.cast("string").alias("archive_status_code"),
            F.lit(None).cast("string").alias("burned_in_pii_tier"),
            s.EXAMINATION_STATUS_CD.cast("string").alias("status_code"),
            F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
            .alias("record_status"),
            s.SRC_ADC_UPDT.alias("record_status_effective_from"),
            F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
            F.lit(None).cast("string").alias("confidentiality_code"),
            F.lit(None).cast("boolean").alias("vip_ind"),
            F.lit(None).cast("boolean").alias("withheld_identity_ind"),
            F.parse_json(F.lit("null")).alias("sensitivity_labels"),
            F.lit("pacs_examination").alias("source_feed"),
            F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
            s.SRC_ADC_UPDT.alias("source_update_timestamp"),
            s.ADC_UPDT.alias("loaded_at"),
            F.lit("sectra-pacs").alias("_source_system"),
            F.lit(SRC_PACS_EXAMINATION).alias("_source_table"),
            s.PACS_EXAMINATION_ID.cast("string").alias("_source_row_id"),
            imaging_event_id.alias("_imaging_patient_event_id"),
            F.when(
                s.LATEST_REPORT_ID.isNotNull(),
                stable_id("document:pacs_report", s.LATEST_REPORT_ID),
            ).alias("_report_patient_event_id"),
        )
    )

def _dicom_file_artifact_canonical():
    """One file-level asset per governed DICOM volume path in the curated pilot."""
    s = read_source(SRC_DICOM_FILE_ATTRIBUTE)
    artifact_id = stable_id("artifact:dicom_file", s.DICOM_PATH)
    parent_artifact_id = F.when(
        s.PACS_EXAMINATION_ID.isNotNull(),
        stable_id("artifact:pacs_study", s.PACS_EXAMINATION_ID),
    )
    imaging_event_id = F.when(
        s.PACS_EXAMINATION_ID.isNotNull(),
        stable_id("imaging_exam:pacs", s.PACS_EXAMINATION_ID),
    )
    skey, ssys = subject_key_with_system(
        [
            ("urn:cerner:person_id", s.PERSON_ID),
            ("urn:sectra:pacs-patient-id", s.PACS_PATIENT_ID),
            ("urn:dicom:patient-id", s.DICOM_PATIENT_ID),
        ],
        SRC_DICOM_FILE_ATTRIBUTE,
        s.DICOM_PATH,
    )
    retracted = F.coalesce(~s.SOURCE_PRESENT_IND, F.lit(False))
    event_time = F.coalesce(
        s.ACQUISITION_DT_TM_CLEAN,
        s.CONTENT_DT_TM_CLEAN,
        s.SERIES_DT_TM_CLEAN,
        s.STUDY_DT_TM_CLEAN,
        s.INSTANCE_CREATION_DT_TM_CLEAN,
    )
    source_code = _code_or_display(s.SOP_CLASS_UID, s.MODALITY)
    source_display = F.coalesce(
        s.IMAGE_TYPE, s.SERIES_DESCRIPTION, s.STUDY_DESCRIPTION, s.SOP_CLASS_UID,
    )
    return (
        s.where(s.DICOM_PATH.isNotNull())
        .select(
            artifact_id.alias("patient_event_id"),
            artifact_id.alias("fact_row_id"),
            artifact_id.alias("artifact_id"),
            parent_artifact_id.alias("parent_artifact_id"),
            skey.alias("subject_key"),
            ssys.alias("subject_id_system"),
            s.PERSON_ID.cast("string").alias("person_id"),
            F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
            .when(s.PACS_PATIENT_ID.isNotNull() | s.DICOM_PATIENT_ID.isNotNull(),
                  F.lit("provisional"))
            .otherwise(F.lit("unresolved")).alias("identity_status"),
            F.lit(None).cast("string").alias("encounter_id"),
            event_time.alias("event_datetime"),
            F.lit(None).cast("timestamp").alias("event_end_datetime"),
            F.lit("urn:dicom:sop-class-uid").alias("source_coding_system"),
            source_code.alias("source_code"),
            source_display.alias("source_display"),
            codeable_concept(
                coding_obj(
                    F.lit("urn:dicom:sop-class-uid"),
                    source_code,
                    source_display,
                    True,
                )
            ).alias("artifact_type"),
            F.lit("image").alias("artifact_class"),
            F.lit("file").alias("artifact_level"),
            event_time.alias("acquisition_datetime"),
            s.MODALITY.alias("modality_code"),
            s.BODY_PART_EXAMINED.alias("body_site_code"),
            F.lit("urn:barts:unity-catalog-volume-path").alias("locator_system"),
            s.DICOM_PATH.alias("locator_value"),
            s.DICOM_PATH.alias("storage_uri"),
            F.when(retracted, F.lit("withdrawn"))
            .otherwise(F.lit("available")).alias("availability_status"),
            s.STUDY_INSTANCE_UID.alias("study_instance_uid"),
            s.SERIES_INSTANCE_UID.alias("series_instance_uid"),
            s.SOP_INSTANCE_UID.alias("sop_instance_uid"),
            F.lit(None).cast("long").alias("series_count"),
            F.lit(1).cast("long").alias("object_count"),
            F.lit(None).cast("long").alias("folder_count"),
            (~retracted).cast("boolean").alias("payload_present_ind"),
            F.regexp_extract(s.DICOM_PATH, r"[^/]+$", 0).alias("file_name"),
            F.lit("application/dicom").alias("content_type"),
            F.lit(None).cast("long").alias("byte_size"),
            s.FILE_SHA256.alias("sha256"),
            s.DICOM_PATH.alias("source_artifact_id"),
            F.lit(None).cast("string").alias("ingest_run_id"),
            F.lit(None).cast("timestamp").alias("last_accessed_datetime"),
            F.lit(None).cast("string").alias("archive_status_code"),
            s.BURNED_IN_PII_TIER.alias("burned_in_pii_tier"),
            s.EXAM_LINK_STATUS.alias("status_code"),
            F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
            .alias("record_status"),
            s.SOURCE_MAX_EXTRACTION_TS.alias("record_status_effective_from"),
            F.when(retracted, s.PIPELINE_UPDT_DT_TM).alias("record_status_effective_to"),
            F.lit(None).cast("string").alias("confidentiality_code"),
            F.lit(None).cast("boolean").alias("vip_ind"),
            F.lit(None).cast("boolean").alias("withheld_identity_ind"),
            F.parse_json(F.lit("null")).alias("sensitivity_labels"),
            F.lit("dicom_file_attribute").alias("source_feed"),
            F.date_format(s.PIPELINE_UPDT_DT_TM, "yyyyMMddHHmmss").alias("load_batch_id"),
            s.SOURCE_MAX_EXTRACTION_TS.alias("source_update_timestamp"),
            s.PIPELINE_UPDT_DT_TM.alias("loaded_at"),
            F.lit("sectra-dicom-volume").alias("_source_system"),
            F.lit(SRC_DICOM_FILE_ATTRIBUTE).alias("_source_table"),
            s.DICOM_PATH.alias("_source_row_id"),
            imaging_event_id.alias("_imaging_patient_event_id"),
            F.lit(None).cast("string").alias("_report_patient_event_id"),
        )
    )

def _artifact_asset_canonical():
    return _pacs_study_artifact_canonical().unionByName(
        _dicom_file_artifact_canonical()
    )

In [0]:
SRC_DICOM_FILE_ATTRIBUTE = "4_prod.bronze.map_dicom_file_attribute"

ARTIFACT_ASSET_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "artifact_id", "parent_artifact_id", "subject_key",
    "subject_id_system", "person_id", "identity_status", "encounter_id",
    "event_datetime", "event_end_datetime", "source_coding_system", "source_code",
    "source_display", "artifact_type", "artifact_class", "artifact_level", "acquisition_datetime",
    "modality_code", "body_site_code", "locator_system", "locator_value",
    "storage_uri", "availability_status", "study_instance_uid", "series_instance_uid",
    "sop_instance_uid", "series_count", "object_count", "folder_count",
    "payload_present_ind", "file_name", "content_type", "byte_size", "sha256",
    "source_artifact_id", "ingest_run_id", "last_accessed_datetime",
    "archive_status_code", "burned_in_pii_tier", "status_code", "record_status",
    "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "sensitivity_labels",
    "source_feed", "load_batch_id", "source_update_timestamp",
    "loaded_at",
]

ARTIFACT_ASSET_COLUMN_COMMENTS = {
    "patient_event_id": "Stable event identifier equal to artifact_id.",
    "fact_row_id": "Storage-row identifier equal to artifact_id.",
    "artifact_id": "Stable clinical-artifact identifier.",
    "parent_artifact_id": "Containing artifact identifier",
    "subject_key": "Always-populated deterministic or source-keyed subject identifier.",
    "subject_id_system": "Identifier system behind subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when governed evidence supplies one.",
    "event_datetime": "Artifact acquisition or creation timestamp.",
    "event_end_datetime": "Artifact acquisition end timestamp when supplied.",
    "source_coding_system": "Verbatim source procedure or artifact coding system.",
    "source_code": "Verbatim source procedure or artifact code.",
    "source_display": "Verbatim source procedure or artifact display.",
    "artifact_type": "Source clinical-artifact CodeableConcept.",
    "artifact_class": "High-level payload class.",
    "artifact_level": "Artifact granularity represented by the row.",
    "acquisition_datetime": "Source acquisition timestamp.",
    "modality_code": "Imaging or acquisition modality when supplied.",
    "body_site_code": "Body-site code when supplied.",
    "locator_system": "Namespace of locator_value.",
    "locator_value": "Stable source locator such as a DICOM Study Instance UID.",
    "storage_uri": "Governed retrievable location when a serving manifest supplies one; never invented by Silver.",
    "availability_status": "Whether Silver has metadata only or a governed retrievable payload.",
    "study_instance_uid": "DICOM Study Instance UID when supplied.",
    "series_instance_uid": "DICOM Series Instance UID when supplied.",
    "sop_instance_uid": "DICOM SOP Instance UID when supplied.",
    "series_count": "Number of series represented by the artifact when supplied.",
    "object_count": "Number of image or file objects represented when supplied.",
    "folder_count": "Source folder count when supplied.",
    "payload_present_ind": "Source evidence that binary or pixel payload exists.",
    "file_name": "Source file name when a governed manifest supplies one.",
    "content_type": "MIME type or governed source format.",
    "byte_size": "Represented payload size in bytes when supplied.",
    "sha256": "Payload SHA-256 when a governed serving manifest supplies one.",
    "source_artifact_id": "Native source identifier for the represented artifact or study.",
    "ingest_run_id": "Upstream manifest or ingest-run identifier when supplied.",
    "last_accessed_datetime": "Source-reported last-access timestamp when supplied.",
    "archive_status_code": "Source archive-state code when supplied.",
    "burned_in_pii_tier": "Upstream burned-in-PII classification when supplied.",
    "status_code": "Source artifact status.",
    "record_status": "Normalized source lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Source confidentiality classification.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Source withheld-identity indicator.",
    "sensitivity_labels": "Ordered source security labels when supplied.",
    "source_feed": "Registered artifact source route.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_artifact.asset"),
    comment="One governed clinical artifact or artifact-collection locator; payloads remain outside Silver.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=ARTIFACT_ASSET_COLUMN_COMMENTS,
)
def artifact_asset():
    return _artifact_asset_canonical().select(*ARTIFACT_ASSET_PUBLIC_COLUMNS)

In [0]:
ARTIFACT_LINK_PUBLIC_COLUMNS = [
    "artifact_link_id", "artifact_id", "patient_event_id", "fact_table", "fact_row_id",
    "role_code", "role_display", "source_system", "source_table", "source_row_id",
    "load_batch_id", "loaded_at",
]

def _artifact_link_canonical():
    studies = _pacs_study_artifact_canonical()
    imaging = studies.select(
        stable_id(
            "artifact_link:pacs", F.col("artifact_id"), F.lit("content_of"),
            F.col("_imaging_patient_event_id"),
        ).alias("artifact_link_id"),
        "artifact_id",
        F.col("_imaging_patient_event_id").alias("patient_event_id"),
        F.lit("journey_clinical.imaging_exam").alias("fact_table"),
        F.col("_imaging_patient_event_id").alias("fact_row_id"),
        F.lit("content_of").alias("role_code"),
        F.lit("Imaging content of examination").alias("role_display"),
        F.col("_source_system").alias("source_system"),
        F.col("_source_table").alias("source_table"),
        F.col("_source_row_id").alias("source_row_id"),
        "load_batch_id", "loaded_at",
    )
    report = studies.where(F.col("_report_patient_event_id").isNotNull()).select(
        stable_id(
            "artifact_link:pacs", F.col("artifact_id"), F.lit("associated_report"),
            F.col("_report_patient_event_id"),
        ).alias("artifact_link_id"),
        "artifact_id",
        F.col("_report_patient_event_id").alias("patient_event_id"),
        F.lit("journey_text.document").alias("fact_table"),
        F.col("_report_patient_event_id").alias("fact_row_id"),
        F.lit("associated_report").alias("role_code"),
        F.lit("Diagnostic report for artifact").alias("role_display"),
        F.col("_source_system").alias("source_system"),
        F.col("_source_table").alias("source_table"),
        F.col("_source_row_id").alias("source_row_id"),
        "load_batch_id", "loaded_at",
    )
    files = _dicom_file_artifact_canonical()
    file_exam = files.where(F.col("_imaging_patient_event_id").isNotNull()).select(
        stable_id(
            "artifact_link:dicom_file", F.col("artifact_id"), F.lit("content_of"),
            F.col("_imaging_patient_event_id"),
        ).alias("artifact_link_id"),
        "artifact_id",
        F.col("_imaging_patient_event_id").alias("patient_event_id"),
        F.lit("journey_clinical.imaging_exam").alias("fact_table"),
        F.col("_imaging_patient_event_id").alias("fact_row_id"),
        F.lit("content_of").alias("role_code"),
        F.lit("DICOM file content of examination").alias("role_display"),
        F.col("_source_system").alias("source_system"),
        F.col("_source_table").alias("source_table"),
        F.col("_source_row_id").alias("source_row_id"),
        "load_batch_id", "loaded_at",
    )
    file_parent = files.where(F.col("parent_artifact_id").isNotNull()).select(
        stable_id(
            "artifact_link:dicom_file", F.col("artifact_id"), F.lit("part_of"),
            F.col("parent_artifact_id"),
        ).alias("artifact_link_id"),
        "artifact_id",
        F.col("parent_artifact_id").alias("patient_event_id"),
        F.lit("journey_artifact.asset").alias("fact_table"),
        F.col("parent_artifact_id").alias("fact_row_id"),
        F.lit("part_of").alias("role_code"),
        F.lit("DICOM file contained by imaging study").alias("role_display"),
        F.col("_source_system").alias("source_system"),
        F.col("_source_table").alias("source_table"),
        F.col("_source_row_id").alias("source_row_id"),
        "load_batch_id", "loaded_at",
    )
    return imaging.unionByName(report).unionByName(file_exam).unionByName(file_parent)

ARTIFACT_LINK_COLUMN_COMMENTS = {
    "artifact_link_id": "Stable relationship identifier.",
    "artifact_id": "Artifact identifier from artifact.asset.",
    "patient_event_id": "Linked event identifier.",
    "fact_table": "Logical typed table or parent artifact holding the linked row.",
    "fact_row_id": "Row identifier inside fact_table.",
    "role_code": "Coded relationship role.",
    "role_display": "Relationship-role display.",
    "source_system": "Source-system identifier.",
    "source_table": "Governed Bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_artifact.link"),
    comment="Traceable relationships from clinical artifacts to typed facts and documents.",
    refresh_policy="incremental",
    column_comments=ARTIFACT_LINK_COLUMN_COMMENTS,
)
def artifact_link():
    return _artifact_link_canonical().select(*ARTIFACT_LINK_PUBLIC_COLUMNS)

In [0]:
# ==== Text documents ====

def _pathology_report_document_base(rows):
    rows = _alias_resolved_accession(rows).alias("r")
    link = spark.read.table(_n("journey_events._pathology_parent_link")).select(
        F.col("source_parent_key").alias("_link_parent_key"),
        F.col("source_encounter_id"),
    ).alias("l")
    i = spark.read.table(_n("journey_events._pathology_accession_identity")).select(
        F.col("pathology_accession_id").alias("_identity_accession_id"),
        "canonical_person_id", "person_resolution_status", "evidence_mrn", "evidence_nhs",
    ).alias("i")
    d = (
        rows.join(link, F.col("r.source_record_key") == F.col("l._link_parent_key"), "left")
        .join(
            i,
            F.col("r._resolved_accession_id") == F.col("i._identity_accession_id"),
            "left",
        )
        .join(
            _pathology_document_alias_map("MRN", "mrn"),
            F.trim(F.col("i.evidence_mrn")) == F.col("_mrn_value"),
            "left",
        )
        .join(
            _pathology_document_alias_map("NHS", "nhs"),
            F.trim(F.col("i.evidence_nhs")) == F.col("_nhs_value"),
            "left",
        )
    )
    mrn_unique = F.coalesce(F.col("_mrn_count"), F.lit(0)) == 1
    nhs_unique = F.coalesce(F.col("_nhs_count"), F.lit(0)) == 1
    alias_conflict = (
        mrn_unique & nhs_unique
        & (F.col("_mrn_person") != F.col("_nhs_person"))
    )
    alias_person = (
        F.when(mrn_unique & nhs_unique
               & (F.col("_mrn_person") == F.col("_nhs_person")), F.col("_nhs_person"))
        .when(nhs_unique & ~mrn_unique, F.col("_nhs_person"))
        .when(mrn_unique & ~nhs_unique, F.col("_mrn_person"))
    )
    resolved_person = F.coalesce(F.col("canonical_person_id").cast("string"), alias_person)
    linkage_route = (
        F.when(F.col("canonical_person_id").isNotNull(), F.lit("direct"))
        .when(alias_conflict, F.lit("none"))
        .when(alias_person.isNotNull() & nhs_unique, F.lit("alias_nhs"))
        .when(alias_person.isNotNull(), F.lit("alias_mrn"))
        .otherwise(F.lit("none"))
    )
    event_id = stable_id("document:pathology_report", F.col("report_version_id"))
    thread_id = stable_id(
        "document_thread:pathology_report", F.col("report_series_id")
    )
    supersedes_id = F.when(
        _present(F.col("supersedes_report_version_id")),
        stable_id(
            "document:pathology_report", F.col("supersedes_report_version_id")
        ),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", resolved_person),
         ("urn:barts:mrn", F.col("evidence_mrn")),
         ("https://fhir.nhs.uk/Id/nhs-number", F.col("evidence_nhs"))],
        SRC_PATHOLOGY_REPORT_VERSIONS, F.col("report_version_id"),
    )
    retracted = F.col("lifecycle_status").isin("cancelled", "entered_in_error")
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        resolved_person.alias("person_id"),
        F.when(resolved_person.isNotNull(), F.lit("resolved"))
         .when(F.col("person_resolution_status") == "conflicting", F.lit("provisional"))
         .when(alias_conflict, F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("source_encounter_id").isNotNull(),
               stable_id("encounter:mill", F.col("source_encounter_id"))).alias("encounter_id"),
        _clamped_ts(F.col("issued_dt")).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:pathology:report-code").alias("source_coding_system"),
        F.col("report_code").cast("string").alias("source_code"),
        F.coalesce(F.col("report_role"), F.col("report_code")).cast("string")
         .alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:barts:pathology:report-code"),
                       F.col("report_code"), F.col("report_role"), True),
        ).alias("_document_type_json"),
        F.col("report_code").cast("string").alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.lit(None).cast("string").alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.col("lifecycle_status").alias("status_code"),
        F.col("report_version_id").alias("version_id"),
        F.col("report_text").alias("document_text"),
        F.lit("[]").alias("_sections_json"),
        F.lit(None).cast("string").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"),
        F.lit(None).cast("string").alias("encoding"),
        F.sha2(F.col("report_text"), 256).alias("text_sha256"),
        F.length(F.col("report_text")).cast("long").alias("text_length"),
        F.when(retracted, F.lit("retracted"))
         .when(F.coalesce(F.col("is_current"), F.lit(False)), F.lit("active"))
         .otherwise(F.lit("superseded")).alias("record_status"),
        F.col("valid_from").alias("record_status_effective_from"),
        F.col("valid_to").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit(None).cast("string").alias("document_class"),
        F.lit(None).cast("string").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        F.lit(None).cast("string").alias("source_parent_event_id"),
        F.lit(None).cast("string").alias("parent_relation"),
        F.lit(None).cast("string").alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:barts:pathology:role").alias("_label_system"),
        F.lower(F.concat_ws(":",
            F.coalesce(F.expr("nullif(trim(discipline), '')"), F.lit("~")),
            F.coalesce(F.expr("nullif(trim(report_role), '')"), F.lit("~")),
        )).alias("_label_key"),
        F.lit("pathology_report").alias("source_feed"),
        F.date_format("ADC_UPDT", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("valid_from").alias("source_update_timestamp"),
        F.col("ADC_UPDT").alias("loaded_at"),
        F.lit("laboratory").alias("_source_system"),
        F.lit(SRC_PATHOLOGY_REPORT_VERSIONS).alias("_source_table"),
        F.col("report_version_id").alias("_source_row_id"),
        F.lit(None).cast("string").alias("_raw_content_sha256"),
        linkage_route.alias("_linkage_route"),
        thread_id.alias("_document_thread_id"),
        supersedes_id.alias("_supersedes_document_id"),
        F.col("version_ordinal").cast("long").alias("_version_ordinal"),
    )

def _pathology_report_document_canonical():
    return _pathology_report_document_base(
        _pathology_report_document_text_rows().where(F.col("is_current"))
    )

def _pathology_report_document_history_canonical():
    return _pathology_report_document_base(
        _pathology_report_document_text_rows().where(
            ~F.coalesce(F.col("is_current"), F.lit(False))
        )
    )

DOCUMENT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "document_type", "title",
    "author_practitioner_id", "author_role", "service_id", "status_code", "version_id",
    "document_thread_id", "supersedes_document_id", "version_ordinal", "is_latest_version",
    "document_text", "sections", "parser_version", "decompressor_version",
    "post_processor_version", "content_type", "encoding", "language",
    "text_sha256", "raw_content_sha256", "text_length", "content_class",
    "date_quality", "text_is_truncated", "linkage_route", "source_class",
    "assembly_status", "chunk_count",
    "corpus_frequency", "is_boilerplate", "source_link_event_id",
    "source_link_system", "author_id_system", "author_source_id",
    "verified_practitioner_id", "verified_datetime", "source_organization_id",
    "source_organization_display",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "sensitivity_labels",
    "document_class", "contributor_system", "succession_status",
    "source_parent_event_id", "parent_relation",
    "source_parent_display", "source_parent_title", "source_parent_tag",
    "source_tag", "source_record_status",
    "document_text_anonymised", "text_is_anonymised",
    "prsb_document_type", "prsb_subtype", "prsb_standard", "prsb_setting",
    "prsb_map_method", "prsb_map_score", "prsb_map_version",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

DOCUMENT_THREADED_COLUMNS = [
    {
        "document_type": "_document_type_json",
        "sections": "_sections_json",
        "sensitivity_labels": "_sensitivity_labels_json",
    }.get(name, name)
    for name in DOCUMENT_PUBLIC_COLUMNS
]

def _order_comment_document_canonical():
    # The >100-character floor is deliberate: the source contains about 98M
    # near-label rows. TEXT_AVAILABLE_IND=false is honest missingness from the
    # mill_long_text feed freeze (2025-09-23), so those rows never mint documents.
    c = read_source(SRC_ORDER_COMMENT).where(
        F.expr(ORDER_COMMENT_DOCUMENT_PREDICATE)
    ).alias("c")
    o = read_source(SRC_ORDERS).select(
        F.col("ORDER_ID").alias("_o_order_id"),
        F.col("PERSON_ID").alias("_o_person_id"),
        F.col("ENCNTR_ID").alias("_o_encntr_id"),
        F.col("CATALOG_DISPLAY").alias("_o_catalog_display"),
        F.col("ORDERED_AS_MNEMONIC").alias("_o_ordered_as"),
        F.col("HNA_ORDER_MNEMONIC").alias("_o_hna_mnemonic"),
        F.col("ORIG_ORDER_DT_TM_CLEAN").alias("_o_orig_dt"),
        F.col("CURRENT_START_DT_TM_CLEAN").alias("_o_start_dt"),
        F.col("SOURCE_ADC_UPDT").alias("_o_loaded_at"),
    ).alias("o")
    s = c.join(o, c.ORDER_ID == F.col("o._o_order_id"), "left")
    natural = F.concat_ws(
        ":", c.ORDER_ID.cast("string"), c.ACTION_SEQUENCE.cast("string"),
        c.COMMENT_TYPE_CD.cast("string"),
    )
    event_id = stable_id(
        "document:order_comment", c.ORDER_ID, c.ACTION_SEQUENCE, c.COMMENT_TYPE_CD
    )
    resolved_person = F.col("o._o_person_id")
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", resolved_person)], SRC_ORDER_COMMENT, natural
    )
    event_time = F.coalesce(
        c.COMMENT_DT_TM_CLEAN, c.COMMENT_UPDT_DT_TM_CLEAN,
        F.col("o._o_orig_dt"), F.col("o._o_start_dt"),
        c.COMMENT_DT_TM, c.COMMENT_UPDT_DT_TM,
    )
    loaded_at = F.greatest(
        c.SOURCE_COMMENT_ADC_UPDT, c.SOURCE_TEXT_ADC_UPDT,
        c.PIPELINE_UPDT_DT_TM, F.col("o._o_loaded_at"),
    )
    source_code = c.COMMENT_TYPE_CD.cast("string")
    source_display = F.coalesce(c.COMMENT_TYPE_DESC, F.lit("Order comment"))
    order_display = F.coalesce(
        F.col("o._o_catalog_display"), F.col("o._o_ordered_as"),
        F.col("o._o_hna_mnemonic"),
    )
    inactive = F.coalesce(c.TEXT_ACTIVE_IND.cast("long"), F.lit(1)) == 0
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        resolved_person.cast("string").alias("person_id"),
        F.when(resolved_person.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("o._o_encntr_id").isNotNull(),
               stable_id("encounter:mill", F.col("o._o_encntr_id"))).alias("encounter_id"),
        event_time.alias("event_datetime"), F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:cerner:order-comment-type").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:cerner:order-comment-type"), source_code,
                       source_display, True)
        ).alias("_document_type_json"),
        F.concat_ws(" — ", source_display, order_display).alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.lit(None).cast("string").alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.when(inactive, F.lit("inactive")).otherwise(F.lit("active")).alias("status_code"),
        F.concat_ws(
            ":", c.COMMENT_UPDT_CNT.cast("string"),
            F.date_format(c.TEXT_UPDT_DT_TM, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"),
        ).alias("version_id"),
        c.COMMENT_TEXT.alias("document_text"), F.lit("[]").alias("_sections_json"),
        F.lit("order-comment-native-v1").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"), F.lit("UTF-8").alias("encoding"),
        F.sha2(c.COMMENT_TEXT, 256).alias("text_sha256"),
        F.length(c.COMMENT_TEXT).cast("long").alias("text_length"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.when(inactive, c.TEXT_UPDT_DT_TM_CLEAN).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit("order_comment").alias("document_class"),
        F.lit("Millennium Orders").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        c.ORDER_ID.cast("string").alias("source_parent_event_id"),
        F.lit("order").alias("parent_relation"),
        order_display.alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:cerner:order-comment-type").alias("_label_system"),
        c.COMMENT_TYPE_CD.cast("string").alias("_label_key"),
        F.lit("order_comment").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.coalesce(c.TEXT_UPDT_DT_TM_CLEAN, c.COMMENT_UPDT_DT_TM_CLEAN)
         .alias("source_update_timestamp"),
        loaded_at.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(SRC_ORDER_COMMENT).alias("_source_table"), natural.alias("_source_row_id"),
        F.lit(None).cast("string").alias("_raw_content_sha256"),
        F.when(resolved_person.isNotNull(), F.lit("direct"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
    )

def _elective_access_comment_document_canonical():
    s = read_source(SRC_EAL_COMMENT).where(
        F.expr(ELECTIVE_ACCESS_COMMENT_PREDICATE)
    ).alias("s")
    natural = F.concat_ws(
        ":", s.SOURCE_SYSTEM_OID.cast("string"), s.WAITING_LIST_OID.cast("string")
    )
    event_id = stable_id(
        "document:elective_access_comment", s.SOURCE_SYSTEM_OID, s.WAITING_LIST_OID
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_EAL_COMMENT, natural
    )
    event_time = F.coalesce(
        s.MODIFIED_DT_TM, s.CREATED_DT_TM, s.WAITING_LIST_STATUS_CHANGE_DT_TM_CLEAN,
        s.DECIDED_TO_ADMIT_DT_TM_CLEAN, s.TCI_DT_TM_CLEAN,
    )
    author_source_id = F.coalesce(s.MODIFIED_BY_PRID, s.CREATED_BY_PRID).cast("string")
    source_code = F.lit("ELECTIVE_ACCESS_COMMENT")
    source_display = F.lit("Elective access scheduling comment")
    missing = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    closed = ~F.coalesce(s.ACTIVE_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        event_time.alias("event_datetime"), F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:luna:eal-comment").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:luna:eal-comment"), source_code, source_display, True)
        ).alias("_document_type_json"),
        F.concat_ws(" — ", source_display, s.WAITING_LIST_NAME).alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.when(author_source_id.isNotNull(), F.lit("list_updater")).alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.when(missing, F.lit("absent"))
         .when(closed, F.lit("closed")).otherwise(F.lit("active")).alias("status_code"),
        s.ROW_HASH.cast("string").alias("version_id"),
        s.COMMENTS.alias("document_text"), F.lit("[]").alias("_sections_json"),
        F.lit("luna-eal-native-v1").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"), F.lit("UTF-8").alias("encoding"),
        F.sha2(s.COMMENTS, 256).alias("text_sha256"),
        F.length(s.COMMENTS).cast("long").alias("text_length"),
        F.when(missing, F.lit("retracted"))
         .when(closed, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        s.CREATED_DT_TM.alias("record_status_effective_from"),
        F.when(missing | closed, event_time).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit("waiting_list_comment").alias("document_class"),
        F.lit("LUNA elective access").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        s.WAITING_LIST_OID.cast("string").alias("source_parent_event_id"),
        F.lit("elective_access_entry").alias("parent_relation"),
        s.WAITING_LIST_NAME.alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:luna:eal-comment").alias("_label_system"),
        F.lit("elective_access_comment").alias("_label_key"),
        F.lit("elective_access_comment").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.MODIFIED_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna").alias("_source_system"), F.lit(SRC_EAL_COMMENT).alias("_source_table"),
        natural.alias("_source_row_id"), F.lit(None).cast("string").alias("_raw_content_sha256"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("direct"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
        F.when(author_source_id.isNotNull(), F.lit("urn:luna:prid")).alias("_author_id_system"),
        author_source_id.alias("_author_source_id"),
    )

def _document_canonical():
    union = (
        _document_lane(_text_document_canonical()).select(*DOCUMENT_LANE_COLUMNS)
        .unionByName(_document_lane(_mill_blob_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_pacs_report_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_endobase_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_order_comment_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_elective_access_comment_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_neonatal_narrative_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
        .unionByName(_document_lane(_pathology_report_document_canonical()).select(*DOCUMENT_LANE_COLUMNS))
    )
    return _document_prsb_enrich(union)

In [0]:
SRC_ORDER_COMMENT = "4_prod.bronze.map_order_comment"
SRC_ORDERS = "4_prod.bronze.map_orders"
SRC_DOC_TYPE_PRSB_MAP = "3_lookup.omop.doc_type_prsb_map"
SRC_PACS_REPORT = "4_prod.bronze.map_pacs_report"
SRC_DOCUMENT_PATIENT_IDENTIFIER = "4_prod.bronze.map_patient_identifier"
SRC_DOCUMENT_LINK_CONTEXT = "8_dev.bronze.document_link_context_s37"
SRC_EAL_COMMENT = "4_prod.bronze.map_elective_access_list"
SRC_PACS_TEXT_BRIDGE = "4_prod.bronze.map_pacs_report_text_bridge"

def _pathology_document_alias_map(alias_type, prefix):
    aliases = (
        read_source(SRC_DOCUMENT_PATIENT_IDENTIFIER)
        .where((F.col("ALIAS_TYPE") == alias_type)
               & F.coalesce(F.col("CURRENT_IND"), F.lit(False))
               & _present(F.col("ALIAS_VALUE")))
        .groupBy(F.trim(F.col("ALIAS_VALUE")).alias(f"_{prefix}_value"))
        .agg(
            F.countDistinct(F.col("PERSON_ID").cast("string")).alias(f"_{prefix}_count"),
            F.max(F.col("PERSON_ID").cast("string")).alias(f"_{prefix}_person"),
        )
    )
    return aliases

def _pathology_report_document_text_rows():
    r = read_source(SRC_PATHOLOGY_REPORT_VERSIONS)
    return r.where(r.report_text.isNotNull() & (F.trim(r.report_text) != ""))

DOCUMENT_PRIMITIVE_COLUMNS = [
    name for name in DOCUMENT_THREADED_COLUMNS if name != "is_latest_version"
]

# PRSB typing columns are joined on by _document_prsb_enrich, never emitted by lanes.
DOCUMENT_PRSB_COLUMNS = [
    "prsb_document_type", "prsb_subtype", "prsb_standard", "prsb_setting",
    "prsb_map_method", "prsb_map_score", "prsb_map_version",
]

DOCUMENT_HYGIENE_COLUMNS = [
    "raw_content_sha256", "content_class", "date_quality", "text_is_truncated",
    "language", "linkage_route", "source_class", "assembly_status", "chunk_count",
]

DOCUMENT_EVIDENCE_COLUMNS = [
    "corpus_frequency", "is_boilerplate", "source_link_event_id",
    "source_link_system", "author_id_system", "author_source_id",
    "verified_practitioner_id", "verified_datetime", "source_organization_id",
    "source_organization_display",
]

DOCUMENT_THREAD_COLUMNS = [
    "document_thread_id", "supersedes_document_id", "version_ordinal",
]

DOCUMENT_EVIDENCE_PRIVATE_TYPES = {
    "_linkage_route": "string",
    "_assembly_status": "string",
    "_chunk_count": "long",
    "_source_link_event_id": "string",
    "_source_link_system": "string",
    "_author_id_system": "string",
    "_author_source_id": "string",
    "_verified_practitioner_id": "string",
    "_verified_datetime": "timestamp",
    "_source_organization_id": "string",
    "_source_organization_native_id": "string",
    "_source_organization_display": "string",
}

DOCUMENT_THREAD_PRIVATE_TYPES = {
    "_document_thread_id": "string",
    "_supersedes_document_id": "string",
    "_version_ordinal": "long",
}

DOCUMENT_LANE_COLUMNS = [
    c for c in DOCUMENT_PRIMITIVE_COLUMNS
    if c not in DOCUMENT_PRSB_COLUMNS
    and c not in DOCUMENT_HYGIENE_COLUMNS
    and c not in DOCUMENT_EVIDENCE_COLUMNS
    and c not in DOCUMENT_THREAD_COLUMNS
] + [
    "_source_system", "_source_table", "_source_row_id", "_label_system", "_label_key",
    "_raw_content_sha256",
] + list(DOCUMENT_EVIDENCE_PRIVATE_TYPES) + list(DOCUMENT_THREAD_PRIVATE_TYPES)

DOCUMENT_VERSION_COLUMNS = [
    c for c in DOCUMENT_PRIMITIVE_COLUMNS if c not in DOCUMENT_THREAD_COLUMNS
] + list(DOCUMENT_THREAD_PRIVATE_TYPES)

_DOCUMENT_CONTROL_CHARS_RE = r"[\x00\x01-\x08\x0B\x0C\x0E-\x1F]"
_DOCUMENT_PUNCT_ONLY_RE = r"^[\p{P}\p{S}\s]+$"
_DOCUMENT_SHORT_CODE_RE = r"^[A-Za-z0-9][A-Za-z0-9 _/-]{0,11}$"

_DOCUMENT_PLACEHOLDERS = [
    "empty", "none", "nil", "null", "n/a", "na", "not available", "no result",
    "no data", "unknown", "not recorded", "deleted",
]

_DOCUMENT_TRUNCATION_CAPS = [1_000_000, 65_535, 32_767, 32_000]

ORDER_COMMENT_DOCUMENT_PREDICATE = (
    "TEXT_AVAILABLE_IND = true "
    "AND length(nullif(trim(COMMENT_TEXT), '')) > 100"
)

ELECTIVE_ACCESS_COMMENT_PREDICATE = (
    "length(nullif(trim(COMMENTS), '')) > 100"
)

def _document_lane(df):
    """Give every lane the same typed private enrichment surface before union."""
    out = df
    private_types = {**DOCUMENT_EVIDENCE_PRIVATE_TYPES, **DOCUMENT_THREAD_PRIVATE_TYPES}
    for name, data_type in private_types.items():
        if name not in out.columns:
            out = out.withColumn(name, F.lit(None).cast(data_type))
    return out

def _hygiene_text(text, strip_controls):
    return F.when(
        strip_controls,
        F.regexp_replace(text, _DOCUMENT_CONTROL_CHARS_RE, ""),
    ).otherwise(text)

def _document_hygiene(df):
    controlled_feed = F.col("source_feed") == F.lit("mill_blob_text")
    out = (
        df.withColumn(
            "document_text",
            _hygiene_text(F.col("document_text"), controlled_feed),
        )
        .withColumn(
            "document_text_anonymised",
            _hygiene_text(F.col("document_text_anonymised"), controlled_feed),
        )
    )
    text = F.col("document_text")
    trimmed = F.trim(F.coalesce(text, F.lit("")))
    lowered = F.lower(trimmed)
    text_length = F.length(text).cast("long")
    content_class = (
        F.when(text.isNull() | (trimmed == ""), F.lit("empty"))
        .when(trimmed.rlike(_DOCUMENT_PUNCT_ONLY_RE), F.lit("punctuation_only"))
        .when(lowered.isin(*_DOCUMENT_PLACEHOLDERS), F.lit("placeholder"))
        .when((F.length(trimmed) <= 12) & trimmed.rlike(_DOCUMENT_SHORT_CODE_RE),
              F.lit("short_code"))
        .otherwise(F.lit("narrative"))
    )
    date_reference = F.coalesce(F.col("loaded_at"), F.col("source_update_timestamp"))
    date_quality = (
        F.when(F.col("event_datetime").isNull(), F.lit("null"))
        .when(F.col("event_datetime") < F.lit("1975-01-01").cast("timestamp"),
              F.lit("epoch_sentinel"))
        .when(
            date_reference.isNotNull()
            & (F.col("event_datetime") > date_reference + F.expr("INTERVAL 1 DAY")),
            F.lit("future"),
        )
        .otherwise(F.lit("ok"))
    )
    post_processor_version = F.when(
        controlled_feed,
        F.when(F.col("post_processor_version").isNull(), F.lit("silver-control-strip-v1"))
        .when(F.col("post_processor_version").contains("silver-control-strip-v1"),
              F.col("post_processor_version"))
        .otherwise(F.concat(F.col("post_processor_version"),
                            F.lit("+silver-control-strip-v1"))),
    ).otherwise(F.col("post_processor_version"))
    source_class = F.lit("clinical")
    assembly_status = F.coalesce(F.col("_assembly_status"), F.lit("single"))
    chunk_count = F.coalesce(F.col("_chunk_count"), F.lit(1).cast("long"))
    tombstone = lowered == F.lit("deleted")
    return (
        out.withColumn("post_processor_version", post_processor_version)
        .withColumn("encoding", F.coalesce(F.col("encoding"), F.lit("UTF-8")))
        .withColumn("language", F.lit("en"))
        .withColumn("text_sha256", F.sha2(F.coalesce(text, F.lit("")), 256))
        .withColumn("raw_content_sha256", F.col("_raw_content_sha256"))
        .withColumn("text_length", text_length)
        .withColumn("content_class", content_class)
        .withColumn("date_quality", date_quality)
        .withColumn("text_is_truncated", F.coalesce(text_length.isin(*_DOCUMENT_TRUNCATION_CAPS),
                                                    F.lit(False)))
        .withColumn("linkage_route", F.coalesce(F.col("_linkage_route"), F.lit("none")))
        .withColumn("source_class", source_class)
        .withColumn("assembly_status", assembly_status)
        .withColumn("chunk_count", chunk_count)
        .withColumn("record_status", F.when(tombstone, F.lit("retracted"))
                    .otherwise(F.col("record_status")))
        .withColumn(
            "record_status_effective_to",
            F.when(tombstone, F.coalesce(F.col("record_status_effective_to"),
                                         F.col("source_update_timestamp"), F.col("loaded_at")))
            .otherwise(F.col("record_status_effective_to")),
        )
    )

def _document_prsb_enrich(df):
    """LEFT-join the governed PRSB doc-type lookup and finalise document typing.
    Runs at the canonical layer: _document_type_json stays a JSON string across the
    join (VARIANT never crosses a join boundary) and is recomputed here from the
    scalar source columns so every lane gets the mapped coding appended uniformly."""
    df = _document_hygiene(df)
    m = read_source(SRC_DOC_TYPE_PRSB_MAP).select(
        F.col("label_system").alias("_m_label_system"),
        F.col("label_key").alias("_m_label_key"),
        F.col("canonical_name").alias("prsb_document_type"),
        F.col("subtype").alias("prsb_subtype"),
        F.col("prsb_standard"),
        F.col("prsb_setting"),
        F.col("method").alias("prsb_map_method"),
        F.col("score").cast("double").alias("prsb_map_score"),
        F.col("map_version").alias("prsb_map_version"),
    )
    j = df.join(
        m,
        (df["_label_system"] == m["_m_label_system"])
        & (df["_label_key"] == m["_m_label_key"]),
        "left",
    )
    prsb_display = F.initcap(F.regexp_replace(F.col("prsb_document_type"), "_", " "))
    mapped = F.struct(
        F.lit("urn:barts:prsb-doc-type").cast("string").alias("coding_system"),
        F.col("prsb_document_type").cast("string").alias("coding_code"),
        prsb_display.cast("string").alias("coding_display"),
        F.lit(False).alias("is_source"),
        F.when(F.col("prsb_document_type").isNotNull(),
               F.lit("doc_type_prsb_map")).cast("string").alias("map_source"),
        F.col("prsb_map_version").cast("string").alias("map_version"),
    )
    source = coding_obj(F.col("source_coding_system"), F.col("source_code"),
                        F.col("source_display"), True)
    typed = j.withColumn("_document_type_json", codeable_concept_json(source, mapped))
    return _document_evidence_enrich(typed)

def _document_evidence_enrich(df):
    """Attach organization evidence; corpus frequency is calculated after all lanes union."""
    organization = spark.read.table(_n("journey_reference.organization")).select(
        F.col("source_organization_id").alias("_mill_organization_native_id"),
        F.col("organization_id").alias("_mill_organization_id"),
        F.col("name").alias("_mill_organization_display"),
    )
    enriched = df.join(
        organization,
        F.col("_source_organization_native_id")
        == F.col("_mill_organization_native_id"),
        "left",
    )
    return (
        enriched
        .withColumn("corpus_frequency", F.lit(None).cast("long"))
        .withColumn("is_boilerplate", F.lit(False))
        .withColumn("source_link_event_id", F.col("_source_link_event_id"))
        .withColumn("source_link_system", F.col("_source_link_system"))
        .withColumn("author_id_system", F.col("_author_id_system"))
        .withColumn("author_source_id", F.col("_author_source_id"))
        .withColumn("verified_practitioner_id", F.col("_verified_practitioner_id"))
        .withColumn("verified_datetime", F.col("_verified_datetime"))
        .withColumn(
            "source_organization_id",
            F.coalesce(F.col("_source_organization_id"), F.col("_mill_organization_id")),
        )
        .withColumn(
            "source_organization_display",
            F.coalesce(
                F.col("_source_organization_display"),
                F.col("_mill_organization_display"),
            ),
        )
    )

def _text_document_canonical():
    s = _text_finding_canonical().where(F.col("_route") == "document_candidate").alias("s")
    return s.select(
        "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
        "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
        "source_coding_system", "source_code", "source_display",
        codeable_concept_json(
            coding_obj(F.col("source_coding_system"), F.col("source_code"),
                       F.col("source_display"), True)
        ).alias("_document_type_json"),
        F.col("source_display").alias("title"),
        F.col("performer_practitioner_id").alias("author_practitioner_id"),
        F.lit("performer").alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.col("result_status_code").alias("status_code"),
        F.col("_source_row_id").alias("version_id"),
        F.col("value_text").alias("document_text"),
        F.lit("[]").alias("_sections_json"),
        F.lit(None).cast("string").alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.lit("text/plain").alias("content_type"),
        F.lit(None).cast("string").alias("encoding"),
        F.sha2(F.coalesce(F.col("value_text"), F.col("patient_event_id")), 256)
         .alias("text_sha256"),
        F.length(F.col("value_text")).cast("long").alias("text_length"),
        "record_status", "record_status_effective_from", "record_status_effective_to",
        "confidentiality_code", "vip_ind", "withheld_identity_ind",
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit(None).cast("string").alias("document_class"),
        F.lit(None).cast("string").alias("contributor_system"),
        F.lit(None).cast("string").alias("succession_status"),
        F.col("parent_event_id").alias("source_parent_event_id"),
        F.lit(None).cast("string").alias("parent_relation"),
        F.lit(None).cast("string").alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:cerner:code_value:pair").alias("_label_system"),
        F.concat_ws(":",
            F.coalesce(F.col("_parent_event_cd"), F.col("source_code")),
            F.col("source_code")).alias("_label_key"),
        F.lit("text_event").alias("source_feed"), "load_batch_id", "source_update_timestamp", "loaded_at", "_source_system", "_source_table",
        "_source_row_id", F.lit(None).cast("string").alias("_raw_content_sha256"),
        F.lit("direct").alias("_linkage_route"),
    )

def _mill_blob_document_canonical():
    raw = read_source(SRC_MILL_BLOB_TEXT).where(F.col("STATUS") == F.lit("Decoded"))
    source_hash = F.sha2(F.to_json(F.struct(*[raw[c] for c in raw.columns])), 256)
    # Source occasionally contains more than one physical extraction row for the same governed
    # document-version key. Collapse those rows with an incrementalizable aggregate rather than a
    # row_number window: latest source/update timestamp wins, then the full-row hash breaks ties.
    # The selected value is the complete source struct, so no column is silently re-aggregated.
    order_key = F.concat_ws(
        "|",
        F.coalesce(F.date_format("UPDT_DT_TM", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"), F.lit("~")),
        F.coalesce(F.date_format("ADC_UPDT", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"), F.lit("~")),
        source_hash,
    )
    source_json = F.to_json(
        F.struct(*[raw[c] for c in raw.columns]), {"ignoreNullFields": "false"}
    )
    # MAX over an orderable struct is deterministic and incrementally composable. The winning
    # source row is round-tripped through its exact Spark schema after selection.
    selected_row = F.max(
        F.struct(order_key.alias("order_key"), source_json.alias("source_json"))
    )
    b = (
        raw.groupBy("EVENT_ID", "UPDT_CNT", "VALID_FROM_DT_TM")
        .agg(selected_row.alias("_selected"))
        .select(F.from_json(F.col("_selected.source_json"), raw.schema).alias("_source"))
        .select("_source.*")
        .alias("b")
    )
    e = read_source(SRC_ENCOUNTER).select(
        F.col("ENCNTR_ID").alias("_context_encounter_id"),
        F.col("PERSON_ID").alias("_context_person_id"),
    ).alias("e")
    s = b.join(e, b.ENCNTR_ID == e._context_encounter_id, "left")
    natural = F.concat_ws(
        ":", b.EVENT_ID.cast("string"), b.UPDT_CNT.cast("string"),
        F.date_format(b.VALID_FROM_DT_TM, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"),
    )
    event_id = stable_id(
        "document:mill_blob", b.EVENT_ID, b.UPDT_CNT, b.VALID_FROM_DT_TM
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", e._context_person_id)], SRC_MILL_BLOB_TEXT, natural
    )
    text_value = b.BLOB_TEXT
    anon_fallback = F.lit(False)
    ended = _closed_timestamp(b.VALID_UNTIL_DT_TM)
    # code set 8: 29/30/31 all decode to 'In Error' — surfaced as retracted, never dropped
    # (the legacy RDE extract silently filters these rows out).
    in_error = b.EVENT_RESULT_STATUS_CD.isin(29, 30, 31)
    record_status = (
        F.when(in_error, F.lit("retracted"))
        .when(ended, F.lit("superseded"))
        .otherwise(F.lit("active"))
    )
    version_time = F.coalesce(b.VALID_FROM_DT_TM, b.UPDT_DT_TM, b.ADC_UPDT)
    # Clinical time preferred; version/lifecycle time retained on record_status_effective_from.
    event_time = F.coalesce(b.CLINSIG_DT_TM, version_time)
    # Label columns are bronze-enriched (Blob 5:Labels); fail OPEN when enrichment
    # hasn't landed for a row — the document still publishes with content-type typing.
    has_event = b.EVENT_CD.isNotNull()
    label_display = F.coalesce(b.EVENT_CD_DISPLAY, b.EVENT_CD_DESC)
    performed_id = F.when(_usable_code(b.PERFORMED_PRSNL_ID),
                          b.PERFORMED_PRSNL_ID.cast("string"))
    updater_id = F.when(_usable_code(b.UPDT_ID), b.UPDT_ID.cast("string"))
    author_source_id = F.coalesce(performed_id, updater_id)
    verifier_source_id = F.when(_usable_code(b.VERIFIED_PRSNL_ID),
                                b.VERIFIED_PRSNL_ID.cast("string"))
    organization_native_id = F.when(_usable_code(b.ORGANIZATION_ID),
                                    b.ORGANIZATION_ID.cast("string"))
    blob_thread_key = F.when(
        _present(b.SERIES_REF_NBR), F.trim(b.SERIES_REF_NBR.cast("string"))
    ).otherwise(b.EVENT_ID.cast("string"))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        e._context_person_id.cast("string").alias("person_id"),
        F.when(e._context_person_id.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(b.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", b.ENCNTR_ID))
         .alias("encounter_id"),
        event_time.alias("event_datetime"),
        F.when(ended, b.VALID_UNTIL_DT_TM).alias("event_end_datetime"),
        F.when(has_event, F.lit("urn:cerner:code_value"))
         .otherwise(F.lit("urn:cerner:blob-content-type")).alias("source_coding_system"),
        F.coalesce(b.EVENT_CD.cast("string"), b.CONTENT_TYPE).alias("source_code"),
        F.coalesce(label_display, b.CONTENT_TYPE).alias("source_display"),
        codeable_concept_json(
            coding_obj(
                F.when(has_event, F.lit("urn:cerner:code_value"))
                 .otherwise(F.lit("urn:cerner:blob-content-type")),
                F.coalesce(b.EVENT_CD.cast("string"), b.CONTENT_TYPE),
                F.coalesce(label_display, b.CONTENT_TYPE), True)
        ).alias("_document_type_json"),
        F.coalesce(
            b.EVENT_TITLE_TEXT,
            F.concat_ws(" ", F.lit("Millennium document"), b.EVENT_ID.cast("string")),
        ).alias("title"),
        F.when(author_source_id.isNotNull(),
               stable_id("practitioner:mill", author_source_id))
         .alias("author_practitioner_id"),
        F.when(performed_id.isNotNull(), F.lit("performer"))
         .when(updater_id.isNotNull(), F.lit("updater")).alias("author_role"),
        F.lit(None).cast("string").alias("service_id"), b.STATUS.alias("status_code"),
        natural.alias("version_id"), text_value.alias("document_text"),
        F.lit("[]").alias("_sections_json"),
        b.parser_version.cast("string").alias("parser_version"),
        b.decompressor_version.cast("string").alias("decompressor_version"),
        b.post_processor_version.cast("string").alias("post_processor_version"),
        b.CONTENT_TYPE.alias("content_type"), b.ENCODING.alias("encoding"),
        F.coalesce(b.raw_sha256, F.sha2(text_value, 256), F.sha2(natural, 256))
         .alias("text_sha256"),
        F.coalesce(b.TEXT_LENGTH, F.length(text_value).cast("long")).alias("text_length"),
        record_status.alias("record_status"),
        version_time.alias("record_status_effective_from"),
        F.when(ended, b.VALID_UNTIL_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        b.EVENT_CLASS_DISPLAY.alias("document_class"),
        b.CONTRIBUTOR_SYSTEM_DISPLAY.alias("contributor_system"),
        b.SUCCESSION_TYPE_DISPLAY.alias("succession_status"),
        b.PARENT_EVENT_ID.cast("string").alias("source_parent_event_id"),
        b.EVENT_RELTN_DISPLAY.alias("parent_relation"),
        b.PARENT_EVENT_CD_DESC.alias("source_parent_display"),
        b.PARENT_EVENT_TITLE_TEXT.alias("source_parent_title"),
        b.PARENT_EVENT_TAG.alias("source_parent_tag"),
        b.EVENT_TAG.alias("source_tag"),
        b.EVENT_RECORD_STATUS_DISPLAY.alias("source_record_status"),
        F.lit(None).cast("string").alias("document_text_anonymised"),
        anon_fallback.alias("text_is_anonymised"),
        F.when(has_event, F.lit("urn:cerner:code_value:pair")).alias("_label_system"),
        F.when(has_event,
               F.concat_ws(":", F.coalesce(b.PARENT_EVENT_CD, b.EVENT_CD).cast("string"),
                           b.EVENT_CD.cast("string"))).alias("_label_key"),
        F.lit("mill_blob_text").alias("source_feed"),
        F.date_format(b.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        b.UPDT_DT_TM.alias("source_update_timestamp"),
        b.ADC_UPDT.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(SRC_MILL_BLOB_TEXT).alias("_source_table"), natural.alias("_source_row_id"),
        b.raw_sha256.alias("_raw_content_sha256"),
        F.when(author_source_id.isNotNull(), F.lit("urn:cerner:prsnl_id"))
         .alias("_author_id_system"),
        author_source_id.alias("_author_source_id"),
        F.when(verifier_source_id.isNotNull(),
               stable_id("practitioner:mill", verifier_source_id))
         .alias("_verified_practitioner_id"),
        F.when(verifier_source_id.isNotNull(), b.VERIFIED_DT_TM)
         .alias("_verified_datetime"),
        organization_native_id.alias("_source_organization_native_id"),
        F.when(e._context_person_id.isNotNull(), F.lit("encounter_join"))
         .when(b.ENCNTR_ID.isNotNull(), F.lit("direct"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
        stable_id("document_thread:mill_blob", blob_thread_key)
         .alias("_document_thread_id"),
        F.lit(None).cast("string").alias("_supersedes_document_id"),
        b.UPDT_CNT.cast("long").alias("_version_ordinal"),
    )

def _pacs_report_document_canonical():
    r = read_source(SRC_PACS_REPORT).alias("r")
    b = read_source(SRC_PACS_TEXT_BRIDGE).select(
        F.col("REPORT_ID").alias("_bridge_report_id"),
        F.col("EVENT_ID").alias("_bridge_event_id"),
        F.col("BRIDGED_TEXT").alias("_bridged_text"),
        F.col("BRIDGED_TEXT_FORMAT").alias("_bridged_format"),
        F.col("BRIDGED_TEXT_PARSER_VERSION").alias("_bridged_parser"),
    ).alias("b")
    r = r.join(b, r.PACS_REPORT_ID == F.col("_bridge_report_id"), "left")
    examination = (
        read_source(SRC_PACS_EXAMINATION)
        .groupBy("PACS_EXAMINATION_ID")
        .agg(F.max(F.when(_present(F.col("INSTITUTION")), F.trim("INSTITUTION")))
             .alias("_exam_institution"))
        .select(
            F.col("PACS_EXAMINATION_ID").alias("_exam_id"),
            "_exam_institution",
        )
    )
    r = r.join(examination, r.PACS_EXAMINATION_ID == F.col("_exam_id"), "left")
    event_context = (
        read_source(SRC_DOCUMENT_LINK_CONTEXT)
        .where(F.col("source_kind") == "clinical_event")
        .select(
            F.col("source_id").alias("_event_context_id"),
            F.col("person_id").alias("_event_context_person_id"),
            F.col("encntr_id").alias("_event_context_encntr_id"),
        )
    )
    r = r.join(
        event_context,
        F.col("_bridge_event_id") == F.col("_event_context_id"),
        "left",
    )
    context_compatible = (
        r.PERSON_ID.isNull() | F.col("_event_context_person_id").isNull()
        | (r.PERSON_ID.cast("string") == F.col("_event_context_person_id").cast("string"))
    )
    resolved_person = F.coalesce(r.PERSON_ID, F.col("_event_context_person_id"))
    resolved_encounter = F.when(context_compatible, F.col("_event_context_encntr_id"))
    event_id = stable_id("document:pacs_report", r.PACS_REPORT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", resolved_person),
         ("urn:sectra:pacs-patient-id", r.PACS_PATIENT_ID)],
        SRC_PACS_REPORT,
        r.PACS_REPORT_ID,
    )
    retracted = F.coalesce(~r.SOURCE_PRESENT_IND, F.lit(False))
    event_time = F.coalesce(r.REPORT_DT_TM, r.REPORT_MODIFIED_UTC, r.ADC_UPDT)
    source_code = F.coalesce(r.EXAM_CODE, r.REPORT_TEXT_FORMAT, F.lit("IMAGING_REPORT"))
    source_display = F.concat_ws(" ", r.EXAM_MODALITY, r.EXAM_CODE, F.lit("imaging report"))
    native_text = F.when(
        r.REPORT_TEXT.isNotNull() & (F.trim(r.REPORT_TEXT) != ""), r.REPORT_TEXT
    )
    source_document_text = F.coalesce(native_text, F.col("_bridged_text"))
    source_document_text_anonymised = F.lit(None).cast("string")
    text_format = F.when(native_text.isNotNull(), r.REPORT_TEXT_FORMAT).otherwise(
        F.col("_bridged_format")
    )
    is_rtf = F.upper(F.trim(F.coalesce(text_format, F.lit("")))).isin(
        "RTF", "APPLICATION/RTF", "TEXT/RTF"
    )
    document_text = F.when(
        is_rtf & source_document_text.isNotNull(), _strip_rtf_text(source_document_text)
    ).otherwise(source_document_text)
    document_text_anonymised = source_document_text_anonymised
    parser_version = (
        F.when(is_rtf & source_document_text.isNotNull(), F.lit(_RTF_PARSER_VERSION))
        .when(native_text.isNotNull(), F.lit("pacs-native-v1"))
        .when(
            F.col("_bridged_text").isNotNull(),
            F.coalesce(F.col("_bridged_parser"), F.lit("pacs-bridge-v1")),
        )
        .otherwise(F.lit("pacs-native-v1"))
    )
    report_doctor_id = F.when(_usable_code(r.REPORT_DOCTOR_ID),
                              r.REPORT_DOCTOR_ID.cast("string"))
    linked_event_id = F.when(_present(F.col("_bridge_event_id")),
                             F.trim(F.col("_bridge_event_id")))
    institution = F.when(_present(F.col("_exam_institution")),
                         F.trim(F.col("_exam_institution")))
    return r.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        resolved_person.cast("string").alias("person_id"),
        F.when(resolved_person.isNotNull(), F.lit("resolved"))
        .when(r.PACS_PATIENT_ID.isNotNull(), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(resolved_encounter.isNotNull(),
               stable_id("encounter:mill", resolved_encounter)).alias("encounter_id"),
        event_time.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:sectra:imaging-report").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:sectra:imaging-report"), source_code,
                       source_display, True)
        ).alias("_document_type_json"),
        F.concat_ws(" ", F.lit("Imaging report"), r.EXAM_CODE,
                    r.PACS_REPORT_ID.cast("string")).alias("title"),
        F.lit(None).cast("string").alias("author_practitioner_id"),
        F.when(report_doctor_id.isNotNull(), F.lit("reporting_doctor"))
         .alias("author_role"),
        F.lit(None).cast("string").alias("service_id"),
        F.when(document_text.isNull() | (F.trim(document_text) == ""),
               F.lit("no_text_at_source"))
        .otherwise(r.REPORT_STATUS_CD.cast("string")).alias("status_code"),
        F.concat_ws(":", r.PACS_REPORT_ID.cast("string"),
                    F.date_format(r.REPORT_MODIFIED_UTC, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"))
        .alias("version_id"),
        document_text.alias("document_text"), F.lit("[]").alias("_sections_json"),
        parser_version.alias("parser_version"),
        F.lit(None).cast("string").alias("decompressor_version"),
        F.lit(None).cast("string").alias("post_processor_version"),
        F.when(is_rtf, F.lit("application/rtf"))
        .otherwise(F.lit("text/plain")).alias("content_type"),
        F.lit("UTF-8").alias("encoding"),
        F.sha2(F.coalesce(document_text, r.PACS_REPORT_ID.cast("string")), 256)
        .alias("text_sha256"),
        F.length(document_text).cast("long").alias("text_length"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
        .alias("record_status"),
        r.SRC_ADC_UPDT.alias("record_status_effective_from"),
        F.when(retracted, r.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("null").alias("_sensitivity_labels_json"),
        F.lit(None).cast("string").alias("document_class"),
        F.lit(None).cast("string").alias("contributor_system"),
        F.when(r.FINAL_SIGNATURE_DT_TM.isNotNull(), F.lit("final"))
         .when(r.PRELIM_SIGNATURE_DT_TM.isNotNull(), F.lit("preliminary"))
         .alias("succession_status"),
        F.lit(None).cast("string").alias("source_parent_event_id"),
        F.lit(None).cast("string").alias("parent_relation"),
        F.lit(None).cast("string").alias("source_parent_display"),
        F.lit(None).cast("string").alias("source_parent_title"),
        F.lit(None).cast("string").alias("source_parent_tag"),
        F.lit(None).cast("string").alias("source_tag"),
        F.lit(None).cast("string").alias("source_record_status"),
        document_text_anonymised.alias("document_text_anonymised"),
        F.lit(False).alias("text_is_anonymised"),
        F.lit("urn:sectra:imaging-modality").alias("_label_system"),
        F.coalesce(F.upper(F.trim(r.EXAM_MODALITY)), F.lit("IMAGING")).alias("_label_key"),
        F.lit("pacs_report").alias("source_feed"),
        F.date_format(r.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        r.REPORT_MODIFIED_UTC.alias("source_update_timestamp"),
        r.ADC_UPDT.alias("loaded_at"), F.lit("sectra-pacs").alias("_source_system"),
        F.lit(SRC_PACS_REPORT).alias("_source_table"),
        r.PACS_REPORT_ID.cast("string").alias("_source_row_id"),
        F.lit(None).cast("string").alias("_raw_content_sha256"),
        linked_event_id.alias("_source_link_event_id"),
        F.when(linked_event_id.isNotNull(), F.lit("urn:cerner:event_id"))
         .alias("_source_link_system"),
        F.when(report_doctor_id.isNotNull(), F.lit("urn:sectra:doctor_id"))
         .alias("_author_id_system"),
        report_doctor_id.alias("_author_source_id"),
        F.when(institution.isNotNull(), stable_id("organization:pacs", institution))
         .alias("_source_organization_id"),
        institution.alias("_source_organization_display"),
        F.when(resolved_encounter.isNotNull()
               | (r.PERSON_ID.isNull() & F.col("_event_context_person_id").isNotNull()),
               F.lit("bridge_event"))
         .when(r.PERSON_ID.isNotNull(), F.lit("direct"))
         .otherwise(F.lit("none")).alias("_linkage_route"),
        stable_id("document_thread:pacs_report", r.PACS_REPORT_ID)
         .alias("_document_thread_id"),
        F.lit(None).cast("string").alias("_supersedes_document_id"),
        F.lit(1).cast("long").alias("_version_ordinal"),
    )

def _document_thread_metadata(df):
    """Attach source-derived thread identity before the batch-built head join."""
    return (
        df.withColumn(
            "document_thread_id",
            F.coalesce(F.col("_document_thread_id"), F.col("patient_event_id")),
        )
        .withColumn("supersedes_document_id", F.col("_supersedes_document_id"))
        .withColumn(
            "version_ordinal",
            F.coalesce(F.col("_version_ordinal"), F.lit(1).cast("long")),
        )
    )

@materialized_view(
    name=_n("journey_text._document_primitive"),
    private=True,
    comment="Internal document union with in-graph corpus frequency and thread metadata.",
)
def _document_primitive():
    history = _document_prsb_enrich(
        _document_lane(_pathology_report_document_history_canonical())
        .select(*DOCUMENT_LANE_COLUMNS)
    )
    versions = (
        _document_canonical().select(*DOCUMENT_VERSION_COLUMNS)
        .unionByName(history.select(*DOCUMENT_VERSION_COLUMNS))
    )
    versions = _document_thread_metadata(versions)

    # Formerly built by a separate post-refresh notebook. Keeping it here removes
    # the circular operational dependency while preserving the sparse contract:
    # counts below 100 remain NULL and 1000+ is marked as boilerplate.
    frequencies = (
        versions.where(F.col("text_sha256").isNotNull())
        .groupBy("text_sha256")
        .agg(F.count(F.lit(1)).cast("long").alias("_corpus_frequency"))
        .where(F.col("_corpus_frequency") >= 100)
    )
    return (
        versions.drop("corpus_frequency", "is_boilerplate")
        .join(frequencies, "text_sha256", "left")
        .withColumn("corpus_frequency", F.col("_corpus_frequency"))
        .withColumn(
            "is_boilerplate",
            F.coalesce(F.col("_corpus_frequency"), F.lit(0)) >= 1000,
        )
        .drop("_corpus_frequency")
        .select(*DOCUMENT_PRIMITIVE_COLUMNS)
    )

In [0]:
@materialized_view(
    name=_n("journey_text._document_threaded"),
    private=True,
    comment="Internal document versions with the latest version selected deterministically in-graph.",
)
def _document_threaded():
    versions = spark.read.table(_n("journey_text._document_primitive"))
    effective_from = F.coalesce(
        F.col("record_status_effective_from"),
        F.col("event_datetime"),
        F.col("source_update_timestamp"),
        F.col("loaded_at"),
    )
    head_key = F.struct(
        F.when(
            F.col("document_text").isNotNull()
            & (F.trim(F.col("document_text")) != ""),
            F.lit(1),
        ).otherwise(F.lit(0)).alias("has_data"),
        F.coalesce(effective_from.cast("double"), F.lit(-1.0e308))
        .alias("effective_from_epoch"),
        F.coalesce(F.col("version_ordinal"), F.lit(-1).cast("long"))
        .alias("version_ordinal"),
        F.coalesce(F.col("version_id"), F.lit("")).alias("version_id"),
    )
    ranked = versions.select(
        "patient_event_id",
        "document_thread_id",
        head_key.alias("head_key"),
    ).alias("r")
    heads = ranked.groupBy("document_thread_id").agg(
        F.max("head_key").alias("selected_head_key")
    ).alias("h")
    selected = ranked.join(
        heads,
        (F.col("r.document_thread_id") == F.col("h.document_thread_id"))
        & (F.col("r.head_key") == F.col("h.selected_head_key")),
        "inner",
    ).select(
        F.col("r.document_thread_id").alias("_head_thread_id"),
        F.col("r.patient_event_id").alias("_latest_patient_event_id"),
    )
    return (
        versions.join(
            selected,
            versions["document_thread_id"] == selected["_head_thread_id"],
            "left",
        )
        .withColumn(
            "is_latest_version",
            F.coalesce(
                F.col("patient_event_id") == F.col("_latest_patient_event_id"),
                F.lit(False),
            ),
        )
        .drop("_head_thread_id", "_latest_patient_event_id")
        .select(*DOCUMENT_THREADED_COLUMNS)
    )

In [0]:
@materialized_view(
    name=_n("journey_text._qc_document"),
    comment="Private JSON bridge and Gold cross-rule flags for document.",
    refresh_policy="incremental",
)
def _qc_document():
    return _cross_qc_primitive(
        spark.read.table(_n("journey_text._document_threaded")),
        "document",
        {
            "document_type": "_document_type_json",
            "sections": "_sections_json",
            "sensitivity_labels": "_sensitivity_labels_json",
        },
    )

In [0]:
DOCUMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide document-version identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic or per-document subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when recoverable.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime": "Document clinical or version timestamp.",
    "event_end_datetime": "Source version-validity end.",
    "source_coding_system": "Verbatim source document-type system.",
    "source_code": "Verbatim source document-type code.",
    "source_display": "Verbatim source document-type display.",
    "document_type": "Source document-type CodeableConcept.",
    "title": "Source document title or label.",
    "author_practitioner_id": "Source author or updater practitioner.",
    "author_role": "Source author role when supplied.",
    "service_id": "Service reference when supplied.",
    "status_code": "Verbatim document extraction or source status.",
    "version_id": "Source document-version token.",
    "document_thread_id": "Stable identifier for all versions of the same logical document; source series identifiers are preferred and single-version lanes use patient_event_id.",
    "supersedes_document_id": "Stable patient_event_id of the directly superseded document version when the source supplies that relationship.",
    "version_ordinal": "Source-supplied version ordinal when available; one for single-version lanes.",
    "is_latest_version": "Exactly one row per document_thread_id selected by non-empty text then effective-from and source version ordinal descending with version_id descending as deterministic tiebreak.",
    "document_text": "Identifiable source document text under default-deny access.",
    "sections": "Deterministically ordered parsed document sections when supplied.",
    "parser_version": "Parser version that produced document_text or sections.",
    "decompressor_version": "Decompressor version used by bronze.",
    "post_processor_version": "Post-processor version used by bronze.",
    "content_type": "Source MIME or content type.",
    "encoding": "Source text encoding",
    "language": "Document language defaulted to English (`en`) because no source language field is available; method=default.",
    "text_sha256": "SHA-256 of retained document_text only; the anonymous alternative never changes this hash or any stable identifier. NULL is represented by the empty-string digest.",
    "raw_content_sha256": "SHA-256 of the original binary payload when supplied by the blob source; distinct from text_sha256.",
    "text_length": "Retained document-text character count recomputed after parsing and post-processing.",
    "content_class": "Deterministic retained-text quality class; rows are flagged rather than dropped.",
    "date_quality": "Event-date quality relative to source provenance: null, pre-1975 epoch sentinel, future beyond source/load time plus one day, or ok.",
    "text_is_truncated": "True when retained text length equals a known source or parser cap (1000000, 65535, 32767, or 32000 characters).",
    "linkage_route": "Deterministic provenance route used to resolve document subject or encounter linkage.",
    "source_class": "Fail-closed source-content class; the public document product contains clinical text only.",
    "assembly_status": "Source-row assembly outcome; NULL for parked SCD working-copy rows.",
    "chunk_count": "Number of source chunks represented by this document row; NULL for parked SCD working-copy rows.",
    "corpus_frequency": "Number of current Journey document rows sharing text_sha256 when the governed release-built frequency is at least 100; NULL means below that storage floor.",
    "is_boilerplate": "True exactly when governed corpus_frequency is at least 1000; absent frequency is false.",
    "source_link_event_id": "Verbatim linked source event identifier when a governed cross-feed bridge supplies one.",
    "source_link_system": "Identifier system for source_link_event_id.",
    "author_id_system": "Identifier system for the verbatim source author identifier.",
    "author_source_id": "Verbatim source author identifier retained separately from any resolved practitioner reference.",
    "verified_practitioner_id": "Stable practitioner reference for the source verifier when resolvable.",
    "verified_datetime": "Source verification timestamp associated with verified_practitioner_id.",
    "source_organization_id": "Stable source-organization reference derived from governed source evidence.",
    "source_organization_display": "Source-organization display from the governed organization dimension or verbatim PACS institution.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source lifecycle start.",
    "record_status_effective_to": "Source lifecycle end.",
    "confidentiality_code": "Source confidentiality classification.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "sensitivity_labels": "Ordered source security labels when supplied.",
    "document_class": "Source event-class display such as Document or mdoc.",
    "contributor_system": "Source contributor-system display such as PowerChart or BLT_TIE_RAD.",
    "succession_status": "Blob succession status display such as Interim or Final.",
    "source_parent_event_id": "Parent clinical-event identifier for document threading when supplied.",
    "parent_relation": "Event relation display such as Root or Child.",
    "source_parent_display": "Parent event-code description; the legacy MainEventDesc document-type label.",
    "source_parent_title": "Parent event title text; the legacy MainTitleText.",
    "source_parent_tag": "Parent event tag text; the legacy MainTagText.",
    "source_tag": "Own event tag text; the legacy ChildTagText.",
    "source_record_status": "Clinical-event record-status display such as Active or Deleted; the legacy Status.",
    "document_text_anonymised": "Reserved NULL field; de-identification is applied at serve time.",
    "text_is_anonymised": "Always false in silver; de-identification is applied at serve time.",
    "prsb_document_type": "PRSB-aligned canonical document type from the governed doc_type_prsb_map lookup.",
    "prsb_subtype": "Canonical subtype qualifier when the mapping supplies one.",
    "prsb_standard": "Source PRSB or Royal-College standard label for the canonical type.",
    "prsb_setting": "Care-setting bucket of the canonical type.",
    "prsb_map_method": "Mapping method provenance.",
    "prsb_map_score": "Embedder cosine score for embedder-method rows.",
    "prsb_map_version": "doc_type_prsb_map version label.",
    "source_feed": "Registered owning source route; all document text is IG-sensitive clinical content.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_text.document"),
    comment="Identifiable clinical document versions with source thread and parser provenance.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=DOCUMENT_COLUMN_COMMENTS,
)
def document():
    return _cross_qc_public(
        spark.read.table(_n("journey_text._qc_document")),
        "document",
        {
            "document_type": "_document_type_json",
            "sections": "_sections_json",
            "sensitivity_labels": "_sensitivity_labels_json",
        },
        DOCUMENT_PUBLIC_COLUMNS,
    )

In [0]:
# ==== Journeys, maternity relationships and temporal membership links ====

SRC_MAT_BIRTH = "4_prod.bronze.map_mat_birth"

PERSON_RELATIONSHIP_PUBLIC_COLUMNS = [
    "person_relationship_id", "source_subject_key", "source_person_id",
    "target_subject_key", "target_person_id", "relationship_type_code",
    "inverse_relationship_type_code", "valid_from", "valid_to", "record_status",
    "construction_rule", "construction_version", "source_system", "source_table",
    "source_row_id", "load_batch_id", "source_update_timestamp",
    "loaded_at",
]

def _person_relationship_canonical():
    b = read_source(SRC_MAT_BIRTH)
    source_row_id = b.BirthRow_ID.cast("string")
    mother_key, _ = subject_key_with_system(
        [("urn:cerner:person_id", b.MotherPerson_ID)], SRC_MAT_BIRTH, source_row_id
    )
    baby_key, _ = subject_key_with_system(
        [
            ("urn:cerner:person_id", b.BabyPerson_ID),
            ("https://fhir.nhs.uk/Id/nhs-number", b.Baby_NHS),
            ("urn:barts:mrn", b.Baby_MRN),
        ],
        SRC_MAT_BIRTH,
        source_row_id,
    )
    deleted = F.coalesce(b.PregnancySource_DELETE_IND.cast("long"), F.lit(0)) != 0
    return b.select(
        stable_id("person_relationship:mother_to_child", b.BirthRow_ID)
        .alias("person_relationship_id"),
        mother_key.alias("source_subject_key"),
        b.MotherPerson_ID.cast("string").alias("source_person_id"),
        baby_key.alias("target_subject_key"),
        b.BabyPerson_ID.cast("string").alias("target_person_id"),
        F.lit("mother_to_child").alias("relationship_type_code"),
        F.lit("child_of_mother").alias("inverse_relationship_type_code"),
        b.BirthDateTime.alias("valid_from"), F.lit(None).cast("timestamp").alias("valid_to"),
        F.when(deleted, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.lit("mat-birth-mother-baby").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.lit("millennium-maternity").alias("source_system"),
        F.lit(SRC_MAT_BIRTH).alias("source_table"), source_row_id.alias("source_row_id"),
        F.date_format(b.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.greatest(b.BirthSourceRecordUpdatedDateTime, b.PregnancySourceRecordUpdatedDateTime,
                   b.NNUSourceLastUpdate).alias("source_update_timestamp"),
        b.ADC_UPDT.alias("loaded_at"),
    ).select(*PERSON_RELATIONSHIP_PUBLIC_COLUMNS)

PERSON_RELATIONSHIP_COLUMN_COMMENTS = {
    "person_relationship_id": "Deterministic relationship primary key.",
    "source_subject_key": "Always-populated subject key for the relationship source person.",
    "source_person_id": "Resolved source-side person identifier when available.",
    "target_subject_key": "Always-populated subject key for the relationship target person.",
    "target_person_id": "Resolved target-side person identifier when available.",
    "relationship_type_code": "Controlled source-to-target relationship type.",
    "inverse_relationship_type_code": "Controlled inverse relationship type.",
    "valid_from": "Relationship validity start from source evidence.",
    "valid_to": "Relationship validity end when source evidence supplies one.",
    "record_status": "Normalized source lifecycle status.",
    "construction_rule": "Governed relationship-construction rule.",
    "construction_version": "Governed relationship-construction version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured bronze source table.",
    "source_row_id": "Stable source birth-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_spine.person_relationship"),
    comment="Source-backed person relationships; maternity contributes mother-to-baby links.",
    refresh_policy="incremental",
    column_comments=PERSON_RELATIONSHIP_COLUMN_COMMENTS,
)
def person_relationship():
    return _person_relationship_canonical()

In [0]:
SRC_MAT_PREGNANCY = "4_prod.bronze.map_mat_pregnancy"

def _pregnancy_journey_canonical():
    base = _pregnancy_constructor_base()
    parent_id = stable_id("journey:pregnancy", F.col("_pregnancy_id"))
    parent = base.select(
        parent_id.alias("journey_id"),
        F.lit(None).cast("string").alias("parent_journey_id"),
        F.col("_subject_key").alias("subject_key"),
        F.col("_subject_id_system").alias("subject_id_system"),
        F.col("_person_id").alias("person_id"),
        F.lit("pregnancy").alias("journey_type_code"),
        F.lit("Pregnancy").alias("journey_type_display"),
        F.col("_period_start").alias("period_start"),
        F.col("_period_end").alias("period_end"),
        F.col("_status_code").alias("status_code"),
        F.lit("http://snomed.info/sct").alias("defining_coding_system"),
        F.lit("77386006").alias("defining_code"),
        F.lit("Pregnancy").alias("defining_display"),
        F.col("_pregnancy_outcome_code").alias("outcome_code"),
        F.col("_pregnancy_outcome_display").alias("outcome_display"),
        F.col("_pregnancy_id").alias("source_journey_identifier"),
        F.lit("pregnancy").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.lit("millennium-maternity").alias("source_system"),
        F.lit(SRC_MAT_PREGNANCY).alias("source_table"),
        F.col("_pregnancy_id").alias("source_row_id"),
        F.date_format("_loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("_source_update_timestamp").alias("source_update_timestamp"),
        F.col("_loaded_at").alias("loaded_at"),
    )

    expanded = base.withColumn("_trimester", F.explode(F.array(F.lit(1), F.lit(2), F.lit(3))))
    raw_start = (
        F.when(F.col("_trimester") == 1, F.col("_period_start"))
        .when(F.col("_trimester") == 2, F.date_add("_period_start", 98).cast("timestamp"))
        .otherwise(F.date_add("_period_start", 196).cast("timestamp"))
    )
    nominal_end = (
        F.when(F.col("_trimester") == 1, F.date_add("_period_start", 97).cast("timestamp"))
        .when(F.col("_trimester") == 2, F.date_add("_period_start", 195).cast("timestamp"))
        .otherwise(F.col("_period_end"))
    )
    child_start = F.when(
        F.col("_period_end").isNull() | (raw_start <= F.col("_period_end")), raw_start
    )
    child_end = F.when(
        child_start.isNotNull(),
        F.when(F.col("_period_end").isNull(), nominal_end)
        .otherwise(F.least(nominal_end, F.col("_period_end"))),
    )
    child = expanded.select(
        stable_id(
            "journey:pregnancy:trimester", F.col("_pregnancy_id"), F.col("_trimester")
        ).alias("journey_id"),
        parent_id.alias("parent_journey_id"),
        F.col("_subject_key").alias("subject_key"),
        F.col("_subject_id_system").alias("subject_id_system"),
        F.col("_person_id").alias("person_id"),
        F.lit("pregnancy_trimester").alias("journey_type_code"),
        F.concat(F.lit("Pregnancy trimester "), F.col("_trimester")).alias("journey_type_display"),
        child_start.alias("period_start"),
        child_end.alias("period_end"),
        F.col("_status_code").alias("status_code"),
        F.lit("urn:journey:pregnancy-structure").alias("defining_coding_system"),
        F.concat(F.lit("trimester-"), F.col("_trimester")).alias("defining_code"),
        F.concat(F.lit("Pregnancy trimester "), F.col("_trimester")).alias("defining_display"),
        F.lit(None).cast("string").alias("outcome_code"),
        F.lit(None).cast("string").alias("outcome_display"),
        F.col("_pregnancy_id").alias("source_journey_identifier"),
        F.lit("pregnancy-trimester").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.lit("millennium-maternity").alias("source_system"),
        F.lit(SRC_MAT_PREGNANCY).alias("source_table"),
        F.concat_ws(":", F.col("_pregnancy_id"), F.lit("trimester"), F.col("_trimester"))
        .alias("source_row_id"),
        F.date_format("_loaded_at", "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("_source_update_timestamp").alias("source_update_timestamp"),
        F.col("_loaded_at").alias("loaded_at"),
    )
    return parent.unionByName(child).select(*JOURNEY_PUBLIC_COLUMNS)

In [0]:
JOURNEY_PUBLIC_COLUMNS = [
    "journey_id", "parent_journey_id", "subject_key", "subject_id_system", "person_id",
    "journey_type_code", "journey_type_display", "period_start", "period_end",
    "status_code", "defining_coding_system", "defining_code", "defining_display",
    "outcome_code", "outcome_display", "source_journey_identifier",
    "construction_rule", "construction_version", "source_system", "source_table",
    "source_row_id", "load_batch_id", "source_update_timestamp",
    "loaded_at",
]

def _pregnancy_constructor_base():
    p = _mat_pregnancy_dedup().alias("p")
    b = (
        read_source(SRC_MAT_BIRTH)
        .groupBy("Pregnancy_ID")
        .agg(
            F.min("BirthDateTime").alias("_first_birth_datetime"),
            F.max("BirthDateTime").alias("_last_birth_datetime"),
            F.max("PregOutcome_CD").cast("string").alias("_pregnancy_outcome_code"),
            F.max("PregOutcome_DESC").alias("_pregnancy_outcome_display"),
            F.max("ADC_UPDT").alias("_birth_loaded_at"),
            F.max("BirthSourceRecordUpdatedDateTime").alias("_birth_source_update"),
        )
        .alias("b")
    )
    joined = p.join(b, p.Pregnancy_ID == b.Pregnancy_ID, "left")
    source_row_id = p.Pregnancy_ID.cast("string")
    skey, ssys = subject_key_with_system(
        [
            ("urn:cerner:person_id", p.Person_ID),
            ("urn:barts:mrn", p.MRN),
            ("https://fhir.nhs.uk/Id/nhs-number", p.NHS_Number),
        ],
        SRC_MAT_PREGNANCY,
        source_row_id,
    )
    start = F.coalesce(
        p.LastMensPeriodDate,
        F.date_sub(p.ExpectedDeliveryDate, 280).cast("timestamp"),
        p.PregnancyFirstContactDate,
        p.FirstAntenatalAPPTDate,
    )
    end = F.coalesce(
        p.MaternityServiceDischargeDate,
        F.col("b._last_birth_datetime"),
        p.ExpectedDeliveryDate,
    )
    deleted = F.coalesce(p.SOURCE_DELETED_IND.cast("boolean"), F.lit(False))
    completed = p.MaternityServiceDischargeDate.isNotNull() | F.col("b._last_birth_datetime").isNotNull()
    status = (
        F.when(deleted, F.lit("superseded"))
        .when(completed, F.lit("completed"))
        .otherwise(F.lit("active"))
    )
    source_update = F.greatest(
        p.MAT_RECORD_UPDATED_DT,
        p.MSDS_RECORD_UPDATED_DT,
        p.BIRTH_ADC_UPDT,
        F.col("b._birth_source_update"),
    )
    loaded_at = F.greatest(p.ADC_UPDT, F.col("b._birth_loaded_at"))
    return joined.select(
        p.Pregnancy_ID.cast("string").alias("_pregnancy_id"),
        skey.alias("_subject_key"),
        ssys.alias("_subject_id_system"),
        p.Person_ID.cast("string").alias("_person_id"),
        start.alias("_period_start"),
        end.alias("_period_end"),
        status.alias("_status_code"),
        F.col("b._pregnancy_outcome_code"),
        F.col("b._pregnancy_outcome_display"),
        source_update.alias("_source_update_timestamp"),
        loaded_at.alias("_loaded_at"),
    )

JOURNEY_COLUMN_COMMENTS = {
    "journey_id": "Deterministic journey primary key.",
    "parent_journey_id": "Parent journey for governed nesting.",
    "subject_key": "Always-populated primary-subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved primary-subject person identifier when available.",
    "journey_type_code": "Controlled journey type.",
    "journey_type_display": "Human-readable journey type.",
    "period_start": "Earliest source-supported journey boundary.",
    "period_end": "Latest source-supported journey boundary.",
    "status_code": "Normalized journey lifecycle state.",
    "defining_coding_system": "Coding system for the constructor-defining concept.",
    "defining_code": "Constructor-defining concept code.",
    "defining_display": "Constructor-defining concept display.",
    "outcome_code": "Source pregnancy outcome code where supplied.",
    "outcome_display": "Source pregnancy outcome display where supplied.",
    "source_journey_identifier": "Verbatim source pregnancy identifier.",
    "construction_rule": "Governed constructor name.",
    "construction_version": "Governed constructor version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured pregnancy source table.",
    "source_row_id": "Stable constructor source-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp supporting construction.",
    "loaded_at": "Latest bronze load timestamp supporting construction.",
}

@materialized_view(
    name=_n("journey_spine.journey"),
    comment="Deterministic longitudinal journeys; pregnancy-v1 supplies parent and trimester rows.",
    cluster_by=["person_id", "period_start"],
    refresh_policy="incremental",
    column_comments=JOURNEY_COLUMN_COMMENTS,
)
def journey():
    return _pregnancy_journey_canonical()

In [0]:
JOURNEY_PARTICIPANT_PUBLIC_COLUMNS = [
    "journey_participant_id", "journey_id", "subject_key", "subject_id_system",
    "person_id", "role_code", "role_display", "valid_from", "valid_to", "record_status",
    "construction_rule", "construction_version", "source_system", "source_table",
    "source_row_id", "load_batch_id", "source_update_timestamp",
    "loaded_at",
]

def _mother_journey_participants():
    j = _pregnancy_journey_canonical()
    return j.select(
        stable_id("journey_participant:mother", F.col("journey_id"), F.col("subject_key"))
        .alias("journey_participant_id"),
        "journey_id", "subject_key", "subject_id_system", "person_id",
        F.lit("mother").alias("role_code"),
        F.lit("Mother / primary subject").alias("role_display"),
        F.col("period_start").alias("valid_from"),
        F.col("period_end").alias("valid_to"),
        F.when(F.col("status_code") == "superseded", F.lit("superseded"))
        .otherwise(F.lit("active")).alias("record_status"),
        F.lit("pregnancy-primary-subject").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        "source_system", "source_table", "source_row_id", "load_batch_id", "source_update_timestamp", "loaded_at",
    )

def _baby_journey_participants():
    b = read_source(SRC_MAT_BIRTH)
    source_row_id = b.BirthRow_ID.cast("string")
    skey, ssys = subject_key_with_system(
        [
            ("urn:cerner:person_id", b.BabyPerson_ID),
            ("https://fhir.nhs.uk/Id/nhs-number", b.Baby_NHS),
            ("urn:barts:mrn", b.Baby_MRN),
        ],
        SRC_MAT_BIRTH,
        source_row_id,
    )
    deleted = F.coalesce(b.PregnancySource_DELETE_IND.cast("long"), F.lit(0)) != 0
    return b.select(
        stable_id("journey_participant:baby", b.Pregnancy_ID, b.BirthRow_ID)
        .alias("journey_participant_id"),
        stable_id("journey:pregnancy", b.Pregnancy_ID).alias("journey_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        b.BabyPerson_ID.cast("string").alias("person_id"),
        F.lit("baby").alias("role_code"), F.lit("Baby").alias("role_display"),
        b.BirthDateTime.alias("valid_from"), F.lit(None).cast("timestamp").alias("valid_to"),
        F.when(deleted, F.lit("superseded")).otherwise(F.lit("active")).alias("record_status"),
        F.lit("pregnancy-birth-participant").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.lit("millennium-maternity").alias("source_system"),
        F.lit(SRC_MAT_BIRTH).alias("source_table"), source_row_id.alias("source_row_id"),
        F.date_format(b.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.greatest(b.BirthSourceRecordUpdatedDateTime, b.PregnancySourceRecordUpdatedDateTime,
                   b.NNUSourceLastUpdate).alias("source_update_timestamp"),
        b.ADC_UPDT.alias("loaded_at"),
    )

JOURNEY_PARTICIPANT_COLUMN_COMMENTS = {
    "journey_participant_id": "Deterministic journey-participant primary key.",
    "journey_id": "Journey reference.",
    "subject_key": "Always-populated participant subject key.",
    "subject_id_system": "Identifier system used for participant subject_key.",
    "person_id": "Resolved participant person identifier when available.",
    "role_code": "Controlled role in the journey.",
    "role_display": "Human-readable participant role.",
    "valid_from": "Role validity start.",
    "valid_to": "Role validity end.",
    "record_status": "Normalized source lifecycle status.",
    "construction_rule": "Governed participant-construction rule.",
    "construction_version": "Governed participant-construction version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured bronze source table.",
    "source_row_id": "Stable source-row identifier supporting the role.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_spine.journey_participant"),
    comment="Journey participant roles with subject-key support for unresolved babies.",
    refresh_policy="incremental",
    column_comments=JOURNEY_PARTICIPANT_COLUMN_COMMENTS,
)
def journey_participant():
    mother = _mother_journey_participants().select(*JOURNEY_PARTICIPANT_PUBLIC_COLUMNS)
    baby = _baby_journey_participants().select(*JOURNEY_PARTICIPANT_PUBLIC_COLUMNS)
    return mother.unionByName(baby)

In [0]:
JOURNEY_LINK_PUBLIC_COLUMNS = [
    "journey_link_id", "journey_id", "member_type", "encounter_id",
    "patient_event_id", "role_code", "member_start_datetime", "member_end_datetime",
    "record_status", "construction_rule", "construction_version", "source_system",
    "source_table", "source_row_id", "load_batch_id", "loaded_at",
]

def _journey_encounter_links():
    j = _pregnancy_journey_canonical().alias("j")
    e = spark.read.table(_n("journey_spine.encounter")).alias("e")
    member_end = F.coalesce(F.col("e.period_end"), F.col("e.period_start"))
    overlap = (
        (F.col("j.subject_key") == F.col("e.subject_key"))
        & F.col("j.period_start").isNotNull()
        & F.col("e.period_start").isNotNull()
        & (F.col("j.period_end").isNull() | (F.col("e.period_start") <= F.col("j.period_end")))
        & (member_end >= F.col("j.period_start"))
    )
    return j.join(e, overlap, "inner").select(
        stable_id("journey_link:temporal_overlap", F.col("j.journey_id"),
                  F.lit("encounter"), F.col("e.encounter_id")).alias("journey_link_id"),
        F.col("j.journey_id"), F.lit("encounter").alias("member_type"),
        F.col("e.encounter_id"), F.lit(None).cast("string").alias("patient_event_id"),
        F.lit("temporal_overlap").alias("role_code"),
        F.col("e.period_start").alias("member_start_datetime"),
        F.col("e.period_end").alias("member_end_datetime"),
        F.lit("active").alias("record_status"),
        F.lit("subject-time-overlap").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.col("j.source_system"), F.col("j.source_table"), F.col("j.source_row_id"),
        F.col("j.load_batch_id"), F.greatest(F.col("j.loaded_at"), F.col("e.loaded_at")).alias("loaded_at"),
    )

def _journey_event_links():
    j = _pregnancy_journey_canonical().alias("j")
    e = spark.read.table(_n("journey_events._event_index")).alias("e")
    member_end = F.coalesce(F.col("e.event_end_datetime"), F.col("e.event_datetime"))
    overlap = (
        (F.col("j.subject_key") == F.col("e.subject_key"))
        & F.col("j.period_start").isNotNull()
        & F.col("e.event_datetime").isNotNull()
        & (F.col("j.period_end").isNull() | (F.col("e.event_datetime") <= F.col("j.period_end")))
        & (member_end >= F.col("j.period_start"))
    )
    return j.join(e, overlap, "inner").select(
        stable_id("journey_link:temporal_overlap", F.col("j.journey_id"),
                  F.lit("patient_event"), F.col("e.patient_event_id")).alias("journey_link_id"),
        F.col("j.journey_id"), F.lit("patient_event").alias("member_type"),
        F.lit(None).cast("string").alias("encounter_id"), F.col("e.patient_event_id"),
        F.lit("temporal_overlap").alias("role_code"),
        F.col("e.event_datetime").alias("member_start_datetime"),
        F.col("e.event_end_datetime").alias("member_end_datetime"),
        F.lit("active").alias("record_status"),
        F.lit("subject-time-overlap").alias("construction_rule"),
        F.lit("pregnancy-v1").alias("construction_version"),
        F.col("j.source_system"), F.col("j.source_table"), F.col("j.source_row_id"),
        F.col("j.load_batch_id"), F.greatest(F.col("j.loaded_at"), F.col("e.loaded_at")).alias("loaded_at"),
    )

def _journey_episode_links():
    """Journey members via shared-episode evidence: encounters sharing an active
    episode with a temporally-linked member encounter join the journey with role
    episode_membership. Relations whose episode is absent from the source
    (SOURCE_EPISODE_ABSENT) are retained on the fact but EXCLUDED here — a container bronze
    marked suspect is not linkage evidence. Both sides pre-collapse BEFORE the member join:
    anchors to one (journey_id, episode_id) row, members to one (episode_id, encounter_id)
    row — the intermediate is journeys x their episodes x members, never anchors x members.
    Anchor-relation load times fold into loaded_at so a newly landed anchor relation is
    visible to watermark consumers on the links it creates."""
    base = _journey_encounter_links().select(
        F.col("journey_id"),
        F.col("encounter_id").alias("anchor_encounter_id"),
        F.col("load_batch_id"),
        F.col("loaded_at").alias("j_loaded_at"),
    )
    ee = spark.read.table(_n("journey_spine.episode_encounter")).where(
        (F.col("record_status") == F.lit("active"))
        & (F.coalesce(F.col("relation_status_code"), F.lit("")) != F.lit("SOURCE_EPISODE_ABSENT"))
    )
    anchor_rel = ee.select(
        F.col("encounter_id").alias("anchor_encounter_id"),
        F.col("episode_id"),
        F.col("loaded_at").alias("anchor_loaded_at"),
    )
    journey_episode = (
        base.join(anchor_rel, "anchor_encounter_id")
            .groupBy("journey_id", "episode_id")
            .agg(
                F.max("load_batch_id").alias("load_batch_id"),
                F.max(F.greatest(F.col("j_loaded_at"), F.col("anchor_loaded_at")))
                 .alias("anchor_loaded_at"),
            )
    )
    member = (
        ee.select(
            F.col("episode_id"),
            F.col("encounter_id").alias("member_encounter_id"),
            F.col("source_row_id"),
            F.col("loaded_at"),
        )
        .groupBy("episode_id", "member_encounter_id")
        .agg(
            F.max("source_row_id").alias("ee_source_row_id"),
            F.max("loaded_at").alias("ee_loaded_at"),
        )
    )
    enc = spark.read.table(_n("journey_spine.encounter")).select(
        F.col("encounter_id").alias("member_encounter_id"),
        F.col("period_start"),
        F.col("period_end"),
        F.col("loaded_at").alias("enc_loaded_at"),
    )
    paths = (
        journey_episode.join(member, "episode_id")
                       .join(enc, "member_encounter_id")
    )
    grouped = paths.groupBy("journey_id", "member_encounter_id").agg(
        F.min("period_start").alias("member_start_datetime"),
        F.max("period_end").alias("member_end_datetime"),
        F.max("ee_source_row_id").alias("source_row_id"),
        F.max("load_batch_id").alias("load_batch_id"),
        F.greatest(F.max("anchor_loaded_at"), F.max("ee_loaded_at"), F.max("enc_loaded_at")).alias("loaded_at"),
    )
    return grouped.select(
        stable_id("journey_link:episode_membership", F.col("journey_id"),
                  F.lit("encounter"), F.col("member_encounter_id")).alias("journey_link_id"),
        F.col("journey_id"),
        F.lit("encounter").alias("member_type"),
        F.col("member_encounter_id").alias("encounter_id"),
        F.lit(None).cast("string").alias("patient_event_id"),
        F.lit("episode_membership").alias("role_code"),
        F.col("member_start_datetime"),
        F.col("member_end_datetime"),
        F.lit("active").alias("record_status"),
        F.lit("shared-episode-membership").alias("construction_rule"),
        F.lit("episode-membership-v1").alias("construction_version"),
        F.lit("millennium").alias("source_system"),
        F.lit(SRC_EPISODE_ENCOUNTER).alias("source_table"),
        F.col("source_row_id"),
        F.col("load_batch_id"),
        F.col("loaded_at"),
    )

JOURNEY_LINK_COLUMN_COMMENTS = {
    "journey_link_id": "Deterministic journey-link primary key.",
    "journey_id": "Journey reference.",
    "member_type": "Kind of linked member.",
    "encounter_id": "Encounter member when member_type is encounter.",
    "patient_event_id": "Event member when member_type is patient_event.",
    "role_code": "Controlled link role.",
    "member_start_datetime": "Linked member start timestamp used by the constructor.",
    "member_end_datetime": "Linked member end timestamp used by the constructor.",
    "record_status": "Normalized link lifecycle state.",
    "construction_rule": "Governed overlap-construction rule; episode_membership populated from the governed episode feeds.",
    "construction_version": "Governed overlap-construction version.",
    "source_system": "Constructor source-system identifier.",
    "source_table": "Fully qualified configured journey-constructor source table.",
    "source_row_id": "Stable constructor source-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Latest bronze load timestamp supporting the constructor.",
}

@materialized_view(
    name=_n("journey_spine.journey_link"),
    comment="N:M journey membership links derived from subject and source-supported time overlap.",
    refresh_policy="incremental",
    column_comments=JOURNEY_LINK_COLUMN_COMMENTS,
)
def journey_link():
    encounters = _journey_encounter_links().select(*JOURNEY_LINK_PUBLIC_COLUMNS)
    events = _journey_event_links().select(*JOURNEY_LINK_PUBLIC_COLUMNS)
    episodes = _journey_episode_links().select(*JOURNEY_LINK_PUBLIC_COLUMNS)
    return encounters.unionByName(events).unionByName(episodes)

In [0]:
SRC_APPOINTMENT_SCHEDULE = "4_prod.bronze.map_appointment_schedule"

def _appointment_schedule_grouped_query():
    s = read_source(SRC_APPOINTMENT_SCHEDULE)
    entry = F.struct(
        F.coalesce(s.SCHEDULE_SEQ, s.SCHEDULE_ID).cast("long").alias("sequence"),
        s.SCHEDULE_ID.cast("string").alias("schedule_id"),
        s.SCHEDULE_SEQ.cast("long").alias("schedule_sequence"),
        s.SCH_STATE_CD.cast("string").alias("status_code"),
        s.SCH_STATE_DESCRIPTION.alias("status_display"),
        s.SOURCE_STATE_MEANING.alias("status_meaning"),
        s.BEG_EFFECTIVE_DT_TM.alias("effective_start"),
        s.END_EFFECTIVE_DT_TM.alias("effective_end"),
        s.LOCATION_CD.cast("string").alias("location_code"),
        F.coalesce(s.LOCATION_DESCRIPTION, s.LOCATION_FREETEXT).alias("location_display"),
        s.SOURCE_PRESENT_IND.cast("boolean").alias("source_present_ind"),
    )
    return (
        s.groupBy("SCH_EVENT_ID")
        .agg(
            F.to_json(F.sort_array(F.collect_list(entry))).alias("_booking_json"),
            F.min("LOCATION_CD").alias("_schedule_location_cd"),
            F.max("SOURCE_ADC_UPDT").alias("_schedule_source_update"),
            F.max("ADC_UPDT").alias("_schedule_loaded_at"),
        )
    )

@materialized_view(
    name=_n("journey_clinical._appointment_schedule_grouped"),
    private=True,
    comment="Internal incremental appointment booking/location history grouped as deterministic JSON.",
    refresh_policy="incremental",
)
def _appointment_schedule_grouped():
    return _appointment_schedule_grouped_query()

In [0]:
def _appointment_resource_grouped_query():
    r = read_source(SRC_APPOINTMENT_RESOURCE)
    entry = F.struct(
        F.coalesce(r.SCHEDULE_SEQ, r.SCH_APPT_ID).cast("long").alias("sequence"),
        r.SCH_APPT_ID.cast("string").alias("appointment_role_id"),
        r.SCHEDULE_ID.cast("string").alias("schedule_id"),
        r.SCH_ROLE_CD.cast("string").alias("role_code"),
        F.coalesce(r.SCH_ROLE_DESCRIPTION, r.ROLE_MEANING).alias("role_display"),
        F.when(r.ALLOCATED_PERSONNEL_ID.isNotNull(),
               stable_id("practitioner:mill", r.ALLOCATED_PERSONNEL_ID))
        .alias("practitioner_id"),
        r.APPT_LOCATION_CD.cast("string").alias("location_code"),
        r.BEG_DT_TM.alias("slot_start"), r.END_DT_TM.alias("slot_end"),
        r.SOURCE_PRESENT_IND.cast("boolean").alias("source_present_ind"),
    )
    return (
        r.groupBy("SCH_EVENT_ID")
        .agg(
            F.to_json(F.sort_array(F.collect_list(entry))).alias("_resource_json"),
            F.max("BEG_DT_TM").alias("_slot_start"),
            F.max("END_DT_TM").alias("_slot_end"),
            F.min("ALLOCATED_PERSONNEL_ID").alias("_allocated_personnel_id"),
            F.min("APPT_LOCATION_CD").alias("_resource_location_cd"),
            F.max("SOURCE_ADC_UPDT").alias("_resource_source_update"),
            F.max("ADC_UPDT").alias("_resource_loaded_at"),
        )
    )

@materialized_view(
    name=_n("journey_clinical._appointment_resource_grouped"),
    private=True,
    comment="Internal incremental appointment resource/slot history grouped as deterministic JSON.",
    refresh_policy="incremental",
)
def _appointment_resource_grouped():
    return _appointment_resource_grouped_query()

In [0]:
# ==== Scheduling activity and SurgiNet case procedures ====

APPOINTMENT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "appointment_type_code",
    "appointment_type_display", "status_code", "status_display", "status_meaning",
    "referral_identifier", "requested_datetime", "original_requested_start",
    "original_requested_end", "first_booked_datetime", "booking_iterations",
    "resource_history", "requested_practitioner_id", "allocated_practitioner_id",
    "location_id", "organization_id", "recurrence_parent_id", "recurrence_type_flag",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _appointment_canonical():
    a = read_source(SRC_APPOINTMENT).alias("a")
    s = spark.read.table(_n("journey_clinical._appointment_schedule_grouped")).alias("s")
    r = spark.read.table(_n("journey_clinical._appointment_resource_grouped")).alias("r")
    joined = a.join(s, "SCH_EVENT_ID", "left").join(r, "SCH_EVENT_ID", "left")
    event_id = stable_id("appointment:mill_scheduling", a.SCH_EVENT_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", a.PERSON_ID)], SRC_APPOINTMENT, a.SCH_EVENT_ID
    )
    source_present = F.coalesce(a.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    inactive = (
        (~source_present)
        | (F.coalesce(a.ACTIVE_IND.cast("long"), F.lit(1)) == 0)
        | ((a.END_EFFECTIVE_DT_TM.isNotNull())
           & (a.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp")))
    )
    loaded_at = F.greatest(a.ADC_UPDT, F.col("s._schedule_loaded_at"),
                           F.col("r._resource_loaded_at"))
    source_update = F.greatest(a.SOURCE_ADC_UPDT, F.col("s._schedule_source_update"),
                               F.col("r._resource_source_update"))
    location_code = F.coalesce(F.col("r._resource_location_cd"),
                               F.col("s._schedule_location_cd"))
    return joined.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        a.PERSON_ID.cast("string").alias("person_id"),
        F.when(a.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.when(a.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", a.ENCNTR_ID))
        .alias("encounter_id"),
        F.coalesce(F.col("r._slot_start"), a.ORIG_REQ_START_DT_TM,
                   a.FIRST_BKD_ASI_DT_TM, a.REFER_DT_TM).alias("event_datetime"),
        F.col("r._slot_end").alias("event_end_datetime"),
        F.lit("urn:cerner:scheduling:appointment-type").alias("source_coding_system"),
        a.APPT_TYPE_CD.cast("string").alias("source_code"),
        F.coalesce(a.APPT_TYPE_DESCRIPTION, a.APPT_SYNONYM_DESCRIPTION).alias("source_display"),
        a.APPT_TYPE_CD.cast("string").alias("appointment_type_code"),
        F.coalesce(a.APPT_TYPE_DESCRIPTION, a.APPT_SYNONYM_DESCRIPTION)
        .alias("appointment_type_display"),
        a.SCH_STATE_CD.cast("string").alias("status_code"),
        a.SCH_STATE_DESCRIPTION.alias("status_display"),
        a.SOURCE_SCHEDULE_MEANING.alias("status_meaning"),
        a.REFERRAL_IDENT.alias("referral_identifier"), a.REFER_DT_TM.alias("requested_datetime"),
        a.ORIG_REQ_START_DT_TM.alias("original_requested_start"),
        a.ORIG_REQ_END_DT_TM.alias("original_requested_end"),
        a.FIRST_BKD_ASI_DT_TM.alias("first_booked_datetime"),
        F.coalesce(F.col("s._booking_json"), F.lit("[]"))
        .alias("_booking_iterations_json"),
        F.coalesce(F.col("r._resource_json"), F.lit("[]"))
        .alias("_resource_history_json"),
        F.when(a.REQUESTED_PERSONNEL_ID.isNotNull(),
               stable_id("practitioner:mill", a.REQUESTED_PERSONNEL_ID))
        .alias("requested_practitioner_id"),
        F.when(F.col("r._allocated_personnel_id").isNotNull(),
               stable_id("practitioner:mill", F.col("r._allocated_personnel_id")))
        .alias("allocated_practitioner_id"),
        F.when(location_code.isNotNull(),
               stable_id("location:mill:nurse_unit", location_code)).alias("location_id"),
        F.when(a.ORGANIZATION_ID.isNotNull(), stable_id("organization:mill", a.ORGANIZATION_ID))
        .alias("organization_id"),
        F.when(a.RECUR_PARENT_ID.isNotNull(),
               stable_id("appointment:mill_scheduling", a.RECUR_PARENT_ID))
        .alias("recurrence_parent_id"),
        a.RECUR_TYPE_FLAG.cast("long").alias("recurrence_type_flag"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        a.BEG_EFFECTIVE_DT_TM.alias("record_status_effective_from"),
        F.when(inactive, F.coalesce(a.SOURCE_ABSENT_DETECTED_TS, a.END_EFFECTIVE_DT_TM,
                                    a.ADC_UPDT)).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("appointment").alias("source_feed"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        source_update.alias("source_update_timestamp"), loaded_at.alias("loaded_at"),
        F.lit("millennium-scheduling").alias("_source_system"),
        F.lit(SRC_APPOINTMENT).alias("_source_table"),
        a.SCH_EVENT_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
APPOINTMENT_PRIMITIVE_COLUMNS = [
    {
        "booking_iterations": "_booking_iterations_json",
        "resource_history": "_resource_history_json",
    }.get(name, name)
    for name in APPOINTMENT_PUBLIC_COLUMNS
]

@materialized_view(
    name=_n("journey_clinical._appointment_primitive"),
    private=True,
    comment="Internal orderable appointment join; booking/resource VARIANT arrays cross as JSON.",
    refresh_policy="incremental",
)
def _appointment_primitive():
    return _appointment_canonical().select(*APPOINTMENT_PRIMITIVE_COLUMNS)

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_appointment"),
    comment="Private JSON bridge and Gold cross-rule flags for appointment.",
    refresh_policy="incremental",
)
def _qc_appointment():
    return _cross_qc_primitive(
        spark.read.table(_n("journey_clinical._appointment_primitive")),
        "appointment",
        {
            "booking_iterations": "_booking_iterations_json",
            "resource_history": "_resource_history_json",
        },
    )

In [0]:
APPOINTMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide appointment identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference supplied by scheduling.",
    "event_datetime": "Booked slot start where available",
    "event_end_datetime": "Booked slot end where available.",
    "source_coding_system": "Verbatim appointment-type coding system.",
    "source_code": "Verbatim appointment-type code.",
    "source_display": "Verbatim appointment-type display.",
    "appointment_type_code": "Source appointment-type code.",
    "appointment_type_display": "Source appointment-type display.",
    "status_code": "Source scheduling-state code.",
    "status_display": "Source scheduling-state display.",
    "status_meaning": "Source scheduling-state meaning.",
    "referral_identifier": "UBRN or other referral alias retained as linkage evidence only.",
    "requested_datetime": "Source referral/request timestamp.",
    "original_requested_start": "Original requested slot start.",
    "original_requested_end": "Original requested slot end.",
    "first_booked_datetime": "First booking timestamp supplied by scheduling.",
    "booking_iterations": "Deterministically ordered booking and location iterations.",
    "resource_history": "Deterministically ordered scheduling-resource and slot history.",
    "requested_practitioner_id": "Requested practitioner reference when supplied.",
    "allocated_practitioner_id": "Deterministic representative allocated practitioner; all allocations remain in resource_history.",
    "location_id": "Deterministic location reference from scheduling history.",
    "organization_id": "Scheduling organization reference.",
    "recurrence_parent_id": "Parent appointment identifier for recurring schedules.",
    "recurrence_type_flag": "Raw source recurrence flag.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source current-row effective start.",
    "record_status_effective_to": "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.appointment"),
    comment="Scheduling appointments with ordered booking and resource history; SCH_EVENT_ID is never treated as a clinical EVENT_ID.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=APPOINTMENT_COLUMN_COMMENTS,
)
def appointment():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_appointment")),
        "appointment",
        {
            "booking_iterations": "_booking_iterations_json",
            "resource_history": "_resource_history_json",
        },
        APPOINTMENT_PUBLIC_COLUMNS,
    )

In [0]:
def _referral_canonical():
    s = read_source(SRC_REFERRAL).alias("s")
    event_id = stable_id("referral:luna", s.SOURCE_SYSTEM_OID, s.REFERRAL_OID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_REFERRAL,
        F.concat_ws(":", s.SOURCE_SYSTEM_OID.cast("string"), s.REFERRAL_OID.cast("string")),
    )
    source_present = F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    inactive = (~source_present) | (
        F.coalesce(s.ACTIVE_IND.cast("boolean"), F.lit(True)) == F.lit(False)
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.REFERRAL_RECEIVED_DATETIME.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:luna:treatment-function").alias("source_coding_system"),
        s.TREATMENT_FUNCTION_CD.alias("source_code"),
        s.TREATMENT_FUNCTION_DESC.alias("source_display"),
        s.UBRN.alias("ubrn"),
        s.WAITING_LIST_OID.cast("string").alias("waiting_list_oid"),
        s.PATHWAY_OID.cast("string").alias("pathway_oid"),
        s.REFERRAL_PRIORITY_CD.alias("referral_priority_code"),
        s.REFERRAL_PRIORITY_DESC.alias("referral_priority_display"),
        s.REFERRAL_SOURCE_CD.alias("referral_source_code"),
        s.REFERRAL_SOURCE_DESC.alias("referral_source_display"),
        s.REFERRAL_STATUS_CD.alias("status_code"),
        s.REFERRAL_STATUS_DESC.alias("status_display"),
        s.REFERRAL_STATUS_CHANGE_REASON_CD.alias("status_change_reason_code"),
        s.REFERRAL_STATUS_CHANGE_REASON_DESC.alias("status_change_reason_display"),
        s.REFERRAL_STATUS_CHANGE_DATETIME.alias("status_change_datetime"),
        s.ENCOUNTER_TYPE_CD.alias("encounter_type_code"),
        s.ENCOUNTER_TYPE_DESC.alias("encounter_type_display"),
        s.SUSPECTED_CANCER_SITE_CD.alias("suspected_cancer_site_code"),
        s.SUSPECTED_CANCER_SITE_DESC.alias("suspected_cancer_site_display"),
        s.TREATMENT_FUNCTION_CD.alias("treatment_function_code"),
        s.TREATMENT_FUNCTION_DESC.alias("treatment_function_display"),
        s.SERVICE_TYPE_REQUESTED_CD.alias("service_type_requested_code"),
        s.SERVICE_TYPE_REQUESTED_DESC.alias("service_type_requested_display"),
        s.SITE_CD.alias("site_code"), s.SITE_DESC.alias("site_display"),
        s.REFERRING_FACILITY_CD.alias("referring_facility_code"),
        s.REFERRING_FACILITY_DESC.alias("referring_facility_display"),
        s.REFERRED_BY_ORG_ID.cast("long").alias("referred_by_org_id"),
        s.BOOKING_TYPE_CD.alias("booking_type_code"),
        s.BOOKING_TYPE_DESC.alias("booking_type_display"),
        s.ADMIN_CATEGORY_CD.alias("admin_category_code"),
        s.ADMIN_CATEGORY_DESC.alias("admin_category_display"),
        s.BUSINESS_UNIT.alias("business_unit"), s.DIVISION.alias("division"),
        s.ORIGINAL_REFERRAL_RECEIVED_DATETIME.alias("original_received_datetime"),
        s.ERS_UBRN_RECEIVED.alias("ers_ubrn_received"),
        s.ERS_PATHWAY_START.alias("ers_pathway_start"),
        s.ERS_SERVICE_NAME.alias("ers_service_name"), s.ERS_SPECIALTY.alias("ers_specialty"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.CREATED_DATETIME.alias("record_status_effective_from"),
        F.when(inactive, F.coalesce(s.SOURCE_ABSENT_DETECTED_TS, s.MODIFIED_DATETIME, s.ADC_UPDT))
        .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("referral").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna").alias("_source_system"), F.lit(SRC_REFERRAL).alias("_source_table"),
        F.concat_ws(":", s.SOURCE_SYSTEM_OID.cast("string"), s.REFERRAL_OID.cast("string"))
        .alias("_source_row_id"),
    )

In [0]:
SRC_REFERRAL = "4_prod.bronze.map_referral"

REFERRAL_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide referral identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; always null for the LUNA referral feed.",
    "event_datetime": "Referral received timestamp.",
    "event_end_datetime": "Event end timestamp; not supplied by LUNA referrals.",
    "source_coding_system": "Verbatim LUNA treatment-function coding system.",
    "source_code": "Verbatim treatment-function code.",
    "source_display": "Verbatim treatment-function display.",
    "ubrn": "Referral linkage evidence only; never a join key.",
    "waiting_list_oid": "Raw LUNA waiting-list object identifier.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "referral_priority_code": "Source referral-priority code.",
    "referral_priority_display": "Source referral-priority display.",
    "referral_source_code": "Source referral-source code.",
    "referral_source_display": "Source referral-source display.",
    "status_code": "Source referral-status code.",
    "status_display": "Source referral-status display.",
    "status_change_reason_code": "Source referral status-change reason code.",
    "status_change_reason_display": "Source referral status-change reason display.",
    "status_change_datetime": "Source referral status-change timestamp.",
    "encounter_type_code": "Source encounter-type code.",
    "encounter_type_display": "Source encounter-type display.",
    "suspected_cancer_site_code": "Source suspected-cancer-site code.",
    "suspected_cancer_site_display": "Source suspected-cancer-site display.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "service_type_requested_code": "Source requested-service-type code.",
    "service_type_requested_display": "Source requested-service-type display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "referred_by_org_id": "Raw referring organization identifier.",
    "booking_type_code": "Source booking-type code.",
    "booking_type_display": "Source booking-type display.",
    "admin_category_code": "Source administrative-category code.",
    "admin_category_display": "Source administrative-category display.",
    "business_unit": "Source business unit.",
    "division": "Source division.",
    "original_received_datetime": "Original referral received timestamp.",
    "ers_ubrn_received": "e-Referral UBRN received date.",
    "ers_pathway_start": "e-Referral pathway start date.",
    "ers_service_name": "e-Referral service name.",
    "ers_specialty": "e-Referral specialty.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source row creation timestamp.",
    "record_status_effective_to": "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.referral"),
    comment="LUNA referrals (request side of threads). UBRN retained as linkage evidence only.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=REFERRAL_COLUMN_COMMENTS,
)
def referral():
    return _referral_canonical().drop("_source_system", "_source_table", "_source_row_id")

In [0]:
def _rtt_pathway_canonical():
    s = read_source(SRC_RTT_PATHWAY).alias("s")
    # PERIOD_OID is never NULL in bronze (0 = clockless-pathway sentinel); coalesce is
    # future-NULL defence only.
    period_key = F.coalesce(s.PERIOD_OID, F.lit(0))
    event_id = stable_id("rtt_period:luna", s.SOURCE_SYSTEM_OID, s.PATHWAY_OID, period_key)
    row_ref = F.concat_ws(
        ":", s.SOURCE_SYSTEM_OID.cast("string"), s.PATHWAY_OID.cast("string"),
        period_key.cast("string"),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_RTT_PATHWAY, row_ref
    )
    source_present = F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    inactive = (
        (~source_present)
        | (F.coalesce(s.PERIOD_ACTIVE_IND, F.lit(True)) == F.lit(False))
        | (F.coalesce(s.CORE_ACTIVE_IND, F.lit(True)) == F.lit(False))
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.START_DATETIME.alias("event_datetime"), s.STOP_DATETIME.alias("event_end_datetime"),
        F.lit("urn:barts:luna:rtt-status").alias("source_coding_system"),
        s.CURRENT_RTT_STATUS_CD.alias("source_code"),
        s.CURRENT_RTT_STATUS_DESC.alias("source_display"),
        s.PATHWAY_OID.cast("long").alias("pathway_oid"),
        s.PERIOD_OID.cast("long").alias("period_oid"),
        s.IS_LATEST_PERIOD.alias("is_latest_period"),
        s.START_RTT_STATUS_CD.alias("start_status_code"),
        s.START_RTT_STATUS_DESC.alias("start_status_display"),
        s.STOP_RTT_STATUS_CD.alias("stop_status_code"),
        s.STOP_RTT_STATUS_DESC.alias("stop_status_display"),
        s.CURRENT_RTT_STATUS_CD.alias("current_status_code"),
        s.CURRENT_RTT_STATUS_DESC.alias("current_status_display"),
        s.SEQ_NO_ASC.cast("long").alias("sequence_asc"),
        s.SEQ_NO_DESC.cast("long").alias("sequence_desc"),
        s.CLOCK_DISCREPANT.alias("clock_discrepant"),
        s.CORE_CLOCK_START_DT_TM.alias("core_clock_start"),
        s.CORE_CLOCK_STOP_DT_TM.alias("core_clock_stop"),
        s.PATHWAY_START_DATE.alias("pathway_start_date"),
        s.PATHWAY_TYPE_CD.alias("pathway_type_code"),
        s.PATHWAY_TYPE_DESC.alias("pathway_type_display"),
        s.BREACH_DATE.alias("breach_date"),
        s.DAYS_WAITED.cast("long").alias("days_waited"),
        s.DAYS_WAITED_ACTIVE.cast("long").alias("days_waited_active"),
        s.OP_APPT_DNA_COUNT.cast("long").alias("op_appt_dna_count"),
        s.TREATMENT_FUNCTION_CD.alias("treatment_function_code"),
        s.TREATMENT_FUNCTION_DESC.alias("treatment_function_display"),
        s.SITE_CD.alias("site_code"), s.SITE_DESC.alias("site_display"),
        s.REFERRING_FACILITY_CD.alias("referring_facility_code"),
        s.REFERRING_FACILITY_DESC.alias("referring_facility_display"),
        F.coalesce(F.to_json(s.ENCOUNTER_TYPES), F.lit("[]")).alias("_encounter_types_json"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.PERIOD_CREATED_DATETIME.alias("record_status_effective_from"),
        F.when(
            inactive,
            F.coalesce(s.SOURCE_ABSENT_DETECTED_TS, s.PERIOD_MODIFIED_DATETIME, s.ADC_UPDT),
        ).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("rtt_pathway").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna").alias("_source_system"), F.lit(SRC_RTT_PATHWAY).alias("_source_table"),
        row_ref.alias("_source_row_id"),
    )

In [0]:
SRC_RTT_PATHWAY = "4_prod.bronze.map_rtt_pathway"

@materialized_view(
    name=_n("journey_clinical._rtt_pathway_primitive"),
    private=True,
    comment="Internal RTT period rows; encounter-type tags cross as JSON.",
    refresh_policy="incremental",
)
def _rtt_pathway_primitive():
    return _rtt_pathway_canonical().drop("_source_system", "_source_table", "_source_row_id")

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_rtt_pathway"),
    comment="Private JSON bridge and Gold cross-rule flags for rtt_pathway.",
    refresh_policy="incremental",
)
def _qc_rtt_pathway():
    return _cross_qc_primitive(
        spark.read.table(_n("journey_clinical._rtt_pathway_primitive")),
        "rtt_pathway",
        {"encounter_types": "_encounter_types_json"},
    )

In [0]:
RTT_PATHWAY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display",
    "pathway_oid", "period_oid", "is_latest_period",
    "start_status_code", "start_status_display", "stop_status_code", "stop_status_display",
    "current_status_code", "current_status_display", "sequence_asc", "sequence_desc",
    "clock_discrepant", "core_clock_start", "core_clock_stop", "pathway_start_date",
    "pathway_type_code", "pathway_type_display", "breach_date", "days_waited",
    "days_waited_active", "op_appt_dna_count", "treatment_function_code",
    "treatment_function_display", "site_code", "site_display",
    "referring_facility_code", "referring_facility_display", "encounter_types",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

RTT_PATHWAY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide RTT period identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; not supplied by the LUNA RTT pathway feed.",
    "event_datetime": "RTT clock-period start timestamp.",
    "event_end_datetime": "RTT clock-period stop timestamp.",
    "source_coding_system": "Verbatim LUNA RTT-status coding system.",
    "source_code": "Verbatim current RTT-status code.",
    "source_display": "Verbatim current RTT-status display.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "period_oid": "Raw LUNA period object identifier; 0 marks the clockless-pathway sentinel and the column is never null.",
    "is_latest_period": "Whether this is the latest period for the pathway.",
    "start_status_code": "Source clock-start RTT-status code.",
    "start_status_display": "Source clock-start RTT-status display.",
    "stop_status_code": "Source clock-stop RTT-status code.",
    "stop_status_display": "Source clock-stop RTT-status display.",
    "current_status_code": "Source current RTT-status code.",
    "current_status_display": "Source current RTT-status display.",
    "sequence_asc": "Ascending pathway-period sequence.",
    "sequence_desc": "Descending pathway-period sequence.",
    "clock_discrepant": "Bronze flag indicating disagreement with the core clock timestamps.",
    "core_clock_start": "Core-system clock start retained as discrepant sidecar data.",
    "core_clock_stop": "Core-system clock stop retained as discrepant sidecar data.",
    "pathway_start_date": "Source pathway start date.",
    "pathway_type_code": "Source pathway-type code.",
    "pathway_type_display": "Source pathway-type display.",
    "breach_date": "Source breach timestamp.",
    "days_waited": "Source total days waited.",
    "days_waited_active": "Source active days waited.",
    "op_appt_dna_count": "Source outpatient did-not-attend count.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "encounter_types": "Deterministically sorted folded LUNA pathway encounter-type tags.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source clock-period creation timestamp.",
    "record_status_effective_to": "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.rtt_pathway"),
    comment="LUNA RTT pathway/clock periods (hybrid grain: one row per period, plus clockless-pathway sentinel rows). Clock discrepancies are carried, never filtered.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=RTT_PATHWAY_COLUMN_COMMENTS,
)
def rtt_pathway():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_rtt_pathway")),
        "rtt_pathway",
        {"encounter_types": "_encounter_types_json"},
        RTT_PATHWAY_PUBLIC_COLUMNS,
    )

In [0]:
def _rtt_activity_canonical():
    s = read_source(SRC_RTT_ACTIVITY).alias("s")
    event_id = stable_id("rtt_activity_event:luna", s.SOURCE_SYSTEM_OID, s.RTT_ACTIVITY_OID)
    row_ref = F.concat_ws(
        ":", s.SOURCE_SYSTEM_OID.cast("string"), s.RTT_ACTIVITY_OID.cast("string")
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_RTT_ACTIVITY, row_ref
    )
    source_present = F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    inactive = (~source_present) | (F.coalesce(s.ACTIVE_IND, F.lit(True)) == F.lit(False))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.RTT_ACTIVITY_DATETIME.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:luna:rtt-status").alias("source_coding_system"),
        s.RTT_STATUS_CD.alias("source_code"), s.RTT_STATUS_DESC.alias("source_display"),
        s.RTT_ACTIVITY_OID.cast("long").alias("rtt_activity_oid"),
        s.PATHWAY_OID.cast("long").alias("pathway_oid"),
        F.when(
            s.REFERRAL_OID.isNotNull(),
            stable_id("referral:luna", s.SOURCE_SYSTEM_OID, s.REFERRAL_OID),
        ).alias("referral_id"),
        s.APPOINTMENT_OID.cast("long").alias("appointment_oid"),
        s.RTT_ACTIVITY_CD.alias("activity_code"), s.RTT_ACTIVITY_DESC.alias("activity_display"),
        s.RTT_ACTIVITY_TYPE_CD.alias("activity_type_code"),
        s.RTT_ACTIVITY_TYPE_DESC.alias("activity_type_display"),
        s.RTT_STATUS_CD.alias("status_code"), s.RTT_STATUS_DESC.alias("status_display"),
        s.RTT_STATUS_SEQUENCE_ASC.cast("long").alias("status_sequence_asc"),
        s.RTT_STATUS_SEQUENCE_DESC.cast("long").alias("status_sequence_desc"),
        s.RTT_ACTIVITY_SEQUENCE_ASC.cast("long").alias("activity_sequence_asc"),
        s.RTT_ACTIVITY_SEQUENCE_DESC.cast("long").alias("activity_sequence_desc"),
        s.IS_ILLOGICAL.alias("is_illogical"),
        s.RTT_ACTIVITY_DATETIME_QUALITY.alias("activity_datetime_quality"),
        s.TREATMENT_FUNCTION_CD.alias("treatment_function_code"),
        s.TREATMENT_FUNCTION_DESC.alias("treatment_function_display"),
        s.SITE_CD.alias("site_code"), s.SITE_DESC.alias("site_display"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.CREATED_DATETIME.alias("record_status_effective_from"),
        F.when(inactive, F.coalesce(s.SOURCE_ABSENT_DETECTED_TS, s.MODIFIED_DATETIME, s.ADC_UPDT))
        .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("rtt_activity").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna").alias("_source_system"),
        F.lit(SRC_RTT_ACTIVITY).alias("_source_table"), row_ref.alias("_source_row_id"),
    )

In [0]:
SRC_RTT_ACTIVITY = "4_prod.bronze.map_rtt_activity"

RTT_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide RTT activity identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; not supplied by the LUNA RTT activity feed.",
    "event_datetime": "RTT activity timestamp.",
    "event_end_datetime": "Event end timestamp; not supplied by LUNA RTT activities.",
    "source_coding_system": "Verbatim LUNA RTT-status coding system.",
    "source_code": "Verbatim RTT-status code.",
    "source_display": "Verbatim RTT-status display.",
    "rtt_activity_oid": "Raw LUNA RTT activity object identifier.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "referral_id": "Minted referral patient_event_id when a source referral object identifier is present.",
    "appointment_oid": "Raw appointment linkage evidence; resolution is gated and no edge is emitted.",
    "activity_code": "Source RTT activity code.",
    "activity_display": "Source RTT activity display.",
    "activity_type_code": "Source RTT activity-type code.",
    "activity_type_display": "Source RTT activity-type display.",
    "status_code": "Source RTT-status code.",
    "status_display": "Source RTT-status display.",
    "status_sequence_asc": "Ascending RTT-status sequence.",
    "status_sequence_desc": "Descending RTT-status sequence.",
    "activity_sequence_asc": "Ascending RTT-activity sequence.",
    "activity_sequence_desc": "Descending RTT-activity sequence.",
    "is_illogical": "Bronze quality flag carried as data and never filtered.",
    "activity_datetime_quality": "Bronze quality classification for the activity timestamp.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source activity creation timestamp.",
    "record_status_effective_to": "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.rtt_activity"),
    comment="LUNA clock-affecting RTT activity/status events. IS_ILLOGICAL is carried as data, never filtered.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=RTT_ACTIVITY_COLUMN_COMMENTS,
)
def rtt_activity():
    return _rtt_activity_canonical().drop("_source_system", "_source_table", "_source_row_id")

In [0]:
SRC_WAITING_LIST = "4_prod.bronze.map_waiting_list"

def _waiting_list_entry_canonical():
    s = read_source(SRC_WAITING_LIST).alias("s")
    entry_id = stable_id("waiting_list:mill_pm", s.PM_WAIT_LIST_ID)
    # SOURCE_VERSION_ID is never NULL in bronze (CURRENT rows always carry -1); coalesce is
    # future-NULL defence only.
    version_row_id = F.concat_ws(
        ":", s.PM_WAIT_LIST_ID.cast("string"), s.ROW_SOURCE,
        F.coalesce(s.SOURCE_VERSION_ID.cast("string"), F.lit("~")),
    )
    fact_row = stable_id(
        "waiting_list:mill_pm", s.PM_WAIT_LIST_ID, s.ROW_SOURCE,
        F.coalesce(s.SOURCE_VERSION_ID, F.lit(-2)),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_WAITING_LIST, version_row_id
    )
    source_present = F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    superseded = (~source_present) | (~F.coalesce(s.IS_CURRENT, F.lit(False)))
    return s.select(
        entry_id.alias("patient_event_id"), fact_row.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
        .alias("encounter_id"),
        s.WAITING_START_DT_TM.alias("event_datetime"),
        s.WAITING_END_DT_TM.alias("event_end_datetime"),
        F.lit("urn:cerner:mill:planned-procedure").alias("source_coding_system"),
        s.PLANNED_PROCEDURE_CD.cast("string").alias("source_code"),
        s.PLANNED_PROCEDURE_DESC.alias("source_display"),
        s.PM_WAIT_LIST_ID.cast("long").alias("pm_wait_list_id"),
        s.ROW_SOURCE.alias("row_source"),
        s.SOURCE_VERSION_ID.cast("long").alias("source_version_id"),
        s.HIST_ACTION.alias("hist_action"), s.VERSION_DT_TM.alias("version_datetime"),
        s.IS_CURRENT.alias("is_current"), s.UPDT_CNT.cast("long").alias("updt_cnt"),
        s.SCH_EVENT_ID.cast("long").alias("sch_event_id"),
        s.STATUS_CD.cast("string").alias("status_code"), s.STATUS_DESC.alias("status_display"),
        s.SUB_STATUS_DESC.alias("sub_status_display"),
        s.ACTIVE_STATUS_DESC.alias("active_status_display"),
        s.URGENCY_DESC.alias("urgency_display"), s.STAND_BY_DESC.alias("stand_by_display"),
        s.ADMIT_CATEGORY_DESC.alias("admit_category_display"),
        s.ADMIT_BOOKING_DESC.alias("admit_booking_display"),
        s.ADMIT_TYPE_DESC.alias("admit_type_display"),
        s.ADMIT_OFFER_OUTCOME_DESC.alias("admit_offer_outcome_display"),
        s.MANAGEMENT_DESC.alias("management_display"),
        s.ATTENDANCE_DESC.alias("attendance_display"),
        s.REASON_FOR_CHANGE_DESC.alias("reason_for_change_display"),
        s.REASON_FOR_REMOVAL_DESC.alias("reason_for_removal_display"),
        s.ANESTHETIC_DESC.alias("anesthetic_display"),
        s.PLANNED_PROCEDURE_CD.cast("string").alias("planned_procedure_code"),
        s.PLANNED_PROCEDURE_DESC.alias("planned_procedure_display"),
        s.REFERRAL_SOURCE_DESC.alias("referral_source_display"),
        s.REFERRAL_TYPE_DESC.alias("referral_type_display"),
        s.SERVICE_TYPE_REQUESTED_DESC.alias("service_type_requested_display"),
        s.FROM_ED_IND.cast("long").alias("from_ed_ind"),
        s.SUSPENDED_DAYS.alias("suspended_days"),
        s.RECOMMEND_DT_TM.alias("recommend_datetime"),
        s.REFERRAL_DT_TM.alias("referral_datetime"),
        s.ORIG_REQUEST_RECEIVED_DT_TM.alias("original_request_received_datetime"),
        s.ADMIT_DECISION_DT_TM.alias("admit_decision_datetime"),
        s.ADMIT_GUARANTEED_DT_TM.alias("admit_guaranteed_datetime"),
        s.PROVISIONAL_ADMIT_DT_TM.alias("provisional_admit_datetime"),
        s.PREV_PROV_ADMIT_DT_TM.alias("previous_provisional_admit_datetime"),
        s.ADJ_WAITING_START_DT_TM.alias("adjusted_waiting_start_datetime"),
        s.SCHEDULE_DT_TM.alias("scheduled_datetime"),
        s.REQUESTED_DT_TM.alias("requested_datetime"),
        s.REMOVAL_DT_TM.alias("removal_datetime"),
        s.LAST_DNA_DT_TM.alias("last_dna_datetime"),
        s.STATUS_DT_TM.alias("status_datetime"),
        s.STATUS_END_DT_TM.alias("status_end_datetime"),
        F.when(
            s.LOC_NURSE_UNIT_CD.isNotNull(),
            stable_id("location:mill:nurse_unit", s.LOC_NURSE_UNIT_CD),
        ).alias("location_id"),
        s.LOC_NURSE_UNIT_DESC.alias("location_display"),
        s.LOC_FACILITY_DESC.alias("facility_display"),
        F.when(superseded, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.BEG_EFFECTIVE_DT_TM.alias("record_status_effective_from"),
        F.when(
            superseded,
            F.coalesce(s.SOURCE_ABSENT_DETECTED_TS, s.END_EFFECTIVE_DT_TM, s.ADC_UPDT),
        ).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("waiting_list").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium-pm").alias("_source_system"),
        F.lit(SRC_WAITING_LIST).alias("_source_table"), version_row_id.alias("_source_row_id"),
    )

In [0]:
WAITING_LIST_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable entry-grain identifier shared by all physical versions of one waiting-list entry.",
    "fact_row_id": "Stable version-grain storage-row identifier; uniqueness belongs here because patient_event_id is shared across versions.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Minted Millennium encounter identifier when supplied.",
    "event_datetime": "Waiting-start timestamp.",
    "event_end_datetime": "Waiting-end timestamp.",
    "source_coding_system": "Verbatim Millennium planned-procedure coding system.",
    "source_code": "Verbatim planned-procedure code.",
    "source_display": "Verbatim planned-procedure display.",
    "pm_wait_list_id": "Native waiting-list entry identifier.",
    "row_source": "Physical bronze row source.",
    "source_version_id": "Native source version identifier; CURRENT rows carry -1.",
    "hist_action": "Source history action when the row is historical.",
    "version_datetime": "Descriptive fallback version timestamp only; it defines no effective interval.",
    "is_current": "Whether this is the current physical version.",
    "updt_cnt": "Native source update counter.",
    "sch_event_id": "Raw scheduling event linkage evidence.",
    "status_code": "Source waiting-list status code.",
    "status_display": "Source waiting-list status display.",
    "sub_status_display": "Source waiting-list sub-status display.",
    "active_status_display": "Source active-status display.",
    "urgency_display": "Source urgency display.",
    "stand_by_display": "Source stand-by display.",
    "admit_category_display": "Source admission-category display.",
    "admit_booking_display": "Source admission-booking display.",
    "admit_type_display": "Source admission-type display.",
    "admit_offer_outcome_display": "Source admission-offer outcome display.",
    "management_display": "Source management display.",
    "attendance_display": "Source attendance display.",
    "reason_for_change_display": "Source reason-for-change display.",
    "reason_for_removal_display": "Source reason-for-removal display.",
    "anesthetic_display": "Source anesthetic display.",
    "planned_procedure_code": "Source planned-procedure code.",
    "planned_procedure_display": "Source planned-procedure display.",
    "referral_source_display": "Source referral-source display.",
    "referral_type_display": "Source referral-type display.",
    "service_type_requested_display": "Source requested-service-type display.",
    "from_ed_ind": "Raw source indicator for origin in the emergency department.",
    "suspended_days": "Source count of suspended days.",
    "recommend_datetime": "Source recommendation timestamp.",
    "referral_datetime": "Source referral timestamp.",
    "original_request_received_datetime": "Original request-received timestamp.",
    "admit_decision_datetime": "Source decision-to-admit timestamp.",
    "admit_guaranteed_datetime": "Source guaranteed-admission timestamp.",
    "provisional_admit_datetime": "Source provisional-admission timestamp.",
    "previous_provisional_admit_datetime": "Previous provisional-admission timestamp.",
    "adjusted_waiting_start_datetime": "Adjusted waiting-start timestamp.",
    "scheduled_datetime": "Source scheduled timestamp.",
    "requested_datetime": "Source requested timestamp.",
    "removal_datetime": "Source removal timestamp.",
    "last_dna_datetime": "Source last did-not-attend timestamp.",
    "status_datetime": "Source status timestamp.",
    "status_end_datetime": "Source status-end timestamp.",
    "location_id": "Minted nurse-unit location identifier.",
    "location_display": "Source nurse-unit display.",
    "facility_display": "Source facility display.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source row effective start.",
    "record_status_effective_to": "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.waiting_list_entry"),
    comment="Waiting-list entries at physical version grain (CURRENT + HIST). patient_event_id is entry-grain and shared across versions; VERSION_DT_TM is descriptive fallback only, never an effective interval.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=WAITING_LIST_ENTRY_COLUMN_COMMENTS,
)
def waiting_list_entry():
    return _waiting_list_entry_canonical().drop(
        "_source_system", "_source_table", "_source_row_id"
    )

In [0]:
SRC_WAITING_LIST_SNAPSHOT = "4_prod.bronze.map_waiting_list_snapshot"

def _waiting_list_snapshot_canonical():
    s = read_source(SRC_WAITING_LIST_SNAPSHOT).alias("s")
    snap_id = stable_id("wl_census:mill_pm", s.SNAPSHOT_DATE, s.PM_WAIT_LIST_ID)
    row_ref = F.concat_ws(":", s.SNAPSHOT_DATE.cast("string"), s.PM_WAIT_LIST_ID.cast("string"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_WAITING_LIST_SNAPSHOT, row_ref
    )
    return s.select(
        snap_id.alias("waiting_list_snapshot_id"),
        s.SNAPSHOT_DATE.alias("snapshot_date"),
        s.SNAPSHOT_CUTOFF_TS.alias("snapshot_cutoff_ts"),
        s.PM_WAIT_LIST_ID.cast("long").alias("pm_wait_list_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved")).otherwise(F.lit("unresolved"))
        .alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
        .alias("encounter_id"),
        s.SCH_EVENT_ID.cast("long").alias("sch_event_id"),
        s.STATUS_CD.cast("string").alias("status_code"), s.STATUS_DESC.alias("status_display"),
        s.SUB_STATUS_DESC.alias("sub_status_display"),
        s.ACTIVE_STATUS_DESC.alias("active_status_display"),
        s.URGENCY_DESC.alias("urgency_display"), s.STAND_BY_DESC.alias("stand_by_display"),
        s.ADMIT_CATEGORY_DESC.alias("admit_category_display"),
        s.ADMIT_BOOKING_DESC.alias("admit_booking_display"),
        s.ADMIT_TYPE_DESC.alias("admit_type_display"),
        s.ADMIT_OFFER_OUTCOME_DESC.alias("admit_offer_outcome_display"),
        s.MANAGEMENT_DESC.alias("management_display"),
        s.ATTENDANCE_DESC.alias("attendance_display"),
        s.REASON_FOR_CHANGE_DESC.alias("reason_for_change_display"),
        s.REASON_FOR_REMOVAL_DESC.alias("reason_for_removal_display"),
        s.ANESTHETIC_DESC.alias("anesthetic_display"),
        s.PLANNED_PROCEDURE_CD.cast("string").alias("planned_procedure_code"),
        s.PLANNED_PROCEDURE_DESC.alias("planned_procedure_display"),
        s.REFERRAL_SOURCE_DESC.alias("referral_source_display"),
        s.REFERRAL_TYPE_DESC.alias("referral_type_display"),
        s.SERVICE_TYPE_REQUESTED_DESC.alias("service_type_requested_display"),
        s.FROM_ED_IND.cast("long").alias("from_ed_ind"),
        s.SUSPENDED_DAYS.alias("suspended_days"),
        s.RECOMMEND_DT_TM.alias("recommend_datetime"),
        s.REFERRAL_DT_TM.alias("referral_datetime"),
        s.ORIG_REQUEST_RECEIVED_DT_TM.alias("original_request_received_datetime"),
        s.ADMIT_DECISION_DT_TM.alias("admit_decision_datetime"),
        s.ADMIT_GUARANTEED_DT_TM.alias("admit_guaranteed_datetime"),
        s.PROVISIONAL_ADMIT_DT_TM.alias("provisional_admit_datetime"),
        s.PREV_PROV_ADMIT_DT_TM.alias("previous_provisional_admit_datetime"),
        s.WAITING_START_DT_TM.alias("waiting_start_datetime"),
        s.WAITING_END_DT_TM.alias("waiting_end_datetime"),
        s.ADJ_WAITING_START_DT_TM.alias("adjusted_waiting_start_datetime"),
        s.SCHEDULE_DT_TM.alias("scheduled_datetime"),
        s.REQUESTED_DT_TM.alias("requested_datetime"),
        s.REMOVAL_DT_TM.alias("removal_datetime"),
        s.LAST_DNA_DT_TM.alias("last_dna_datetime"),
        s.STATUS_DT_TM.alias("status_datetime"),
        s.STATUS_END_DT_TM.alias("status_end_datetime"),
        F.when(
            s.LOC_NURSE_UNIT_CD.isNotNull(),
            stable_id("location:mill:nurse_unit", s.LOC_NURSE_UNIT_CD),
        ).alias("location_id"),
        s.LOC_NURSE_UNIT_DESC.alias("location_display"),
        s.LOC_FACILITY_DESC.alias("facility_display"),
        F.lit("administrative").alias("fact_category"),
        F.lit("waiting_list_census").alias("source_feed"),
        s.SNAPSHOT_CUTOFF_TS.alias("loaded_at"),
    )

WAITING_LIST_SNAPSHOT_COLUMN_COMMENTS = {
    "waiting_list_snapshot_id": "Stable census-row identifier.",
    "snapshot_date": "Census snapshot date.",
    "snapshot_cutoff_ts": "Immutable census cutoff timestamp.",
    "pm_wait_list_id": "Native waiting-list entry identifier.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Minted Millennium encounter identifier when supplied.",
    "sch_event_id": "Raw scheduling event linkage evidence.",
    "status_code": "Source waiting-list status code.",
    "status_display": "Source waiting-list status display.",
    "sub_status_display": "Source waiting-list sub-status display.",
    "active_status_display": "Source active-status display.",
    "urgency_display": "Source urgency display.",
    "stand_by_display": "Source stand-by display.",
    "admit_category_display": "Source admission-category display.",
    "admit_booking_display": "Source admission-booking display.",
    "admit_type_display": "Source admission-type display.",
    "admit_offer_outcome_display": "Source admission-offer outcome display.",
    "management_display": "Source management display.",
    "attendance_display": "Source attendance display.",
    "reason_for_change_display": "Source reason-for-change display.",
    "reason_for_removal_display": "Source reason-for-removal display.",
    "anesthetic_display": "Source anesthetic display.",
    "planned_procedure_code": "Source planned-procedure code.",
    "planned_procedure_display": "Source planned-procedure display.",
    "referral_source_display": "Source referral-source display.",
    "referral_type_display": "Source referral-type display.",
    "service_type_requested_display": "Source requested-service-type display.",
    "from_ed_ind": "Raw source indicator for origin in the emergency department.",
    "suspended_days": "Source count of suspended days.",
    "recommend_datetime": "Source recommendation timestamp.",
    "referral_datetime": "Source referral timestamp.",
    "original_request_received_datetime": "Original request-received timestamp.",
    "admit_decision_datetime": "Source decision-to-admit timestamp.",
    "admit_guaranteed_datetime": "Source guaranteed-admission timestamp.",
    "provisional_admit_datetime": "Source provisional-admission timestamp.",
    "previous_provisional_admit_datetime": "Previous provisional-admission timestamp.",
    "waiting_start_datetime": "Source waiting-start timestamp.",
    "waiting_end_datetime": "Source waiting-end timestamp.",
    "adjusted_waiting_start_datetime": "Adjusted waiting-start timestamp.",
    "scheduled_datetime": "Source scheduled timestamp.",
    "requested_datetime": "Source requested timestamp.",
    "removal_datetime": "Source removal timestamp.",
    "last_dna_datetime": "Source last did-not-attend timestamp.",
    "status_datetime": "Source status timestamp.",
    "status_end_datetime": "Source status-end timestamp.",
    "location_id": "Minted nurse-unit location identifier.",
    "location_display": "Source nurse-unit display.",
    "facility_display": "Source facility display.",
    "fact_category": "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the census row.",
    "loaded_at": "Census cutoff timestamp used as the immutable load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.waiting_list_snapshot"),
    comment="Immutable weekly waiting-list census (one row per SNAPSHOT_DATE x PM_WAIT_LIST_ID). Snapshots never enter patient_event.",
    refresh_policy="incremental",
    column_comments=WAITING_LIST_SNAPSHOT_COLUMN_COMMENTS,
)
def waiting_list_snapshot():
    return _waiting_list_snapshot_canonical()

In [0]:
def _allergy_canonical_pregate():
    s = read_source(SRC_ALLERGY)
    event_id = stable_id("allergy:mill", s.ALLERGY_INSTANCE_ID)
    raw_source_code = F.coalesce(
        s.SUBSTANCE_SNOMED_CODE,
        s.SUBSTANCE_SOURCE_IDENTIFIER,
        s.SUBSTANCE_NOM_ID.cast("long").cast("string"),
    )
    source_display = F.coalesce(
        s.SUBSTANCE_FTDESC, s.SUBSTANCE_SHORT_STRING, s.SUBSTANCE_SOURCE_STRING
    )
    source_code = _code_or_display(raw_source_code, source_display)
    source_system = F.coalesce(s.REC_SRC_VOCAB_DESC, F.lit("urn:cerner:nomenclature"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_ALLERGY, s.ALLERGY_INSTANCE_ID
    )
    cancelled = F.upper(F.coalesce(s.REACTION_STATUS_DESC, F.lit(""))) == F.lit("CANCELLED")
    ended = F.coalesce(
        s.END_EFFECTIVE_DT_TM < F.lit("2100-01-01").cast("timestamp"), F.lit(False)
    ) | (F.coalesce(s.ACTIVE_IND.cast("long"), F.lit(1)) == 0)
    record_status = (
        F.when(cancelled, F.lit("retracted"))
        .when(ended, F.lit("superseded"))
        .otherwise(F.lit("active"))
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
        .alias("encounter_id"),
        F.coalesce(s.ONSET_DT_TM_CLEAN, s.CREATED_DT_TM_CLEAN).alias("event_datetime"),
        F.when(cancelled, s.CANCEL_DT_TM_CLEAN)
        .when(ended, s.END_EFFECTIVE_DT_TM).alias("event_end_datetime"),
        source_system.alias("source_coding_system"), source_code.alias("source_code"),
        source_display.alias("source_display"),
        codeable_concept(
            coding_obj(source_system, source_code, source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), s.SUBSTANCE_SNOMED_CODE,
                       source_display, False, "bronze.map_allergy", None),
        ).alias("substance_code"),
        s.SUBSTANCE_TYPE_CD.cast("string").alias("substance_type_code"),
        s.SUBSTANCE_TYPE_DESC.alias("substance_type_display"),
        s.REACTION_CLASS_CD.cast("string").alias("reaction_class_code"),
        s.REACTION_CLASS_DESC.alias("reaction_class_display"),
        s.REACTION_STATUS_CD.cast("string").alias("reaction_status_code"),
        s.REACTION_STATUS_DESC.alias("reaction_status_display"),
        s.SEVERITY_CD.cast("string").alias("severity_code"),
        s.SEVERITY_DESC.alias("severity_display"),
        s.ABSENCE_ASSERTION_IND.cast("boolean").alias("absence_assertion_ind"),
        s.ONSET_PRECISION_DESC.alias("onset_precision_display"),
        F.coalesce(s.SOURCE_OF_INFO_DESC, s.SOURCE_OF_INFO_FT).alias("source_of_info_display"),
        s.VERIFIED_STATUS_FLAG.cast("string").alias("verified_status_flag"),
        s.REVIEWED_DT_TM_CLEAN.alias("reviewed_datetime"),
        s.CANCEL_REASON_DESC.alias("cancel_reason_display"),
        record_status.alias("record_status"),
        F.coalesce(s.REACTION_STATUS_DT_TM_CLEAN, s.ACTIVE_STATUS_DT_TM,
                   s.BEG_EFFECTIVE_DT_TM).alias("record_status_effective_from"),
        F.when(cancelled, s.CANCEL_DT_TM_CLEAN)
        .when(ended, s.END_EFFECTIVE_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.lit(SRC_ALLERGY).alias("_source_table"),
        s.ALLERGY_INSTANCE_ID.cast("string").alias("_source_row_id"),
        s.SUBSTANCE_SNOMED_CODE.cast("string").alias("_snomed_code"),
        source_display.alias("_snomed_display"),
    )

def _allergy_canonical():
    return _allergy_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_ALLERGY = "4_prod.bronze.map_allergy"

@materialized_view(
    name=_n("journey_clinical._qc_allergy_intolerance"),
    comment="Private JSON bridge and Gold cross-rule flags for allergy_intolerance.",
    refresh_policy="incremental",
)
def _qc_allergy_intolerance():
    return _cross_qc_primitive(
        _allergy_canonical(),
        "allergy_intolerance",
        {"substance_code": "_qc_substance_code_json"},
    )

In [0]:
# ==== Further clinical feeds ====

ALLERGY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "substance_code",
    "substance_type_code", "substance_type_display", "reaction_class_code",
    "reaction_class_display", "reaction_status_code", "reaction_status_display",
    "severity_code", "severity_display", "absence_assertion_ind", "onset_precision_display",
    "source_of_info_display", "verified_status_flag", "reviewed_datetime",
    "cancel_reason_display", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "load_batch_id", "source_update_timestamp", "loaded_at",
]

ALLERGY_INTOLERANCE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide event identifier.",
    "fact_row_id": "Storage-row identifier; equal to patient_event_id for this fact.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Recording encounter reference when supplied.",
    "event_datetime": "Sentinel-cleaned onset",
    "event_end_datetime": "Reaction-status end when resolved or cancelled.",
    "source_coding_system": "Verbatim substance code system.",
    "source_code": "Substance code (SNOMED then source identifier then nomenclature id)",
    "source_display": "Verbatim substance display text.",
    "substance_code": "Source and mapped substance codings as a one-level CodeableConcept VARIANT.",
    "substance_type_code": "Source substance type code.",
    "substance_type_display": "Substance type (Drug/Food/Environment/...).",
    "reaction_class_code": "Source reaction class code.",
    "reaction_class_display": "Reaction class (Allergy/Intolerance/Side Effect/...).",
    "reaction_status_code": "Source reaction status code.",
    "reaction_status_display": "Reaction status (Active/Cancelled/Resolved/Proposed).",
    "severity_code": "Source severity code.",
    "severity_display": "Severity (Mild/Moderate/Severe).",
    "absence_assertion_ind": "True when the row asserts ABSENCE of allergy (e.g. no known allergies) rather than a positive assertion.",
    "onset_precision_display": "Source onset precision.",
    "source_of_info_display": "Source-of-information display.",
    "verified_status_flag": "Pharmacy-verified flag verbatim.",
    "reviewed_datetime": "Sentinel-cleaned last review timestamp.",
    "cancel_reason_display": "Cancel reason when cancelled.",
    "record_status": "Normalized silver lifecycle status; Cancelled maps to retracted.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Bronze source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "person_id_resolved": "True when person_id is NULL or resolves to journey_spine.person; published for Gold cross-rule attribution.",
    "encounter_id_resolved": "True when encounter_id is NULL or resolves to journey_spine.encounter; published for Gold cross-rule attribution.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.allergy_intolerance"),
    comment="One Millennium allergy/intolerance assertion with cancel/review lifecycle.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=ALLERGY_INTOLERANCE_COLUMN_COMMENTS,
)
def allergy_intolerance():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_allergy_intolerance")),
        "allergy_intolerance",
        {"substance_code": "_qc_substance_code_json"},
        ALLERGY_PUBLIC_COLUMNS,
    )

In [0]:
def _transfusion_canonical_pregate():
    s = read_source(SRC_BLOODTRACK_TRANSFUSION)
    event_id = stable_id("transfusion:bloodtrack", s.TRANSFUSION_KEY)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)],
        SRC_BLOODTRACK_TRANSFUSION,
        s.TRANSFUSION_KEY,
    )
    inactive = ~F.coalesce(s.IS_CURRENT_IN_SOURCE, F.lit(True))
    source_code = _code_or_display(s.PRODUCT_CODE, s.BLOOD_PRODUCT_GROUP)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        F.coalesce(s.BEGIN_TS, s.END_TS).alias("event_datetime"),
        s.END_TS.alias("event_end_datetime"),
        F.lit("urn:isbt:product-code").alias("source_coding_system"),
        source_code.alias("source_code"), s.BLOOD_PRODUCT_GROUP.alias("source_display"),
        s.BEGIN_TS.alias("begin_datetime"), s.END_TS.alias("end_datetime"),
        s.TRANSFUSION_STATUS.alias("transfusion_status"),
        s.ELAPSED_MINUTES.cast("decimal(38,6)").alias("elapsed_minutes"),
        s.UNIT_NUMBER.alias("unit_number"), s.BLOOD_PRODUCT_GROUP.alias("blood_product_group"),
        s.BLOOD_UNIT_GROUP.alias("blood_unit_group"),
        s.PATIENT_BLOOD_GROUP.alias("patient_blood_group"),
        s.END_QUANTITY_VALUE.cast("decimal(38,6)").alias("quantity_value"),
        s.END_QUANTITY_RAW.alias("quantity_raw"),
        s.BEGIN_LOCATION_NAME.alias("begin_location"),
        s.END_LOCATION_NAME.alias("end_location"),
        s.UNIT_IS_IRRADIATED.alias("unit_is_irradiated"),
        s.UNIT_IS_CMV_NEG.alias("unit_is_cmv_neg"),
        s.REQUIRES_IRRADIATED.alias("requires_irradiated"),
        s.REQUIRES_CMV_NEG.alias("requires_cmv_neg"),
        s.AMBIGUITY_IND.alias("ambiguity_ind"),
        s.PRODUCT_CONCEPT_ID.cast("string").alias("product_concept_id"),
        s.PRODUCT_CONCEPT_NAME.alias("product_concept_name"),
        s.UNIT_GROUP_CONCEPT_ID.cast("string").alias("unit_group_concept_id"),
        s.PATIENT_GROUP_CONCEPT_ID.cast("string").alias("patient_group_concept_id"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        F.coalesce(s.BEGIN_TS, s.END_TS).alias("record_status_effective_from"),
        F.when(inactive, s.PIPELINE_LOADED_AT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.PIPELINE_LOADED_AT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        s.PIPELINE_LOADED_AT.alias("loaded_at"),
        F.lit("bloodtrack").alias("_source_system"),
        F.lit(SRC_BLOODTRACK_TRANSFUSION).alias("_source_table"),
        s.TRANSFUSION_KEY.cast("string").alias("_source_row_id"),
        s.PRODUCT_PROC_CONCEPT_ID.cast("string").alias("_product_proc_concept_id"),
        s.PRODUCT_PROC_CONCEPT_NAME.alias("_product_proc_concept_name"),
        s.UNIT_GROUP_CONCEPT_NAME.alias("_unit_group_concept_name"),
        s.PATIENT_GROUP_CONCEPT_NAME.alias("_patient_group_concept_name"),
    )

def _transfusion_canonical():
    return _transfusion_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_BLOODTRACK_TRANSFUSION = "4_prod.bronze.map_bloodtrack_transfusion"

TRANSFUSION_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "begin_datetime",
    "end_datetime", "transfusion_status", "elapsed_minutes", "unit_number",
    "blood_product_group", "blood_unit_group", "patient_blood_group", "quantity_value",
    "quantity_raw", "begin_location", "end_location", "unit_is_irradiated",
    "unit_is_cmv_neg", "requires_irradiated", "requires_cmv_neg", "ambiguity_ind",
    "product_concept_id", "product_concept_name", "unit_group_concept_id",
    "patient_group_concept_id", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "load_batch_id", "source_update_timestamp", "loaded_at",
]

TRANSFUSION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable transfusion event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Transfusion begin timestamp falling back to end.",
    "event_end_datetime": "Transfusion end timestamp.",
    "source_coding_system": "BloodTrack product coding system.",
    "source_code": "ISBT product code with product display fallback.",
    "source_display": "Blood product description.",
    "begin_datetime": "Paired transfusion begin timestamp.",
    "end_datetime": "Paired transfusion end timestamp.",
    "transfusion_status": "Pairing and clock-quality outcome.",
    "elapsed_minutes": "Elapsed transfusion minutes when calculable.",
    "unit_number": "Blood unit number.",
    "blood_product_group": "Normalized blood product group.",
    "blood_unit_group": "Blood group recorded on the unit.",
    "patient_blood_group": "Patient blood group at transfusion.",
    "quantity_value": "Parsed transfused quantity.",
    "quantity_raw": "Verbatim quantity text.",
    "begin_location": "Begin workflow location.",
    "end_location": "End workflow location.",
    "unit_is_irradiated": "Unit irradiation flag.",
    "unit_is_cmv_neg": "Unit CMV-negative flag.",
    "requires_irradiated": "Patient requires irradiated product.",
    "requires_cmv_neg": "Patient requires CMV-negative product.",
    "ambiguity_ind": "Pairing ambiguity indicator.",
    "product_concept_id": "Mapped SNOMED device concept identifier.",
    "product_concept_name": "Mapped product concept display.",
    "unit_group_concept_id": "Mapped unit blood-group concept identifier.",
    "patient_group_concept_id": "Mapped patient blood-group concept identifier.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.transfusion"),
    comment="One paired BloodTrack unit-recipient transfusion episode.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=TRANSFUSION_COLUMN_COMMENTS,
)
def transfusion():
    return _transfusion_canonical().select(*TRANSFUSION_PUBLIC_COLUMNS)

In [0]:
SRC_BLOODTRACK_TRANSACTION = "4_prod.bronze.map_bloodtrack_transaction"

TRANSFUSION_EVENT_COLUMN_COMMENTS = {
    "transfusion_event_id": "Stable scan-event identifier.",
    "person_id": "Resolved Millennium person identifier.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "identity_status": "Subject resolution state.",
    "event_datetime": "Effective transaction timestamp.",
    "workflow_step": "BloodTrack workflow step.",
    "transaction_success_ind": "Whether the transaction succeeded.",
    "unit_number": "Blood unit number.",
    "product_code": "Product code scanned.",
    "product_description": "Product display.",
    "bloodtrack_unit_id": "BloodTrack unit identifier.",
    "device_name": "Scanning device name.",
    "source_location": "Source workflow location name.",
    "linkage_status": "Patient linkage status.",
    "response_code": "Device or workflow response code.",
    "response_text": "Device or workflow response text.",
    "blood_unit_state": "Blood unit state.",
    "blood_unit_fate": "Blood unit fate.",
    "alert_present_ind": "Alert presence indicator.",
    "comment_present_ind": "Comment presence indicator.",
    "record_status": "Normalized source-record lifecycle.",
    "source_table": "Registered source table.",
    "source_row_id": "Verbatim bronze row identifier.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.transfusion_event"),
    comment="BloodTrack scan-grain workflow evidence including failed and safety-check attempts.",
    refresh_policy="incremental",
    column_comments=TRANSFUSION_EVENT_COLUMN_COMMENTS,
)
def transfusion_event():
    s = read_source(SRC_BLOODTRACK_TRANSACTION)
    event_id = stable_id("transfusion_event:bloodtrack", s.BLOODTRACK_TRANSACTION_KEY)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID),
         ("urn:bloodtrack:source-patient-number", s.SOURCE_PATIENT_NUMBER)],
        SRC_BLOODTRACK_TRANSACTION,
        s.BLOODTRACK_TRANSACTION_KEY,
    )
    inactive = ~F.coalesce(s.IS_CURRENT_IN_SOURCE, F.lit(True))
    return s.select(
        event_id.alias("transfusion_event_id"),
        s.PERSON_ID.cast("string").alias("person_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.SOURCE_PATIENT_NUMBER), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.coalesce(s.EVENT_TS_EFFECTIVE, s.SERVER_TRANSACTION_TS).alias("event_datetime"),
        s.WORKFLOW_STEP.alias("workflow_step"),
        s.TRANSACTION_SUCCESS_IND.alias("transaction_success_ind"),
        s.UNIT_NUMBER.alias("unit_number"), s.PRODUCT_CODE.alias("product_code"),
        s.PRODUCT_DESCRIPTION.alias("product_description"),
        s.BLOODTRACK_UNIT_ID.cast("string").alias("bloodtrack_unit_id"),
        s.DEVICE_NAME.alias("device_name"), s.SOURCE_LOCATION_NAME.alias("source_location"),
        s.LINKAGE_STATUS.alias("linkage_status"), s.RESPONSE_CODE.alias("response_code"),
        s.RESPONSE_TEXT.alias("response_text"), s.BLOOD_UNIT_STATE.alias("blood_unit_state"),
        s.BLOOD_UNIT_FATE.alias("blood_unit_fate"),
        s.ALERT_PRESENT_IND.alias("alert_present_ind"),
        s.COMMENT_PRESENT_IND.alias("comment_present_ind"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        F.lit(SRC_BLOODTRACK_TRANSACTION).alias("source_table"),
        s.BLOODTRACK_TRANSACTION_KEY.cast("string").alias("source_row_id"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
def _cancer_treatment_canonical_pregate():
    s = read_source(SRC_CANCER_TREATMENT)
    event_id = stable_id("cancer_treatment:sact", s.TREATMENT_KEY)
    source_display = F.coalesce(s.AriaAgentName, s.IqemoSactName)
    source_code = _code_or_display(s.drug, source_display)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:mrn", s.MRN),
         ("https://fhir.nhs.uk/Id/nhs-number", s.NHS_Number)],
        SRC_CANCER_TREATMENT,
        s.TREATMENT_KEY,
    )
    inactive = ~F.coalesce(s.SOURCE_PRESENT_IND.cast("boolean"), F.lit(True))
    normalized_record_type = (
        F.when(s.record_type.rlike("(?i)matched"), F.lit("matched"))
        .when(s.record_type.rlike("(?i)aria"), F.lit("aria_only"))
        .otherwise(F.lit("iqemo_only"))
    )
    event_time = s.start_date.cast("timestamp")
    end_time = F.coalesce(s.EndDate, s.FinalTreatmentDate).cast("timestamp")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.MRN) | _present(s.NHS_Number), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        event_time.alias("event_datetime"), end_time.alias("event_end_datetime"),
        F.lit("urn:barts:sact:drug-token").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("urn:barts:sact:drug-token"), source_code, source_display, True),
            coding_obj(F.lit("urn:omop:concept_id"), s.drug_concept_id,
                       s.drug_concept_name, False,
                       "lookup.cancer_treatment_term_map:applied", None),
        ).alias("drug_code"),
        s.TreatmentPlan.alias("treatment_plan"), s.RegimenName.alias("regimen_name"),
        s.Indication.alias("indication"), normalized_record_type.alias("record_type"),
        s.RxDose.cast("decimal(38,6)").alias("dose_value"),
        s.RxTotal.cast("decimal(38,6)").alias("dose_total"),
        s.AdmnDosageUnit.cast("string").alias("dose_unit_code"),
        s.AdmnRoute.cast("string").alias("route_code"),
        event_time.alias("start_date"), s.EndDate.cast("timestamp").alias("end_date"),
        s.FinalTreatmentDate.cast("timestamp").alias("final_treatment_date"),
        s.CourseFinished.cast("boolean").alias("course_finished"),
        s.PlannedCycles.cast("long").alias("planned_cycles"),
        s.DefaultCycles.cast("long").alias("default_cycles"),
        s.ChemoRadiation.cast("boolean").alias("chemo_radiation"),
        s.OPCSProcurementCode.alias("procurement_opcs_code"),
        s.OPCSDeliveryCode.alias("delivery_opcs_code"),
        s.drug_similarity.cast("decimal(38,6)").alias("drug_similarity"),
        s.iqemo_chemotherapy_course_id.cast("string").alias("iqemo_course_id"),
        F.concat_ws(":", s.aria_pt_id, s.aria_rx_id.cast("string"),
                    s.aria_item_no.cast("string")).alias("aria_rx_key"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.when(inactive, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SRC_ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("sact").alias("_source_system"),
        F.lit(SRC_CANCER_TREATMENT).alias("_source_table"),
        s.TREATMENT_KEY.cast("string").alias("_source_row_id"),
        s.drug_concept_id.cast("string").alias("_drug_omop_code"),
        s.drug_concept_name.alias("_drug_omop_display"),
        s.procurement_snomed_concept_id.cast("string").alias("_procurement_omop_code"),
        s.procurement_snomed_concept_id.cast("string").alias("_procurement_omop_display"),
        s.OPCSProcurementCode.alias("_procurement_opcs4_code"),
        s.OPCSProcurementCode.alias("_procurement_opcs4_display"),
        s.delivery_snomed_concept_id.cast("string").alias("_delivery_omop_code"),
        s.delivery_snomed_concept_id.cast("string").alias("_delivery_omop_display"),
        s.OPCSDeliveryCode.alias("_delivery_opcs4_code"),
        s.OPCSDeliveryCode.alias("_delivery_opcs4_display"),
    )

def _cancer_treatment_canonical():
    return _cancer_treatment_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_CANCER_TREATMENT = "4_prod.bronze.map_cancer_treatment"

@materialized_view(
    name=_n("journey_clinical._qc_cancer_treatment"),
    comment="Private JSON bridge and Gold cross-rule flags for cancer_treatment.",
    refresh_policy="incremental",
)
def _qc_cancer_treatment():
    return _cross_qc_primitive(
        _cancer_treatment_canonical(),
        "cancer_treatment",
        {"drug_code": "_qc_drug_code_json"},
    )

In [0]:
CANCER_TREATMENT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "drug_code",
    "treatment_plan", "regimen_name", "indication", "record_type", "dose_value",
    "dose_total", "dose_unit_code", "route_code", "start_date", "end_date",
    "final_treatment_date", "course_finished", "planned_cycles", "default_cycles",
    "chemo_radiation", "procurement_opcs_code", "delivery_opcs_code", "drug_similarity",
    "iqemo_course_id", "aria_rx_key", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "load_batch_id", "source_update_timestamp", "loaded_at",
]

CANCER_TREATMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable treatment event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Treatment start timestamp.",
    "event_end_datetime": "Treatment end or final-treatment timestamp.",
    "source_coding_system": "SACT drug-token coding system.",
    "source_code": "Normalized drug token with agent-name display fallback.",
    "source_display": "Source agent name.",
    "drug_code": "Source and mapped drug codings.",
    "treatment_plan": "Source treatment plan.",
    "regimen_name": "Treatment regimen name.",
    "indication": "Treatment indication.",
    "record_type": "ARIA/iQemo linkage class.",
    "dose_value": "Constituent dose value.",
    "dose_total": "Total planned or delivered dose.",
    "dose_unit_code": "Dose unit code.",
    "route_code": "Administration route code.",
    "start_date": "Source start date.",
    "end_date": "Source end date.",
    "final_treatment_date": "Final treatment date.",
    "course_finished": "Course-finished indicator.",
    "planned_cycles": "Planned cycle count.",
    "default_cycles": "Default regimen cycle count.",
    "chemo_radiation": "Concurrent chemo-radiation indicator.",
    "procurement_opcs_code": "OPCS procurement code.",
    "delivery_opcs_code": "OPCS delivery code.",
    "drug_similarity": "Drug-link similarity score.",
    "iqemo_course_id": "iQemo course identifier.",
    "aria_rx_key": "ARIA prescription key.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.cancer_treatment"),
    comment="One SACT constituent-drug treatment fact, separate from Millennium medication administration.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CANCER_TREATMENT_COLUMN_COMMENTS,
)
def cancer_treatment():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_cancer_treatment")),
        "cancer_treatment",
        {"drug_code": "_qc_drug_code_json"},
        CANCER_TREATMENT_PUBLIC_COLUMNS,
    )

In [0]:
SRC_CANCER_TREATMENT_CYCLE = "4_prod.bronze.map_cancer_treatment_cycle"

CANCER_TREATMENT_CYCLE_COLUMN_COMMENTS = {
    "cancer_treatment_cycle_id": "Stable composite cycle identifier.",
    "iqemo_course_id": "iQemo chemotherapy course identifier.",
    "cycle_sequence_id": "Verbatim treatment-cycle sequence token.",
    "regimen_cycle_id": "Regimen cycle identifier.",
    "cycle_code": "Source cycle code.",
    "prescribed_datetime": "Sentinel-cleaned prescribed timestamp.",
    "pharmacy_confirmed_datetime": "Sentinel-cleaned pharmacy-confirmed timestamp.",
    "cycle_start_datetime": "Sentinel-cleaned cycle start timestamp.",
    "cancellation_datetime": "Sentinel-cleaned cancellation timestamp.",
    "cycle_status_code": "Source cycle status code.",
    "treatment_response_id": "Treatment-response identifier.",
    "line_of_treatment": "Line of treatment.",
    "regimen_number": "Regimen number.",
    "course_link_status": "Parent course-link status.",
    "outcome_comments": "Source outcome comments.",
    "record_status": "Reference-row lifecycle.",
    "source_table": "Registered source table.",
    "source_row_id": "Composite source row identifier.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.cancer_treatment_cycle"),
    comment="One iQemo chemotherapy cycle child reference.",
    refresh_policy="incremental",
    column_comments=CANCER_TREATMENT_CYCLE_COLUMN_COMMENTS,
)
def cancer_treatment_cycle():
    s = read_source(SRC_CANCER_TREATMENT_CYCLE)
    row_key = F.concat_ws(":", s.CHEMOTHERAPY_COURSE_ID.cast("string"),
                          s.TREATMENT_CYCLE_ID.cast("string"))
    return s.select(
        stable_id("cancer_cycle:iqemo", s.CHEMOTHERAPY_COURSE_ID,
                  s.TREATMENT_CYCLE_ID).alias("cancer_treatment_cycle_id"),
        s.CHEMOTHERAPY_COURSE_ID.cast("string").alias("iqemo_course_id"),
        s.TREATMENT_CYCLE_ID.cast("string").alias("cycle_sequence_id"),
        s.RegimenCycleID.cast("string").alias("regimen_cycle_id"),
        s.TreatmentCycleCode.alias("cycle_code"),
        s.PRESCRIBED_DATE_CLEAN.alias("prescribed_datetime"),
        s.PHARMACY_CONFIRMED_DATE_CLEAN.alias("pharmacy_confirmed_datetime"),
        s.START_DATE_CLEAN.alias("cycle_start_datetime"),
        s.CANCELLATION_DATE_CLEAN.alias("cancellation_datetime"),
        s.CycleStatus.cast("string").alias("cycle_status_code"),
        s.TreatmentResponseID.cast("string").alias("treatment_response_id"),
        s.LINE_OF_TREATMENT_ID.cast("string").alias("line_of_treatment"),
        s.REGIMEN_NUMBER.cast("string").alias("regimen_number"),
        s.MAP_CANCER_TREATMENT_LINK_STATUS.alias("course_link_status"),
        s.OutcomeComments.alias("outcome_comments"),
        F.lit("active").alias("record_status"),
        F.lit(SRC_CANCER_TREATMENT_CYCLE).alias("source_table"),
        row_key.alias("source_row_id"),
        F.date_format(s.PIPELINE_UPDT_DT_TM, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("loaded_at"),
    )

In [0]:
def _condition_stage_canonical_pregate():
    s = read_source(SRC_ARIA_STAGING)
    event_id = stable_id("condition_stage:aria", s.ARIA_PT_ID, s.ARIA_DX_ID)
    source_display = F.coalesce(s.DX_NAME, s.DX_DESC)
    source_code = _code_or_display(s.ICD_CODE, source_display)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:aria:mrn", s.ARIA_MRN)],
        SRC_ARIA_STAGING,
        F.concat_ws(":", s.ARIA_PT_ID, s.ARIA_DX_ID.cast("string")),
    )
    inactive = (
        (F.coalesce(s.CUR_ENTRY_IND, F.lit("Y")) != F.lit("Y"))
        | ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.ARIA_MRN), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.ONSET_DATE_CLEAN.cast("timestamp").alias("event_datetime"),
        s.RESOLUTION_DATE_CLEAN.cast("timestamp").alias("event_end_datetime"),
        F.lit("http://hl7.org/fhir/sid/icd-10").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept(
            coding_obj(F.lit("http://hl7.org/fhir/sid/icd-10"), source_code,
                       source_display, True),
            coding_obj(F.lit("urn:omop:concept_id"), s.DIAGNOSIS_CONCEPT_ID,
                       s.DIAGNOSIS_CONCEPT_NAME, False,
                       "bronze.map_aria_diagnosis_staging", None),
        ).alias("stage_code"),
        s.STAGE_OF_DISEASE.alias("stage_of_disease"), s.STG_CRIT_DESC.alias("stage_criteria"),
        s.DX_TYP.alias("dx_type"), s.CONFIRM_DX.alias("dx_confirmed"),
        s.MTHD_OF_DX.alias("dx_method"),
        (F.upper(F.coalesce(s.HX_OF_IND, F.lit("N"))) == "Y").alias("history_ind"),
        (F.upper(F.coalesce(s.CUR_ENTRY_IND, F.lit("N"))) == "Y")
        .alias("current_entry_ind"),
        (F.upper(F.coalesce(s.CS_OF_DTH_IND, F.lit("N"))) == "Y")
        .alias("cause_of_death_ind"),
        s.ONSET_DATE_CLEAN.cast("timestamp").alias("onset_datetime"),
        s.RESOLUTION_DATE_CLEAN.cast("timestamp").alias("resolution_datetime"),
        s.CLINICAL_DESC.alias("clinical_description"), s.DX_CMT.alias("dx_comment"),
        s.PERSON_LINK_STATUS.alias("person_link_status"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        s.EVLV_TSTAMP_CLEAN.alias("record_status_effective_from"),
        F.when(inactive, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("aria").alias("_source_system"), F.lit(SRC_ARIA_STAGING).alias("_source_table"),
        F.concat_ws(":", s.ARIA_PT_ID, s.ARIA_DX_ID.cast("string")).alias("_source_row_id"),
        s.DIAGNOSIS_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.DIAGNOSIS_CONCEPT_NAME.alias("_omop_display"),
    )

def _condition_stage_canonical():
    return _condition_stage_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_ARIA_STAGING = "4_prod.bronze.map_aria_diagnosis_staging"

@materialized_view(
    name=_n("journey_clinical._qc_condition_stage"),
    comment="Private JSON bridge and Gold cross-rule flags for condition_stage.",
    refresh_policy="incremental",
)
def _qc_condition_stage():
    return _cross_qc_primitive(
        _condition_stage_canonical(),
        "condition_stage",
        {"stage_code": "_qc_stage_code_json"},
    )

In [0]:
CONDITION_STAGE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "stage_code",
    "stage_of_disease", "stage_criteria", "dx_type", "dx_confirmed", "dx_method",
    "history_ind", "current_entry_ind", "cause_of_death_ind", "onset_datetime",
    "resolution_datetime", "clinical_description", "dx_comment", "person_link_status",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

CONDITION_STAGE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable condition-stage event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Sentinel-cleaned diagnosis onset timestamp.",
    "event_end_datetime": "Sentinel-cleaned resolution timestamp.",
    "source_coding_system": "ICD-10 coding system.",
    "source_code": "ICD diagnosis code with diagnosis-name fallback.",
    "source_display": "Diagnosis name or description.",
    "stage_code": "Source ICD and mapped OMOP diagnosis codings.",
    "stage_of_disease": "Source disease stage.",
    "stage_criteria": "Staging criteria description.",
    "dx_type": "Diagnosis type.",
    "dx_confirmed": "Diagnosis confirmation state.",
    "dx_method": "Diagnosis method.",
    "history_ind": "History indicator.",
    "current_entry_ind": "Current-entry indicator.",
    "cause_of_death_ind": "Cause-of-death indicator.",
    "onset_datetime": "Source onset timestamp.",
    "resolution_datetime": "Source resolution timestamp.",
    "clinical_description": "Clinical diagnosis description.",
    "dx_comment": "Diagnosis comment.",
    "person_link_status": "Bronze person-link status.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.condition_stage"),
    comment="One frozen ARIA diagnosis-and-staging assertion.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CONDITION_STAGE_COLUMN_COMMENTS,
)
def condition_stage():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_condition_stage")),
        "condition_stage",
        {"stage_code": "_qc_stage_code_json"},
        CONDITION_STAGE_PUBLIC_COLUMNS,
    )

In [0]:
SRC_ENDOBASE_EXAM = "4_prod.bronze.map_endobase_exam"
SRC_ENDOBASE_EXAM_TERM = "4_prod.bronze.map_endobase_exam_term"

def _endoscopy_finding_canonical_pregate():
    s, t, event_time, event_time_source = _endobase_term_base()
    s = s.where(~F.coalesce(t.FREE_TEXT_IND, F.lit(False)))
    event_id = stable_id("endoscopy_finding:endobase", t.ENDOBASE_EXAM_TERM_ID)
    source_code = t._finding_source_code
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", t.PERSON_ID)],
        SRC_ENDOBASE_EXAM_TERM,
        t.ENDOBASE_EXAM_TERM_ID,
    )
    inactive = ~F.coalesce(t.SOURCE_PRESENT_IND, F.lit(True))
    loaded_at = F.greatest(t.ADC_UPDT, F.coalesce(F.col("x._x_loaded_at"), t.ADC_UPDT))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        t.PERSON_ID.cast("string").alias("person_id"),
        F.when(t.PERSON_ID.isNotNull(), F.lit("resolved"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(F.col("x._x_encntr_id").isNotNull(),
               stable_id("encounter:mill", F.col("x._x_encntr_id"))).alias("encounter_id"),
        event_time.alias("event_datetime"), F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:endobase:dgvs-term").alias("source_coding_system"),
        source_code.alias("source_code"), t.TERM_TEXT.alias("source_display"),
        t._finding_code_json.alias("_finding_code_json"),
        F.when(t.ENDOBASE_EXAM_ID.isNotNull(),
               stable_id("procedure:endobase_exam", t.ENDOBASE_EXAM_ID))
        .alias("endoscopy_exam_event_id"),
        t.SECTION_TAB_ID.cast("string").alias("section_id"),
        t.SUBSECTION_TAB_ID.cast("string").alias("subsection_id"),
        t.PARENT_EXAM_TERM_ID.cast("string").alias("parent_term_id"),
        t.DISPLAY_ORDER.cast("long").alias("display_order"),
        t.CONFIRMED_IND.alias("confirmed_ind"), t.TEXT_CHANGED_IND.alias("text_changed_ind"),
        t.FREE_TEXT_IND.alias("free_text_ind"),
        t.TERM_MAPPING_STATUS.alias("term_mapping_status"),
        t.PERSON_LINK_STATUS.alias("person_link_status"), t.CREATED_TS.alias("authored_datetime"),
        event_time_source.alias("event_time_source"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        t.CREATED_TS.alias("record_status_effective_from"),
        F.when(inactive, t.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(loaded_at, "yyyyMMddHHmmss").alias("load_batch_id"),
        t.ADC_UPDT.alias("source_update_timestamp"), loaded_at.alias("loaded_at"),
        F.lit("endobase").alias("_source_system"),
        F.lit(SRC_ENDOBASE_EXAM_TERM).alias("_source_table"),
        t.ENDOBASE_EXAM_TERM_ID.cast("string").alias("_source_row_id"),
        t.SNOMED_CODE.cast("string").alias("_snomed_code"),
        t.TERM_TEXT.alias("_snomed_display"),
        t.OMOP_CONCEPT_ID.cast("string").alias("_omop_code"),
        t.TERM_TEXT.alias("_omop_display"),
    )

def _endoscopy_finding_canonical():
    return _endoscopy_finding_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
def _endobase_term_base():
    t0 = read_source(SRC_ENDOBASE_EXAM_TERM)
    term_source_code = _code_or_display(t0.DGVS_TERM_ID.cast("string"), t0.TERM_TEXT)
    source_coding = coding_obj(F.lit("urn:endobase:dgvs-term"), term_source_code,
                               t0.TERM_TEXT, True)
    snomed_coding = coding_obj(F.lit("http://snomed.info/sct"), t0.SNOMED_CODE,
                               t0.TERM_TEXT, False,
                               "bronze.map_endobase_exam_term:c4", None)
    omop_coding = coding_obj(F.lit("urn:omop:concept_id"), t0.OMOP_CONCEPT_ID,
                             t0.TERM_TEXT, False,
                             "bronze.map_endobase_exam_term:c4", None)
    finding_code_json = codeable_concept_json(source_coding, snomed_coding, omop_coding)
    t = (
        t0.withColumn("_finding_source_code", term_source_code)
        .withColumn("_finding_code_json", finding_code_json)
        .alias("t")
    )
    x = read_source(SRC_ENDOBASE_EXAM).select(
        F.col("ENDOBASE_EXAM_ID").alias("_x_exam_id"),
        F.coalesce(F.col("PERFORMED_TS_CLEAN"), F.col("EXAM_TS_CLEAN"),
                   F.col("TRUE_START_TS_CLEAN"), F.col("START_TS_CLEAN"))
        .alias("_x_exam_ts"),
        F.col("MILL_ENCNTR_ID").alias("_x_encntr_id"),
        F.col("ADC_UPDT").alias("_x_loaded_at"),
    ).alias("x")
    s = t.join(x, t.ENDOBASE_EXAM_ID == F.col("x._x_exam_id"), "left")
    event_time = F.coalesce(F.col("x._x_exam_ts"), t.CREATED_TS)
    event_time_source = F.when(F.col("x._x_exam_ts").isNotNull(), F.lit("exam")) \
        .otherwise(F.lit("authored"))
    return s, t, event_time, event_time_source

@materialized_view(
    name=_n("journey_clinical._endoscopy_finding_stage"),
    private=True,
    comment="Incremental boundary for joined Endobase finding scalars and deterministic coding JSON.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
)
def _endoscopy_finding_stage():
    return _endoscopy_finding_canonical()

In [0]:
ENDOSCOPY_FINDING_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "finding_code",
    "endoscopy_exam_event_id", "section_id", "subsection_id", "parent_term_id",
    "display_order", "confirmed_ind", "text_changed_ind", "free_text_ind",
    "term_mapping_status", "person_link_status", "authored_datetime", "event_time_source",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

ENDOSCOPY_FINDING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable endoscopy-finding identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Parent exam encounter reference.",
    "event_datetime": "Parent exam time",
    "event_end_datetime": "Finding end timestamp when supplied.",
    "source_coding_system": "DGVS term coding system.",
    "source_code": "DGVS term identifier with term-text fallback.",
    "source_display": "Endobase term text.",
    "finding_code": "Source DGVS and mapped SNOMED/OMOP codings.",
    "endoscopy_exam_event_id": "Parent Endobase procedure event identifier.",
    "section_id": "Report section identifier.",
    "subsection_id": "Report subsection identifier.",
    "parent_term_id": "Parent term identifier.",
    "display_order": "Source display order.",
    "confirmed_ind": "Finding confirmation indicator.",
    "text_changed_ind": "Source text-changed indicator.",
    "free_text_ind": "Free-text route indicator.",
    "term_mapping_status": "C4 term-mapping status.",
    "person_link_status": "Bronze person-link status.",
    "authored_datetime": "Report-authoring timestamp.",
    "event_time_source": "Whether event time came from the exam or authored fallback.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@materialized_view(
    name=_n("journey_clinical.endoscopy_finding"),
    comment="One coded Endobase report term with parent-exam clinical time.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=ENDOSCOPY_FINDING_COLUMN_COMMENTS,
)
def endoscopy_finding():
    return (
        spark.read.table(_n("journey_clinical._endoscopy_finding_stage"))
        .withColumn("finding_code", F.parse_json(F.col("_finding_code_json")))
        .select(*ENDOSCOPY_FINDING_PUBLIC_COLUMNS)
    )

In [0]:
def _device_canonical_pregate():
    d = read_source(SRC_MEDICONNECT_DEVICE)
    m = read_source(SRC_MEDICONNECT_TYPE_MAP).select(
        F.col("DEVICE_TYPE").cast("string").alias("_m_device_type"),
        F.col("DEVICE_ROLE").alias("_m_device_role"),
        F.col("SNOMED_CONCEPT_ID").cast("string").alias("_m_snomed_code"),
        F.col("SNOMED_CONCEPT_NAME").alias("_m_snomed_display"),
        F.col("MAPPING_STATUS").alias("_m_mapping_status"),
    )
    s = d.join(m, d.DEVICE_TYPE.cast("string") == m._m_device_type, "left")
    event_id = stable_id("device:mediconnect", s.MC_DEVICE_ID)
    raw_type = F.when(s.DEVICE_TYPE.cast("long") != 0, s.DEVICE_TYPE.cast("string"))
    source_display = F.coalesce(s.TYPE, s.MODEL_NAME)
    source_code = _code_or_display(raw_type, source_display)
    snomed_code = F.coalesce(s.DEVICE_SNOMED_CONCEPT_ID.cast("string"), s._m_snomed_code)
    snomed_display = F.coalesce(s.DEVICE_SNOMED_CONCEPT_NAME, s._m_snomed_display)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:mediconnect:patient-id", s.PATIENTID)],
        SRC_MEDICONNECT_DEVICE,
        s.MC_DEVICE_ID,
    )
    inactive = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    implant_time = s.IMPLANTED_DATE_CLEAN.cast("timestamp")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.PATIENTID), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        implant_time.alias("event_datetime"), F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:barts:mediconnect:device-type").alias("source_coding_system"),
        source_code.alias("source_code"), source_display.alias("source_display"),
        codeable_concept_json(
            coding_obj(F.lit("urn:barts:mediconnect:device-type"), source_code,
                       source_display, True),
            coding_obj(F.lit("http://snomed.info/sct"), snomed_code,
                       snomed_display, False,
                       "lookup.mediconnect_device_type_map:applied", None),
        ).alias("_device_code_json"),
        F.coalesce(s.DEVICE_ROLE, s._m_device_role).alias("device_role"),
        s.MODEL_NAME.alias("model_name"),
        s.MODEL_CODE.alias("model_code"),
        F.coalesce(s.MANUFACTURER_CLEAN, s.MANUFACTURER).alias("manufacturer"),
        s.MANUFACTURER_PARENT.alias("manufacturer_parent"), s.SERIAL_NO.alias("serial_number"),
        implant_time.alias("implanted_datetime"),
        s.IMPLANTED_DATE_QUALITY.alias("implant_date_quality"),
        s.EXPLANTED_IND.alias("explanted_ind"), s.LEAD_CHAMBER.alias("lead_chamber"),
        s.LEAD_LOCATION.alias("lead_location"), s.POCKET_SITE.alias("pocket_site"),
        s.STATUS.alias("status_display"),
        F.coalesce(s.DEVICE_MAPPING_STATUS, s._m_mapping_status).alias("device_mapping_status"),
        s.COMMENT.alias("comment_text"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        implant_time.alias("record_status_effective_from"),
        F.when(inactive, F.coalesce(s.SOURCE_ABSENT_DETECTED_TS, s.ADC_UPDT))
        .alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("mediconnect").alias("_source_system"),
        F.lit(SRC_MEDICONNECT_DEVICE).alias("_source_table"),
        s.MC_DEVICE_ID.cast("string").alias("_source_row_id"),
        snomed_code.alias("_snomed_code"), snomed_display.alias("_snomed_display"),
    )

def _device_canonical():
    return _device_canonical_pregate().where(_usable_code(F.col("source_code")))

def _registry_entry_canonical():
    lanes = [_registry_entry_lane(*cfg) for cfg in IWEB_REGISTRY_CONFIG]
    out = lanes[0]
    for lane_df in lanes[1:]:
        out = out.unionByName(lane_df)
    return out

In [0]:
SRC_MEDICONNECT_DEVICE = "4_prod.bronze.mediconnect_device"

IWEB_REGISTRY_SLOTS = {
    "acs_transfer": "4_prod.bronze.iweb_acs_transfer",
    "cardiac_mdt": "4_prod.bronze.iweb_cardiac_mdt",
    "mortality_review": "4_prod.bronze.iweb_cardiac_mortality_review",
    "surgery_episode": "4_prod.bronze.iweb_cardiac_surgery_episode",
    "surgery_followup": "4_prod.bronze.iweb_cardiac_surgery_followup",
    "surgery_procedure": "4_prod.bronze.iweb_cardiac_surgery_procedure",
    "coronary_procedure": "4_prod.bronze.iweb_coronary_procedure",
    "coronary_lesion": "4_prod.bronze.iweb_coronary_lesion",
    "eracs_episode": "4_prod.bronze.iweb_eracs_episode",
    "noncoronary_procedure": "4_prod.bronze.iweb_noncoronary_procedure",
}

def _iweb_field(s, field, module):
    # iWeb bronze nests module fields into typed structs (2026-08-21 restructure of
    # iweb_cardiac_surgery_episode); dev fixtures predate it and still carry them at the
    # top level. Resolve either shape and fail closed if the field vanishes entirely.
    if field in s.columns:
        return s[field]
    if module in s.columns and field in s.schema[module].dataType.fieldNames():
        return s[module][field]
    raise KeyError(f"iweb field {field} absent at top level and inside {module}")

IWEB_REGISTRY_CONFIG = [
    ("acs_transfer", "acs_transfer",
     lambda s: F.coalesce(s.ARRIVE_HERE, s.ADMISSIONDATE), lambda s: s.DATE_OF_DISCHARGE,
     None, None, lambda s: s.DATE_OF_DEATH, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("cardiac_mdt", "cardiac_mdt",
     lambda s: s.MD_TDATE, lambda s: F.lit(None).cast("timestamp"),
     None, lambda s: s.REGISTRY_TYPE, lambda s: s.DATE_OF_DEATH, None,
     lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("mortality_review", "mortality_review",
     lambda s: F.coalesce(s.DATE_AND_TIME_OF_DEATH, s.DATE_OF_DEATH_CLEAN.cast("timestamp")),
     lambda s: F.lit(None).cast("timestamp"), None, None,
     lambda s: s.DATE_OF_DEATH_DEMOG, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("surgery_episode", "surgery_episode",
     lambda s: F.coalesce(s.DATE_AND_TIME_OF_OPERATION, s.DATE_OF_OPERATION),
     lambda s: _iweb_field(s, "DATE_OF_DISCHARGE_OR_DEATH", "POST1_MODULE"),
     None, None, lambda s: s.DATE_OF_DEATH,
     None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("surgery_followup", "surgery_followup",
     lambda s: s.DISCHARGE_DT, lambda s: F.lit(None).cast("timestamp"),
     None, None, lambda s: s.DATE_OF_DEATH, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("surgery_procedure", "surgery_procedure",
     None, lambda s: F.lit(None).cast("timestamp"),
     ("surgery_episode", "PARENT_ENTRY_ID", "DATE_AND_TIME_OF_OPERATION"),
     None, lambda s: s.DATE_OF_DEATH, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("coronary_procedure", "coronary_procedure",
     lambda s: s.PRE_DATEANDTIMEOFOPERATION_CLEAN,
     lambda s: s.POST_DATE_OF_DISCHARGE_TS_CLEAN,
     None, None, lambda s: s.PRE_DATE_OF_DEATH_CLEAN, lambda s: s.NHS_NUMBER,
     lambda s: F.lit(None).cast("timestamp")),
    ("coronary_lesion", "coronary_lesion",
     None, lambda s: F.lit(None).cast("timestamp"),
     ("coronary_procedure", "PARENT_ENTRY_ID", "PRE_DATEANDTIMEOFOPERATION_CLEAN"),
     None, lambda s: s.DATE_OF_DEATH, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("eracs_episode", "eracs_episode",
     lambda s: F.coalesce(s.ADMISSION_DT_TM, s.DATE_OF_ENTRY),
     lambda s: s.ACTUAL_DISCHARGE_DT_TM, None, None, lambda s: s.DATE_OF_DEATH,
     None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
    ("noncoronary_procedure", "noncoronary_procedure",
     lambda s: s.DATEOFPROCEDURE, lambda s: s.DATE_OF_DISCHARGE_DEATH,
     None, None, lambda s: s.DATE_OF_DEATH, None, lambda s: s.SOURCE_ABSENT_DETECTED_TS),
]

def _registry_entry_lane(family, slot_key, event_fn, end_fn, parent, rtype_fn, dod_fn,
                         nhs_fn, absent_ts_fn):
    base = read_source(IWEB_REGISTRY_SLOTS[slot_key])
    source_columns = list(base.columns)
    base = base.withColumn(
        "_registry_payload_json",
        F.to_json(F.struct(*[base[c] for c in source_columns])),
    )
    s = base
    registry_type = rtype_fn(s) if rtype_fn else F.lit(None).cast("string")
    row_key = F.concat_ws(":", F.coalesce(registry_type, F.lit("~")),
                          s.ENTRY_ID.cast("string"))
    event_id = stable_id("registry_entry:iweb", F.lit(family),
                         F.coalesce(registry_type, F.lit("~")), s.ENTRY_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:mrn", s.MRN)],
        IWEB_REGISTRY_SLOTS[slot_key], row_key,
    )
    if parent:
        parent_family, parent_fk, parent_event_col = parent
        p = read_source(IWEB_REGISTRY_SLOTS[parent_family]).select(
            F.col("ENTRY_ID").alias("_p_entry_id"),
            F.col(parent_event_col).alias("_p_event_ts"),
        )
        s = s.join(p, s[parent_fk] == F.col("_p_entry_id"), "left")
        parent_id = stable_id("registry_entry:iweb", F.lit(parent_family),
                              F.lit("~"), s[parent_fk])
        event_expr = F.col("_p_event_ts")
    else:
        parent_id = F.lit(None).cast("string")
        event_expr = event_fn(s)
    nhs_expr = nhs_fn(s) if nhs_fn else F.lit(None).cast("string")
    inactive = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
        .when(_present(s.MRN) | _present(nhs_expr), F.lit("provisional"))
        .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        event_expr.cast("timestamp").alias("event_datetime"),
        end_fn(s).cast("timestamp").alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        F.lit(family).alias("registry_family"), registry_type.alias("registry_type"),
        s.ENTRY_ID.cast("string").alias("entry_id"),
        parent_id.alias("parent_registry_entry_id"), s.LINKAGE_STATUS.alias("linkage_status"),
        s.MRN.alias("mrn"), nhs_expr.alias("nhs_number"),
        dod_fn(s).cast("timestamp").alias("date_of_death"),
        s._registry_payload_json.alias("_registry_payload_json"),
        s.DATE_LAST_CHANGED.alias("source_edit_datetime"),
        F.when(inactive, F.lit("superseded")).otherwise(F.lit("active"))
        .alias("record_status"),
        F.lit(None).cast("timestamp").alias("record_status_effective_from"),
        F.when(inactive, absent_ts_fn(s)).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.DATE_LAST_CHANGED.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("iweb").alias("_source_system"),
        F.lit(IWEB_REGISTRY_SLOTS[slot_key]).alias("_source_table"), row_key.alias("_source_row_id"),
    )

@materialized_view(
    name=_n("journey_clinical._registry_device_stage"),
    private=True,
    comment="Incremental boundary for joined registry/device scalars and deterministic VARIANT JSON.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
)
def _registry_device_stage():
    registry = _registry_entry_canonical().withColumn("_stage_kind", F.lit("registry"))
    device = _device_canonical().withColumn("_stage_kind", F.lit("device"))
    return registry.unionByName(device, allowMissingColumns=True)

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_registry_entry"),
    comment="Private JSON bridge and Gold cross-rule flags for registry_entry.",
    refresh_policy="incremental",
)
def _qc_registry_entry():
    source = spark.read.table(_n("journey_clinical._registry_device_stage")).where(
        F.col("_stage_kind") == "registry"
    )
    return _cross_qc_primitive(
        source,
        "registry_entry",
        {"registry_payload": "_registry_payload_json"},
    )

In [0]:
REGISTRY_ENTRY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "registry_family", "registry_type", "entry_id", "parent_registry_entry_id",
    "linkage_status", "mrn", "nhs_number", "date_of_death", "registry_payload",
    "source_edit_datetime", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "load_batch_id", "source_update_timestamp", "loaded_at",
]

REGISTRY_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable registry event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Family-specific clinical timestamp; child rows inherit the parent timestamp.",
    "event_end_datetime": "Family-specific clinical end timestamp.",
    "registry_family": "Parameterized registry family.",
    "registry_type": "Registry subtype for composite-key MDT rows.",
    "entry_id": "Verbatim iWeb entry identifier.",
    "parent_registry_entry_id": "Stable parent registry event identifier for child lanes.",
    "linkage_status": "Bronze person-link status.",
    "mrn": "Source MRN where supplied.",
    "nhs_number": "Source NHS number where supplied.",
    "date_of_death": "Family-specific demographic date of death.",
    "registry_payload": "Full verbatim source row as key/value pairs.",
    "source_edit_datetime": "Source edit timestamp; never treated as clinical time.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Source edit timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.registry_entry"),
    comment="Parameterized iWeb cardiac-registry entries across ten source families.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=REGISTRY_ENTRY_COLUMN_COMMENTS,
)
def registry_entry():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_registry_entry")),
        "registry_entry",
        {"registry_payload": "_registry_payload_json"},
        REGISTRY_ENTRY_PUBLIC_COLUMNS,
    )

In [0]:
@materialized_view(
    name=_n("journey_clinical._qc_device"),
    comment="Private JSON bridge and Gold cross-rule flags for device.",
    refresh_policy="incremental",
)
def _qc_device():
    source = spark.read.table(_n("journey_clinical._registry_device_stage")).where(
        F.col("_stage_kind") == "device"
    )
    return _cross_qc_primitive(
        source,
        "device",
        {"device_code": "_device_code_json"},
    )

In [0]:
DEVICE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "device_code", "device_role",
    "model_name", "model_code", "manufacturer", "manufacturer_parent", "serial_number",
    "implanted_datetime", "implant_date_quality", "explanted_ind", "lead_chamber",
    "lead_location", "pocket_site", "status_display", "device_mapping_status",
    "comment_text", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "load_batch_id", "source_update_timestamp", "loaded_at",
]

DEVICE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable device event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated deterministic subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Sentinel-cleaned implantation timestamp.",
    "event_end_datetime": "Device lifecycle end timestamp when available.",
    "source_coding_system": "MediConnect device-type coding system.",
    "source_code": "Device type with type/model display fallback for unknown type zero.",
    "source_display": "Device type or model display.",
    "device_code": "Source and mapped device codings.",
    "device_role": "Device role.",
    "model_name": "Device model name.",
    "model_code": "Device model code.",
    "manufacturer": "Device manufacturer.",
    "manufacturer_parent": "Parent manufacturer.",
    "serial_number": "Device serial number.",
    "implanted_datetime": "Source implantation timestamp.",
    "implant_date_quality": "Implant-date quality class.",
    "explanted_ind": "Inferred explant indicator; not independently confirmed.",
    "lead_chamber": "Lead chamber.",
    "lead_location": "Lead location.",
    "pocket_site": "Device pocket site.",
    "status_display": "Source device status display.",
    "device_mapping_status": "Device terminology mapping status.",
    "comment_text": "Source device comment.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth": "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d": "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@materialized_view(
    name=_n("journey_clinical.device"),
    comment="One frozen MediConnect implanted-device registry row.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=DEVICE_COLUMN_COMMENTS,
)
def device():
    return _cross_qc_public(
        spark.read.table(_n("journey_clinical._qc_device")),
        "device",
        {"device_code": "_device_code_json"},
        DEVICE_PUBLIC_COLUMNS,
    )

In [0]:
# ==== Pathway and administrative feeds ====

def _clamped_ts(col):
    """Return NULL for source timestamps outside the governed [1950, 2100) range."""
    return F.when(
        (col >= F.lit("1950-01-01").cast("timestamp"))
        & (col < F.lit("2100-01-01").cast("timestamp")),
        col,
    )

def _community_care_contact_canonical():
    s = read_source(SRC_COMMUNITY_CONTACT)
    event_id = stable_id("community_care_contact:csds", s.community_care_contact_key)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.person_id),
         ("urn:barts:community_patient_key", s.community_patient_key)],
        SRC_COMMUNITY_CONTACT,
        s.community_care_contact_key,
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    raw_event = F.coalesce(
        s.care_contact_datetime_local.cast("timestamp"), s.care_contact_date.cast("timestamp")
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.person_id.cast("string").alias("person_id"),
        F.when(s.person_id.isNotNull(), F.lit("resolved"))
         .when(s.community_patient_key.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(raw_event).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.when(s.consultation_type_code.isNotNull(),
               F.lit("urn:barts:community:consultation-type")).alias("source_coding_system"),
        s.consultation_type_code.cast("string").alias("source_code"),
        s.consultation_type_description.alias("source_display"),
        s.care_contact_date.cast("timestamp").alias("care_contact_date"),
        s.source_database_id.cast("string").alias("community_database_id"),
        s.care_contact_id.alias("care_contact_id"),
        s.community_patient_key.alias("community_patient_key"),
        s.service_request_id.alias("service_request_id"),
        s.source_service_id.cast("string").alias("service_id"),
        s.source_service_name.alias("service_name"),
        s.source_team_id.alias("team_id"),
        s.care_contact_id_variant_count.alias("care_contact_id_variant_count"),
        s.person_match_status.alias("person_match_status"),
        s.clinical_contact_duration_minutes.alias("duration_minutes"),
        s.clinical_contact_duration_quality_status.alias("duration_quality_status"),
        s.earliest_reasonable_offer_date.cast("timestamp").alias("earliest_reasonable_offer_date"),
        s.earliest_clinically_appropriate_date.cast("timestamp")
         .alias("earliest_clinically_appropriate_date"),
        s.commissioner_ods_code.alias("commissioner_ods_code"),
        s.commissioner_organization_id.cast("string").alias("commissioner_organization_id"),
        s.commissioner_organization_name.alias("commissioner_organization_name"),
        s.normalized_consultation_mechanism_code.alias("consultation_mechanism_code"),
        s.normalized_consultation_mechanism_description.alias("consultation_mechanism_display"),
        s.activity_location_type_code.alias("location_type_code"),
        s.activity_location_type_description.alias("location_type_display"),
        s.service_team_type_code.alias("service_team_type_code"),
        s.service_team_type_description.alias("service_team_type_display"),
        s.service_team_type_mapping_status.alias("service_team_type_mapping_status"),
        s.source_consultation_term.alias("source_consultation_term"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        raw_event.alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("community_care_contact").alias("source_feed"),
        F.lit(None).cast("string").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        F.lit(None).cast("timestamp").alias("loaded_at"),
        F.lit("bh-community").alias("_source_system"),
        F.lit(SRC_COMMUNITY_CONTACT).alias("_source_table"),
        s.community_care_contact_key.alias("_source_row_id"),
    )

In [0]:
SRC_COMMUNITY_CONTACT = "4_prod.bronze.map_community_care_contact"

COMMUNITY_CONTACT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "care_contact_date",
    "community_database_id", "care_contact_id", "community_patient_key",
    "service_request_id", "service_id", "service_name", "team_id",
    "care_contact_id_variant_count", "person_match_status", "duration_minutes",
    "duration_quality_status", "earliest_reasonable_offer_date",
    "earliest_clinically_appropriate_date", "commissioner_ods_code",
    "commissioner_organization_id", "commissioner_organization_name",
    "consultation_mechanism_code", "consultation_mechanism_display",
    "location_type_code", "location_type_display", "service_team_type_code",
    "service_team_type_display", "service_team_type_mapping_status",
    "source_consultation_term", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

COMMUNITY_CARE_CONTACT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "care_contact_date": "Published field.",
    "community_database_id": "Published field.",
    "care_contact_id": "Published field.",
    "community_patient_key": "Published field.",
    "service_request_id": "Published field.",
    "service_id": "Published field.",
    "service_name": "Published field.",
    "team_id": "Published field.",
    "care_contact_id_variant_count": "Published field.",
    "person_match_status": "Published field.",
    "duration_minutes": "Published field.",
    "duration_quality_status": "Published field.",
    "earliest_reasonable_offer_date": "Published field.",
    "earliest_clinically_appropriate_date": "Published field.",
    "commissioner_ods_code": "Published field.",
    "commissioner_organization_id": "Published field.",
    "commissioner_organization_name": "Published field.",
    "consultation_mechanism_code": "Published field.",
    "consultation_mechanism_display": "Published field.",
    "location_type_code": "Published field.",
    "location_type_display": "Published field.",
    "service_team_type_code": "Published field.",
    "service_team_type_display": "Published field.",
    "service_team_type_mapping_status": "Published field.",
    "source_consultation_term": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.community_care_contact"),
    comment="One CSDS community-care contact with sentinel-safe event time.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=COMMUNITY_CARE_CONTACT_COLUMN_COMMENTS,
)
def community_care_contact():
    return _community_care_contact_canonical().select(*COMMUNITY_CONTACT_PUBLIC_COLUMNS)

In [0]:
def _community_care_activity_canonical_pregate():
    s = read_source(SRC_COMMUNITY_ACTIVITY)
    event_id = stable_id("community_care_activity:csds", s.community_care_activity_key)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.person_id),
         ("urn:barts:community_patient_key", s.community_patient_key)],
        SRC_COMMUNITY_ACTIVITY,
        s.community_care_activity_key,
    )
    source_code = _code_or_display(
        s.community_care_activity_type_code, s.community_care_activity_type_description
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    raw_event = s.care_activity_date.cast("timestamp")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.person_id.cast("string").alias("person_id"),
        F.when(s.person_id.isNotNull(), F.lit("resolved"))
         .when(s.community_patient_key.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(raw_event).alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.when(source_code.isNotNull(), F.lit("urn:barts:community:activity-type"))
         .alias("source_coding_system"),
        source_code.alias("source_code"),
        s.community_care_activity_type_description.alias("source_display"),
        raw_event.alias("care_activity_date"),
        s.care_activity_date_quality_status.alias("care_activity_date_quality_status"),
        F.when(s.community_care_contact_key.isNotNull(),
               stable_id("community_care_contact:csds", s.community_care_contact_key))
         .alias("community_contact_id"),
        s.contact_match_status.alias("contact_match_status"),
        s.contact_candidate_count.alias("contact_candidate_count"),
        s.same_date_contact_candidate_count.alias("same_date_contact_candidate_count"),
        s.source_database_id.cast("string").alias("community_database_id"),
        s.care_activity_id.alias("care_activity_id"),
        s.community_patient_key.alias("community_patient_key"),
        s.source_service_id.cast("string").alias("service_id"),
        s.source_service_name.alias("service_name"),
        s.source_care_professional_local_id.alias("care_professional_local_id"),
        s.clinical_contact_duration_minutes.alias("duration_minutes"),
        s.clinical_contact_duration_quality_status.alias("duration_quality_status"),
        s.source_clinical_term.alias("source_clinical_term"),
        s.source_clinical_term_key.alias("source_clinical_term_key"),
        s.source_observation_type_id.cast("string").alias("observation_type_id"),
        s.source_code_category_id.cast("string").alias("code_category_id"),
        s.observation_value_raw.alias("observation_value_raw"),
        s.observation_value_numeric.alias("observation_value_numeric"),
        s.unit_source_value.alias("unit_source_value"),
        s.normalized_ucum_code.alias("normalized_ucum_code"),
        s.unit_omop_concept_id.cast("string").alias("unit_concept_id"),
        s.unit_omop_concept_name.alias("unit_concept_name"),
        s.unit_mapping_status.alias("unit_mapping_status"),
        s.unit_mapping_method.alias("unit_mapping_method"),
        s.snomed_candidate_count.alias("snomed_candidate_count"),
        s.snomed_candidate_concept_id.cast("string").alias("snomed_candidate_concept_id"),
        s.snomed_candidate_code.alias("snomed_candidate_code"),
        s.snomed_candidate_name.alias("snomed_candidate_name"),
        s.snomed_candidate_domain.alias("snomed_candidate_domain"),
        s.snomed_candidate_class.alias("snomed_candidate_class"),
        s.snomed_candidate_method.alias("snomed_candidate_method"),
        s.snomed_candidate_status.alias("snomed_candidate_status"),
        s.person_match_status.alias("person_match_status"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        raw_event.alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("community_care_activity").alias("source_feed"),
        F.lit(None).cast("string").alias("load_batch_id"),
        F.lit(None).cast("timestamp").alias("source_update_timestamp"),
        F.lit(None).cast("timestamp").alias("loaded_at"),
        F.lit("bh-community").alias("_source_system"),
        F.lit(SRC_COMMUNITY_ACTIVITY).alias("_source_table"),
        s.community_care_activity_key.alias("_source_row_id"),
        s.snomed_candidate_code.alias("_snomed_candidate_code"),
        s.snomed_candidate_name.alias("_snomed_candidate_name"),
        s.snomed_candidate_concept_id.cast("string").alias("_snomed_candidate_omop_code"),
        s.snomed_candidate_name.alias("_snomed_candidate_omop_display"),
    )

def _community_care_activity_canonical():
    return _community_care_activity_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_COMMUNITY_ACTIVITY = "4_prod.bronze.map_community_care_activity"

COMMUNITY_ACTIVITY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "care_activity_date",
    "care_activity_date_quality_status", "community_contact_id", "contact_match_status",
    "contact_candidate_count", "same_date_contact_candidate_count", "community_database_id",
    "care_activity_id", "community_patient_key", "service_id", "service_name",
    "care_professional_local_id", "duration_minutes", "duration_quality_status",
    "source_clinical_term", "source_clinical_term_key", "observation_type_id",
    "code_category_id", "observation_value_raw", "observation_value_numeric",
    "unit_source_value", "normalized_ucum_code", "unit_concept_id", "unit_concept_name",
    "unit_mapping_status", "unit_mapping_method", "snomed_candidate_count",
    "snomed_candidate_concept_id", "snomed_candidate_code", "snomed_candidate_name",
    "snomed_candidate_domain", "snomed_candidate_class", "snomed_candidate_method",
    "snomed_candidate_status", "person_match_status", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

COMMUNITY_CARE_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "care_activity_date": "Published field.",
    "care_activity_date_quality_status": "Published field.",
    "community_contact_id": "Published field.",
    "contact_match_status": "Published field.",
    "contact_candidate_count": "Published field.",
    "same_date_contact_candidate_count": "Published field.",
    "community_database_id": "Published field.",
    "care_activity_id": "Published field.",
    "community_patient_key": "Published field.",
    "service_id": "Published field.",
    "service_name": "Published field.",
    "care_professional_local_id": "Published field.",
    "duration_minutes": "Published field.",
    "duration_quality_status": "Published field.",
    "source_clinical_term": "Published field.",
    "source_clinical_term_key": "Published field.",
    "observation_type_id": "Published field.",
    "code_category_id": "Published field.",
    "observation_value_raw": "Published field.",
    "observation_value_numeric": "Published field.",
    "unit_source_value": "Published field.",
    "normalized_ucum_code": "Published field.",
    "unit_concept_id": "Published field.",
    "unit_concept_name": "Published field.",
    "unit_mapping_status": "Published field.",
    "unit_mapping_method": "Published field.",
    "snomed_candidate_count": "Published field.",
    "snomed_candidate_concept_id": "Published field.",
    "snomed_candidate_code": "Published field.",
    "snomed_candidate_name": "Published field.",
    "snomed_candidate_domain": "Published field.",
    "snomed_candidate_class": "Published field.",
    "snomed_candidate_method": "Published field.",
    "snomed_candidate_status": "Published field.",
    "person_match_status": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.community_care_activity"),
    comment="One admitted CSDS community-care activity with candidate terminology provenance.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=COMMUNITY_CARE_ACTIVITY_COLUMN_COMMENTS,
)
def community_care_activity():
    return _community_care_activity_canonical().select(*COMMUNITY_ACTIVITY_PUBLIC_COLUMNS)

In [0]:
def _hrg_grouping_canonical():
    return _hrg_arm(read_source(SRC_SLAM_APC_HRG), "slam_apc_hrg").unionByName(
        _hrg_arm(read_source(SRC_SLAM_OP_HRG), "slam_op_hrg")
    )

SRC_SLAM_APC_HRG = "4_prod.bronze.map_slam_apc_hrg"
SRC_SLAM_OP_HRG = "4_prod.bronze.map_slam_op_hrg"

HRG_GROUPING_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "cds_record_id",
    "person_link_method", "admission_datetime", "discharge_datetime",
    "episode_start_datetime", "episode_end_datetime", "hosp_prov_spell_num",
    "provider_org_cd", "episode_order", "episode_duration_days", "main_specialty_cd",
    "treatment_function_cd", "admission_method_cd", "admission_source_cd",
    "admission_source_desc", "discharge_method_cd", "discharge_dest_cd",
    "discharge_dest_desc", "patient_class_cd", "patient_class_desc", "source_age",
    "source_sex_cd", "neonatal_care_level_cd", "critical_care_days", "rehab_days",
    "icd_diagnosis_codes_json", "opcs_procedure_codes_json", "fce_hrg_cd",
    "fce_hrg_desc", "fce_grouping_method_flag", "fce_dominant_proc_cd",
    "fce_dominant_proc_desc", "fce_pbc_cd", "fce_calc_episode_duration",
    "fce_reporting_episode_duration", "dominant_episode_flag", "spell_hrg_cd",
    "spell_hrg_desc", "spell_grouping_method_flag", "spell_dominant_proc_cd",
    "spell_dominant_proc_desc", "spell_primary_diag_cd", "spell_primary_diag_desc",
    "spell_secondary_diag_cd", "spell_secondary_diag_desc", "spell_episode_count",
    "spell_los", "spell_reporting_los", "spell_critical_care_days", "spell_ssc_cd",
    "spell_best_practice_cd", "first_attend_cd", "first_attend_desc",
    "grouping_method_flag", "dominant_proc_cd", "dominant_proc_desc", "grouper_errors",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

def _hrg_arm(s, arm):
    is_apc = arm == "slam_apc_hrg"
    cds_id = s.CDS_APC_ID if is_apc else s.CDS_OPA_ID
    raw_event = s.EPISODE_START_DT_TM if is_apc else s.ATTENDANCE_DT_TM
    raw_end = s.EPISODE_END_DT_TM if is_apc else F.lit(None).cast("timestamp")
    source_code = s.FCE_HRG_CD if is_apc else s.HRG_CD
    source_display = s.FCE_HRG_DESC if is_apc else s.HRG_DESC
    event_id = stable_id("hrg_grouping:slam", F.lit(arm), cds_id)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)],
        SRC_SLAM_APC_HRG if is_apc else SRC_SLAM_OP_HRG,
        cds_id,
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    ns = lambda: F.lit(None).cast("string")
    ni = lambda: F.lit(None).cast("int")
    nt = lambda: F.lit(None).cast("timestamp")
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        _clamped_ts(raw_event.cast("timestamp")).alias("event_datetime"),
        _clamped_ts(raw_end.cast("timestamp")).alias("event_end_datetime"),
        F.when(source_code.isNotNull(), F.lit("urn:barts:slam:hrg"))
         .alias("source_coding_system"),
        source_code.cast("string").alias("source_code"),
        source_display.alias("source_display"),
        cds_id.cast("string").alias("cds_record_id"),
        s.PERSON_LINK_METHOD.alias("person_link_method"),
        (s.ADMISSION_DT_TM if is_apc else nt()).alias("admission_datetime"),
        (s.DISCHARGE_DT_TM if is_apc else nt()).alias("discharge_datetime"),
        (s.EPISODE_START_DT_TM if is_apc else nt()).alias("episode_start_datetime"),
        (s.EPISODE_END_DT_TM if is_apc else nt()).alias("episode_end_datetime"),
        (s.HOSP_PROV_SPELL_NUM if is_apc else ns()).alias("hosp_prov_spell_num"),
        (s.PROVIDER_ORG_CD if is_apc else ns()).alias("provider_org_cd"),
        (s.EPISODE_ORDER if is_apc else ni()).alias("episode_order"),
        (s.EPISODE_DURATION_DAYS if is_apc else ni()).alias("episode_duration_days"),
        s.MAIN_SPECIALTY_CD.alias("main_specialty_cd"),
        s.TREATMENT_FUNCTION_CD.alias("treatment_function_cd"),
        (s.ADMISSION_METHOD_CD if is_apc else ns()).alias("admission_method_cd"),
        (s.ADMISSION_SOURCE_CD if is_apc else ns()).alias("admission_source_cd"),
        (s.ADMISSION_SOURCE_DESC if is_apc else ns()).alias("admission_source_desc"),
        (s.DISCHARGE_METHOD_CD if is_apc else ns()).alias("discharge_method_cd"),
        (s.DISCHARGE_DEST_CD if is_apc else ns()).alias("discharge_dest_cd"),
        (s.DISCHARGE_DEST_DESC if is_apc else ns()).alias("discharge_dest_desc"),
        (s.PATIENT_CLASS_CD if is_apc else ns()).alias("patient_class_cd"),
        (s.PATIENT_CLASS_DESC if is_apc else ns()).alias("patient_class_desc"),
        s.SOURCE_AGE.alias("source_age"), s.SOURCE_SEX_CD.alias("source_sex_cd"),
        (s.NEONATAL_CARE_LEVEL_CD if is_apc else ni()).alias("neonatal_care_level_cd"),
        (s.CRITICAL_CARE_DAYS if is_apc else ni()).alias("critical_care_days"),
        (s.REHAB_DAYS if is_apc else ni()).alias("rehab_days"),
        (F.to_json(s.ICD_DIAG_CODES) if is_apc else ns()).alias("icd_diagnosis_codes_json"),
        F.to_json(s.OPCS_PROC_CODES).alias("opcs_procedure_codes_json"),
        (s.FCE_HRG_CD if is_apc else ns()).alias("fce_hrg_cd"),
        (s.FCE_HRG_DESC if is_apc else ns()).alias("fce_hrg_desc"),
        (s.FCE_GROUPING_METHOD_FLAG if is_apc else ns()).alias("fce_grouping_method_flag"),
        (s.FCE_DOMINANT_PROC_CD if is_apc else ns()).alias("fce_dominant_proc_cd"),
        (s.FCE_DOMINANT_PROC_DESC if is_apc else ns()).alias("fce_dominant_proc_desc"),
        (s.FCE_PBC_CD if is_apc else ns()).alias("fce_pbc_cd"),
        (s.FCE_CALC_EPISODE_DUR if is_apc else ni()).alias("fce_calc_episode_duration"),
        (s.FCE_REPORTING_EPISODE_DUR if is_apc else ni())
         .alias("fce_reporting_episode_duration"),
        (s.DOMINANT_EPISODE_FLAG if is_apc else ns()).alias("dominant_episode_flag"),
        (s.SPELL_HRG_CD if is_apc else ns()).alias("spell_hrg_cd"),
        (s.SPELL_HRG_DESC if is_apc else ns()).alias("spell_hrg_desc"),
        (s.SPELL_GROUPING_METHOD_FLAG if is_apc else ns()).alias("spell_grouping_method_flag"),
        (s.SPELL_DOMINANT_PROC_CD if is_apc else ns()).alias("spell_dominant_proc_cd"),
        (s.SPELL_DOMINANT_PROC_DESC if is_apc else ns()).alias("spell_dominant_proc_desc"),
        (s.SPELL_PRIMARY_DIAG_CD if is_apc else ns()).alias("spell_primary_diag_cd"),
        (s.SPELL_PRIMARY_DIAG_DESC if is_apc else ns()).alias("spell_primary_diag_desc"),
        (s.SPELL_SECONDARY_DIAG_CD if is_apc else ns()).alias("spell_secondary_diag_cd"),
        (s.SPELL_SECONDARY_DIAG_DESC if is_apc else ns()).alias("spell_secondary_diag_desc"),
        (s.SPELL_EPISODE_COUNT if is_apc else ni()).alias("spell_episode_count"),
        (s.SPELL_LOS if is_apc else ni()).alias("spell_los"),
        (s.SPELL_REPORTING_LOS if is_apc else ni()).alias("spell_reporting_los"),
        (s.SPELL_CRITICAL_CARE_DAYS if is_apc else ni()).alias("spell_critical_care_days"),
        (s.SPELL_SSC_CD if is_apc else ns()).alias("spell_ssc_cd"),
        (s.SPELL_BEST_PRACTICE_CD if is_apc else ns()).alias("spell_best_practice_cd"),
        (ns() if is_apc else s.FIRST_ATTEND_CD).alias("first_attend_cd"),
        (ns() if is_apc else s.FIRST_ATTEND_DESC).alias("first_attend_desc"),
        (ns() if is_apc else s.GROUPING_METHOD_FLAG).alias("grouping_method_flag"),
        (ns() if is_apc else s.DOMINANT_PROC_CD).alias("dominant_proc_cd"),
        (ns() if is_apc else s.DOMINANT_PROC_DESC).alias("dominant_proc_desc"),
        s.GROUPER_ERRORS.alias("grouper_errors"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        raw_event.cast("timestamp").alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"), F.lit(arm).alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.SOURCE_RECORD_UPDATED_DT.alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("slam").alias("_source_system"),
        F.lit(SRC_SLAM_APC_HRG if is_apc else SRC_SLAM_OP_HRG).alias("_source_table"),
        cds_id.cast("string").alias("_source_row_id"),
    )

HRG_GROUPING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "cds_record_id": "Published field.",
    "person_link_method": "Published field.",
    "admission_datetime": "Published field.",
    "discharge_datetime": "Published field.",
    "episode_start_datetime": "Published field.",
    "episode_end_datetime": "Published field.",
    "hosp_prov_spell_num": "Published field.",
    "provider_org_cd": "Published field.",
    "episode_order": "Published field.",
    "episode_duration_days": "Published field.",
    "main_specialty_cd": "Published field.",
    "treatment_function_cd": "Published field.",
    "admission_method_cd": "Published field.",
    "admission_source_cd": "Published field.",
    "admission_source_desc": "Published field.",
    "discharge_method_cd": "Published field.",
    "discharge_dest_cd": "Published field.",
    "discharge_dest_desc": "Published field.",
    "patient_class_cd": "Published field.",
    "patient_class_desc": "Published field.",
    "source_age": "Published field.",
    "source_sex_cd": "Published field.",
    "neonatal_care_level_cd": "Published field.",
    "critical_care_days": "Published field.",
    "rehab_days": "Published field.",
    "icd_diagnosis_codes_json": "Published field.",
    "opcs_procedure_codes_json": "Published field.",
    "fce_hrg_cd": "Published field.",
    "fce_hrg_desc": "Published field.",
    "fce_grouping_method_flag": "Published field.",
    "fce_dominant_proc_cd": "Published field.",
    "fce_dominant_proc_desc": "Published field.",
    "fce_pbc_cd": "Published field.",
    "fce_calc_episode_duration": "Published field.",
    "fce_reporting_episode_duration": "Published field.",
    "dominant_episode_flag": "Published field.",
    "spell_hrg_cd": "Published field.",
    "spell_hrg_desc": "Published field.",
    "spell_grouping_method_flag": "Published field.",
    "spell_dominant_proc_cd": "Published field.",
    "spell_dominant_proc_desc": "Published field.",
    "spell_primary_diag_cd": "Published field.",
    "spell_primary_diag_desc": "Published field.",
    "spell_secondary_diag_cd": "Published field.",
    "spell_secondary_diag_desc": "Published field.",
    "spell_episode_count": "Published field.",
    "spell_los": "Published field.",
    "spell_reporting_los": "Published field.",
    "spell_critical_care_days": "Published field.",
    "spell_ssc_cd": "Published field.",
    "spell_best_practice_cd": "Published field.",
    "first_attend_cd": "Published field.",
    "first_attend_desc": "Published field.",
    "grouping_method_flag": "Published field.",
    "dominant_proc_cd": "Published field.",
    "dominant_proc_desc": "Published field.",
    "grouper_errors": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.hrg_grouping"),
    comment="SLAM APC and outpatient HRG grouping results with sentinel-safe event time.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=HRG_GROUPING_COLUMN_COMMENTS,
)
def hrg_grouping():
    return _hrg_grouping_canonical().select(*HRG_GROUPING_PUBLIC_COLUMNS)

In [0]:
def _costed_activity_canonical():
    s = read_source(SRC_SLAM_COSTED_ACTIVITY)
    event_id = stable_id("costed_activity:slam", s.EXTRACT_CD, s.ACTIVITY_RECORD_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)],
        SRC_SLAM_COSTED_ACTIVITY,
        F.concat_ws(":", s.EXTRACT_CD, s.ACTIVITY_RECORD_ID.cast("string")),
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.ACTIVITY_START_DT_TM.alias("event_datetime"),
        s.ACTIVITY_END_DT_TM.alias("event_end_datetime"),
        F.when(s.POD_CD.isNotNull(), F.lit("urn:barts:slam:pod"))
         .alias("source_coding_system"),
        s.POD_CD.alias("source_code"), F.lit(None).cast("string").alias("source_display"),
        s.EXTRACT_CD.alias("extract_cd"),
        s.ACTIVITY_RECORD_ID.cast("string").alias("activity_record_id"),
        s.FEED_TYPE.alias("feed_type"), s.PLEMI.alias("plemi"),
        s.NHS_NUMBER_STATUS_CD.alias("nhs_number_status_cd"), s.CDS_ID.alias("cds_id"),
        s.ATTENDANCE_ID.alias("attendance_id"), s.ARRIVAL_DT.alias("arrival_date"),
        s.ARRIVAL_TM.alias("arrival_time"), s.DEPARTURE_DT.alias("departure_date"),
        s.DEPARTURE_TM.alias("departure_time"), s.DEPARTURE_TYPE_CD.alias("departure_type_cd"),
        s.PROVIDER_ORG_CD.alias("provider_org_cd"), s.PATIENT_ORG_CD.alias("patient_org_cd"),
        s.PATHWAY_ID.alias("pathway_id"), s.POD_CD.alias("pod_cd"),
        s.TREATMENT_FUNCTION_CD.alias("treatment_function_cd"), s.SOURCE_LOS.alias("source_los"),
        s.CF_BAND_CD.alias("cf_band_cd"), s.EPISODE_NUMBER.alias("episode_number"),
        s.EPISODE_START_DT_TM.alias("episode_start_datetime"),
        s.EPISODE_END_DT_TM.alias("episode_end_datetime"),
        s.EPISODE_TYPE_CD.alias("episode_type_cd"), s.HOSP_SPELL_ID.alias("hosp_spell_id"),
        s.HRG_CD.alias("hrg_cd"), s.HRG_DESC.alias("hrg_desc"),
        s.FCE_HRG_CD.alias("fce_hrg_cd"), s.FCE_HRG_DESC.alias("fce_hrg_desc"),
        s.SPELL_HRG_CD.alias("spell_hrg_cd"), s.SPELL_HRG_DESC.alias("spell_hrg_desc"),
        s.APPOINTMENT_DT.alias("appointment_date"), s.APPOINTMENT_TM.alias("appointment_time"),
        s.CRITICAL_CARE_UNIT_FUNCTION_CD.alias("critical_care_unit_function_cd"),
        s.ORGANS_SUPPORTED.alias("organs_supported"),
        s.CRITICAL_CARE_PERIOD_TYPE_CD.alias("critical_care_period_type_cd"),
        s.CRITICAL_CARE_LEVEL_IND.alias("critical_care_level_ind"),
        s.UNBUNDLED_ACTIVITY_DT_TM.alias("unbundled_activity_datetime"),
        s.UNBUNDLED_ACTIVITY_CD.alias("unbundled_activity_cd"),
        s.UNBUNDLED_HRG_CD.alias("unbundled_hrg_cd"),
        s.UNBUNDLED_HRG_DESC.alias("unbundled_hrg_desc"),
        s.PARTIAL_COSTING_IND.alias("partial_costing_ind"), s.CARE_DT_TM.alias("care_datetime"),
        s.CARE_ID.alias("care_id"), s.CLINICAL_CONTACT_DURATION.alias("clinical_contact_duration"),
        s.CHS_CURRENCY_CD.alias("chs_currency_cd"), s.TEAM_TYPE_CD.cast("string").alias("team_type_cd"),
        s.CONTACT_SUBJECT_CD.alias("contact_subject_cd"), s.CONSULT_TYPE_CD.alias("consult_type_cd"),
        s.CONSULT_MEDIUM_CD.alias("consult_medium_cd"), s.LOCATION_CD.alias("location_cd"),
        s.GP_THERAPY_IND.alias("gp_therapy_ind"), s.SERVICE_REQUEST_ID.alias("service_request_id"),
        s.COST_LINE_COUNT.alias("cost_line_count"),
        s.TOTAL_COST_SUM.cast("decimal(38,6)").alias("total_cost_sum"),
        s.TOTAL_O_COST_SUM.cast("decimal(38,6)").alias("total_o_cost_sum"),
        s.PERSON_LINK_METHOD.alias("person_link_method"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.ACTIVITY_START_DT_TM.alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("slam_costed_activity").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("slam-plics").alias("_source_system"),
        F.lit(SRC_SLAM_COSTED_ACTIVITY).alias("_source_table"),
        F.concat_ws(":", s.EXTRACT_CD, s.ACTIVITY_RECORD_ID.cast("string"))
         .alias("_source_row_id"),
    )

In [0]:
SRC_SLAM_COSTED_ACTIVITY = "4_prod.bronze.map_slam_costed_activity"

COSTED_ACTIVITY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "extract_cd",
    "activity_record_id", "feed_type", "plemi", "nhs_number_status_cd", "cds_id",
    "attendance_id", "arrival_date", "arrival_time", "departure_date", "departure_time",
    "departure_type_cd", "provider_org_cd", "patient_org_cd", "pathway_id", "pod_cd",
    "treatment_function_cd", "source_los", "cf_band_cd", "episode_number",
    "episode_start_datetime", "episode_end_datetime", "episode_type_cd", "hosp_spell_id",
    "hrg_cd", "hrg_desc", "fce_hrg_cd", "fce_hrg_desc", "spell_hrg_cd",
    "spell_hrg_desc", "appointment_date", "appointment_time",
    "critical_care_unit_function_cd", "organs_supported", "critical_care_period_type_cd",
    "critical_care_level_ind", "unbundled_activity_datetime", "unbundled_activity_cd",
    "unbundled_hrg_cd", "unbundled_hrg_desc", "partial_costing_ind", "care_datetime",
    "care_id", "clinical_contact_duration", "chs_currency_cd", "team_type_cd",
    "contact_subject_cd", "consult_type_cd", "consult_medium_cd", "location_cd",
    "gp_therapy_ind", "service_request_id", "cost_line_count", "total_cost_sum",
    "total_o_cost_sum", "person_link_method", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

COSTED_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "extract_cd": "Published field.",
    "activity_record_id": "Published field.",
    "feed_type": "Published field.",
    "plemi": "Published field.",
    "nhs_number_status_cd": "Published field.",
    "cds_id": "Published field.",
    "attendance_id": "Published field.",
    "arrival_date": "Published field.",
    "arrival_time": "Published field.",
    "departure_date": "Published field.",
    "departure_time": "Published field.",
    "departure_type_cd": "Published field.",
    "provider_org_cd": "Published field.",
    "patient_org_cd": "Published field.",
    "pathway_id": "Published field.",
    "pod_cd": "Published field.",
    "treatment_function_cd": "Published field.",
    "source_los": "Published field.",
    "cf_band_cd": "Published field.",
    "episode_number": "Published field.",
    "episode_start_datetime": "Published field.",
    "episode_end_datetime": "Published field.",
    "episode_type_cd": "Published field.",
    "hosp_spell_id": "Published field.",
    "hrg_cd": "Published field.",
    "hrg_desc": "Published field.",
    "fce_hrg_cd": "Published field.",
    "fce_hrg_desc": "Published field.",
    "spell_hrg_cd": "Published field.",
    "spell_hrg_desc": "Published field.",
    "appointment_date": "Published field.",
    "appointment_time": "Published field.",
    "critical_care_unit_function_cd": "Published field.",
    "organs_supported": "Published field.",
    "critical_care_period_type_cd": "Published field.",
    "critical_care_level_ind": "Published field.",
    "unbundled_activity_datetime": "Published field.",
    "unbundled_activity_cd": "Published field.",
    "unbundled_hrg_cd": "Published field.",
    "unbundled_hrg_desc": "Published field.",
    "partial_costing_ind": "Published field.",
    "care_datetime": "Published field.",
    "care_id": "Published field.",
    "clinical_contact_duration": "Published field.",
    "chs_currency_cd": "Published field.",
    "team_type_cd": "Published field.",
    "contact_subject_cd": "Published field.",
    "consult_type_cd": "Published field.",
    "consult_medium_cd": "Published field.",
    "location_cd": "Published field.",
    "gp_therapy_ind": "Published field.",
    "service_request_id": "Published field.",
    "cost_line_count": "Published field.",
    "total_cost_sum": "Published field.",
    "total_o_cost_sum": "Published field.",
    "person_link_method": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.costed_activity"),
    comment="One frozen PLICS costed activity from FY19/20 through FY21/22 extracts.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=COSTED_ACTIVITY_COLUMN_COMMENTS,
)
def costed_activity():
    return _costed_activity_canonical().select(*COSTED_ACTIVITY_PUBLIC_COLUMNS)

In [0]:
SRC_SLAM_COST_LINE_ITEM = "4_prod.bronze.map_slam_cost_line_item"

COST_LINE_ITEM_COLUMN_COMMENTS = {
    "cost_line_item_id": "Stable row identifier.",
    "costed_activity_id": "Published field.",
    "extract_cd": "Published field.",
    "activity_record_id": "Published field.",
    "line_hash": "Published field.",
    "activity_cost_item_cd": "Published field.",
    "resource_cost_item_cd": "Published field.",
    "activity_count": "Published field.",
    "unbundled_subtype_cd": "Published field.",
    "unbundled_currency_cd": "Published field.",
    "unbundled_currency_datetime": "Published field.",
    "total_cost": "Published field.",
    "total_o_cost": "Published field.",
    "source_duplicate_count": "Published field.",
    "record_status": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_reference.cost_line_item"),
    comment="PLICS cost lines (frozen FY19/20-FY21/22 extracts); event plane excluded by design.",
    refresh_policy="incremental",
    column_comments=COST_LINE_ITEM_COLUMN_COMMENTS,
)
def cost_line_item():
    s = read_source(SRC_SLAM_COST_LINE_ITEM)
    return s.select(
        stable_id("cost_line:slam", s.EXTRACT_CD, s.ACTIVITY_RECORD_ID, s.LINE_HASH)
         .alias("cost_line_item_id"),
        stable_id("costed_activity:slam", s.EXTRACT_CD, s.ACTIVITY_RECORD_ID)
         .alias("costed_activity_id"),
        s.EXTRACT_CD.alias("extract_cd"),
        s.ACTIVITY_RECORD_ID.cast("string").alias("activity_record_id"),
        s.LINE_HASH.alias("line_hash"), s.ACTIVITY_COST_ITEM_CD.alias("activity_cost_item_cd"),
        s.RESOURCE_COST_ITEM_CD.alias("resource_cost_item_cd"),
        s.ACTIVITY_COUNT.alias("activity_count"), s.UNBUNDLED_SUBTYPE_CD.alias("unbundled_subtype_cd"),
        s.UNBUNDLED_CURRENCY_CD.alias("unbundled_currency_cd"),
        s.UNBUNDLED_CURRENCY_DT_TM.alias("unbundled_currency_datetime"),
        s.TOTAL_COST.cast("decimal(38,6)").alias("total_cost"),
        s.TOTAL_O_COST.cast("decimal(38,6)").alias("total_o_cost"),
        s.SOURCE_DUPLICATE_COUNT.alias("source_duplicate_count"),
        F.when(~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True)), F.lit("retracted"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
def _drug_expenditure_canonical():
    s = read_source(SRC_FINANCE_HCD)
    event_id = stable_id("drug_expenditure:hcd", s.ROW_HASH)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_FINANCE_HCD, s.ROW_HASH
    )
    coded = _usable_code(s.DMD_CODE)
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"), s.EFFECTIVE_DT_TM.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.when(coded, F.lit("http://snomed.info/sct"))
         .otherwise(F.lit("urn:barts:hcd:chargeable-item")).alias("source_coding_system"),
        F.when(coded, s.DMD_CODE.cast("string"))
         .otherwise(s.CHARGEABLE_ITEM).alias("source_code"),
        F.coalesce(s.DMD_CONCEPT_NAME, s.CHARGEABLE_ITEM).alias("source_display"),
        s.TRANSACTION_ID.alias("transaction_id"), s.FINANCIAL_YEAR.alias("financial_year"),
        s.FINANCIAL_MONTH.alias("financial_month"), s.REPORTING_YEAR.alias("reporting_year"),
        s.REPORTING_MONTH.alias("reporting_month"), s.PROVIDER_ORG_CD.alias("provider_org_cd"),
        s.SITE_CD.alias("site_cd"), s.SITE_NAME.alias("site_name"),
        s.SPECIALTY_CD.alias("specialty_cd"), s.CONSULTANT_CD.alias("consultant_cd"),
        s.PATIENT_TYPE.alias("patient_type"), s.POD_CD.alias("pod_cd"),
        s.CHARGEABLE_ITEM.alias("chargeable_item"), s.ADDITIONAL_INFO.alias("additional_info"),
        s.DMD_RAW.alias("dmd_raw"), s.DMD_CODE.alias("dmd_code"),
        s.DMD_CONCEPT_ID.cast("string").alias("dmd_concept_id"),
        s.DMD_CONCEPT_NAME.alias("dmd_concept_name"),
        s.DRUG_STANDARD_CONCEPT_ID.cast("string").alias("drug_standard_concept_id"),
        s.DRUG_STANDARD_CONCEPT_NAME.alias("drug_standard_concept_name"),
        s.DMD_MAPPING_STATUS.alias("dmd_mapping_status"), s.DMD_TAXONOMY_CD.alias("dmd_taxonomy_cd"),
        s.ROUTE_OF_ADMINISTRATION.alias("route_of_administration"), s.STRENGTH.alias("strength"),
        s.VOLUME.alias("volume"), s.PACK_SIZE.alias("pack_size"), s.QUANTITY.alias("quantity"),
        s.UNIT_OF_MEASURE.alias("unit_of_measure"), s.DISPENSING_ROUTE.alias("dispensing_route"),
        s.DISPENSING_LOCATION.alias("dispensing_location"), s.INDICATION.alias("indication"),
        s.FUNDING_REFERENCE.alias("funding_reference"), s.HCDR_CATEGORY_CD.alias("hcdr_category_cd"),
        s.HCDR_CATEGORY_DESC.alias("hcdr_category_desc"), s.CCG_RESIDENCE_CD.alias("ccg_residence_cd"),
        s.CCG_GP_CD.alias("ccg_gp_cd"), s.COMMISSIONER_CD.alias("commissioner_cd"),
        s.COMMISSIONER_TYPE.alias("commissioner_type"), s.SERVICE_LINE.alias("service_line"),
        s.SERVICE_CATEGORY_CD.alias("service_category_cd"),
        s.UNIT_PRICE_SUPPLIER.alias("unit_price_supplier"),
        s.UNIT_PRICE_COMMISSIONER.alias("unit_price_commissioner"), s.VAT.alias("vat"),
        s.VAT_CD.alias("vat_cd"), s.INCOME.alias("income"), s.COST.alias("cost"),
        s.MARGIN.alias("margin"), s.LLOYDS_DISPENSING_FEE.alias("lloyds_dispensing_fee"),
        s.PRODUCTION_FEE.alias("production_fee"),
        s.FIXED_PATIENT_INCOME.alias("fixed_patient_income"),
        s.COST_CENTRE_DESC.alias("cost_centre_desc"), s.DRUG_FEED.alias("drug_feed"),
        s.DATA_SET.alias("data_set"), s.DRUG_CATEGORY.alias("drug_category"),
        s.LEDGER_CD.alias("ledger_cd"), s.EXCLUSION_FLAG.alias("exclusion_flag"),
        s.EXCLUSION_REASON.alias("exclusion_reason"), s.LEDGER_LV3_CD.alias("ledger_lv3_cd"),
        s.LEDGER_LV3_DESC.alias("ledger_lv3_desc"), s.LEDGER_LV6_CD.alias("ledger_lv6_cd"),
        s.LEDGER_LV6_DESC.alias("ledger_lv6_desc"), s.LEDGER_LV7_CD.alias("ledger_lv7_cd"),
        s.LEDGER_LV7_DESC.alias("ledger_lv7_desc"), s.LEDGER_LV9_CD.alias("ledger_lv9_cd"),
        s.LEDGER_LV9_DESC.alias("ledger_lv9_desc"), s.SLR_CD.alias("slr_cd"),
        s.DIABETIC_FLAG.alias("diabetic_flag"), s.IMCOE_FLAG.alias("imcoe_flag"),
        s.SOURCE_DUPLICATE_COUNT.alias("source_duplicate_count"),
        s.PERSON_LINK_METHOD.alias("person_link_method"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.EFFECTIVE_DT_TM.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("finance_hcd_expenditure").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("slr-finance").alias("_source_system"),
        F.lit(SRC_FINANCE_HCD).alias("_source_table"), s.ROW_HASH.alias("_source_row_id"),
        s.DMD_CODE.cast("string").alias("_dmd_code"), s.DMD_CONCEPT_NAME.alias("_dmd_display"),
        s.DRUG_STANDARD_CONCEPT_ID.cast("string").alias("_omop_code"),
        s.DRUG_STANDARD_CONCEPT_NAME.alias("_omop_display"),
    )

In [0]:
SRC_FINANCE_HCD = "4_prod.tmp.journey_finance_hcd_expenditure_s42"

DRUG_EXPENDITURE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "transaction_id",
    "financial_year", "financial_month", "reporting_year", "reporting_month",
    "provider_org_cd", "site_cd", "site_name", "specialty_cd", "consultant_cd",
    "patient_type", "pod_cd", "chargeable_item", "additional_info", "dmd_raw",
    "dmd_code", "dmd_concept_id", "dmd_concept_name", "drug_standard_concept_id",
    "drug_standard_concept_name", "dmd_mapping_status", "dmd_taxonomy_cd",
    "route_of_administration", "strength", "volume", "pack_size", "quantity",
    "unit_of_measure", "dispensing_route", "dispensing_location", "indication",
    "funding_reference", "hcdr_category_cd", "hcdr_category_desc", "ccg_residence_cd",
    "ccg_gp_cd", "commissioner_cd", "commissioner_type", "service_line",
    "service_category_cd", "unit_price_supplier", "unit_price_commissioner", "vat",
    "vat_cd", "income", "cost", "margin", "lloyds_dispensing_fee", "production_fee",
    "fixed_patient_income", "cost_centre_desc", "drug_feed", "data_set", "drug_category",
    "ledger_cd", "exclusion_flag", "exclusion_reason", "ledger_lv3_cd", "ledger_lv3_desc",
    "ledger_lv6_cd", "ledger_lv6_desc", "ledger_lv7_cd", "ledger_lv7_desc",
    "ledger_lv9_cd", "ledger_lv9_desc", "slr_cd", "diabetic_flag", "imcoe_flag",
    "source_duplicate_count", "person_link_method", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

DRUG_EXPENDITURE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "transaction_id": "Published field.",
    "financial_year": "Published field.",
    "financial_month": "Published field.",
    "reporting_year": "Published field.",
    "reporting_month": "Published field.",
    "provider_org_cd": "Published field.",
    "site_cd": "Published field.",
    "site_name": "Published field.",
    "specialty_cd": "Published field.",
    "consultant_cd": "Published field.",
    "patient_type": "Published field.",
    "pod_cd": "Published field.",
    "chargeable_item": "Published field.",
    "additional_info": "Published field.",
    "dmd_raw": "Published field.",
    "dmd_code": "Published field.",
    "dmd_concept_id": "Published field.",
    "dmd_concept_name": "Published field.",
    "drug_standard_concept_id": "Published field.",
    "drug_standard_concept_name": "Published field.",
    "dmd_mapping_status": "Published field.",
    "dmd_taxonomy_cd": "Published field.",
    "route_of_administration": "Published field.",
    "strength": "Published field.",
    "volume": "Published field.",
    "pack_size": "Published field.",
    "quantity": "Published field.",
    "unit_of_measure": "Published field.",
    "dispensing_route": "Published field.",
    "dispensing_location": "Published field.",
    "indication": "Published field.",
    "funding_reference": "Published field.",
    "hcdr_category_cd": "Published field.",
    "hcdr_category_desc": "Published field.",
    "ccg_residence_cd": "Published field.",
    "ccg_gp_cd": "Published field.",
    "commissioner_cd": "Published field.",
    "commissioner_type": "Published field.",
    "service_line": "Published field.",
    "service_category_cd": "Published field.",
    "unit_price_supplier": "Published field.",
    "unit_price_commissioner": "Published field.",
    "vat": "Published field.",
    "vat_cd": "Published field.",
    "income": "Published field.",
    "cost": "Published field.",
    "margin": "Published field.",
    "lloyds_dispensing_fee": "Published field.",
    "production_fee": "Published field.",
    "fixed_patient_income": "Published field.",
    "cost_centre_desc": "Published field.",
    "drug_feed": "Published field.",
    "data_set": "Published field.",
    "drug_category": "Published field.",
    "ledger_cd": "Published field.",
    "exclusion_flag": "Published field.",
    "exclusion_reason": "Published field.",
    "ledger_lv3_cd": "Published field.",
    "ledger_lv3_desc": "Published field.",
    "ledger_lv6_cd": "Published field.",
    "ledger_lv6_desc": "Published field.",
    "ledger_lv7_cd": "Published field.",
    "ledger_lv7_desc": "Published field.",
    "ledger_lv9_cd": "Published field.",
    "ledger_lv9_desc": "Published field.",
    "slr_cd": "Published field.",
    "diabetic_flag": "Published field.",
    "imcoe_flag": "Published field.",
    "source_duplicate_count": "Published field.",
    "person_link_method": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.drug_expenditure"),
    comment="One HCD drug-expenditure row keyed by immutable ROW_HASH.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=DRUG_EXPENDITURE_COLUMN_COMMENTS,
)
def drug_expenditure():
    return _drug_expenditure_canonical().select(*DRUG_EXPENDITURE_PUBLIC_COLUMNS)

In [0]:
def _medication_supply_canonical_pregate():
    s = read_source(SRC_HOMECARE_REQUEST)
    person_id = s.PERSON_ID.cast("bigint")
    event_id = stable_id("medication_supply:jac", s.HOMECARE_REQUEST_ITEM_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", person_id), ("urn:jac:lnkpid", s.LNKPID)],
        SRC_HOMECARE_REQUEST,
        s.HOMECARE_REQUEST_ITEM_ID,
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    coded = _usable_code(s.DMD_VTM_CODE)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        person_id.cast("string").alias("person_id"),
        F.when(person_id.isNotNull(), F.lit("resolved"))
         .when(s.LNKPID.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.REQUEST_CREATED.cast("timestamp").alias("event_datetime"),
        s.REQUEST_COMPLETED.cast("timestamp").alias("event_end_datetime"),
        F.when(coded, F.lit("http://snomed.info/sct"))
         .otherwise(F.lit("urn:jac:homecare:item")).alias("source_coding_system"),
        _code_or_display(s.DMD_VTM_CODE, s.ITEM_DESCRIPTION).alias("source_code"),
        F.coalesce(s.DMD_VTM_NAME, s.ITEM_DESCRIPTION).alias("source_display"),
        s.REQUEST_KEY.alias("request_key"), s.ITEM_SEQ.alias("item_seq"),
        s.REQUEST_STATUS.alias("request_status_code"),
        s.REQUEST_STATUS_DESC.alias("request_status_display"),
        s.ITEM_STATUS.alias("item_status_code"), s.ITEM_STATUS_DESC.alias("item_status_display"),
        s.ITEM_TYPE.alias("item_type"), s.ITEM_DESCRIPTION.alias("item_description"),
        s.PACK_DESCRIPTION.alias("pack_description"), s.REQUEST_QUANTITY.alias("quantity_requested"),
        s.ORIGINAL_QUANTITY.alias("quantity_original"),
        s.QUANTITY_DELIVERED.alias("quantity_delivered"), s.ORDER_UNIT.alias("order_unit"),
        s.LABEL_DIRECTIONS.alias("label_directions"), s.NFD_REASON.alias("nfd_reason"),
        s.SUPPLY_START_DATE.cast("timestamp").alias("supply_start_date"),
        s.SUPPLY_INTERVAL.alias("supply_interval"), s.SUPPLY_PERIOD.alias("supply_period"),
        s.TOTAL_ITEM_COUNT.alias("request_item_count"),
        s.COMPLETE_ITEM_COUNT.alias("request_complete_item_count"),
        s.REQUEST_RELEASED.cast("timestamp").alias("request_released_date"),
        s.LOCATION_NAME.alias("location_name"), s.COSTCENTRE_NAME.alias("cost_centre_name"),
        s.INDICATION.alias("indication"), s.CLINIC.alias("clinic"),
        s.DMD_VTM_CONCEPT_ID.cast("string").alias("dmd_vtm_concept_id"),
        s.DMD_VTM_CODE.cast("string").alias("dmd_vtm_code"),
        s.DMD_VTM_NAME.alias("dmd_vtm_name"), s.DRUG_MAPPING_METHOD.alias("drug_mapping_method"),
        s.CARE_SITE_CD.cast("string").alias("care_site_cd"),
        s.CARE_SITE_MATCH_METHOD.alias("care_site_match_method"),
        s.LNKPID.alias("lnkpid"), s.NAME_KEY.alias("name_key"),
        s.MRN_CANDIDATES.alias("mrn_candidates"), s.NHS_CANDIDATES.alias("nhs_candidates"),
        s.PERSON_MATCH_METHOD.alias("person_match_method"),
        s.PERSON_MATCH_STATUS.alias("person_match_status"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.REQUEST_CREATED.cast("timestamp").alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"), F.lit("homecare_request").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        F.coalesce(s.ITEM_SOURCE_RECORD_UPDATED_DT, s.SOURCE_RECORD_UPDATED_DT)
         .alias("source_update_timestamp"),
        s.ADC_UPDT.alias("loaded_at"), F.lit("jac-homecare").alias("_source_system"),
        F.lit(SRC_HOMECARE_REQUEST).alias("_source_table"),
        s.HOMECARE_REQUEST_ITEM_ID.alias("_source_row_id"),
        s.DMD_VTM_CODE.cast("string").alias("_dmd_code"), s.DMD_VTM_NAME.alias("_dmd_display"),
    )

def _medication_supply_canonical():
    return _medication_supply_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_HOMECARE_REQUEST = "4_prod.bronze.map_homecare_request"

MEDICATION_SUPPLY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "request_key", "item_seq",
    "request_status_code", "request_status_display", "item_status_code", "item_status_display",
    "item_type", "item_description", "pack_description", "quantity_requested",
    "quantity_original", "quantity_delivered", "order_unit", "label_directions", "nfd_reason",
    "supply_start_date", "supply_interval", "supply_period", "request_item_count",
    "request_complete_item_count", "request_released_date", "location_name", "cost_centre_name",
    "indication", "clinic", "dmd_vtm_concept_id", "dmd_vtm_code", "dmd_vtm_name",
    "drug_mapping_method", "care_site_cd", "care_site_match_method", "lnkpid", "name_key",
    "mrn_candidates", "nhs_candidates", "person_match_method", "person_match_status",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

MEDICATION_SUPPLY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "request_key": "Published field.",
    "item_seq": "Published field.",
    "request_status_code": "Published field.",
    "request_status_display": "Published field.",
    "item_status_code": "Published field.",
    "item_status_display": "Published field.",
    "item_type": "Published field.",
    "item_description": "Published field.",
    "pack_description": "Published field.",
    "quantity_requested": "Published field.",
    "quantity_original": "Published field.",
    "quantity_delivered": "Published field.",
    "order_unit": "Published field.",
    "label_directions": "Published field.",
    "nfd_reason": "Published field.",
    "supply_start_date": "Published field.",
    "supply_interval": "Published field.",
    "supply_period": "Published field.",
    "request_item_count": "Published field.",
    "request_complete_item_count": "Published field.",
    "request_released_date": "Published field.",
    "location_name": "Published field.",
    "cost_centre_name": "Published field.",
    "indication": "Published field.",
    "clinic": "Published field.",
    "dmd_vtm_concept_id": "Published field.",
    "dmd_vtm_code": "Published field.",
    "dmd_vtm_name": "Published field.",
    "drug_mapping_method": "Published field.",
    "care_site_cd": "Published field.",
    "care_site_match_method": "Published field.",
    "lnkpid": "Direct identifier published and IG-governed at serve time",
    "name_key": "Direct identifier published and IG-governed at serve time",
    "mrn_candidates": "Published field.",
    "nhs_candidates": "Published field.",
    "person_match_method": "Published field.",
    "person_match_status": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.medication_supply"),
    comment="One JAC homecare supply-request item; supply, never administration.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=MEDICATION_SUPPLY_COLUMN_COMMENTS,
)
def medication_supply():
    return _medication_supply_canonical().select(*MEDICATION_SUPPLY_PUBLIC_COLUMNS)

In [0]:
SRC_EAL_PROCEDURE = "4_prod.bronze.map_elective_access_list_procedure"

def _elective_access_entry_canonical():
    s = read_source(SRC_EAL)
    prim = read_source(SRC_EAL_PROCEDURE).where(
        F.coalesce(F.col("SOURCE_PRESENT_IND"), F.lit(True))
        & (F.col("PROCEDURE_TYPE_SEQ") == 1)
        & (F.col("PROCEDURE_SEQ") == 1)
    ).select("WAITING_LIST_OID", "PROCEDURE_CODE", "PROCEDURE_DESC", "PROCEDURE_CATALOG")
    prio = read_source(SRC_EAL_ATTRIBUTE).where(
        F.coalesce(F.col("SOURCE_PRESENT_IND"), F.lit(True))
    ).select(
        "WAITING_LIST_OID",
        F.col("FIELD_VALUE_VAR").alias("_p_code"),
        F.col("FIELD_VALUE_DATE_CLEAN").alias("_p_date"),
    )
    j = s.join(prim, "WAITING_LIST_OID", "left").join(prio, "WAITING_LIST_OID", "left")
    event_id = stable_id("elective_access:luna", s.SOURCE_SYSTEM_OID, s.WAITING_LIST_OID)
    luna_pid = F.when(
        s.SOURCE_SYSTEM_OID.isNotNull() & s.PATIENT_OID.isNotNull(),
        F.concat_ws(":", s.SOURCE_SYSTEM_OID.cast("string"), s.PATIENT_OID.cast("string")),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:luna:patient-oid", luna_pid)],
        SRC_EAL,
        s.WAITING_LIST_OID.cast("string"),
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    superseded = ~F.coalesce(s.ACTIVE_IND, F.lit(True))
    coded = _usable_code(j.PROCEDURE_CODE)
    return j.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .when(luna_pid.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.CREATED_DT_TM.alias("event_datetime"), s.ADMIT_DT_TM_CLEAN.alias("event_end_datetime"),
        F.when(coded, F.lit("http://fhir.hl7.org.uk/CodeSystem/OPCS-4"))
         .alias("source_coding_system"),
        F.when(coded, j.PROCEDURE_CODE.cast("string")).alias("source_code"),
        j.PROCEDURE_DESC.alias("source_display"),
        s.WAITING_LIST_OID.cast("string").alias("waiting_list_oid"),
        s.SOURCE_SYSTEM_OID.alias("source_system_oid"),
        s.PATHWAY_OID.cast("string").alias("pathway_oid"),
        s.REFERRAL_OID.cast("string").alias("referral_oid"),
        s.PATIENT_OID.cast("string").alias("patient_oid"),
        s.WAITING_LIST_ID.alias("waiting_list_id"),
        s.LEGACY_WAITING_LIST_ID.alias("legacy_waiting_list_id"),
        s.WAITING_LIST_NAME.alias("waiting_list_name"), s.WAITING_LIST_CODE.alias("waiting_list_code"),
        s.DEPARTMENT.alias("department"), s.DIVISION.alias("division"),
        s.BUSINESS_UNIT.alias("business_unit"), F.col("_p_code").alias("clinical_priority_code"),
        F.col("_p_date").alias("clinical_priority_recorded_datetime"),
        s.SITE_RVID.alias("site_rvid"), s.TREATMENT_FUNCTION_RVID.alias("treatment_function_rvid"),
        s.ADMIN_CATEGORY_RVID.alias("admin_category_rvid"),
        s.INTENDED_MANAGEMENT_RVID.alias("intended_management_rvid"),
        s.ADMIT_METHOD_RVID.alias("admit_method_rvid"),
        s.WAITING_LIST_PRIORITY_RVID.alias("priority_rvid"),
        s.WAITING_LIST_STATUS_RVID.alias("status_rvid"),
        s.ELECTIVE_ADMISSION_TYPE_RVID.alias("elective_admission_type_rvid"),
        s.ENCOUNTER_TYPE_RVID.alias("encounter_type_rvid"),
        s.REMOVAL_REASON_RVID.alias("removal_reason_rvid"), s.DIVISION_RVID.alias("division_rvid"),
        s.TCI_LOCATION_RVID.alias("tci_location_rvid"),
        s.ADMIT_OFFER_OUTCOME_RVID.alias("admit_offer_outcome_rvid"),
        s.LEAD_CLINICIAN_PRID.alias("lead_clinician_prid"),
        s.WAITING_LIST_STATUS_REASON.alias("status_reason"),
        s.WAITING_LIST_STATUS_CHANGE_DT_TM_CLEAN.alias("status_change_datetime"),
        s.DECIDED_TO_ADMIT_DT_TM_CLEAN.alias("decided_to_admit_datetime"),
        s.TCI_DT_TM_CLEAN.alias("tci_datetime"), s.TCI_DT_TM_FUTURE_IND.alias("tci_future_ind"),
        s.TCI_CREATED_DT_TM.alias("tci_created_datetime"),
        s.GUARANTEED_ACTIVITY_DT_TM_CLEAN.alias("guaranteed_activity_datetime"),
        s.ACTUAL_GUARANTEED_ACTIVITY_DT_TM_CLEAN.alias("actual_guaranteed_activity_datetime"),
        s.PLANNED_DT_TM_CLEAN.alias("planned_datetime"),
        s.EARLIEST_REASONABLE_OFFER_DT_TM_CLEAN.alias("earliest_reasonable_offer_datetime"),
        s.ADMIT_DT_TM_CLEAN.alias("admit_datetime"), s.COMMENTS.alias("comments"),
        s.ACTIVE_IND.alias("active_ind"), s.CREATED_DT_TM.alias("created_datetime"),
        s.CREATED_BY_PRID.alias("created_by_prid"), s.MODIFIED_DT_TM.alias("modified_datetime"),
        s.MODIFIED_BY_PRID.alias("modified_by_prid"), s.PERSON_LINK_STATUS.alias("person_link_status"),
        s.PERSON_LINK_METHOD.alias("person_link_method"),
        s.IDENTIFIER_LINK_STATUS.alias("identifier_link_status"),
        s.LINKAGE_HISTORICAL_FALLBACK_IND.alias("linkage_historical_fallback_ind"),
        s.LINKAGE_FALLBACK_CONFLICT_IND.alias("linkage_fallback_conflict_ind"),
        s.NHS_NUMBER_VALID_IND.alias("nhs_number_valid_ind"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.CREATED_DT_TM.alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS)
         .when(superseded, s.MODIFIED_DT_TM).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("elective_access_list").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.MODIFIED_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna").alias("_source_system"), F.lit(SRC_EAL).alias("_source_table"),
        s.WAITING_LIST_OID.cast("string").alias("_source_row_id"),
        F.when(coded, j.PROCEDURE_CODE.cast("string")).alias("_opcs4_code"),
        j.PROCEDURE_DESC.alias("_opcs4_display"),
    )

In [0]:
SRC_EAL = "4_prod.bronze.map_elective_access_list"
SRC_EAL_ATTRIBUTE = "4_prod.bronze.map_elective_access_list_attribute"

ELECTIVE_ACCESS_ENTRY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "waiting_list_oid",
    "source_system_oid", "pathway_oid", "referral_oid", "patient_oid", "waiting_list_id",
    "legacy_waiting_list_id", "waiting_list_name", "waiting_list_code", "department",
    "division", "business_unit", "clinical_priority_code",
    "clinical_priority_recorded_datetime", "site_rvid", "treatment_function_rvid",
    "admin_category_rvid", "intended_management_rvid", "admit_method_rvid", "priority_rvid",
    "status_rvid", "elective_admission_type_rvid", "encounter_type_rvid",
    "removal_reason_rvid", "division_rvid", "tci_location_rvid", "admit_offer_outcome_rvid",
    "lead_clinician_prid", "status_reason", "status_change_datetime",
    "decided_to_admit_datetime", "tci_datetime", "tci_future_ind", "tci_created_datetime",
    "guaranteed_activity_datetime", "actual_guaranteed_activity_datetime", "planned_datetime",
    "earliest_reasonable_offer_datetime", "admit_datetime", "comments", "active_ind",
    "created_datetime", "created_by_prid", "modified_datetime", "modified_by_prid",
    "person_link_status", "person_link_method", "identifier_link_status",
    "linkage_historical_fallback_ind", "linkage_fallback_conflict_ind", "nhs_number_valid_ind",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

ELECTIVE_ACCESS_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "waiting_list_oid": "Published field.",
    "source_system_oid": "Published field.",
    "pathway_oid": "Published field.",
    "referral_oid": "Published field.",
    "patient_oid": "Published field.",
    "waiting_list_id": "Published field.",
    "legacy_waiting_list_id": "Published field.",
    "waiting_list_name": "Published field.",
    "waiting_list_code": "Published field.",
    "department": "Published field.",
    "division": "Published field.",
    "business_unit": "Published field.",
    "clinical_priority_code": "Published field.",
    "clinical_priority_recorded_datetime": "Published field.",
    "site_rvid": "Published field.",
    "treatment_function_rvid": "Published field.",
    "admin_category_rvid": "Published field.",
    "intended_management_rvid": "Published field.",
    "admit_method_rvid": "Published field.",
    "priority_rvid": "Published field.",
    "status_rvid": "Published field.",
    "elective_admission_type_rvid": "Published field.",
    "encounter_type_rvid": "Published field.",
    "removal_reason_rvid": "Published field.",
    "division_rvid": "Published field.",
    "tci_location_rvid": "Published field.",
    "admit_offer_outcome_rvid": "Published field.",
    "lead_clinician_prid": "Published field.",
    "status_reason": "Published field.",
    "status_change_datetime": "Published field.",
    "decided_to_admit_datetime": "Published field.",
    "tci_datetime": "Published field.",
    "tci_future_ind": "Published field.",
    "tci_created_datetime": "Published field.",
    "guaranteed_activity_datetime": "Published field.",
    "actual_guaranteed_activity_datetime": "Published field.",
    "planned_datetime": "Published field.",
    "earliest_reasonable_offer_datetime": "Published field.",
    "admit_datetime": "Published field.",
    "comments": "Direct identifier published and IG-governed at serve time",
    "active_ind": "Published field.",
    "created_datetime": "Published field.",
    "created_by_prid": "Published field.",
    "modified_datetime": "Published field.",
    "modified_by_prid": "Published field.",
    "person_link_status": "Published field.",
    "person_link_method": "Published field.",
    "identifier_link_status": "Published field.",
    "linkage_historical_fallback_ind": "Published field.",
    "linkage_fallback_conflict_ind": "Published field.",
    "nhs_number_valid_ind": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.elective_access_entry"),
    comment="One LUNA elective-access entry with primary OPCS and clinical-priority folds.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=ELECTIVE_ACCESS_ENTRY_COLUMN_COMMENTS,
)
def elective_access_entry():
    return _elective_access_entry_canonical().select(*ELECTIVE_ACCESS_ENTRY_PUBLIC_COLUMNS)

In [0]:
ELECTIVE_ACCESS_PROCEDURE_COLUMN_COMMENTS = {
    "elective_access_procedure_id": "Stable row identifier.",
    "elective_access_entry_id": "Published field.",
    "waiting_list_oid": "Published field.",
    "procedure_code": "Published field.",
    "procedure_desc": "Published field.",
    "procedure_catalog": "Published field.",
    "procedure_rvid": "Published field.",
    "procedure_type_seq": "Published field.",
    "procedure_seq": "Published field.",
    "active_ind": "Published field.",
    "parent_present_ind": "Published field.",
    "source_system_oid": "Published field.",
    "source_system_oid_inherited_ind": "Published field.",
    "record_status": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_reference.elective_access_procedure"),
    comment="All LUNA elective-access planned-procedure slots, including bare and tombstoned rows.",
    refresh_policy="incremental",
    column_comments=ELECTIVE_ACCESS_PROCEDURE_COLUMN_COMMENTS,
)
def elective_access_procedure():
    s = read_source(SRC_EAL_PROCEDURE)
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    superseded = ~F.coalesce(s.ACTIVE_IND, F.lit(True))
    return s.select(
        stable_id("elective_access_procedure:luna", s.WAITING_LIST_OID,
                  s.PROCEDURE_TYPE_SEQ, s.PROCEDURE_SEQ)
         .alias("elective_access_procedure_id"),
        stable_id("elective_access:luna", s.SOURCE_SYSTEM_OID, s.WAITING_LIST_OID)
         .alias("elective_access_entry_id"),
        s.WAITING_LIST_OID.cast("string").alias("waiting_list_oid"),
        s.PROCEDURE_CODE.alias("procedure_code"), s.PROCEDURE_DESC.alias("procedure_desc"),
        s.PROCEDURE_CATALOG.alias("procedure_catalog"), s.PROCEDURE_RVID.alias("procedure_rvid"),
        s.PROCEDURE_TYPE_SEQ.alias("procedure_type_seq"), s.PROCEDURE_SEQ.alias("procedure_seq"),
        s.ACTIVE_IND.alias("active_ind"), s.PARENT_PRESENT_IND.alias("parent_present_ind"),
        s.SOURCE_SYSTEM_OID.alias("source_system_oid"),
        s.SOURCE_SYSTEM_OID_INHERITED_IND.alias("source_system_oid_inherited_ind"),
        F.when(retracted, F.lit("retracted"))
         .when(superseded, F.lit("superseded"))
         .otherwise(F.lit("active")).alias("record_status"),
        s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
def _pathway_tracking_canonical():
    s = read_source(SRC_CANCER_PTL)
    event_id = stable_id("pathway_tracking:pathfinder", s.PTL_UNIQUE_ID)
    luna_pid = F.when(
        s.SOURCE_SYSTEM_OID.isNotNull() & s.PATIENT_OID.isNotNull(),
        F.concat_ws(":", s.SOURCE_SYSTEM_OID.cast("string"), s.PATIENT_OID.cast("string")),
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID), ("urn:barts:luna:patient-oid", luna_pid)],
        SRC_CANCER_PTL,
        s.PTL_UNIQUE_ID.cast("string"),
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .when(luna_pid.isNotNull(), F.lit("provisional"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.LATEST_ACTIVITY_DATE_CLEAN.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        s.PTL_UNIQUE_ID.cast("string").alias("ptl_unique_id"),
        s.PTL_GROUP_ID.alias("ptl_group_id"), s.ARCHIVED_PATHWAY.alias("archived_pathway"),
        s.PATHWAY_OID.cast("string").alias("pathway_oid"),
        s.REFERRAL_OID.cast("string").alias("referral_oid"),
        s.PATIENT_OID.cast("string").alias("patient_oid"),
        s.PTL_ACTIVITY_OID.cast("string").alias("ptl_activity_oid"),
        s.PARENT_PTL_UNIQUE_ID.cast("string").alias("parent_ptl_unique_id"),
        s.PARENT_PTL_ACTIVITY_OID.cast("string").alias("parent_ptl_activity_oid"),
        s.PARENT_PRESENT_IND.alias("parent_present_ind"),
        s.LATEST_ACTIVITY_TYPE.alias("latest_activity_type"),
        s.LATEST_ACTIVITY_OID.cast("string").alias("latest_activity_oid"),
        s.LATEST_ACTIVITY_DATE_FUTURE_IND.alias("latest_activity_date_future_ind"),
        s.DAYS_WAITED.alias("days_waited"), s.SPECIALTY.alias("specialty"),
        s.TREATMENT_FUNCTION.alias("treatment_function"),
        s.TREATMENT_FUNCTION_CODE.alias("treatment_function_code"), s.SITE.alias("site"),
        s.SITE_GROUP.alias("site_group"), s.DIVISION.alias("division"),
        s.LEAD_CLINICIAN.alias("lead_clinician"),
        s.LEAD_CLINICIAN_PRID.alias("lead_clinician_prid"), s.SITE_RVID.alias("site_rvid"),
        s.SOURCE_KEY_STATUS.alias("source_key_status"), s.PATIENT_SPINE_IND.alias("patient_spine_ind"),
        s.PATHWAY_SPINE_IND.alias("pathway_spine_ind"),
        s.REFERRAL_SPINE_IND.alias("referral_spine_ind"),
        s.PATIENT_SPINE_LINK_STATUS.alias("patient_spine_link_status"),
        s.PATHWAY_SPINE_LINK_STATUS.alias("pathway_spine_link_status"),
        s.REFERRAL_SPINE_LINK_STATUS.alias("referral_spine_link_status"),
        s.NHS_NUMBER_VALID_IND.alias("nhs_number_valid_ind"),
        s.LINKAGE_HISTORICAL_FALLBACK_IND.alias("linkage_historical_fallback_ind"),
        s.LINKAGE_FALLBACK_CONFLICT_IND.alias("linkage_fallback_conflict_ind"),
        s.PERSON_LINK_STATUS.alias("person_link_status"),
        s.PERSON_LINK_METHOD.alias("person_link_method"),
        s.IDENTIFIER_LINK_STATUS.alias("identifier_link_status"),
        s.SOURCE_SYSTEM_OID.alias("source_system_oid"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.LATEST_ACTIVITY_DATE_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, s.SOURCE_ABSENT_DETECTED_TS).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"), F.lit("cancer_ptl").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("luna-pathfinder").alias("_source_system"),
        F.lit(SRC_CANCER_PTL).alias("_source_table"),
        s.PTL_UNIQUE_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_CANCER_PTL = "4_prod.bronze.map_cancer_ptl"

PATHWAY_TRACKING_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "ptl_unique_id", "ptl_group_id",
    "archived_pathway", "pathway_oid", "referral_oid", "patient_oid", "ptl_activity_oid",
    "parent_ptl_unique_id", "parent_ptl_activity_oid", "parent_present_ind",
    "latest_activity_type", "latest_activity_oid", "latest_activity_date_future_ind",
    "days_waited", "specialty", "treatment_function", "treatment_function_code", "site",
    "site_group", "division", "lead_clinician", "lead_clinician_prid", "site_rvid",
    "source_key_status", "patient_spine_ind", "pathway_spine_ind", "referral_spine_ind",
    "patient_spine_link_status", "pathway_spine_link_status", "referral_spine_link_status",
    "nhs_number_valid_ind", "linkage_historical_fallback_ind", "linkage_fallback_conflict_ind",
    "person_link_status", "person_link_method", "identifier_link_status", "source_system_oid",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category", "source_feed",
    "load_batch_id", "source_update_timestamp", "loaded_at",
]

PATHWAY_TRACKING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "ptl_unique_id": "Published field.",
    "ptl_group_id": "Published field.",
    "archived_pathway": "Published field.",
    "pathway_oid": "Published field.",
    "referral_oid": "Published field.",
    "patient_oid": "Published field.",
    "ptl_activity_oid": "Published field.",
    "parent_ptl_unique_id": "Published field.",
    "parent_ptl_activity_oid": "Published field.",
    "parent_present_ind": "Published field.",
    "latest_activity_type": "Published field.",
    "latest_activity_oid": "Published field.",
    "latest_activity_date_future_ind": "Published field.",
    "days_waited": "Published field.",
    "specialty": "Published field.",
    "treatment_function": "Published field.",
    "treatment_function_code": "Published field.",
    "site": "Published field.",
    "site_group": "Published field.",
    "division": "Published field.",
    "lead_clinician": "Published field.",
    "lead_clinician_prid": "Published field.",
    "site_rvid": "Published field.",
    "source_key_status": "Published field.",
    "patient_spine_ind": "Published field.",
    "pathway_spine_ind": "Published field.",
    "referral_spine_ind": "Published field.",
    "patient_spine_link_status": "Published field.",
    "pathway_spine_link_status": "Published field.",
    "referral_spine_link_status": "Published field.",
    "nhs_number_valid_ind": "Published field.",
    "linkage_historical_fallback_ind": "Published field.",
    "linkage_fallback_conflict_ind": "Published field.",
    "person_link_status": "Published field.",
    "person_link_method": "Published field.",
    "identifier_link_status": "Published field.",
    "source_system_oid": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_clinical.pathway_tracking"),
    comment="Whole-trust Pathfinder PTL activity rows; scope is PTL_GROUP_ID, not cancer.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=PATHWAY_TRACKING_COLUMN_COMMENTS,
)
def pathway_tracking():
    return _pathway_tracking_canonical().select(*PATHWAY_TRACKING_PUBLIC_COLUMNS)

In [0]:
def _pregnancy_reconciliation_source():
    s = read_source(SRC_MAT_MSDS_UNMATCHED)
    return s.select(
        stable_id("pregnancy_reconciliation:msds", s.UNMATCHED_KEY)
         .alias("pregnancy_reconciliation_id"),
        s.UNMATCHED_REASON.alias("unmatched_reason"),
        s.PREGNANCYID_RAW.alias("pregnancy_id_raw"),
        s.Pregnancy_ID.cast("string").alias("pregnancy_id"),
        s.LPIDMother.alias("lpid_mother"),
        s.AntenatalAppointmentDate.alias("antenatal_appointment_date"),
        s.PregnancyFirstContactDate.alias("pregnancy_first_contact_date"),
        s.ExpectedDeliveryDate.alias("expected_delivery_date"),
        s.LastMensPeriodDate.alias("last_mens_period_date"),
        s.FolicAcidSupplement_CD.alias("folic_acid_supplement_cd"),
        s.PreviousLiveBirths.alias("previous_live_births"),
        s.PreviousStillBirths.alias("previous_still_births"),
        s.PreviousLossesUnder24Weeks.alias("previous_losses_under_24_weeks"),
        s.PreviousCaesareanSections.alias("previous_caesarean_sections"),
        s.SOURCE_SYSTEM.alias("source_system_code"), s.IS_VALID.alias("is_valid"),
        s.MSDS_SOURCE_VERSION.alias("msds_source_version"), F.lit("active").alias("record_status"),
        s.RECORD_UPDATED_DT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
    )

In [0]:
SRC_MAT_MSDS_UNMATCHED = "4_prod.bronze.map_mat_pregnancy_msds_unmatched"

PREGNANCY_RECONCILIATION_COLUMN_COMMENTS = {
    "pregnancy_reconciliation_id": "Stable row identifier.",
    "unmatched_reason": "Published field.",
    "pregnancy_id_raw": "Published field.",
    "pregnancy_id": "Published field.",
    "lpid_mother": "Direct identifier published and IG-governed at serve time",
    "antenatal_appointment_date": "Published field.",
    "pregnancy_first_contact_date": "Published field.",
    "expected_delivery_date": "Published field.",
    "last_mens_period_date": "Published field.",
    "folic_acid_supplement_cd": "Published field.",
    "previous_live_births": "Published field.",
    "previous_still_births": "Published field.",
    "previous_losses_under_24_weeks": "Published field.",
    "previous_caesarean_sections": "Published field.",
    "source_system_code": "Published field.",
    "is_valid": "Published field.",
    "msds_source_version": "Published field.",
    "record_status": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@materialized_view(
    name=_n("journey_reference.pregnancy_reconciliation"),
    comment="Unmatched MSDS pregnancy rows documenting the person-spine coverage gap.",
    refresh_policy="incremental",
    column_comments=PREGNANCY_RECONCILIATION_COLUMN_COMMENTS,
)
def pregnancy_reconciliation():
    return _pregnancy_reconciliation_source()

In [0]:
def _critical_care_period_canonical():
    s = read_source(SRC_CC_PERIOD)
    ranked = s.select(
        "*",
        F.struct(
            F.coalesce(F.col("CURRENT_IND").cast("int"), F.lit(0)).alias("_cur"),
            F.coalesce(
                F.col("Record_Updated_Dt").cast("timestamp"),
                F.lit("1900-01-01").cast("timestamp"),
            ).alias("_upd"),
            F.coalesce(F.col("Crit_Care_Period_Id"), F.lit(-1)).alias("_sid"),
        ).alias("_pick"),
    )
    best = ranked.groupBy("PERIOD_BUSINESS_KEY").agg(F.max("_pick").alias("_best"))
    d = ranked.join(best, ["PERIOD_BUSINESS_KEY"]).where(F.col("_pick") == F.col("_best"))
    event_id = stable_id("critical_care_period:cds", d.PERIOD_BUSINESS_KEY)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", d.PERSON_ID)], SRC_CC_PERIOD, d.PERIOD_BUSINESS_KEY
    )
    retracted = ~F.coalesce(d.SOURCE_PRESENT_IND, F.lit(True))
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        d.PERSON_ID.cast("string").alias("person_id"),
        F.when(d.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.when(d.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", d.ENCNTR_ID))
         .alias("encounter_id"),
        d.CC_Period_Start_Dt_Tm_CLEAN.alias("event_datetime"),
        d.CC_Period_Disch_Dt_Tm_CLEAN.alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        d.CC_Period_Disch_Dt_Tm_CLEAN.alias("period_end_datetime"),
        d.PERIOD_BUSINESS_KEY.cast("string").alias("period_business_key"),
        (~F.coalesce(d.CURRENT_IND, F.lit(False))).alias("no_current_version_ind"),
        d.BUSINESS_KEY_STATUS.alias("business_key_status"),
        d.Is_Valid.alias("source_valid_ind"), d.CC_TYPE_DESC.alias("care_type"),
        d.CC_UNIT_FUNCTION_DESC.alias("unit_function"), d.Unit_Id.alias("unit_id"),
        d.Source_System.alias("cds_source_system"), d.CC_Level2_Days.alias("level2_days"),
        d.CC_Level3_Days.alias("level3_days"),
        d.CC_No_Organ_Systems.alias("organ_systems_supported"),
        d.Gestation_Length.alias("gestation_length"),
        d.CC_DISCH_STATUS_DESC.alias("discharge_status"),
        d.CC_DISCH_DEST_DESC.alias("discharge_destination"),
        d.ENCNTR_ID.cast("string").alias("source_encounter_id"),
        F.when(d.CC_ENCNTR_ID.isNotNull(), stable_id("encounter:mill", d.CC_ENCNTR_ID))
         .alias("cc_encounter_id"),
        d.CDS_APC_ID.alias("cds_apc_id"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        d.CC_Period_Start_Dt_Tm_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, d.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("critical_care_period").alias("source_feed"),
        F.date_format(d.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        d.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"),
        d.ADC_UPDT.alias("loaded_at"), F.lit("cds-ccmds").alias("_source_system"),
        F.lit(SRC_CC_PERIOD).alias("_source_table"),
        d.PERIOD_BUSINESS_KEY.cast("string").alias("_source_row_id"),
    )

In [0]:
# ==== Remaining planes and supporting products ====

SRC_CC_PERIOD = "4_prod.bronze.map_critical_care_period"

CRITICAL_CARE_PERIOD_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "period_end_datetime",
    "period_business_key", "no_current_version_ind", "business_key_status",
    "source_valid_ind", "care_type", "unit_function", "unit_id", "cds_source_system",
    "level2_days", "level3_days", "organ_systems_supported", "gestation_length",
    "discharge_status", "discharge_destination", "source_encounter_id",
    "cc_encounter_id", "cds_apc_id", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

CRITICAL_CARE_PERIOD_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "period_end_datetime": "Published field period_end_datetime.",
    "period_business_key": "Published field period_business_key.",
    "no_current_version_ind": "Published field no_current_version_ind.",
    "business_key_status": "Published field business_key_status.",
    "source_valid_ind": "Published field source_valid_ind.",
    "care_type": "Published field care_type.",
    "unit_function": "Published field unit_function.",
    "unit_id": "Published field unit_id.",
    "cds_source_system": "Published field cds_source_system.",
    "level2_days": "Published field level2_days.",
    "level3_days": "Published field level3_days.",
    "organ_systems_supported": "Published field organ_systems_supported.",
    "gestation_length": "Published field gestation_length.",
    "discharge_status": "Published field discharge_status.",
    "discharge_destination": "Published field discharge_destination.",
    "source_encounter_id": "Published field source_encounter_id.",
    "cc_encounter_id": "Published field cc_encounter_id.",
    "cds_apc_id": "Published field cds_apc_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.critical_care_period"),
    comment="One critical-care period per CCMDS business key; version history stays in bronze.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CRITICAL_CARE_PERIOD_COLUMN_COMMENTS,
)
def critical_care_period():
    return _critical_care_period_canonical().select(*CRITICAL_CARE_PERIOD_PUBLIC_COLUMNS)

In [0]:
def _critical_care_activity_canonical_pregate():
    s = read_source(SRC_CC_ACTIVITY)
    event_id = stable_id("critical_care_activity:cds", s.ROW_HASH)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_CC_ACTIVITY, s.ROW_HASH
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.Activity_Date_CLEAN.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:nhs:ccmds:activity").alias("source_coding_system"),
        s.Activity_Code.cast("string").alias("source_code"),
        s.ACTIVITY_DESC.alias("source_display"),
        s.PERIOD_LINK_STATUS.alias("period_link_status"),
        s.PERIOD_BUSINESS_KEY.cast("string").alias("period_business_key"),
        s.PARENT_PERIOD_SURROGATE_ID.cast("string").alias("parent_period_id"),
        s.CDS_APC_ID.alias("cds_apc_id"),
        F.when(s.CC_Type == 1, F.lit("adult"))
         .when(s.CC_Type == 2, F.lit("neonatal"))
         .otherwise(s.CC_Type.cast("string")).alias("cc_type"),
        s.SOURCE_DUPLICATE_COUNT.alias("source_duplicate_count"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        s.Activity_Date_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("critical_care_activity").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("cds-ccmds").alias("_source_system"),
        F.lit(SRC_CC_ACTIVITY).alias("_source_table"),
        s.ROW_HASH.cast("string").alias("_source_row_id"),
    )

def _critical_care_activity_canonical():
    return _critical_care_activity_canonical_pregate().where(
        _usable_code(F.col("source_code"))
    )

In [0]:
SRC_CC_ACTIVITY = "4_prod.bronze.map_critical_care_activity"

CRITICAL_CARE_ACTIVITY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "period_link_status",
    "period_business_key", "parent_period_id", "cds_apc_id", "cc_type",
    "source_duplicate_count", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

CRITICAL_CARE_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "period_link_status": "Published field period_link_status.",
    "period_business_key": "Published field period_business_key.",
    "parent_period_id": "Published field parent_period_id.",
    "cds_apc_id": "Published field cds_apc_id.",
    "cc_type": "Published field cc_type.",
    "source_duplicate_count": "Published field source_duplicate_count.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.critical_care_activity"),
    comment="Admitted CCMDS critical-care activity rows with period-link evidence.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CRITICAL_CARE_ACTIVITY_COLUMN_COMMENTS,
)
def critical_care_activity():
    return _critical_care_activity_canonical().select(*CRITICAL_CARE_ACTIVITY_PUBLIC_COLUMNS)

In [0]:
def _critical_care_admission_canonical():
    s = read_source(SRC_CC_ADMISSION)
    event_id = stable_id("critical_care_admission:medicus", s.ADMISSION_KEY)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_CC_ADMISSION, s.ADMISSION_KEY
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.date_unit_adm_CLEAN.alias("event_datetime"),
        s.date_unit_discharge_CLEAN.alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        s.ADMISSION_KEY.alias("admission_key"), s.SOURCE_UNIT.alias("source_unit"),
        s.UNIT_RAW.alias("unit_raw"), s.SOURCE_SITE.alias("source_site"),
        s.date_hospital_adm_CLEAN.alias("hospital_admission_datetime"),
        s.date_unit_adm_CLEAN.alias("unit_admission_datetime"),
        s.date_unit_discharge_CLEAN.alias("unit_discharge_datetime"),
        s.date_hospital_discharge_CLEAN.alias("hospital_discharge_datetime"),
        s.dgn_adm1_code.alias("admission_diagnosis_code"),
        s.maxorgansupp.alias("max_organ_support"), s.cause_of_death.alias("cause_of_death"),
        s.date_of_death_CLEAN.alias("date_of_death"),
        s.updated_at.alias("source_updated_at"), s.PERSON_LINK_STATUS.alias("person_link_status"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        s.date_unit_adm_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("critical_care_admission").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.updated_at.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("medicus").alias("_source_system"),
        F.lit(SRC_CC_ADMISSION).alias("_source_table"),
        s.ADMISSION_KEY.alias("_source_row_id"),
    )

In [0]:
SRC_CC_ADMISSION = "4_prod.bronze.map_critical_care_admission"

CRITICAL_CARE_ADMISSION_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "admission_key",
    "source_unit", "unit_raw", "source_site", "hospital_admission_datetime",
    "unit_admission_datetime", "unit_discharge_datetime", "hospital_discharge_datetime",
    "admission_diagnosis_code", "max_organ_support", "cause_of_death",
    "date_of_death", "source_updated_at", "person_link_status", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

CRITICAL_CARE_ADMISSION_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "admission_key": "Published field admission_key.",
    "source_unit": "Published field source_unit.",
    "unit_raw": "Published field unit_raw.",
    "source_site": "Published field source_site.",
    "hospital_admission_datetime": "Published field hospital_admission_datetime.",
    "unit_admission_datetime": "Published field unit_admission_datetime.",
    "unit_discharge_datetime": "Published field unit_discharge_datetime.",
    "hospital_discharge_datetime": "Published field hospital_discharge_datetime.",
    "admission_diagnosis_code": "Published field admission_diagnosis_code.",
    "max_organ_support": "Published field max_organ_support.",
    "cause_of_death": "Published field cause_of_death.",
    "date_of_death": "Published field date_of_death.",
    "source_updated_at": "Published field source_updated_at.",
    "person_link_status": "Published field person_link_status.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.critical_care_admission"),
    comment="One Medicus critical-care admission per source admission key.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CRITICAL_CARE_ADMISSION_COLUMN_COMMENTS,
)
def critical_care_admission():
    return _critical_care_admission_canonical().select(*CRITICAL_CARE_ADMISSION_PUBLIC_COLUMNS)

In [0]:
def _cc_daily_score_canonical_pregate():
    s = read_source(SRC_CC_DAILY_SCORE)
    event_id = stable_id(
        "critical_care_daily_score:medicus",
        s.SOURCE_UNIT, s.SOURCE_SITE, s.SOURCE_DAILY_ID, s.SCORE_TYPE,
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)],
        SRC_CC_DAILY_SCORE,
        F.concat_ws("|", s.SOURCE_UNIT, s.SOURCE_SITE, s.SOURCE_DAILY_ID, s.SCORE_TYPE),
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    event_time = F.coalesce(s.DATE_DAILY_CLEAN, s.DATE_SCORE_CALC_CLEAN)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"), event_time.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:medicus:critical-care-score").alias("source_coding_system"),
        s.SCORE_TYPE.alias("source_code"), s.SCORE_TYPE.alias("source_display"),
        s.SCORE_VALUE.alias("score_value"), s.SCORE_VALUE_RAW.alias("score_value_raw"),
        s.DATE_DAILY_CLEAN.alias("score_date"),
        s.DATE_SCORE_CALC_CLEAN.alias("score_calc_date"),
        s.ADMISSION_KEY.alias("admission_key"),
        stable_id("critical_care_admission:medicus", s.ADMISSION_KEY)
         .alias("critical_care_admission_id"),
        s.DAY_LATEST_IND.alias("day_latest_ind"), s.ROW_CLASS.alias("row_class"),
        s.PERSON_LINK_STATUS.alias("person_link_status"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("critical_care_daily_score").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.UPDATED_AT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("medicus").alias("_source_system"),
        F.lit(SRC_CC_DAILY_SCORE).alias("_source_table"),
        F.concat_ws("|", s.SOURCE_UNIT, s.SOURCE_SITE, s.SOURCE_DAILY_ID, s.SCORE_TYPE)
         .alias("_source_row_id"),
    )

def _cc_daily_score_canonical():
    return _cc_daily_score_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_CC_DAILY_SCORE = "4_prod.bronze.map_critical_care_daily_score"

CRITICAL_CARE_DAILY_SCORE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "score_value",
    "score_value_raw", "score_date", "score_calc_date", "admission_key",
    "critical_care_admission_id", "day_latest_ind", "row_class", "person_link_status",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

CRITICAL_CARE_DAILY_SCORE_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "score_value": "Published field score_value.",
    "score_value_raw": "Published field score_value_raw.",
    "score_date": "Published field score_date.",
    "score_calc_date": "Published field score_calc_date.",
    "admission_key": "Published field admission_key.",
    "critical_care_admission_id": "Published field critical_care_admission_id.",
    "day_latest_ind": "Published field day_latest_ind.",
    "row_class": "Published field row_class.",
    "person_link_status": "Published field person_link_status.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.critical_care_daily_score"),
    comment="Long-format Medicus critical-care scores; DAY_LATEST_IND and ROW_CLASS are flags, not dedup rules.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=CRITICAL_CARE_DAILY_SCORE_COLUMN_COMMENTS,
)
def critical_care_daily_score():
    return _cc_daily_score_canonical().select(*CRITICAL_CARE_DAILY_SCORE_PUBLIC_COLUMNS)

In [0]:
SRC_NEO_EPISODE = "4_prod.bronze.map_neonatal_episode"

def _neonatal_episode_canonical():
    s = read_source(SRC_NEO_EPISODE)
    event_id = stable_id("neonatal_episode:badgernet", s.EntityID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.BABY_PERSON_ID)], SRC_NEO_EPISODE, s.EntityID
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    event_time = F.coalesce(s.AdmitTime_CLEAN, s.BirthTimeBaby_CLEAN)
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.BABY_PERSON_ID.cast("string").alias("person_id"),
        F.when(s.BABY_PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"), event_time.alias("event_datetime"),
        s.DischTime_CLEAN.alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        s.EntityID.alias("entity_id"), s.BadgerUniqueID.alias("badger_unique_id"),
        s.MOTHER_PERSON_ID.cast("string").alias("mother_person_id"),
        s.CareLocationID.alias("care_location_id"),
        s.CareLocationName.alias("care_location_name"),
        s.BirthTimeBaby_CLEAN.alias("birth_datetime"),
        s.BirthTimeBaby.alias("birth_datetime_raw"), s.AdmitTime_CLEAN.alias("admit_datetime"),
        s.DischTime_CLEAN.alias("discharge_datetime"),
        s.GestationWeeks.alias("gestation_weeks"), s.GestationDays.alias("gestation_days"),
        s.Birthweight.alias("birthweight"), s.Sex.alias("sex"),
        s.FinalNNUOutcome.alias("final_nnu_outcome"), s.UnitLevel.alias("unit_level"),
        s.PERSON_LINK_STATUS.alias("person_link_status"),
        s.RecordTimestamp.alias("record_timestamp"), s.LastUpdate.alias("last_update"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("neonatal_episode").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.LastUpdate.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("badgernet").alias("_source_system"),
        F.lit(SRC_NEO_EPISODE).alias("_source_table"), s.EntityID.alias("_source_row_id"),
    )

In [0]:
NEONATAL_EPISODE_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "entity_id",
    "badger_unique_id", "mother_person_id", "care_location_id", "care_location_name",
    "birth_datetime", "birth_datetime_raw", "admit_datetime", "discharge_datetime",
    "gestation_weeks", "gestation_days", "birthweight", "sex", "final_nnu_outcome",
    "unit_level", "person_link_status", "record_timestamp", "last_update",
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

NEONATAL_EPISODE_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "badger_unique_id": "Published field badger_unique_id.",
    "mother_person_id": "Published field mother_person_id.",
    "care_location_id": "Published field care_location_id.",
    "care_location_name": "Published field care_location_name.",
    "birth_datetime": "Published field birth_datetime.",
    "birth_datetime_raw": "Published field birth_datetime_raw.",
    "admit_datetime": "Published field admit_datetime.",
    "discharge_datetime": "Published field discharge_datetime.",
    "gestation_weeks": "Published field gestation_weeks.",
    "gestation_days": "Published field gestation_days.",
    "birthweight": "Published field birthweight.",
    "sex": "Published field sex.",
    "final_nnu_outcome": "Published field final_nnu_outcome.",
    "unit_level": "Published field unit_level.",
    "person_link_status": "Published field person_link_status.",
    "record_timestamp": "Published field record_timestamp.",
    "last_update": "Published field last_update.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.neonatal_episode"),
    comment="One BadgerNet neonatal episode with baby subject and mother reference.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=NEONATAL_EPISODE_COLUMN_COMMENTS,
)
def neonatal_episode():
    return _neonatal_episode_canonical().select(*NEONATAL_EPISODE_PUBLIC_COLUMNS)

In [0]:
def _neonatal_care_day_canonical_pregate():
    s = read_source(SRC_NEO_CRITICAL_CARE)
    event_id = stable_id("neonatal_care_day:badgernet", s.EntityID, s.ActivityDate)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.BABY_PERSON_ID)],
        SRC_NEO_CRITICAL_CARE,
        F.concat_ws("|", s.EntityID, s.ActivityDate),
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    activity_array = ",".join(f"CCAC{i}" for i in range(1, 21))
    drug_array = ",".join(f"HCDRUG{i}" for i in range(1, 21))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.BABY_PERSON_ID.cast("string").alias("person_id"),
        F.when(s.BABY_PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.ActivityDate.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:nhs:nccmds:activity").alias("source_coding_system"),
        s.CCAC1.alias("source_code"), s.CCAC1.alias("source_display"),
        s.EntityID.alias("entity_id"), s.ActivityDate.alias("activity_date"),
        s.WardLocation.alias("ward_location"),
        s.CriticalCareUnitFunction.alias("unit_function"),
        s.CriticalCareStartDate_CLEAN.alias("critical_care_start"),
        s.CriticalCareDischargeDate_CLEAN.alias("critical_care_discharge"),
        F.expr(f"to_json(filter(array({activity_array}), x -> x is not null))")
         .alias("activity_codes_json"),
        F.expr(f"to_json(filter(array({drug_array}), x -> x is not null))")
         .alias("high_cost_drugs_json"),
        s.EPISODE_LINK_STATUS.alias("episode_link_status"),
        stable_id("neonatal_episode:badgernet", s.EntityID).alias("neonatal_episode_id"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        s.ActivityDate.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("neonatal_critical_care").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("badgernet").alias("_source_system"),
        F.lit(SRC_NEO_CRITICAL_CARE).alias("_source_table"),
        F.concat_ws("|", s.EntityID, s.ActivityDate).alias("_source_row_id"),
    )

def _neonatal_care_day_canonical():
    return _neonatal_care_day_canonical_pregate().where(_usable_code(F.col("source_code")))

In [0]:
SRC_NEO_CRITICAL_CARE = "4_prod.bronze.map_neonatal_critical_care"

NEONATAL_CARE_DAY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "entity_id",
    "activity_date", "ward_location", "unit_function", "critical_care_start",
    "critical_care_discharge", "activity_codes_json", "high_cost_drugs_json",
    "episode_link_status", "neonatal_episode_id", "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

NEONATAL_CARE_DAY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "activity_date": "Published field activity_date.",
    "ward_location": "Published field ward_location.",
    "unit_function": "Published field unit_function.",
    "critical_care_start": "Published field critical_care_start.",
    "critical_care_discharge": "Published field critical_care_discharge.",
    "activity_codes_json": "Published field activity_codes_json.",
    "high_cost_drugs_json": "Published field high_cost_drugs_json.",
    "episode_link_status": "Published field episode_link_status.",
    "neonatal_episode_id": "Published field neonatal_episode_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.neonatal_care_day"),
    comment="One admitted BadgerNet neonatal critical-care activity day.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=NEONATAL_CARE_DAY_COLUMN_COMMENTS,
)
def neonatal_care_day():
    return _neonatal_care_day_canonical().select(*NEONATAL_CARE_DAY_PUBLIC_COLUMNS)

In [0]:
def _neonatal_examination_canonical_pregate():
    s = read_source(SRC_NEO_EXAMINATION)
    event_id = stable_id("neonatal_examination:badgernet", s.EntityID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.BABY_PERSON_ID)], SRC_NEO_EXAMINATION, s.EntityID
    )
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    source_code = F.when(F.coalesce(s.EXAM_POPULATED_IND, F.lit(False)), F.lit("NIPE"))
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.BABY_PERSON_ID.cast("string").alias("person_id"),
        F.when(s.BABY_PERSON_ID.isNotNull(), F.lit("resolved"))
         .otherwise(F.lit("unresolved")).alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        s.DateOfExamination_CLEAN.alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit("urn:badgernet:examination").alias("source_coding_system"),
        source_code.alias("source_code"), F.lit("Newborn and Infant Physical Examination")
         .alias("source_display"),
        s.EntityID.alias("entity_id"),
        s.DateOfExamination_CLEAN.alias("examination_datetime"),
        s.EXAM_DATE_DERIVED.alias("exam_date_derived"),
        s.IncludeInDischargeLetter.alias("include_in_discharge_letter"),
        s.HeadCircumference.alias("head_circumference"), s.Spine.alias("spine_finding"),
        s.Heart.alias("heart_finding"), s.Genitalia.alias("genitalia_finding"),
        s.Hips.alias("hips_finding"), s.HipsRight.alias("right_hip_finding"),
        s.Eyes.alias("eyes_finding"), s.SpineComments.alias("spine_comments"),
        s.HeartComments.alias("heart_comments"),
        s.GenitaliaComments.alias("genitalia_comments"),
        F.coalesce(s.HipsOverallComments, s.HipsComments).alias("hips_comments"),
        s.EyesComments.alias("eyes_comments"),
        stable_id("neonatal_episode:badgernet", s.EntityID).alias("neonatal_episode_id"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active"))
         .alias("record_status"),
        s.DateOfExamination_CLEAN.alias("record_status_effective_from"),
        F.when(retracted, s.ADC_UPDT).alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("neonatal_examination").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.ADC_UPDT.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("badgernet").alias("_source_system"),
        F.lit(SRC_NEO_EXAMINATION).alias("_source_table"), s.EntityID.alias("_source_row_id"),
    )

def _neonatal_examination_canonical():
    return _neonatal_examination_canonical_pregate().where(
        _usable_code(F.col("source_code"))
    )

In [0]:
SRC_NEO_EXAMINATION = "4_prod.bronze.map_neonatal_examination"

NEONATAL_EXAMINATION_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "entity_id",
    "examination_datetime", "exam_date_derived", "include_in_discharge_letter",
    "head_circumference", "spine_finding", "heart_finding", "genitalia_finding",
    "hips_finding", "right_hip_finding", "eyes_finding", "spine_comments",
    "heart_comments", "genitalia_comments", "hips_comments", "eyes_comments",
    "neonatal_episode_id", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind",
    "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

NEONATAL_EXAMINATION_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "examination_datetime": "Published field examination_datetime.",
    "exam_date_derived": "Published field exam_date_derived.",
    "include_in_discharge_letter": "Published field include_in_discharge_letter.",
    "head_circumference": "Published field head_circumference.",
    "spine_finding": "Published field spine_finding.",
    "heart_finding": "Published field heart_finding.",
    "genitalia_finding": "Published field genitalia_finding.",
    "hips_finding": "Published field hips_finding.",
    "right_hip_finding": "Published field right_hip_finding.",
    "eyes_finding": "Published field eyes_finding.",
    "spine_comments": "Published field spine_comments.",
    "heart_comments": "Published field heart_comments.",
    "genitalia_comments": "Published field genitalia_comments.",
    "hips_comments": "Published field hips_comments.",
    "eyes_comments": "Published field eyes_comments.",
    "neonatal_episode_id": "Published field neonatal_episode_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.neonatal_examination"),
    comment="Populated BadgerNet neonatal examinations; empty episode stubs are excluded-and-counted.",
    cluster_by=["person_id", "event_datetime"],
    refresh_policy="incremental",
    column_comments=NEONATAL_EXAMINATION_COLUMN_COMMENTS,
)
def neonatal_examination():
    return _neonatal_examination_canonical().select(*NEONATAL_EXAMINATION_PUBLIC_COLUMNS)

In [0]:
def _mat_pregnancy_dedup():
    """One deterministic full row per Pregnancy_ID; no flow window function."""
    p = read_source(SRC_MAT_PREGNANCY)
    ranked = p.select(
        "*",
        F.struct(
            (~F.coalesce(p.SOURCE_DELETED_IND, F.lit(False))).cast("int").alias("_live"),
            F.coalesce(p.ADC_UPDT, F.lit("1900-01-01").cast("timestamp")).alias("_upd"),
            F.coalesce(p.Person_ID, F.lit(-1)).alias("_pid"),
            F.coalesce(p.ROW_HASH, F.lit(-1)).alias("_rh"),
        ).alias("_pick"),
    )
    best = ranked.groupBy("Pregnancy_ID").agg(F.max("_pick").alias("_best"))
    return (
        ranked.join(best, ["Pregnancy_ID"])
        .where(F.col("_pick") == F.col("_best"))
        .drop("_pick", "_best")
    )

def _maternity_join(slot):
    s = _maternity_source_dedup(slot).alias("s")
    p = _mat_pregnancy_spine().alias("p")
    return s.join(p, s.PREGNANCY_ID_PARSED == p.Pregnancy_ID, "left")

def _maternity_identity_status(embedded, spine):
    return (
        F.when(embedded.isNotNull(), F.lit("resolved"))
        .when(spine.isNotNull(), F.lit("recovered"))
        .otherwise(F.lit("unresolved"))
    )

MATERNITY_LINK_COLUMNS = [
    "pregnancy_id", "journey_pregnancy_id", "source_link_status",
    "pregnancy_orphan_ind", "spine_person_mismatch_ind",
]

def _baby_delivery_canonical():
    d = _maternity_join(SRC_MATERNITY_BABY)
    person = F.coalesce(F.col("_spine_person_id"), F.col("s.PERSON_ID"))
    event_id = stable_id("baby_delivery:msds", F.col("s.ROW_HASH"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", person)], SRC_MATERNITY_BABY, F.col("s.ROW_HASH")
    )
    mismatch = (
        F.col("s._source_person_conflict_ind")
        | (
            F.col("s.PERSON_ID").isNotNull()
            & F.col("_spine_person_id").isNotNull()
            & (F.col("s.PERSON_ID") != F.col("_spine_person_id"))
        )
    )
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        person.cast("string").alias("person_id"),
        _maternity_identity_status(F.col("s.PERSON_ID"), F.col("_spine_person_id"))
         .alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"),
        F.col("s.PERSONBIRTHDATETIMEBABY_CLEAN").alias("event_datetime"),
        F.col("s.DISCHARGEDATETIMEBABYHSP_CLEAN").alias("event_end_datetime"),
        F.lit("urn:nhs:msds:delivery-method").alias("source_coding_system"),
        F.col("s.DELIVERYMETHODCODE").alias("source_code"),
        F.col("s.DELIVERYMETHODCODE").alias("source_display"),
        F.col("s.BABY_PERSON_ID").cast("string").alias("baby_person_id"),
        F.col("s.BIRTHORDERMATERNITYSUS").alias("birth_order"),
        F.col("s.PREGOUTCOME").alias("pregnancy_outcome"),
        F.col("s.DELIVERYMETHODCODE").alias("delivery_method_code"),
        F.col("s.PERSONPHENSEX").alias("phenotypic_sex"),
        F.col("s.GESTATIONLENGTHBIRTH").alias("gestation_length_birth"),
        F.col("s.BIRTHWEIGHT").alias("birthweight"), F.col("s.APGARSCORE5").alias("apgar_5"),
        F.col("s.PERSONDEATHDATETIMEBABY_CLEAN").alias("baby_death_datetime"),
        F.col("s.BABYFIRSTFEEDDATETIME_CLEAN").alias("first_feed_datetime"),
        F.col("s.BABYFIRSTFEEDINDCODE").alias("first_feed_code"),
        F.col("s.BABYFIRSTFEEDBREASTMILKSTATUS").alias("first_feed_breast_milk_status"),
        F.col("s.BABYBREASTMILKSTATUSDISCHARGE").alias("breast_milk_status_discharge"),
        F.col("s.SKINTOSKINCONTACT1HOURIND").alias("skin_to_skin_ind"),
        F.col("s.DISCHARGEDATETIMEBABYHSP_CLEAN").alias("baby_discharge_datetime"),
        F.col("s.ORGSITEIDACTUALDELIVERY").alias("delivery_org_site"),
        F.col("s.SETTINGPLACEBIRTH").alias("birth_setting"),
        F.col("s.PLACETYPEACTUALDELIVERY").alias("birth_place_type"),
        F.col("s.PLACETYPEACTUALMIDWIFERY").alias("midwifery_place_type"),
        F.col("s.LABOURDELIVERYID").alias("labour_delivery_id"),
        stable_id("labour_delivery:msds", F.col("s.LABOURDELIVERYID"))
         .alias("labour_delivery_ref"),
        F.col("s.PREGNANCY_ID_PARSED").cast("string").alias("pregnancy_id"),
        stable_id("journey:pregnancy", F.col("s.PREGNANCY_ID_PARSED"))
         .alias("journey_pregnancy_id"),
        F.col("s.PERSON_LINK_STATUS").alias("source_link_status"),
        F.col("p.Pregnancy_ID").isNull().alias("pregnancy_orphan_ind"),
        mismatch.alias("spine_person_mismatch_ind"), F.lit("active").alias("record_status"),
        F.col("s.PERSONBIRTHDATETIMEBABY_CLEAN").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("maternity_baby_delivery").alias("source_feed"),
        F.date_format(F.col("s.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("s.RECORD_UPDATED_DT").alias("source_update_timestamp"),
        F.col("s.ADC_UPDT").alias("loaded_at"), F.lit("msds").alias("_source_system"),
        F.lit(SRC_MATERNITY_BABY).alias("_source_table"),
        F.col("s.ROW_HASH").cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_MATERNITY_BABY = "4_prod.bronze.map_maternity_baby_delivery"

def _mat_pregnancy_spine():
    d = _mat_pregnancy_dedup()
    return d.select(
        "Pregnancy_ID",
        F.col("Person_ID").alias("_spine_person_id"),
        F.col("SOURCE_DELETED_IND").alias("_spine_deleted_ind"),
    )

def _maternity_source_dedup(slot):
    """Collapse source-row identity fan-out without choosing an arbitrary person.

    The MSDS adapters can publish the same ROW_HASH more than once when one clinical row
    is projected onto conflicting person matches. All non-PERSON_ID columns are identical
    for that source identity. Keep one deterministic clinical row, retain PERSON_ID only
    when the evidence is single-valued, and expose the conflict to the maternity facts.
    """
    s = read_source(slot)
    non_person_cols = [c for c in s.columns if c != "PERSON_ID"]
    grouped = s.groupBy(*non_person_cols).agg(
        F.countDistinct("PERSON_ID").alias("_person_id_count"),
        F.max("PERSON_ID").alias("_single_person_id"),
    )
    conflict = F.col("_person_id_count") > 1
    return (
        grouped
        .withColumn(
            "PERSON_LINK_STATUS",
            F.when(conflict, F.lit("CONFLICTING")).otherwise(F.col("PERSON_LINK_STATUS")),
        )
        .withColumn(
            "PERSON_ID",
            F.when(F.col("_person_id_count") == 1, F.col("_single_person_id")),
        )
        .withColumn("_source_person_conflict_ind", conflict)
        .drop("_person_id_count", "_single_person_id")
    )

BABY_DELIVERY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "baby_person_id",
    "birth_order", "pregnancy_outcome", "delivery_method_code", "phenotypic_sex",
    "gestation_length_birth", "birthweight", "apgar_5", "baby_death_datetime",
    "first_feed_datetime", "first_feed_code", "first_feed_breast_milk_status",
    "breast_milk_status_discharge", "skin_to_skin_ind", "baby_discharge_datetime",
    "delivery_org_site", "birth_setting", "birth_place_type", "midwifery_place_type",
    "labour_delivery_id", "labour_delivery_ref", *MATERNITY_LINK_COLUMNS,
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

BABY_DELIVERY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "baby_person_id": "Published field baby_person_id.",
    "birth_order": "Published field birth_order.",
    "pregnancy_outcome": "Published field pregnancy_outcome.",
    "delivery_method_code": "Published field delivery_method_code.",
    "phenotypic_sex": "Published field phenotypic_sex.",
    "gestation_length_birth": "Published field gestation_length_birth.",
    "birthweight": "Published field birthweight.",
    "apgar_5": "Published field apgar_5.",
    "baby_death_datetime": "Published field baby_death_datetime.",
    "first_feed_datetime": "Published field first_feed_datetime.",
    "first_feed_code": "Published field first_feed_code.",
    "first_feed_breast_milk_status": "Published field first_feed_breast_milk_status.",
    "breast_milk_status_discharge": "Published field breast_milk_status_discharge.",
    "skin_to_skin_ind": "Published field skin_to_skin_ind.",
    "baby_discharge_datetime": "Published field baby_discharge_datetime.",
    "delivery_org_site": "Published field delivery_org_site.",
    "birth_setting": "Published field birth_setting.",
    "birth_place_type": "Published field birth_place_type.",
    "midwifery_place_type": "Published field midwifery_place_type.",
    "labour_delivery_id": "Published field labour_delivery_id.",
    "labour_delivery_ref": "Published field labour_delivery_ref.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.baby_delivery"),
    comment="One MSDS baby-delivery row with mother identity re-derived through pregnancy.",
    table_properties={"delta.dataSkippingStatsColumns": "person_id,event_datetime"},
    cluster_by=["person_id", "event_datetime"], refresh_policy="incremental",
    column_comments=BABY_DELIVERY_COLUMN_COMMENTS,
)
def baby_delivery():
    return _baby_delivery_canonical().select(*BABY_DELIVERY_PUBLIC_COLUMNS)

In [0]:
def _labour_delivery_canonical():
    d = _maternity_join(SRC_MATERNITY_LABOUR)
    person = F.coalesce(F.col("_spine_person_id"), F.col("s.PERSON_ID"))
    event_id = stable_id("labour_delivery:msds", F.col("s.LABOURDELIVERYID"))
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", person)], SRC_MATERNITY_LABOUR, F.col("s.LABOURDELIVERYID")
    )
    event_time = _clamped_ts(F.coalesce(
        F.col("s.LABOURONSETDATETIME_CLEAN"), F.col("s.CAESAREANDATETIME_CLEAN"),
        F.col("s.STARTDATETIMEMOTHERDELIVERYHPS_CLEAN"),
    ))
    mismatch = (
        F.col("s._source_person_conflict_ind")
        | (
            F.col("s.PERSON_ID").isNotNull() & F.col("_spine_person_id").isNotNull()
            & (F.col("s.PERSON_ID") != F.col("_spine_person_id"))
        )
    )
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        person.cast("string").alias("person_id"),
        _maternity_identity_status(F.col("s.PERSON_ID"), F.col("_spine_person_id"))
         .alias("identity_status"),
        F.lit(None).cast("string").alias("encounter_id"), event_time.alias("event_datetime"),
        F.col("s.DISCHARGEDATETIMEMOTHERHSP_CLEAN").alias("event_end_datetime"),
        F.lit("urn:nhs:msds:labour-onset-method").alias("source_coding_system"),
        F.col("s.LABOURONSETMETHOD").alias("source_code"),
        F.col("s.LABOURONSETMETHOD").alias("source_display"),
        F.col("s.LABOURDELIVERYID").alias("labour_delivery_id"),
        F.col("s.LABOURONSETMETHOD").alias("labour_onset_method"),
        F.col("s.LABOURONSETPRESENTATION").alias("labour_onset_presentation"),
        F.col("s.CAESAREANDATETIME_CLEAN").alias("caesarean_datetime"),
        F.col("s.DECISIONTODELIVERDATETIME_CLEAN").alias("decision_to_deliver_datetime"),
        F.col("s.ROMDATETIME_CLEAN").alias("rom_datetime"),
        F.col("s.ROMMETHOD").alias("rom_method"), F.col("s.ROMREASON").alias("rom_reason"),
        F.col("s.LABOURONSETSECONDSTAGEDATETIME_CLEAN").alias("second_stage_datetime"),
        F.col("s.LABOURTHIRDSTAGEENDDATETIME_CLEAN").alias("third_stage_end_datetime"),
        F.col("s.EPISIOTOMYREASON").alias("episiotomy_reason"),
        F.col("s.PLACENTADELIVERYMETHOD").alias("placenta_delivery_method"),
        F.col("s.ADMMETHCODEMOTHDELHSP").alias("mother_admission_method"),
        F.col("s.DISCHARGEDATETIMEMOTHERHSP_CLEAN").alias("mother_discharge_datetime"),
        F.col("s.DISCHMETHCODEMOTHPOSTDELHSP").alias("mother_discharge_method"),
        F.col("s.DISCHDESTCODEMOTHPOSTDELHSP").alias("mother_discharge_destination"),
        F.col("s.ORGSITEIDINTRA").alias("intrapartum_org_site"),
        F.col("s.SETTINGINTRACARE").alias("intrapartum_setting"),
        F.col("s.ORGIDPOSTNATALPATHLEADPROVIDER").alias("postnatal_lead_provider"),
        F.col("s.PREGNANCY_ID_PARSED").cast("string").alias("pregnancy_id"),
        stable_id("journey:pregnancy", F.col("s.PREGNANCY_ID_PARSED"))
         .alias("journey_pregnancy_id"),
        F.col("s.PERSON_LINK_STATUS").alias("source_link_status"),
        F.col("p.Pregnancy_ID").isNull().alias("pregnancy_orphan_ind"),
        mismatch.alias("spine_person_mismatch_ind"), F.lit("active").alias("record_status"),
        event_time.alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("clinical").alias("fact_category"),
        F.lit("maternity_labour_delivery").alias("source_feed"),
        F.date_format(F.col("s.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("s.RECORD_UPDATED_DT").alias("source_update_timestamp"),
        F.col("s.ADC_UPDT").alias("loaded_at"), F.lit("msds").alias("_source_system"),
        F.lit(SRC_MATERNITY_LABOUR).alias("_source_table"),
        F.col("s.LABOURDELIVERYID").alias("_source_row_id"),
    )

In [0]:
SRC_MATERNITY_LABOUR = "4_prod.bronze.map_maternity_labour_delivery"

LABOUR_DELIVERY_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "labour_delivery_id",
    "labour_onset_method", "labour_onset_presentation", "caesarean_datetime",
    "decision_to_deliver_datetime", "rom_datetime", "rom_method", "rom_reason",
    "second_stage_datetime", "third_stage_end_datetime", "episiotomy_reason",
    "placenta_delivery_method", "mother_admission_method", "mother_discharge_datetime",
    "mother_discharge_method", "mother_discharge_destination", "intrapartum_org_site",
    "intrapartum_setting", "postnatal_lead_provider", *MATERNITY_LINK_COLUMNS,
    "record_status", "record_status_effective_from", "record_status_effective_to",
    "confidentiality_code", "vip_ind", "withheld_identity_ind", "fact_category",
    "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

LABOUR_DELIVERY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "labour_delivery_id": "Published field labour_delivery_id.",
    "labour_onset_method": "Published field labour_onset_method.",
    "labour_onset_presentation": "Published field labour_onset_presentation.",
    "caesarean_datetime": "Published field caesarean_datetime.",
    "decision_to_deliver_datetime": "Published field decision_to_deliver_datetime.",
    "rom_datetime": "Published field rom_datetime.",
    "rom_method": "Published field rom_method.",
    "rom_reason": "Published field rom_reason.",
    "second_stage_datetime": "Published field second_stage_datetime.",
    "third_stage_end_datetime": "Published field third_stage_end_datetime.",
    "episiotomy_reason": "Published field episiotomy_reason.",
    "placenta_delivery_method": "Published field placenta_delivery_method.",
    "mother_admission_method": "Published field mother_admission_method.",
    "mother_discharge_datetime": "Published field mother_discharge_datetime.",
    "mother_discharge_method": "Published field mother_discharge_method.",
    "mother_discharge_destination": "Published field mother_discharge_destination.",
    "intrapartum_org_site": "Published field intrapartum_org_site.",
    "intrapartum_setting": "Published field intrapartum_setting.",
    "postnatal_lead_provider": "Published field postnatal_lead_provider.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.labour_delivery"),
    comment="One MSDS labour-delivery row; code zero remains a legitimate value.",
    table_properties={"delta.dataSkippingStatsColumns": "person_id,event_datetime"},
    cluster_by=["person_id", "event_datetime"], refresh_policy="incremental",
    column_comments=LABOUR_DELIVERY_COLUMN_COMMENTS,
)
def labour_delivery():
    return _labour_delivery_canonical().select(*LABOUR_DELIVERY_PUBLIC_COLUMNS)

In [0]:
def _maternity_care_contact_canonical():
    d = _maternity_join(SRC_MATERNITY_CONTACT)
    person = F.coalesce(F.col("_spine_person_id"), F.col("s.PERSON_ID"))
    event_id = stable_id(
        "maternity_care_contact:msds", F.col("s.CARECONID"), F.col("s.PREGNANCYID")
    )
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", person)], SRC_MATERNITY_CONTACT,
        F.concat_ws("|", F.col("s.CARECONID"), F.col("s.PREGNANCYID")),
    )
    mismatch = (
        F.col("s._source_person_conflict_ind")
        | (
            F.col("s.PERSON_ID").isNotNull() & F.col("_spine_person_id").isNotNull()
            & (F.col("s.PERSON_ID") != F.col("_spine_person_id"))
        )
    )
    return d.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"), person.cast("string").alias("person_id"),
        _maternity_identity_status(F.col("s.PERSON_ID"), F.col("_spine_person_id"))
         .alias("identity_status"), F.lit(None).cast("string").alias("encounter_id"),
        F.col("s.CCONTACTDATETIME_CLEAN").alias("event_datetime"),
        F.lit(None).cast("timestamp").alias("event_end_datetime"),
        F.lit(None).cast("string").alias("source_coding_system"),
        F.lit(None).cast("string").alias("source_code"),
        F.lit(None).cast("string").alias("source_display"),
        F.col("s.CARECONID").alias("care_contact_id"), F.col("s.ATTENDCODE").alias("attend_code"),
        F.col("s.CONSULTTYPE").alias("consult_type"),
        F.col("s.CCSUBJECT").alias("contact_subject"), F.col("s.MEDIUM").alias("medium"),
        F.col("s.CONTACTDURATION").alias("duration"),
        F.col("s.ADMINCATCODE").alias("admin_category"),
        F.col("s.GPTHERAPYIND").alias("gp_therapy_ind"),
        F.col("s.CANCELDATE_CLEAN").alias("cancel_datetime"),
        F.col("s.CANCELREASON").alias("cancel_reason"),
        F.col("s.REPLAPPTOFFDATE_CLEAN").alias("replacement_offer_datetime"),
        F.col("s.REPLAPPTDATE_CLEAN").alias("replacement_appointment_datetime"),
        F.col("s.ORGIDCOMM").alias("organization_id"),
        F.col("s.ORGSITEIDOFTREAT").alias("site_id"), F.col("s.LOCCODE").alias("location_code"),
        F.col("s.PREGNANCY_ID_PARSED").cast("string").alias("pregnancy_id"),
        stable_id("journey:pregnancy", F.col("s.PREGNANCY_ID_PARSED")).alias("journey_pregnancy_id"),
        F.col("s.PERSON_LINK_STATUS").alias("source_link_status"),
        F.col("p.Pregnancy_ID").isNull().alias("pregnancy_orphan_ind"),
        mismatch.alias("spine_person_mismatch_ind"), F.lit("active").alias("record_status"),
        F.col("s.CCONTACTDATETIME_CLEAN").alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("maternity_care_contact").alias("source_feed"),
        F.date_format(F.col("s.ADC_UPDT"), "yyyyMMddHHmmss").alias("load_batch_id"),
        F.col("s.RECORD_UPDATED_DT").alias("source_update_timestamp"),
        F.col("s.ADC_UPDT").alias("loaded_at"), F.lit("msds").alias("_source_system"),
        F.lit(SRC_MATERNITY_CONTACT).alias("_source_table"),
        F.concat_ws("|", F.col("s.CARECONID"), F.col("s.PREGNANCYID"))
         .alias("_source_row_id"),
    )

In [0]:
SRC_MATERNITY_CONTACT = "4_prod.bronze.map_maternity_care_contact"

MATERNITY_CONTACT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "care_contact_id",
    "attend_code", "consult_type", "contact_subject", "medium", "duration",
    "admin_category", "gp_therapy_ind", "cancel_datetime", "cancel_reason",
    "replacement_offer_datetime", "replacement_appointment_datetime", "organization_id",
    "site_id", "location_code", *MATERNITY_LINK_COLUMNS, "record_status",
    "record_status_effective_from", "record_status_effective_to", "confidentiality_code",
    "vip_ind", "withheld_identity_ind", "fact_category", "source_feed", "load_batch_id",
    "source_update_timestamp", "loaded_at",
]

MATERNITY_CARE_CONTACT_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "care_contact_id": "Published field care_contact_id.",
    "attend_code": "Published field attend_code.",
    "consult_type": "Published field consult_type.",
    "contact_subject": "Published field contact_subject.",
    "medium": "Published field medium.",
    "duration": "Published field duration.",
    "admin_category": "Published field admin_category.",
    "gp_therapy_ind": "Published field gp_therapy_ind.",
    "cancel_datetime": "Published field cancel_datetime.",
    "cancel_reason": "Published field cancel_reason.",
    "replacement_offer_datetime": "Published field replacement_offer_datetime.",
    "replacement_appointment_datetime": "Published field replacement_appointment_datetime.",
    "organization_id": "Published field organization_id.",
    "site_id": "Published field site_id.",
    "location_code": "Published field location_code.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.maternity_care_contact"),
    comment="One MSDS maternity care contact with pregnancy-spine identity recovery.",
    table_properties={"delta.dataSkippingStatsColumns": "person_id,event_datetime"},
    cluster_by=["person_id", "event_datetime"], refresh_policy="incremental",
    column_comments=MATERNITY_CARE_CONTACT_COLUMN_COMMENTS,
)
def maternity_care_contact():
    return _maternity_care_contact_canonical().select(*MATERNITY_CONTACT_PUBLIC_COLUMNS)

In [0]:
SRC_RESEARCH_STUDY = "4_prod.bronze.map_research_study"

RESEARCH_STUDY_COLUMN_COMMENTS = {
    "research_study_id": "Published field research_study_id.",
    "source_protocol_id": "Published field source_protocol_id.",
    "study_mnemonic": "Published field study_mnemonic.",
    "study_mnemonic_key": "Published field study_mnemonic_key.",
    "protocol_type_code": "Published field protocol_type_code.",
    "protocol_type_desc": "Published field protocol_type_desc.",
    "protocol_phase_code": "Published field protocol_phase_code.",
    "protocol_phase_desc": "Published field protocol_phase_desc.",
    "protocol_status_code": "Published field protocol_status_code.",
    "protocol_status_desc": "Published field protocol_status_desc.",
    "protocol_purpose_code": "Published field protocol_purpose_code.",
    "protocol_purpose_desc": "Published field protocol_purpose_desc.",
    "parent_protocol_id": "Published field parent_protocol_id.",
    "previous_protocol_id": "Published field previous_protocol_id.",
    "root_protocol_ind": "Published field root_protocol_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "open_ended_ind": "Published field open_ended_ind.",
    "display_ind": "Published field display_ind.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.research_study"),
    comment="Cerner research protocol amendments with parent and previous protocol lineage.",
    refresh_policy="incremental",
    column_comments=RESEARCH_STUDY_COLUMN_COMMENTS,
)
def research_study():
    s = read_source(SRC_RESEARCH_STUDY)
    return s.select(
        stable_id("research_study:cerner", s.PROT_MASTER_ID).alias("research_study_id"),
        s.PROT_MASTER_ID.cast("string").alias("source_protocol_id"),
        s.STUDY_MNEMONIC.alias("study_mnemonic"), s.STUDY_MNEMONIC_KEY.alias("study_mnemonic_key"),
        s.PROT_TYPE_CD.cast("long").alias("protocol_type_code"),
        s.PROT_TYPE_DESC.alias("protocol_type_desc"),
        s.PROT_PHASE_CD.cast("long").alias("protocol_phase_code"),
        s.PROT_PHASE_DESC.alias("protocol_phase_desc"),
        s.PROT_STATUS_CD.cast("long").alias("protocol_status_code"),
        s.PROT_STATUS_DESC.alias("protocol_status_desc"),
        s.PROT_PURPOSE_CD.cast("long").alias("protocol_purpose_code"),
        s.PROT_PURPOSE_DESC.alias("protocol_purpose_desc"),
        s.PARENT_PROT_MASTER_ID.cast("string").alias("parent_protocol_id"),
        s.PREV_PROT_MASTER_ID.cast("string").alias("previous_protocol_id"),
        (s.PARENT_PROT_MASTER_ID == s.PROT_MASTER_ID).alias("root_protocol_ind"),
        s.BEG_EFFECTIVE_DT_TM.alias("beg_effective"), s.END_EFFECTIVE_DT_TM.alias("end_effective"),
        (s.END_EFFECTIVE_DT_TM >= F.lit("2100-01-01").cast("timestamp")).alias("open_ended_ind"),
        s.DISPLAY_IND.cast("long").alias("display_ind"), F.lit("active").alias("record_status"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("cerner-research").alias("_source_system"),
        F.lit(SRC_RESEARCH_STUDY).alias("_source_table"),
        s.PROT_MASTER_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
def _research_enrollment_canonical():
    s = read_source(SRC_RESEARCH_SUBJECT)
    event_id = stable_id("research_enrollment:cerner", s.PT_PROT_REG_ID)
    skey, ssys = subject_key_with_system(
        [("urn:cerner:person_id", s.PERSON_ID)], SRC_RESEARCH_SUBJECT, s.PT_PROT_REG_ID
    )
    return s.select(
        event_id.alias("patient_event_id"), event_id.alias("fact_row_id"),
        skey.alias("subject_key"), ssys.alias("subject_id_system"),
        s.PERSON_ID.cast("string").alias("person_id"), F.lit("resolved").alias("identity_status"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        s.ON_STUDY_DT_TM_CLEAN.alias("event_datetime"),
        s.OFF_STUDY_DT_TM_CLEAN.alias("event_end_datetime"),
        F.lit("urn:cerner:research-status").alias("source_coding_system"),
        s.STATUS_ENUM.cast("long").cast("string").alias("source_code"),
        s.STATUS_DESC.alias("source_display"),
        stable_id("research_study:cerner", s.PROT_MASTER_ID).alias("research_study_id"),
        s.REG_ID.cast("string").alias("registration_id"),
        s.PROT_ACCESSION_NBR.alias("protocol_accession_number"),
        s.PROT_ARM_ID.cast("string").alias("protocol_arm_id"),
        s.STATUS_ENUM.cast("long").alias("status_code"), s.STATUS_DESC.alias("status_desc"),
        s.OFF_STUDY_DT_TM_CLEAN.alias("off_study_datetime"),
        s.TX_START_DT_TM_CLEAN.alias("treatment_start_datetime"),
        s.TX_COMPLETION_DT_TM_CLEAN.alias("treatment_completion_datetime"),
        s.REMOVAL_REASON_DESC_CV.alias("removal_reason_desc"),
        s.REMOVAL_REASON_FT.alias("removal_reason_text"),
        s.REASON_OFF_TX_DESC_CV.alias("off_treatment_reason_desc"),
        s.REASON_OFF_TX_FT.alias("off_treatment_reason_text"),
        s.EPISODE_ID.cast("string").alias("source_episode_id"),
        s.ENROLLING_ORGANIZATION_ID.cast("string").alias("enrolling_organization_id"),
        F.lit("active").alias("record_status"),
        s.ON_STUDY_DT_TM_CLEAN.alias("record_status_effective_from"),
        F.lit(None).cast("timestamp").alias("record_status_effective_to"),
        F.lit(None).cast("string").alias("confidentiality_code"),
        F.lit(None).cast("boolean").alias("vip_ind"),
        F.lit(None).cast("boolean").alias("withheld_identity_ind"),
        F.lit("administrative").alias("fact_category"),
        F.lit("research_subject").alias("source_feed"),
        F.date_format(s.ADC_UPDT, "yyyyMMddHHmmss").alias("load_batch_id"),
        s.PIPELINE_UPDT_DT_TM.alias("source_update_timestamp"), s.ADC_UPDT.alias("loaded_at"),
        F.lit("cerner-research").alias("_source_system"),
        F.lit(SRC_RESEARCH_SUBJECT).alias("_source_table"),
        s.PT_PROT_REG_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_RESEARCH_SUBJECT = "4_prod.bronze.map_research_subject"

RESEARCH_ENROLLMENT_PUBLIC_COLUMNS = [
    "patient_event_id", "fact_row_id", "subject_key", "subject_id_system", "person_id",
    "identity_status", "encounter_id", "event_datetime", "event_end_datetime",
    "source_coding_system", "source_code", "source_display", "research_study_id",
    "registration_id", "protocol_accession_number", "protocol_arm_id", "status_code",
    "status_desc", "off_study_datetime", "treatment_start_datetime",
    "treatment_completion_datetime", "removal_reason_desc", "removal_reason_text",
    "off_treatment_reason_desc", "off_treatment_reason_text", "source_episode_id",
    "enrolling_organization_id", "record_status", "record_status_effective_from",
    "record_status_effective_to", "confidentiality_code", "vip_ind", "withheld_identity_ind",
    "fact_category", "source_feed", "load_batch_id", "source_update_timestamp", "loaded_at",
]

RESEARCH_ENROLLMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "research_study_id": "Published field research_study_id.",
    "registration_id": "Published field registration_id.",
    "protocol_accession_number": "Published field protocol_accession_number.",
    "protocol_arm_id": "Published field protocol_arm_id.",
    "status_code": "Published field status_code.",
    "status_desc": "Published field status_desc.",
    "off_study_datetime": "Published field off_study_datetime.",
    "treatment_start_datetime": "Published field treatment_start_datetime.",
    "treatment_completion_datetime": "Published field treatment_completion_datetime.",
    "removal_reason_desc": "Published field removal_reason_desc.",
    "removal_reason_text": "Published field removal_reason_text.",
    "off_treatment_reason_desc": "Published field off_treatment_reason_desc.",
    "off_treatment_reason_text": "Published field off_treatment_reason_text.",
    "source_episode_id": "Published field source_episode_id.",
    "enrolling_organization_id": "Published field enrolling_organization_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@materialized_view(
    name=_n("journey_clinical.research_enrollment"),
    comment="One Cerner research-subject registration using clean sentinel-safe dates.",
    cluster_by=["person_id", "event_datetime"], refresh_policy="incremental",
    column_comments=RESEARCH_ENROLLMENT_COLUMN_COMMENTS,
)
def research_enrollment():
    return _research_enrollment_canonical().select(*RESEARCH_ENROLLMENT_PUBLIC_COLUMNS)

In [0]:
SRC_ADDRESS = "4_prod.bronze.map_address"
SRC_ADDRESS_EPC = "4_prod.bronze.map_address_epc"

PERSON_ADDRESS_COLUMN_COMMENTS = {
    "person_address_id": "Published field person_address_id.",
    "source_address_id": "Published field source_address_id.",
    "parent_entity": "Published field parent_entity.",
    "person_id": "Published field person_id.",
    "organization_id": "Published field organization_id.",
    "address_type_code": "Published field address_type_code.",
    "active_ind": "Published field active_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "open_ended_ind": "Published field open_ended_ind.",
    "street_address": "Published field street_address.",
    "city": "Published field city.",
    "postcode": "Published field postcode.",
    "postcode_masked": "Published field postcode_masked.",
    "postcode_outward": "Published field postcode_outward.",
    "uprn": "Published field uprn.",
    "lsoa": "Published field lsoa.",
    "msoa": "Published field msoa.",
    "local_authority_code": "Published field local_authority_code.",
    "imd_decile": "Published field imd_decile.",
    "imd_quintile": "Published field imd_quintile.",
    "latitude": "Published field latitude.",
    "longitude": "Published field longitude.",
    "uprn_match_quality": "Published field uprn_match_quality.",
    "epc_current_energy_rating": "Published field epc_current_energy_rating.",
    "epc_potential_energy_rating": "Published field epc_potential_energy_rating.",
    "epc_property_type": "Published field epc_property_type.",
    "epc_built_form": "Published field epc_built_form.",
    "epc_construction_age_band": "Published field epc_construction_age_band.",
    "epc_tenure": "Published field epc_tenure.",
    "epc_mains_gas_flag": "Published field epc_mains_gas_flag.",
    "epc_total_floor_area": "Published field epc_total_floor_area.",
    "epc_inspection_date": "Published field epc_inspection_date.",
    "epc_lodgement_date": "Published field epc_lodgement_date.",
    "epc_fuel_poverty_risk": "Published field epc_fuel_poverty_risk.",
    "epc_cold_hazard_proxy": "Published field epc_cold_hazard_proxy.",
    "epc_spatial_heating_poverty": "Published field epc_spatial_heating_poverty.",
    "epc_off_gas_grid": "Published field epc_off_gas_grid.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.person_address"),
    comment="Address assignments with EPC extension; direct address/postcode/UPRN fields are IG-sensitive.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=PERSON_ADDRESS_COLUMN_COMMENTS,
)
def person_address():
    a = read_source(SRC_ADDRESS).alias("a")
    e = read_source(SRC_ADDRESS_EPC).select(
        F.col("ADDRESS_ID").alias("_epc_address_id"),
        F.col("CURRENT_ENERGY_RATING").alias("epc_current_energy_rating"),
        F.col("POTENTIAL_ENERGY_RATING").alias("epc_potential_energy_rating"),
        F.col("PROPERTY_TYPE").alias("epc_property_type"),
        F.col("BUILT_FORM").alias("epc_built_form"),
        F.col("CONSTRUCTION_AGE_BAND").alias("epc_construction_age_band"),
        F.col("TENURE").alias("epc_tenure"),
        F.col("MAINS_GAS_FLAG").alias("epc_mains_gas_flag"),
        F.col("TOTAL_FLOOR_AREA").alias("epc_total_floor_area"),
        F.col("EPC_INSPECTION_DATE").alias("epc_inspection_date"),
        F.col("EPC_LODGEMENT_DATE").alias("epc_lodgement_date"),
        F.col("fuel_poverty_risk").alias("epc_fuel_poverty_risk"),
        F.col("hhsrs_cold_hazard_proxy").alias("epc_cold_hazard_proxy"),
        F.col("spatial_heating_poverty").alias("epc_spatial_heating_poverty"),
        F.col("off_gas_grid").alias("epc_off_gas_grid"),
    ).alias("e")
    d = a.join(e, a.ADDRESS_ID == e._epc_address_id, "left")
    return d.select(
        stable_id("person_address:mill", a.ADDRESS_ID).alias("person_address_id"),
        a.ADDRESS_ID.cast("string").alias("source_address_id"),
        a.PARENT_ENTITY_NAME.alias("parent_entity"),
        F.when(a.PARENT_ENTITY_NAME == "PERSON", a.PARENT_ENTITY_ID.cast("string"))
         .alias("person_id"),
        F.when(a.PARENT_ENTITY_NAME == "ORGANIZATION", a.PARENT_ENTITY_ID.cast("string"))
         .alias("organization_id"),
        a.ADDRESS_TYPE_CODE.cast("string").alias("address_type_code"),
        a.ACTIVE_IND.alias("active_ind"), a.BEG_EFFECTIVE_DT_TM.alias("beg_effective"),
        a.END_EFFECTIVE_DT_TM.alias("end_effective"),
        (a.END_EFFECTIVE_DT_TM >= F.lit("2100-01-01").cast("timestamp"))
         .alias("open_ended_ind"),
        a.full_street_address.alias("street_address"), a.CITY.alias("city"),
        a.SOURCE_ZIPCODE.alias("postcode"), a.masked_zipcode.alias("postcode_masked"),
        a.POSTCODE_OUTWARD.alias("postcode_outward"), a.UPRN.cast("string").alias("uprn"),
        a.LSOA.alias("lsoa"), a.MSOA21CD.alias("msoa"),
        a.LADCD.alias("local_authority_code"), a.IMD_Decile.alias("imd_decile"),
        a.IMD_Quintile.alias("imd_quintile"), a.LATITUDE.alias("latitude"),
        a.LONGITUDE.alias("longitude"), a.match_quality.alias("uprn_match_quality"),
        F.col("epc_current_energy_rating"), F.col("epc_potential_energy_rating"),
        F.col("epc_property_type"), F.col("epc_built_form"),
        F.col("epc_construction_age_band"), F.col("epc_tenure"),
        F.col("epc_mains_gas_flag"), F.col("epc_total_floor_area"),
        F.col("epc_inspection_date"), F.col("epc_lodgement_date"),
        F.col("epc_fuel_poverty_risk"), F.col("epc_cold_hazard_proxy"),
        F.col("epc_spatial_heating_poverty"), F.col("epc_off_gas_grid"),
        a.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_ADDRESS).alias("_source_table"),
        a.ADDRESS_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_DEATH = "4_prod.bronze.map_death"

PERSON_DEATH_EVIDENCE_COLUMN_COMMENTS = {
    "person_death_evidence_id": "Published field person_death_evidence_id.",
    "person_id": "Published field person_id.",
    "deceased_datetime_raw": "Published field deceased_datetime_raw.",
    "deceased_datetime": "Published field deceased_datetime.",
    "calculated_death_date": "Published field calculated_death_date.",
    "precision_flag": "Published field precision_flag.",
    "precision_desc": "Published field precision_desc.",
    "source_desc": "Published field source_desc.",
    "method_desc": "Published field method_desc.",
    "death_date_estimate_source": "Published field death_date_estimate_source.",
    "cause_of_death": "Published field cause_of_death.",
    "autopsy_desc": "Published field autopsy_desc.",
    "age_at_death": "Published field age_at_death.",
    "last_encounter_datetime": "Published field last_encounter_datetime.",
    "last_clinical_event_datetime": "Published field last_clinical_event_datetime.",
    "last_known_activity_datetime": "Published field last_known_activity_datetime.",
    "clinical_event_count": "Published field clinical_event_count.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.person_death_evidence"),
    comment="Corroborating death-register evidence; person-spine deceased fields remain authoritative.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=PERSON_DEATH_EVIDENCE_COLUMN_COMMENTS,
)
def person_death_evidence():
    s = read_source(SRC_DEATH)
    return s.select(
        stable_id("person_death_evidence:mill", s.PERSON_ID).alias("person_death_evidence_id"),
        s.PERSON_ID.cast("string").alias("person_id"),
        s.DECEASED_DT_TM.alias("deceased_datetime_raw"),
        _clamped_ts(s.DECEASED_DT_TM).alias("deceased_datetime"),
        s.CALC_DEATH_DATE.alias("calculated_death_date"),
        s.DECEASED_DT_TM_PREC_FLAG.alias("precision_flag"),
        s.DECEASED_DT_TM_PREC_DESC.alias("precision_desc"),
        s.DECEASED_SOURCE_DESC.alias("source_desc"),
        s.DECEASED_METHOD_DESC.alias("method_desc"),
        s.DEATH_DATE_ESTIMATE_SOURCE.alias("death_date_estimate_source"),
        s.CAUSE_OF_DEATH.alias("cause_of_death"), s.AUTOPSY_DESC.alias("autopsy_desc"),
        s.AGE_AT_DEATH.alias("age_at_death"), s.LAST_ENCNTR_DT_TM.alias("last_encounter_datetime"),
        s.LAST_CLINICAL_EVENT_DT_TM.alias("last_clinical_event_datetime"),
        s.LAST_KNOWN_ACTIVITY_DT_TM.alias("last_known_activity_datetime"),
        s.CLINICAL_EVENT_COUNT.alias("clinical_event_count"),
        F.lit("active").alias("record_status"), s.ADC_UPDT.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(SRC_DEATH).alias("_source_table"), s.PERSON_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_DEVICE_MAPPING = "4_prod.bronze.map_device_mapping"

DEVICE_MAPPING_COLUMN_COMMENTS = {
    "device_mapping_id": "Published field device_mapping_id.",
    "source_event_id": "Published field source_event_id.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "implant_description": "Published field implant_description.",
    "normalized_description": "Published field normalized_description.",
    "cleaned_description": "Published field cleaned_description.",
    "udi_di": "Published field udi_di.",
    "udi_issuer": "Published field udi_issuer.",
    "gs1_identifier": "Published field gs1_identifier.",
    "hibcc_device_id": "Published field hibcc_device_id.",
    "serial_number": "Published field serial_number.",
    "expiry_date": "Published field expiry_date.",
    "gmdn_code": "Published field gmdn_code.",
    "gmdn_name": "Published field gmdn_name.",
    "snomed_concept_id": "Published field snomed_concept_id.",
    "snomed_name": "Published field snomed_name.",
    "standard_concept_id": "Published field standard_concept_id.",
    "standard_concept_name": "Published field standard_concept_name.",
    "standard_vocabulary_id": "Published field standard_vocabulary_id.",
    "device_type": "Published field device_type.",
    "mapping_layer": "Published field mapping_layer.",
    "mapping_status": "Published field mapping_status.",
    "mapping_confidence": "Published field mapping_confidence.",
    "confidence_tier": "Published field confidence_tier.",
    "mapping_rule_id": "Published field mapping_rule_id.",
    "matched_field": "Published field matched_field.",
    "matched_value": "Published field matched_value.",
    "mapping_candidate_count": "Published field mapping_candidate_count.",
    "mapping_distinct_concept_count": "Published field mapping_distinct_concept_count.",
    "mapping_ambiguous_ind": "Published field mapping_ambiguous_ind.",
    "matched_opcs_code": "Published field matched_opcs_code.",
    "procedure_support_ind": "Published field procedure_support_ind.",
    "mapping_schema_version": "Published field mapping_schema_version.",
    "normalization_version": "Published field normalization_version.",
    "brand_rules_version": "Published field brand_rules_version.",
    "mapped_at": "Published field mapped_at.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.device_mapping"),
    comment="Best-effort implant-device decode with complete mapping provenance; identifiers are IG-sensitive.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=DEVICE_MAPPING_COLUMN_COMMENTS,
)
def device_mapping():
    s = read_source(SRC_DEVICE_MAPPING)
    return s.select(
        stable_id("device_mapping:mill", s.EVENT_ID).alias("device_mapping_id"),
        s.EVENT_ID.cast("string").alias("source_event_id"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID))
         .alias("encounter_id"),
        s.IMPLANT_DESCRIPTION.alias("implant_description"),
        s.NORMALIZED_DESCRIPTION.alias("normalized_description"),
        s.CLEANED_DESCRIPTION.alias("cleaned_description"),
        s.UDI_DI.alias("udi_di"), s.UDI_ISSUER.alias("udi_issuer"),
        s.SOURCE_GS1_IDENTIFIER.alias("gs1_identifier"),
        s.SOURCE_HIBCC_DEVICE_ID.alias("hibcc_device_id"),
        s.EFFECTIVE_SERIAL_NUMBER.alias("serial_number"),
        s.EFFECTIVE_EXPIRY_DATE.alias("expiry_date"),
        s.gmdncode.cast("string").alias("gmdn_code"), s.gmdn_name.alias("gmdn_name"),
        s.snomed_concept_id.cast("string").alias("snomed_concept_id"),
        s.snomed_name.alias("snomed_name"),
        s.STANDARD_CONCEPT_ID.cast("string").alias("standard_concept_id"),
        s.STANDARD_CONCEPT_NAME.alias("standard_concept_name"),
        s.STANDARD_VOCABULARY_ID.alias("standard_vocabulary_id"),
        s.device_type.alias("device_type"), s.mapping_layer.alias("mapping_layer"),
        s.MAPPING_STATUS.alias("mapping_status"), s.mapping_confidence.alias("mapping_confidence"),
        s.confidence_tier.alias("confidence_tier"), s.MAPPING_RULE_ID.alias("mapping_rule_id"),
        s.MATCHED_FIELD.alias("matched_field"), s.MATCHED_VALUE.alias("matched_value"),
        s.MAPPING_CANDIDATE_COUNT.alias("mapping_candidate_count"),
        s.MAPPING_DISTINCT_CONCEPT_COUNT.alias("mapping_distinct_concept_count"),
        s.MAPPING_AMBIGUOUS_IND.alias("mapping_ambiguous_ind"),
        s.MATCHED_OPCS_CODE.alias("matched_opcs_code"),
        s.PROCEDURE_SUPPORT_IND.alias("procedure_support_ind"),
        s.MAPPING_SCHEMA_VERSION.alias("mapping_schema_version"),
        s.NORMALIZATION_VERSION.alias("normalization_version"),
        s.BRAND_RULES_VERSION.alias("brand_rules_version"), s.MAPPED_AT.alias("mapped_at"),
        s.PIPELINE_LOADED_AT.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(SRC_DEVICE_MAPPING).alias("_source_table"),
        s.EVENT_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_ENC_ACTIVITY_BOUNDS = "4_prod.bronze.map_encounter_activity_bounds"
SRC_ENC_EVENT_BOUNDS = "4_prod.bronze.map_encounter_event_bounds"

ENCOUNTER_BOUNDS_COLUMN_COMMENTS = {
    "encounter_bounds_id": "Published field encounter_bounds_id.",
    "encounter_id": "Published field encounter_id.",
    "source_encounter_id": "Published field source_encounter_id.",
    "first_clinical_event_datetime": "Published field first_clinical_event_datetime.",
    "last_clinical_event_datetime": "Published field last_clinical_event_datetime.",
    "clinical_event_count": "Published field clinical_event_count.",
    "first_contemporaneous_event_datetime": "Published field first_contemporaneous_event_datetime.",
    "last_contemporaneous_event_datetime": "Published field last_contemporaneous_event_datetime.",
    "contemporaneous_event_count": "Published field contemporaneous_event_count.",
    "first_order_datetime": "Published field first_order_datetime.",
    "last_order_datetime": "Published field last_order_datetime.",
    "order_count": "Published field order_count.",
    "ward_move_count": "Published field ward_move_count.",
    "ward_occupancy_minutes": "Published field ward_occupancy_minutes.",
    "last_ward_in_datetime": "Published field last_ward_in_datetime.",
    "last_ward_out_datetime": "Published field last_ward_out_datetime.",
    "last_ward_still_open_ind": "Published field last_ward_still_open_ind.",
    "event_first_datetime": "Published field event_first_datetime.",
    "event_last_datetime": "Published field event_last_datetime.",
    "in_activity_bounds_ind": "Published field in_activity_bounds_ind.",
    "in_event_bounds_ind": "Published field in_event_bounds_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.encounter_bounds"),
    comment="Full-outer encounter activity/event bounds with source-arm presence flags.",
    cluster_by=["encounter_id"], refresh_policy="incremental",
    column_comments=ENCOUNTER_BOUNDS_COLUMN_COMMENTS,
)
def encounter_bounds():
    a = read_source(SRC_ENC_ACTIVITY_BOUNDS).alias("a")
    b = read_source(SRC_ENC_EVENT_BOUNDS).alias("b")
    d = a.join(b, a.ENCNTR_ID == b.ENCNTR_ID, "full")
    source_id = F.coalesce(a.ENCNTR_ID, b.ENCNTR_ID)
    return d.select(
        stable_id("encounter_bounds:mill", source_id).alias("encounter_bounds_id"),
        stable_id("encounter:mill", source_id).alias("encounter_id"),
        source_id.cast("string").alias("source_encounter_id"),
        a.FIRST_CLINICAL_EVENT_DT_TM.alias("first_clinical_event_datetime"),
        a.LAST_CLINICAL_EVENT_DT_TM.alias("last_clinical_event_datetime"),
        a.CLINICAL_EVENT_COUNT.alias("clinical_event_count"),
        a.FIRST_CONTEMPORANEOUS_EVENT_DT_TM.alias("first_contemporaneous_event_datetime"),
        a.LAST_CONTEMPORANEOUS_EVENT_DT_TM.alias("last_contemporaneous_event_datetime"),
        a.CONTEMPORANEOUS_EVENT_COUNT.alias("contemporaneous_event_count"),
        a.FIRST_ORDER_DT_TM.alias("first_order_datetime"),
        a.LAST_ORDER_DT_TM.alias("last_order_datetime"), a.ORDER_COUNT.alias("order_count"),
        a.WARD_MOVE_COUNT.alias("ward_move_count"),
        a.WARD_OCCUPANCY_MINUTES.alias("ward_occupancy_minutes"),
        a.LAST_WARD_IN_DT_TM.alias("last_ward_in_datetime"),
        a.LAST_WARD_OUT_DT_TM.alias("last_ward_out_datetime"),
        a.LAST_WARD_STILL_OPEN_IND.alias("last_ward_still_open_ind"),
        b.FIRST_EVENT_DT_TM.alias("event_first_datetime"),
        b.LAST_EVENT_DT_TM.alias("event_last_datetime"),
        a.ENCNTR_ID.isNotNull().alias("in_activity_bounds_ind"),
        b.ENCNTR_ID.isNotNull().alias("in_event_bounds_ind"),
        F.lit(None).cast("timestamp").alias("loaded_at"),
        F.lit("millennium").alias("_source_system"),
        F.concat_ws("|", F.lit(SRC_ENC_ACTIVITY_BOUNDS), F.lit(SRC_ENC_EVENT_BOUNDS))
         .alias("_source_table"), source_id.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_PRSNL_ALIAS = "4_prod.bronze.map_medical_personnel_alias"

PRACTITIONER_IDENTIFIER_COLUMN_COMMENTS = {
    "practitioner_identifier_id": "Published field practitioner_identifier_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "source_alias_id": "Published field source_alias_id.",
    "alias_type_meaning": "Published field alias_type_meaning.",
    "alias_type_display": "Published field alias_type_display.",
    "alias": "Published field alias.",
    "active_ind": "Published field active_ind.",
    "effective_now_ind": "Published field effective_now_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "contributor_system_code": "Published field contributor_system_code.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.practitioner_identifier"),
    comment="Staff aliases including postcode-type identifiers; alias values are IG-sensitive.",
    cluster_by=["practitioner_id"], refresh_policy="incremental",
    column_comments=PRACTITIONER_IDENTIFIER_COLUMN_COMMENTS,
)
def practitioner_identifier():
    s = read_source(SRC_PRSNL_ALIAS)
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return s.select(
        stable_id("practitioner_identifier:mill", s.PRSNL_ALIAS_ID)
         .alias("practitioner_identifier_id"),
        stable_id("practitioner:mill", s.PERSON_ID).alias("practitioner_id"),
        s.PERSON_ID.cast("string").alias("practitioner_person_id"),
        s.PRSNL_ALIAS_ID.cast("string").alias("source_alias_id"),
        s.ALIAS_TYPE_CDF_MEANING.alias("alias_type_meaning"),
        s.ALIAS_TYPE_DISPLAY.alias("alias_type_display"), s.ALIAS.alias("alias"),
        s.ACTIVE_IND.alias("active_ind"), s.EFFECTIVE_NOW_IND.alias("effective_now_ind"),
        s.BEG_EFFECTIVE_DT_TM.alias("beg_effective"), s.END_EFFECTIVE_DT_TM.alias("end_effective"),
        s.CONTRIBUTOR_SYSTEM_CD.cast("string").alias("contributor_system_code"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_PRSNL_ALIAS).alias("_source_table"),
        s.PRSNL_ALIAS_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_PRSNL_GROUP = "4_prod.bronze.map_medical_personnel_group"

PRACTITIONER_GROUP_COLUMN_COMMENTS = {
    "practitioner_group_id": "Published field practitioner_group_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "in_practitioner_dimension_ind": "Published field in_practitioner_dimension_ind.",
    "source_group_id": "Published field source_group_id.",
    "group_name": "Published field group_name.",
    "group_label": "Published field group_label.",
    "group_type_meaning": "Published field group_type_meaning.",
    "group_type_display": "Published field group_type_display.",
    "primary_ind": "Published field primary_ind.",
    "relation_active_ind": "Published field relation_active_ind.",
    "relation_beg_effective": "Published field relation_beg_effective.",
    "relation_end_effective": "Published field relation_end_effective.",
    "group_active_ind": "Published field group_active_ind.",
    "group_beg_effective": "Published field group_beg_effective.",
    "group_end_effective": "Published field group_end_effective.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.practitioner_group"),
    comment="Staff-group memberships; personnel-dimension orphans are retained and flagged.",
    cluster_by=["practitioner_id"], refresh_policy="incremental",
    column_comments=PRACTITIONER_GROUP_COLUMN_COMMENTS,
)
def practitioner_group():
    s = read_source(SRC_PRSNL_GROUP).alias("s")
    p = read_source(SRC_PRACTITIONER).select(
        F.col("PERSON_ID").alias("_practitioner_person_id")
    ).distinct().alias("p")
    d = s.join(p, s.PERSON_ID == p._practitioner_person_id, "left")
    retracted = ~F.coalesce(s.SOURCE_PRESENT_IND, F.lit(True))
    return d.select(
        stable_id("practitioner_group:mill", s.PRSNL_GROUP_RELTN_ID)
         .alias("practitioner_group_id"),
        stable_id("practitioner:mill", s.PERSON_ID).alias("practitioner_id"),
        s.PERSON_ID.cast("string").alias("practitioner_person_id"),
        F.col("_practitioner_person_id").isNotNull().alias("in_practitioner_dimension_ind"),
        s.PRSNL_GROUP_ID.cast("string").alias("source_group_id"),
        s.PRSNL_GROUP_NAME.alias("group_name"), s.GROUP_LABEL.alias("group_label"),
        s.CDF_MEANING.alias("group_type_meaning"), s.GROUP_TYPE_DISPLAY.alias("group_type_display"),
        s.PRIMARY_IND.alias("primary_ind"), s.RELATION_ACTIVE_IND.alias("relation_active_ind"),
        s.RELATION_BEG_EFFECTIVE_DT_TM.alias("relation_beg_effective"),
        s.RELATION_END_EFFECTIVE_DT_TM.alias("relation_end_effective"),
        s.GROUP_ACTIVE_IND.alias("group_active_ind"),
        s.GROUP_BEG_EFFECTIVE_DT_TM.alias("group_beg_effective"),
        s.GROUP_END_EFFECTIVE_DT_TM.alias("group_end_effective"),
        F.when(retracted, F.lit("retracted")).otherwise(F.lit("active")).alias("record_status"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("millennium").alias("_source_system"), F.lit(SRC_PRSNL_GROUP).alias("_source_table"),
        s.PRSNL_GROUP_RELTN_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_PRSNL_LOC_EVIDENCE = "4_prod.bronze.map_medical_personnel_location_evidence"

PRACTITIONER_LOCATION_EVIDENCE_COLUMN_COMMENTS = {
    "practitioner_location_evidence_id": "Published field practitioner_location_evidence_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "location_code": "Published field location_code.",
    "event_count": "Published field event_count.",
    "first_event_datetime_raw": "Published field first_event_datetime_raw.",
    "first_event_datetime": "Published field first_event_datetime.",
    "last_event_datetime_raw": "Published field last_event_datetime_raw.",
    "last_event_datetime": "Published field last_event_datetime.",
    "location_rank": "Published field location_rank.",
    "top_count_tie_count": "Published field top_count_tie_count.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.practitioner_location_evidence"),
    comment="Staff location evidence with raw and sentinel-clamped event bounds.",
    cluster_by=["practitioner_id"], refresh_policy="incremental",
    column_comments=PRACTITIONER_LOCATION_EVIDENCE_COLUMN_COMMENTS,
)
def practitioner_location_evidence():
    s = read_source(SRC_PRSNL_LOC_EVIDENCE)
    return s.select(
        stable_id("practitioner_location_evidence:mill", s.PERSON_ID, s.LOC_NURSE_UNIT_CD)
         .alias("practitioner_location_evidence_id"),
        stable_id("practitioner:mill", s.PERSON_ID).alias("practitioner_id"),
        s.PERSON_ID.cast("string").alias("practitioner_person_id"),
        s.LOC_NURSE_UNIT_CD.cast("string").alias("location_code"),
        s.EVENT_COUNT.alias("event_count"), s.FIRST_EVENT_DT_TM.alias("first_event_datetime_raw"),
        _clamped_ts(s.FIRST_EVENT_DT_TM).alias("first_event_datetime"),
        s.LAST_EVENT_DT_TM.alias("last_event_datetime_raw"),
        _clamped_ts(s.LAST_EVENT_DT_TM).alias("last_event_datetime"),
        s.LOCATION_RANK.alias("location_rank"), s.TOP_COUNT_TIE_COUNT.alias("top_count_tie_count"),
        s.EVIDENCE_ADC_UPDT.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(SRC_PRSNL_LOC_EVIDENCE).alias("_source_table"),
        F.concat_ws("|", s.PERSON_ID, s.LOC_NURSE_UNIT_CD).alias("_source_row_id"),
    )

In [0]:
def _theatre_parent_keys():
    return read_source(SRC_THEATRE_CASE).select(
        F.col("SURG_CASE_ID").alias("_parent_surg_case_id")
    ).distinct()

SRC_THEATRE_ATTENDANCE = "4_prod.bronze.map_theatre_case_attendance"

THEATRE_ATTENDANCE_COLUMN_COMMENTS = {
    "theatre_attendance_id": "Published field theatre_attendance_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "attendee_practitioner_id": "Published field attendee_practitioner_id.",
    "attendee_personnel_id": "Published field attendee_personnel_id.",
    "role_description": "Published field role_description.",
    "signing_attendee_ind": "Published field signing_attendee_ind.",
    "in_datetime": "Published field in_datetime.",
    "in_datetime_quality": "Published field in_datetime_quality.",
    "out_datetime": "Published field out_datetime.",
    "out_datetime_quality": "Published field out_datetime_quality.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "surgical_area_desc": "Published field surgical_area_desc.",
    "active_ind": "Published field active_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.theatre_attendance"),
    comment="SurgiNet case attendance; parent-orphan rows remain visible and flagged.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=THEATRE_ATTENDANCE_COLUMN_COMMENTS,
)
def theatre_attendance():
    s = read_source(SRC_THEATRE_ATTENDANCE).alias("s")
    d = s.join(_theatre_parent_keys(), s.SURG_CASE_ID == F.col("_parent_surg_case_id"), "left")
    return d.select(
        stable_id("theatre_attendance:surginet", s.CASE_ATTENDANCE_ID).alias("theatre_attendance_id"),
        s.SURG_CASE_ID.cast("string").alias("theatre_case_id"),
        F.when(F.col("_parent_surg_case_id").isNotNull(), F.lit("linked"))
         .otherwise(F.lit("orphan")).alias("case_link_status"),
        stable_id("practitioner:mill", s.CASE_ATTENDEE_ID).alias("attendee_practitioner_id"),
        s.CASE_ATTENDEE_ID.cast("string").alias("attendee_personnel_id"),
        s.ROLE_PERF_DESCRIPTION.alias("role_description"),
        s.SIGNING_ATTENDEE_IND.alias("signing_attendee_ind"),
        s.IN_DT_TM.alias("in_datetime"), s.IN_DT_TM_QUALITY.alias("in_datetime_quality"),
        s.OUT_DT_TM.alias("out_datetime"), s.OUT_DT_TM_QUALITY.alias("out_datetime_quality"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        s.SURG_AREA_DESCRIPTION.alias("surgical_area_desc"), s.ACTIVE_IND.alias("active_ind"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("surginet").alias("_source_system"), F.lit(SRC_THEATRE_ATTENDANCE).alias("_source_table"),
        s.CASE_ATTENDANCE_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_THEATRE_TIMES = "4_prod.bronze.map_theatre_case_times"

THEATRE_CASE_MILESTONE_COLUMN_COMMENTS = {
    "theatre_case_milestone_id": "Published field theatre_case_milestone_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "task_assay_code": "Published field task_assay_code.",
    "task_assay_description": "Published field task_assay_description.",
    "stage_description": "Published field stage_description.",
    "case_time_datetime": "Published field case_time_datetime.",
    "case_time_quality": "Published field case_time_quality.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "active_ind": "Published field active_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.theatre_case_milestone"),
    comment="SurgiNet case-time milestones; source-case-time meaning is dead upstream.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=THEATRE_CASE_MILESTONE_COLUMN_COMMENTS,
)
def theatre_case_milestone():
    s = read_source(SRC_THEATRE_TIMES).alias("s")
    d = s.join(_theatre_parent_keys(), s.SURG_CASE_ID == F.col("_parent_surg_case_id"), "left")
    return d.select(
        stable_id("theatre_milestone:surginet", s.CASE_TIMES_ID).alias("theatre_case_milestone_id"),
        s.SURG_CASE_ID.cast("string").alias("theatre_case_id"),
        F.when(F.col("_parent_surg_case_id").isNotNull(), F.lit("linked"))
         .otherwise(F.lit("orphan")).alias("case_link_status"),
        s.TASK_ASSAY_CD.cast("string").alias("task_assay_code"),
        s.TASK_ASSAY_DESCRIPTION.alias("task_assay_description"),
        s.STAGE_DESCRIPTION.alias("stage_description"), s.CASE_TIME_DT_TM.alias("case_time_datetime"),
        s.CASE_TIME_DT_TM_QUALITY.alias("case_time_quality"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        s.ACTIVE_IND.alias("active_ind"), s.ADC_UPDT.alias("loaded_at"), F.lit("surginet").alias("_source_system"),
        F.lit(SRC_THEATRE_TIMES).alias("_source_table"),
        s.CASE_TIMES_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
SRC_THEATRE_IMPLANT = "4_prod.bronze.map_theatre_implant_log"

THEATRE_IMPLANT_COLUMN_COMMENTS = {
    "theatre_implant_id": "Published field theatre_implant_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "item_id": "Published field item_id.",
    "manufacturer": "Published field manufacturer.",
    "model_number": "Published field model_number.",
    "catalog_number": "Published field catalog_number.",
    "serial_number": "Published field serial_number.",
    "lot_number": "Published field lot_number.",
    "batch_number": "Published field batch_number.",
    "implant_site": "Published field implant_site.",
    "implant_size": "Published field implant_size.",
    "quantity": "Published field quantity.",
    "expiry_date": "Published field expiry_date.",
    "free_text_item_desc": "Published field free_text_item_desc.",
    "implanted_by_practitioner_id": "Published field implanted_by_practitioner_id.",
    "mill_implant_event_id": "Published field mill_implant_event_id.",
    "implant_link_method": "Published field implant_link_method.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "document_type_desc": "Published field document_type_desc.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.theatre_implant"),
    comment="SurgiNet theatre implant log; serial/lot/batch identifiers are IG-sensitive.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=THEATRE_IMPLANT_COLUMN_COMMENTS,
)
def theatre_implant():
    s = read_source(SRC_THEATRE_IMPLANT).alias("s")
    d = s.join(_theatre_parent_keys(), s.SURG_CASE_ID == F.col("_parent_surg_case_id"), "left")
    return d.select(
        stable_id("theatre_implant:surginet", s.IMPLANT_LOG_ST_ID).alias("theatre_implant_id"),
        s.SURG_CASE_ID.cast("string").alias("theatre_case_id"),
        F.when(F.col("_parent_surg_case_id").isNotNull(), F.lit("linked"))
         .otherwise(F.lit("orphan")).alias("case_link_status"),
        s.ITEM_ID.alias("item_id"), s.MANUFACTURER.alias("manufacturer"),
        s.MODEL_NUMBER.alias("model_number"), s.CATALOG_NUMBER.alias("catalog_number"),
        s.SERIAL_NUMBER.alias("serial_number"), s.LOT_NUMBER.alias("lot_number"),
        s.BATCH_NUMBER.alias("batch_number"), s.IMPLANT_SITE.alias("implant_site"),
        s.IMPLANT_SIZE.alias("implant_size"), s.QUANTITY.alias("quantity"),
        s.EXP_DATE.alias("expiry_date"), s.FREE_TEXT_ITEM_DESC.alias("free_text_item_desc"),
        stable_id("practitioner:mill", s.IMPLANTED_BY_ID).alias("implanted_by_practitioner_id"),
        s.MILL_IMPLANT_EVENT_ID.cast("string").alias("mill_implant_event_id"),
        s.IMPLANT_LINK_METHOD.alias("implant_link_method"),
        s.PERSON_ID.cast("string").alias("person_id"),
        F.when(s.ENCNTR_ID.isNotNull(), stable_id("encounter:mill", s.ENCNTR_ID)).alias("encounter_id"),
        s.DOC_TYPE_DESCRIPTION.alias("document_type_desc"),
        s.ADC_UPDT.alias("loaded_at"),
        F.lit("surginet").alias("_source_system"), F.lit(SRC_THEATRE_IMPLANT).alias("_source_table"),
        s.IMPLANT_LOG_ST_ID.cast("string").alias("_source_row_id"),
    )

In [0]:
def _attribute_reference(slot, namespace, id_name, entity_kind):
    s = read_source(slot)
    entity_id = s.ENTITY_ID
    entity_ref = (
        stable_id("encounter:mill", entity_id) if entity_kind == "encounter"
        else entity_id.cast("string")
    )
    cols = [
        stable_id(namespace, s.SOURCE_PK).alias(id_name),
        entity_ref.alias(f"{entity_kind}_id"),
        entity_id.cast("string").alias(f"source_{entity_kind}_id"),
        s.ATTRIBUTE_NAME.alias("attribute_name"), s.VALUE_KIND.alias("value_kind"),
        s.VALUE_CD.cast("string").alias("value_code"), s.VALUE_DISPLAY.alias("value_display"),
        s.VALUE_CODE_SET.cast("string").alias("value_code_set"),
        s.VALUE_DT_TM_CLEAN.alias("value_datetime"), s.VALUE_NUMERIC.alias("value_numeric"),
        s.VALUE_ANSWERED_IND.alias("answered_ind"), s.ACTIVE_IND.alias("active_ind"),
        s.CURRENT_IND.alias("current_ind"),
        s.BEG_EFFECTIVE_DT_TM_CLEAN.alias("beg_effective"),
        s.END_EFFECTIVE_DT_TM_CLEAN.alias("end_effective"), s.LINK_STATUS.alias("link_status"),
        s.PIPELINE_UPDT_DT_TM.alias("loaded_at"), F.lit("millennium").alias("_source_system"),
        F.lit(slot).alias("_source_table"), s.SOURCE_PK.cast("string").alias("_source_row_id"),
    ]
    return s.select(*cols)

In [0]:
SRC_ENCOUNTER_ATTRIBUTE = "4_prod.bronze.map_encounter_attribute"

ENCOUNTER_ATTRIBUTE_COLUMN_COMMENTS = {
    "encounter_attribute_id": "Published field encounter_attribute_id.",
    "encounter_id": "Published field encounter_id.",
    "source_encounter_id": "Published field source_encounter_id.",
    "attribute_name": "Published field attribute_name.",
    "value_kind": "Published field value_kind.",
    "value_code": "Published field value_code.",
    "value_display": "Published field value_display.",
    "value_code_set": "Published field value_code_set.",
    "value_datetime": "Published field value_datetime.",
    "value_numeric": "Published field value_numeric.",
    "answered_ind": "Published field answered_ind.",
    "active_ind": "Published field active_ind.",
    "current_ind": "Published field current_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "link_status": "Published field link_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.encounter_attribute"),
    comment="Allowlisted encounter attributes with current/effective evidence.",
    cluster_by=["encounter_id"], refresh_policy="incremental",
    column_comments=ENCOUNTER_ATTRIBUTE_COLUMN_COMMENTS,
)
def encounter_attribute():
    return _attribute_reference(
        SRC_ENCOUNTER_ATTRIBUTE, "encounter_attribute:mill", "encounter_attribute_id", "encounter"
    )

In [0]:
SRC_PERSON_ATTRIBUTE = "4_prod.bronze.map_person_attribute"

PERSON_ATTRIBUTE_COLUMN_COMMENTS = {
    "person_attribute_id": "Published field person_attribute_id.",
    "person_id": "Published field person_id.",
    "source_person_id": "Published field source_person_id.",
    "attribute_name": "Published field attribute_name.",
    "value_kind": "Published field value_kind.",
    "value_code": "Published field value_code.",
    "value_display": "Published field value_display.",
    "value_code_set": "Published field value_code_set.",
    "value_datetime": "Published field value_datetime.",
    "value_numeric": "Published field value_numeric.",
    "answered_ind": "Published field answered_ind.",
    "active_ind": "Published field active_ind.",
    "current_ind": "Published field current_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "link_status": "Published field link_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@materialized_view(
    name=_n("journey_reference.person_attribute"),
    comment="Allowlisted person attributes; NO_FIXED_ABODE and CHILD_IN_PUBLIC_CARE are IG-sensitive.",
    cluster_by=["person_id"], refresh_policy="incremental",
    column_comments=PERSON_ATTRIBUTE_COLUMN_COMMENTS,
)
def person_attribute():
    return _attribute_reference(
        SRC_PERSON_ATTRIBUTE, "person_attribute:mill", "person_attribute_id", "person"
    )